# Moving-block holdout bootstrap for XANES reference selection

This notebook develops the fitting and model-selection machinery behind MrFitty on the
`example/arsenic` data: given one unknown spectrum and a pool of reference spectra, which
combination of references best explains it, and how confident can we be in that choice?

Both questions are answered by the same estimator: a **moving-block holdout bootstrap**.
A bootstrap answers "how confident" by refitting. Rather than trusting a single fit, it
manufactures many slightly different versions of the same spectrum — the same fitted curve,
plus a reshuffled copy of the leftover noise — refits each one, and takes the spread of the
resulting mixing coefficients and prediction errors across those refits as the measure of
confidence. The **holdout** half borrows from cross-validation: each refit is scored only at
energies it was not fitted to, so a model earns its score by predicting parts of the
spectrum it has not seen.

Both halves have to respect one property of this data: the residuals of a XANES fit are
strongly autocorrelated — where the fitted curve runs above the measured spectrum it stays
above it across a stretch of adjacent energies, because what the model missed is a smooth
feature many points wide rather than independent point-to-point noise — so an ordinary
bootstrap, which assumes independent residuals, understates the uncertainty. Resampling in
contiguous *blocks* carries that dependence into the bootstrap, and holding out contiguous
*blocks* keeps the scored energies from being near-duplicates of the fitted ones next door.
"Moving" block means the blocks may start at any energy and may overlap, rather than coming
from one fixed partition; how long they should be is a real question, and one of the studies
below answers it.

Why prediction error rather than goodness of fit: adding a reference can only improve the
fit, since three references cannot fit worse than two, so residual size cannot tell you how
many references are justified. Prediction error can — a reference that is merely absorbing
noise helps at the energies it was fitted to and hurts at the ones it wasn't (the holdout
blocks).

## The pipeline

1. **Read the spectra.** `ReferenceSpectrum.read_all` loads the reference pool and the
   unknown spectra; `filter_spectra_by_name` selects the sample and reference subset a
   given cell works with.
2. **Build the design matrix.** Every reference is measured on its own energy grid, so
   `interpolate_references_at_sample_energies` resamples each one onto the sample's grid
   over the energy range common to the sample and all references — no extrapolation. It
   returns the design matrix `A` (one column per reference), the response vector `b` (the
   sample's normalized fluorescence), and which spectra limit each end of the range.
   `plot_interpolated_references` draws that output, and a small `pytest` suite pins the
   function's behavior against hand-built synthetic spectra.
3. **Fit.** `fit_nnls` solves the non-negative least squares problem — mixing fractions
   cannot be negative — with OLS available alongside it for comparison. The residuals go
   through an autocorrelation function (ACF) and a histogram, which is where the
   autocorrelation that motivates the whole block scheme becomes visible.
4. **Randomly generate the moving-block structure for the bootstrap.**
   `select_holdout_blocks_v5` pre-generates, for all
   `n_bootstrap` iterations at once, the holdout masks (~1/3 of the energies, in
   contiguous blocks) and the moving-block starts used to resample residuals. Generating it
   once and sharing it is what makes prediction errors comparable across combinations.
5. **Bootstrap the prediction error.** For each iteration,
   `do_moving_block_holdout_bootstrap` adds resampled residual blocks to the fitted
   spectrum to build a bootstrap spectrum, refits by NNLS on the training positions, and
   scores root-mean-square error (RMSE) against the *real* spectrum at the held-out
   positions. The spread of those 1000 prediction errors is the uncertainty estimate.
6. **Search the combinations.** `do_ref_subsets_moving_block_holdout_bootstrap` runs
   steps 3–5 for every combination of references at each size in `M` (2,324 combinations
   for M = 1, 2, 3 over 24 references), tuning the block length from the residuals by
   default. The combination with the lowest median prediction error wins, and
   `plot_ref_subsets_summary` / `plot_best_subset_bootstrap_summaries` report the ranking,
   the winning fits, and their bootstrap coefficient intervals — each alongside the
   reference trees described below, so how good a fit is and how distinctive the
   references it chose are can be read off one figure. That ranking is a point estimate
   and says nothing about whether the gaps between subsets are real;
   `plot_best_peci_subset_bootstrap_summaries` reports the subsets whose prediction error
   cannot be distinguished from the winner's instead, which is a weaker and better
   supported claim.

## The studies

Six sections examine choices the pipeline makes, each ending in a **Findings** write-up:

- **Confidence interval estimators** — which of three interval estimators, all of which
  this repository already calls a "95% CI", should decide whether two subsets differ. They
  give the same answer; one of them frequently gives none at all.
- **What counts as equally good** — whether a confidence interval on the median paired
  difference, or the spread of the differences themselves, should decide ties. The first
  sharpens as the bootstrap runs longer, which makes its answer partly a statement about
  the iteration count.
- **Correlation vs. cosine reference distance** — which distance the reference tree should
  be built on. The two agree almost perfectly about which references are neighbors and
  disagree about where the tree is cut, which changes whether one selected combination reads
  as spanning the reference set or as coming from a single cluster.
- **`select_holdout_blocks` v1–v5** — how holdout blocks should be laid out so every energy
  is held out equally often. The ranking turns entirely on whether the block length divides
  the number of energy points; v5, which tiles exactly at any block length, is the one used
  above.
- **Choosing the block length** — `politis_white_block_length` (Politis & White 2004) plus
  `choose_block_length`, which reduces one estimate per combination to a single length with
  a low quantile. It returns 10 for this data against the `round(n ** (1/3)) = 6` rule of
  thumb, and the sweep shows the selected model is unchanged from L = 3 to 15.
- **Linear vs. cubic spline interpolation** — the resampling in step 2 is an assumption
  underneath everything downstream. The two methods disagree substantially at the
  absorption edge and yet select the same references, which is the useful negative result
  about the spline hardcoded in `mrfitty/base.py`.

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import scipy.optimize

import mrfitty
from mrfitty.base import (
    AdaptiveEnergyRangeBuilder,
    InterpolatedReferenceSpectraSet,
    ReferenceSpectrum,
)

%matplotlib inline

In [ ]:
import fnmatch


def filter_spectra_by_name(spectra_list, *patterns):
    """Return spectra whose file_name matches any of the glob-style patterns."""
    matches = [s for s in spectra_list if any(fnmatch.fnmatch(s.file_name, p) for p in patterns)]
    if not matches:
        raise ValueError(
            f"No spectra matched the given pattern(s): {patterns!r}. "
            f"Available file names: {[s.file_name for s in spectra_list]}"
        )
    return matches


In [ ]:
src_path, _ = os.path.split(mrfitty.__path__[0])
sample_data_dir_path = os.path.join(src_path, 'example', 'arsenic')
print('sample data is installed at "{}"'.format(sample_data_dir_path))
os.path.exists(sample_data_dir_path)

In [ ]:
reference_spectra_glob = os.path.join(sample_data_dir_path, 'reference/*.e')
print('reference spectra glob: {}'.format(reference_spectra_glob))
sample_spectra_glob = os.path.join(sample_data_dir_path, 'unknown/*.e')
print('sample spectra glob: {}'.format(sample_spectra_glob))

In [ ]:
reference_spectra_list = sorted(list(ReferenceSpectrum.read_all([reference_spectra_glob])[0]), key=lambda s: s.file_name)
print('reference spectra file count: {}'.format(len(reference_spectra_list)))
sample_spectra_list = sorted(list(ReferenceSpectrum.read_all([sample_spectra_glob])[0]), key=lambda s: s.file_name)
print('sample spectra file count: {}'.format(len(sample_spectra_list)))

In [ ]:
def fit_nnls(A, b):
    coef, _ = scipy.optimize.nnls(A, b)
    fitted = A @ coef
    residuals = fitted - b
    return coef, fitted, residuals


In [ ]:
def fit_ols(A, b):
    coef = np.linalg.solve(A.T @ A, A.T @ b)
    fitted = A @ coef
    residuals = fitted - b
    return coef, fitted, residuals


In [ ]:
import statsmodels.api as sm


def fit_ols_with_statistics(A, b):
    result = sm.OLS(b, A).fit()
    coef = result.params
    fitted = result.fittedvalues
    residuals = fitted - b
    return coef, fitted, residuals, result


In [ ]:
unknown_spectrum = filter_spectra_by_name(sample_spectra_list, "OTT3_55*")[0]
print(f'unknown spectrum: {unknown_spectrum.file_name}')
reference_spectra = filter_spectra_by_name(
    reference_spectra_list,
    "Arsenopyrite_Jul*", "orpiment_all*", "arsenate*_diop*") 
ref_names = tuple([r.file_name for r in reference_spectra])
print(f"reference spectra:\n  {'\n  '.join(ref_names)}")

In [ ]:
# Interpolation methods for resampling reference spectra onto the sample's energy grid.
#
# make_interp_spline is scipy's current spline-construction API -- the modern
# replacement for the legacy interp1d / splrep interface -- and covers both methods
# we want through its degree parameter k, so linear and cubic differ only in one
# argument rather than in which function is called.
#
# Each factory takes (energies, norm) and returns a callable evaluated at arbitrary
# energies, so the *factory* is the unit passed to
# interpolate_references_at_sample_energies.
#
# make_cubic_spline_interpolant is the default there, and it is not a change in
# behavior: ReferenceSpectrum.__init__ (mrfitty/base.py) builds an
# InterpolatedUnivariateSpline, which is also a k=3 interpolating spline, and the two
# agree to ~1e-15 in norm units on this data. Building the interpolant here rather
# than reading the one stored on the spectrum is what makes the method selectable.
from scipy.interpolate import make_interp_spline


def make_linear_interpolant(energies, norm):
    """Piecewise-linear interpolant through (energies, norm)."""
    return make_interp_spline(energies, norm, k=1)


def make_cubic_spline_interpolant(energies, norm):
    """Interpolating cubic spline through (energies, norm)."""
    return make_interp_spline(energies, norm, k=3)

In [ ]:
def interpolate_references_at_sample_energies(
    reference_spectra, sample_spectrum, make_interpolant=make_cubic_spline_interpolant,
):
    """Interpolate reference spectra onto the sample spectrum's energy grid.

    The usable energy range is the intersection of the sample spectrum's range
    and every reference spectrum's range, so no extrapolation occurs.  Each
    reference is resampled by building an interpolant through its own measured
    energies and evaluating it at the sample energies.

    Parameters
    ----------
    reference_spectra : list of ReferenceSpectrum
        Pool of reference spectra to interpolate.  Each must expose a
        ``data_df`` attribute indexed by energy (eV) with a ``norm`` column.
    sample_spectrum : Spectrum or ReferenceSpectrum
        The unknown spectrum to be fitted.  Must expose a ``data_df`` attribute
        whose index contains energy values (eV) and whose ``norm`` column
        contains the normalized fluorescence values used as the regression
        response vector.
    make_interpolant : callable, optional
        Factory called as ``make_interpolant(energies, norm)`` returning a
        callable that evaluates the reference at arbitrary energies.  Defaults
        to ``make_cubic_spline_interpolant``, matching the cubic spline
        ``ReferenceSpectrum`` builds internally.  Pass
        ``make_linear_interpolant`` to resample linearly instead.  This choice
        sits upstream of the design matrix, the fit coefficients, and the
        prediction error, so it can affect which reference combination is
        selected -- see the interpolation method comparison at the end of this
        notebook.

    Returns
    -------
    valid_energies : ndarray, shape (n,)
        Energy values (eV) at which interpolation was performed — the
        intersection of the sample spectrum's range and all reference ranges.
        Depends only on the measured energy ranges, not on ``make_interpolant``.
    A : ndarray, shape (n, n_refs)
        Design matrix A for linear regression: column i holds reference i
        interpolated at ``valid_energies``.
    b : ndarray, shape (n,)
        Sample spectrum normalized fluorescence values at ``valid_energies``.
        This is the response vector for linear regression against A.
    low_limiters : list
        The spectrum object(s) whose lower bound sets ``valid_energies[0]``
        (the highest lower bound).  More than one when several tie exactly.
        The sample spectrum is a candidate alongside the references.
    high_limiters : list
        The spectrum object(s) whose upper bound sets ``valid_energies[-1]``
        (the lowest upper bound).  More than one when several tie exactly.
    """
    energies = sample_spectrum.data_df.index.values
    print(f'sample_spectrum: {sample_spectrum.file_name}')
    print(f'energy range: {energies[0]:.2f}–{energies[-1]:.2f} eV ({len(energies)} points)')
    print(f'references: {len(reference_spectra)}')
    print(f'interpolation: {make_interpolant.__name__}')

    # restrict to energies covered by the sample_spectrum AND every reference to avoid extrapolation.
    # collect (spectrum, low, high) for the sample and every reference so we can report and return
    # which spectra limit each end of the common range.
    spectrum_ranges = [
        (s, s.data_df.index.values[0], s.data_df.index.values[-1])
        for s in (sample_spectrum, *reference_spectra)
    ]

    # energy_min is set by the highest lower bound; energy_max by the lowest upper bound.
    # report every spectrum tied at each limiting energy, not just the first.
    energy_min = max(low for _, low, _ in spectrum_ranges)
    energy_max = min(high for _, _, high in spectrum_ranges)
    low_limiters = [s for s, low, _ in spectrum_ranges if low == energy_min]
    high_limiters = [s for s, _, high in spectrum_ranges if high == energy_max]

    print(f'lowest valid energy {energy_min:.2f} eV limited by '
          f'{", ".join(s.file_name for s in low_limiters)}')
    print(f'highest valid energy {energy_max:.2f} eV limited by '
          f'{", ".join(s.file_name for s in high_limiters)}')

    valid_mask = (energies >= energy_min) & (energies <= energy_max)
    valid_energies = energies[valid_mask]
    n_excluded = (~valid_mask).sum()
    if n_excluded:
        print(f'excluded {n_excluded} energies outside common range '
              f'({energy_min:.2f}–{energy_max:.2f} eV)')
    print(f'interpolating at {len(valid_energies)} energies '
          f'({valid_energies[0]:.2f}–{valid_energies[-1]:.2f} eV)')

    A = np.zeros((len(valid_energies), len(reference_spectra)))
    for i, ref in enumerate(reference_spectra):
        # build the interpolant from the reference's own measured points rather than
        # using a pre-built one, so make_interpolant actually selects the method
        reference_interpolant = make_interpolant(
            ref.data_df.index.values, ref.data_df['norm'].values,
        )
        A[:, i] = reference_interpolant(valid_energies)
        print(f'  {ref.file_name}: norm [{A[:, i].min():.4f}, {A[:, i].max():.4f}]')

    b = sample_spectrum.data_df['norm'].values[valid_mask]

    return valid_energies, A, b, low_limiters, high_limiters

## Tests for `interpolate_references_at_sample_energies`

The tests below exercise the function with tiny, hand-built synthetic spectra so
that every expected value is known exactly.  They cover:

- the common energy range being the intersection of the sample and all references,
- the shapes and values of the returned design matrix `A` and response vector `b`,
- correct identification of the low- and high-energy limiting spectra,
- reporting **all** spectra tied at a limiting energy,
- the sample spectrum itself acting as a limiter,
- nothing being excluded when the ranges coincide.

They are written as plain `pytest` functions (no fixtures, no arguments), so they
are collected automatically by `pytest`; the final cell also runs them directly
inside the notebook.

In [ ]:
# ---------------------------------------------------------------------------
# Test fixtures: a minimal stand-in for ReferenceSpectrum / Spectrum
# ---------------------------------------------------------------------------
# interpolate_references_at_sample_energies() only ever touches two things on
# the spectra it is handed:
#   * .file_name              - a label used in the printed / returned report
#   * .data_df                - a pandas DataFrame indexed by energy (eV) with a
#                               'norm' column of normalized fluorescence values
# It builds its own interpolant from .data_df via the make_interpolant argument,
# so a real spectrum's pre-built .interpolant is never read and the fixture does
# not need to supply one.
# FakeSpectrum supplies exactly those two, so the tests can drive the function
# with tiny, fully-controlled inputs instead of reading real .e files from disk.
import numpy as np
import pandas as pd


class FakeSpectrum:
    def __init__(self, file_name, energies, norm):
        # store energy/norm as float arrays so integer test grids behave like real data
        energies = np.asarray(energies, dtype=float)
        norm = np.asarray(norm, dtype=float)
        self.file_name = file_name
        # energy is the index; 'norm' is the single column the function reads
        self.data_df = pd.DataFrame({'norm': norm}, index=energies)

In [ ]:
def test_returns_intersection_range_shapes_and_values():
    # Sample grid spans 0..10 on integer eV; its 'norm' is a simple ramp (10*E)
    # so the expected response vector b is trivial to read off.
    sample = FakeSpectrum('sample', np.arange(0, 11), np.arange(0, 11) * 10.0)

    # The references sit on grids shifted a fraction of an eV off the sample grid,
    # so none of their nodes line up with the sample energies and the function
    # must genuinely interpolate them.  Each reference's 'norm' is still a linear
    # function of energy, and linear interpolation reproduces a linear function
    # exactly, so the expected values at the sample energies stay simple.
    offset = 0.3
    ref_a_energies = np.arange(2, 11) - offset   # 1.7 .. 9.7 -> highest lower bound (1.7 eV)
    ref_a = FakeSpectrum('ref_a', ref_a_energies, 2.0 * ref_a_energies)   # norm = 2*E
    ref_b_energies = np.arange(0, 9) + offset    # 0.3 .. 8.3 -> lowest upper bound (8.3 eV)
    ref_b = FakeSpectrum('ref_b', ref_b_energies, ref_b_energies + 1.0)   # norm = E + 1

    valid_energies, A, b, low_limiters, high_limiters = \
        interpolate_references_at_sample_energies([ref_a, ref_b], sample)

    # Common range is the intersection [max(lows), min(highs)] = [1.7, 8.3]; the
    # valid energies are the SAMPLE grid points falling inside it, i.e. 2..8.
    np.testing.assert_array_equal(valid_energies, np.array([2, 3, 4, 5, 6, 7, 8], dtype=float))

    # A: one row per valid energy, one column per reference, in the input order.
    assert A.shape == (7, 2)
    # Column 0 is ref_a interpolated onto valid_energies (2*E), column 1 is ref_b (E+1).
    # These hold despite the offset grids because linear interp of linear data is exact.
    np.testing.assert_allclose(A[:, 0], 2.0 * valid_energies)
    np.testing.assert_allclose(A[:, 1], valid_energies + 1.0)

    # b is the sample's 'norm' (10*E) sampled at the valid energies - no interpolation.
    np.testing.assert_allclose(b, 10.0 * valid_energies)

In [ ]:
def test_identifies_low_and_high_limiters():
    # 'norm' values are irrelevant here, so use zeros; only the energy bounds matter.
    sample = FakeSpectrum('sample', np.arange(0, 11), np.zeros(11))
    ref_a = FakeSpectrum('ref_a', np.arange(2, 11), np.zeros(9))   # starts latest (2 eV)
    ref_b = FakeSpectrum('ref_b', np.arange(0, 9), np.zeros(9))    # ends earliest (8 eV)

    _, _, _, low_limiters, high_limiters = \
        interpolate_references_at_sample_energies([ref_a, ref_b], sample)

    # ref_a alone has the highest lower bound, so it alone limits the low end;
    # ref_b alone has the lowest upper bound, so it alone limits the high end.
    assert low_limiters == [ref_a]
    assert high_limiters == [ref_b]

In [ ]:
def test_reports_all_tied_limiters():
    sample = FakeSpectrum('sample', np.arange(0, 11), np.zeros(11))
    # ref_a and ref_c share the exact same starting energy (2 eV) ...
    ref_a = FakeSpectrum('ref_a', np.arange(2, 11), np.zeros(9))
    ref_c = FakeSpectrum('ref_c', np.arange(2, 11), np.zeros(9))
    # ... and ref_b and ref_d share the exact same ending energy (8 eV).
    ref_b = FakeSpectrum('ref_b', np.arange(0, 9), np.zeros(9))
    ref_d = FakeSpectrum('ref_d', np.arange(0, 9), np.zeros(9))

    _, _, _, low_limiters, high_limiters = \
        interpolate_references_at_sample_energies([ref_a, ref_c, ref_b, ref_d], sample)

    # Every spectrum tied at a limiting energy is reported (not just the first),
    # in the order the function scans them: sample first, then the references as given.
    assert low_limiters == [ref_a, ref_c]
    assert high_limiters == [ref_b, ref_d]

In [ ]:
def test_sample_spectrum_can_be_the_limiter():
    # The sample is NARROWER than both references (3..7 vs 0..10), so the sample
    # itself limits BOTH ends and none of its grid points are excluded.
    sample = FakeSpectrum('sample', np.arange(3, 8), np.zeros(5))
    ref_a = FakeSpectrum('ref_a', np.arange(0, 11), np.zeros(11))
    ref_b = FakeSpectrum('ref_b', np.arange(0, 11), np.zeros(11))

    valid_energies, A, b, low_limiters, high_limiters = \
        interpolate_references_at_sample_energies([ref_a, ref_b], sample)

    # No sample energies fall outside the common range, so the full grid is kept.
    np.testing.assert_array_equal(valid_energies, np.arange(3, 8, dtype=float))
    assert A.shape == (5, 2)
    # The sample is the tightest spectrum at each end, so it is the sole limiter.
    assert low_limiters == [sample]
    assert high_limiters == [sample]

In [ ]:
def test_identical_ranges_exclude_nothing():
    # When the sample and reference share the same grid, the whole grid is valid
    # and the design matrix has exactly one column.
    grid = np.arange(0, 6)
    sample = FakeSpectrum('sample', grid, np.arange(0, 6) * 1.0)
    ref_a = FakeSpectrum('ref_a', grid, np.arange(0, 6) * 1.0)

    valid_energies, A, b, low_limiters, high_limiters = \
        interpolate_references_at_sample_energies([ref_a], sample)

    np.testing.assert_array_equal(valid_energies, grid.astype(float))
    assert A.shape == (6, 1)
    # With identical bounds, both the sample and the reference tie at each edge.
    assert low_limiters == [sample, ref_a]
    assert high_limiters == [sample, ref_a]

In [ ]:
def test_interpolation_method_is_selectable():
    # Every other fixture in this file uses linear (or zero) 'norm' data, which both
    # a linear and a cubic interpolant reproduce exactly -- so those tests cannot tell
    # the two methods apart. This one uses data with genuine curvature.
    #
    # The reference sits on a COARSE grid (every 2 eV) and is sampled on a FINER grid
    # (every 1 eV), so half the sample energies fall strictly between reference nodes
    # and must actually be interpolated. That is the same situation as the real data,
    # where several references are measured every 1.05 eV against a 0.5 eV sample grid.
    def cubic_norm(energy):
        return 0.01 * energy ** 3 - 0.2 * energy ** 2 + energy + 1.0

    sample_energies = np.arange(0, 21, 1)
    sample = FakeSpectrum('sample', sample_energies, np.zeros(len(sample_energies)))
    reference_energies = np.arange(0, 21, 2)
    reference = FakeSpectrum('curved_ref', reference_energies, cubic_norm(reference_energies))

    _, A_cubic, _, _, _ = interpolate_references_at_sample_energies(
        [reference], sample, make_interpolant=make_cubic_spline_interpolant,
    )
    _, A_linear, _, _, _ = interpolate_references_at_sample_energies(
        [reference], sample, make_interpolant=make_linear_interpolant,
    )

    # An interpolating cubic spline through samples of a cubic polynomial reproduces
    # that polynomial exactly, so the cubic arm is right to floating-point tolerance.
    np.testing.assert_allclose(A_cubic[:, 0], cubic_norm(sample_energies), atol=1e-10)

    # The linear arm chords across each 2 eV gap and so must NOT match, and the
    # disagreement has to be large enough to matter rather than a rounding artifact.
    assert not np.allclose(A_linear[:, 0], cubic_norm(sample_energies), atol=1e-10)
    assert np.abs(A_linear[:, 0] - A_cubic[:, 0]).max() > 0.05

    # Both methods interpolate rather than extrapolate, so they agree exactly wherever
    # a sample energy coincides with a reference node -- the difference is confined to
    # the points between nodes.
    on_node = np.isin(sample_energies, reference_energies)
    np.testing.assert_allclose(A_linear[on_node, 0], A_cubic[on_node, 0], atol=1e-10)
    assert np.abs(A_linear[~on_node, 0] - A_cubic[~on_node, 0]).max() > 0.05

In [ ]:
def test_default_interpolation_is_cubic():
    # Pins the default. Everything in this notebook that was produced before
    # make_interpolant existed -- the v1-v5 holdout comparison, the fits, the
    # prediction error tables -- was computed with the cubic spline
    # ReferenceSpectrum builds internally, so the default has to stay cubic for
    # those results to remain reproducible.
    def cubic_norm(energy):
        return 0.01 * energy ** 3 - 0.2 * energy ** 2 + energy + 1.0

    sample_energies = np.arange(0, 21, 1)
    sample = FakeSpectrum('sample', sample_energies, np.zeros(len(sample_energies)))
    reference_energies = np.arange(0, 21, 2)
    reference = FakeSpectrum('curved_ref', reference_energies, cubic_norm(reference_energies))

    _, A_default, _, _, _ = interpolate_references_at_sample_energies([reference], sample)
    _, A_cubic, _, _, _ = interpolate_references_at_sample_energies(
        [reference], sample, make_interpolant=make_cubic_spline_interpolant,
    )

    np.testing.assert_array_equal(A_default, A_cubic)

In [ ]:
# Under pytest these test_* functions are collected and run automatically.
# Inside the notebook we invoke each one directly and report pass/fail so the
# suite can be exercised interactively as well.
_test_fns = [obj for name, obj in sorted(globals().items())
             if name.startswith('test_') and callable(obj)]
for _fn in _test_fns:
    _fn()
    print(f'PASSED: {_fn.__name__}')
print(f'\n{len(_test_fns)} tests passed')

### Visualizing the interpolated design matrix

`plot_interpolated_references` draws the output of
`interpolate_references_at_sample_energies` on a single axis: the sample response
vector `b` and every interpolated reference column of `A`, all against
`valid_energies`. The edges of the common energy range are marked with dashed
lines annotated with the spectra that limit them.

In [ ]:
def plot_interpolated_references(
    valid_energies, A, b, low_limiters, high_limiters,
    reference_spectra, sample_spectrum, ax=None,
):
    """Visualize the output of interpolate_references_at_sample_energies.

    Built to stay legible from 3 references to 30+.  The sample is bold black;
    references are colored by the *role* they play, not one hue each, so the
    legend never explodes:
      * references that limit the low edge  -> blue
      * references that limit the high edge -> orange
      * references that limit both edges    -> green
      * every other reference               -> faint gray context (one legend row)
    Color is never the only cue: the legend names each role and its count, and
    the red dashed bound lines name the limiting spectra (or their count when many
    tie at the same energy).

    Parameters
    ----------
    valid_energies, A, b, low_limiters, high_limiters
        The five values returned by interpolate_references_at_sample_energies.
    reference_spectra : list of ReferenceSpectrum
        The same reference pool passed to that function; ``A``'s columns follow
        this order.
    sample_spectrum : Spectrum or ReferenceSpectrum
        The sample whose response vector ``b`` is plotted.
    ax : matplotlib Axes, optional
        Axis to draw on. A new figure/axis is created when omitted.

    Returns
    -------
    ax : matplotlib Axes
        The axis the plot was drawn on.
    """
    from collections import Counter
    from matplotlib.lines import Line2D

    if ax is None:
        _, ax = plt.subplots(figsize=(11, 5))

    low_ids = {id(s) for s in low_limiters}
    high_ids = {id(s) for s in high_limiters}
    role_color = {'low': 'tab:blue', 'high': 'tab:orange', 'low+high': 'tab:green'}

    def role_of(ref):
        r = [name for name, ids in (('low', low_ids), ('high', high_ids)) if id(ref) in ids]
        return '+'.join(r) if r else None

    # context (non-limiter) references first, faint gray, underneath everything
    n_other = 0
    for i, ref in enumerate(reference_spectra):
        if role_of(ref) is None:
            ax.plot(valid_energies, A[:, i], color='0.45', alpha=0.5, linewidth=0.8, zorder=1)
            n_other += 1

    # limiting references, colored by role; all co-limiters share the role color
    role_counts = Counter()
    for i, ref in enumerate(reference_spectra):
        role = role_of(ref)
        if role is None:
            continue
        ax.plot(valid_energies, A[:, i], color=role_color[role], alpha=0.75, linewidth=1.3, zorder=3)
        role_counts[role] += 1

    # the full sample spectrum at ALL its energies: the portion outside the common
    # range (where no references are interpolated) is drawn dimmed and dashed so
    # every sample energy is visible, ...
    sample_energies = sample_spectrum.data_df.index.values
    sample_norm = sample_spectrum.data_df['norm'].values
    faint_sample, = ax.plot(sample_energies, sample_norm, color='black', alpha=0.5,
                            linewidth=1, linestyle='--', zorder=2)
    # ... while the in-range response vector b is drawn bold and solid on top.
    sample_line, = ax.plot(valid_energies, b, color='black', linewidth=2, zorder=4)
    handles = [sample_line, faint_sample]
    labels = [f'{sample_spectrum.file_name} (sample, in range)',
              'sample (all energies)']

    # one legend row per non-empty limiter role (names live on the bound lines below)
    for role in ('low', 'high', 'low+high'):
        n = role_counts.get(role, 0)
        if n:
            handles.append(Line2D([], [], color=role_color[role], alpha=0.75, linewidth=1.3))
            labels.append(f'{role} limiter{"" if n == 1 else "s"} ({n})')

    # one proxy entry stands in for every faded reference
    if n_other:
        handles.append(Line2D([], [], color='0.45', alpha=0.7, linewidth=0.8))
        labels.append(f'other references ({n_other})')

    # exact common-range bounds (red dashed) and the sampled-energy extent (gray dotted).
    # every low/high limiter shares the limiting energy, so read it off the first one;
    # summarise the limiter names so a big tie does not blow out the legend width.
    def summarize(spectra):
        names = [s.file_name for s in spectra]
        return ', '.join(names) if len(names) <= 2 else f'{len(names)} spectra'

    energy_min = low_limiters[0].data_df.index.values[0]
    energy_max = high_limiters[0].data_df.index.values[-1]
    lo = ax.axvline(energy_min, color='tab:red', linestyle='--', linewidth=1, zorder=2)
    hi = ax.axvline(energy_max, color='tab:red', linestyle='--', linewidth=1, zorder=2)
    ax.axvline(valid_energies[0], color='gray', linestyle=':', linewidth=1, zorder=2)
    ax.axvline(valid_energies[-1], color='gray', linestyle=':', linewidth=1, zorder=2)
    handles += [lo, hi, Line2D([], [], color='gray', linestyle=':', linewidth=1)]
    labels += [
        f'low bound {energy_min:.1f} eV (limited by {summarize(low_limiters)})',
        f'high bound {energy_max:.1f} eV (limited by {summarize(high_limiters)})',
        f'sampled extent [{valid_energies[0]:.1f}, {valid_energies[-1]:.1f}] eV',
    ]

    ax.set_xlabel('Energy (eV)')
    ax.set_ylabel('Normalized Fluorescence')
    ax.set_title(f'Interpolated references vs {sample_spectrum.file_name} '
                 f'({len(valid_energies)} energies, {len(reference_spectra)} references)')
    # legend outside the axes on the right so it never covers the data
    ax.legend(handles, labels, loc='upper left', bbox_to_anchor=(1.02, 1),
              fontsize='small', borderaxespad=0.0)
    return ax

In [ ]:
# Demonstrate on the three references used elsewhere in the notebook ...
valid_energies, A, b, low_limiters, high_limiters = \
    interpolate_references_at_sample_energies(reference_spectra, unknown_spectrum)
fig, ax = plt.subplots(figsize=(11, 5))
plot_interpolated_references(
    valid_energies, A, b, low_limiters, high_limiters,
    reference_spectra, unknown_spectrum, ax=ax,
)
plt.show()

# ... and on the full reference pool, to confirm it stays readable with many
# references (only the sample and the limiting spectra are highlighted; the rest
# recede to faint gray and collapse into a single legend entry).
ve, A_all, b_all, low_all, high_all = \
    interpolate_references_at_sample_energies(reference_spectra_list, unknown_spectrum)
fig, ax = plt.subplots(figsize=(11, 5))
plot_interpolated_references(
    ve, A_all, b_all, low_all, high_all,
    reference_spectra_list, unknown_spectrum, ax=ax,
)
plt.show()

## Where a selected combination sits in the reference set

The fit ranks combinations by prediction error and names a winner, but a winning
combination is not self-explanatory. NNLS cannot tell apart two references that are nearly
the same vector — it can trade one against the other almost freely — so three references
drawn from a single tight cluster are a weaker claim about the sample than three that are
nothing alike, even at identical prediction error.

`cluster_reference_spectra` clusters the columns of the design matrix, which is the
reference set exactly as the fit saw it: same interpolation, same common energy range, same
grid. It also works out where the tree can be cut without that being an eyeball
judgment, by asking how close two references get when there is nothing bringing them
together: the reference values within each energy are repeatedly shuffled between
references — destroying any resemblance while leaving every energy the values it actually
measured — and the shuffled set is reclustered. The cutoff is the distance that only the
tightest 5% of those chance merges reach, so a real merge below it is closer than
randomness readily produces. `plot_reference_dendrogram` then draws the tree and, for each
combination handed to it, shades the smallest subtree containing all of that combination's
references and brackets it at the height where that subtree merges. The height is the
answer: near zero means the combination came out of one cluster, at the root means it spans
the reference set.

The functions are defined here, next to the interpolation that builds the design matrix they
cluster. Their figures appear further down, where there are fits to place: the summary of the
combination search draws both trees plain, each per-combination summary marks that
combination in them, and the correlation-vs-cosine comparison after the fits settles which
distance the trees should use.

In [ ]:
# Hierarchical clustering of the reference spectra that make up the design matrix.
#
# The fit reports which combination of references wins; it says nothing about how those
# references relate to each other. That matters for reading the result: NNLS cannot tell
# apart two references that are nearly the same vector, so a combination drawn from one
# tight cluster is a weaker claim than the same-sized combination drawn from opposite
# ends of the tree.
#
# This clusters the columns of A rather than re-reading the spectra from disk, so the
# tree describes exactly the references the fit saw -- same interpolation, same common
# energy range, same grid.
import hashlib

import scipy.cluster.hierarchy as hc
from scipy.spatial.distance import pdist, squareform


def permute_within_rows(A, rng):
    """Shuffle the reference values within each energy row, independently row by row.

    This builds the comparison case the significance cutoff below is measured against --
    the *null model*, meaning a version of this data in which the thing being looked for
    is absent by construction. The thing being looked for here is resemblance between
    references, so each energy keeps exactly the values it measured while which reference
    holds which value is randomized. Every reference still looks like a plausible spectrum
    row by row, but any tendency for two of them to rise and fall together is gone.
    Clustering such a randomized copy shows how close references come to each other for no
    reason at all, which is the yardstick a real merge has to beat.

    `Generator.permuted(A, axis=1)` does this and returns a copy. It replaces the
    DataFrame idiom in the older clustering notebooks,
    `df.values[i, :] = shuffle(df.values[i, :])`, which is unreliable: `.values` may hand
    back a copy under pandas copy-on-write, in which case the assignment silently does
    nothing and the "randomized" copy is the real data again. The cutoff is then measured
    against the very merge heights it is supposed to judge, and certifies whatever it is
    given. `mrfitty/combination_fit.py` fixes that with `.iloc`; working on a numpy copy
    avoids the question.
    """
    return rng.permuted(A, axis=1)


def phase_randomize_columns(A, rng):
    """Surrogate references that keep each reference's own character but not its relatives.

    Another way to build the randomized copies, in place of `permute_within_rows`, done the
    standard surrogate-data way:
    Fourier transform each reference, replace the phases with uniform random ones, transform
    back. Every reference keeps its own power spectrum -- and so its mean, its variance and
    its smoothness -- while what it shares with the other references is destroyed.

    The intent was a stricter comparison than shuffling values within an energy: one that asks
    "are these two references more alike than two arbitrary spectra with this character?"
    rather than "is there any shared structure at all?". Measured, it is the *easier* of the
    two to beat -- see the surrogate comparison below, which is why `cluster_reference_spectra`
    still defaults to `permute_within_rows`. It is kept because the comparison is worth
    being able to re-run on another reference set.
    """
    n_energies = A.shape[0]
    spectrum = np.fft.rfft(A, axis=0)
    phases = rng.uniform(0.0, 2.0 * np.pi, size=spectrum.shape)
    phases[0, :] = 0.0                   # DC stays real, so each reference keeps its mean
    if n_energies % 2 == 0:
        phases[-1, :] = 0.0              # and so does Nyquist, where the length gives one
    return np.fft.irfft(np.abs(spectrum) * np.exp(1j * phases), n=n_energies, axis=0)


def cluster_reference_spectra(
    A, ref_names, rng, metric='correlation', method='complete',
    resample_count=1000, percentile=95.0, surrogate=permute_within_rows, verbose=True,
):
    """Hierarchically cluster the reference spectra that are the columns of A.

    Parameters
    ----------
    A : ndarray, shape (n_energies, n_refs)
        The design matrix from `interpolate_references_at_sample_energies`. Its columns
        are the references, so the clustering transposes it -- `pdist` treats *rows* as
        the observations. That A is free of NaN is load-bearing rather than incidental:
        `pdist` propagates a single NaN across a reference's entire distance row, and
        `linkage` then fails opaquely. A is NaN-free because it is built on the energy
        range common to the sample and every reference.
    ref_names : list of str
        Reference file names, parallel to A's columns.
    rng : numpy Generator
        Drives the randomized comparison copies only. The tree itself is deterministic,
        so a different seed moves the cutoff and nothing else.
    metric : str
        Any `pdist` metric. 'correlation' (the default, and what
        `mrfitty/combination_fit.py` uses) centers each reference before comparing, so it
        measures shape alone; 'cosine' does not center, so it measures the angle from the
        origin -- closer to the collinearity that makes two references interchangeable in
        a non-negative fit. Which one to use is settled by the correlation-vs-cosine
        comparison that follows the fits below.
    method : str
        Linkage method, passed to `scipy.cluster.hierarchy.linkage`.
    resample_count, percentile : int, float
        How the significance cutoff is built. `resample_count` times, the references are
        randomized against each other by `permute_within_rows` and reclustered; every
        merge height those randomized trees produce is pooled, and the cutoff is the
        `percentile`th of that pool. Read the result as "two references this close would
        hardly ever happen by chance": at the 95th percentile, only one merge in twenty of
        the randomized ones was that tight, so merges below the cutoff are the ones worth
        calling clusters. 1,000 replicates take about a tenth of a second at this size, so
        there is no reason to run fewer -- and no reason to reach for `run_arms`, whose
        process round trip would cost more than the work.
    surrogate : callable
        `surrogate(A, rng) -> ndarray`, how a randomized copy is built. The default,
        `permute_within_rows`, shuffles values between references at each energy;
        `phase_randomize_columns` is the alternative, and the comparison below measures what
        the choice costs.

    Returns
    -------
    dict carrying the tree and everything drawn or reported from it, so that nothing is
    recomputed at draw time: 'distances' (condensed), 'Z', 'cutoff_distance',
    'chance_merge_heights' (every merge height the randomized copies produced -- the
    distribution the cutoff is one percentile of), 'cophenetic_correlation',
    'cophenetic_distances', 'labels' (flat clusters at the cutoff), 'n_clusters', the
    parameters, and 'first_permutation_digest' -- see below.
    """
    A = np.ascontiguousarray(A, dtype=float)
    ref_names = list(ref_names)
    if A.shape[1] != len(ref_names):
        raise ValueError(f'A has {A.shape[1]} columns but {len(ref_names)} reference names were given')
    if A.shape[1] < 2:
        raise ValueError('clustering needs at least two references')
    if not np.isfinite(A).all():
        raise ValueError('A holds non-finite values, which pdist would propagate across whole '
                         'distance rows; build A over the common energy range first')

    # Catch the degenerate columns each metric cannot handle, naming the reference, rather
    # than letting pdist return NaN and linkage fail somewhere further down.
    if metric == 'correlation':
        degenerate = [ref_names[i] for i in np.flatnonzero(A.std(axis=0) == 0)]
        reason = 'is constant, so its correlation with anything is undefined'
    elif metric == 'cosine':
        degenerate = [ref_names[i] for i in np.flatnonzero(np.linalg.norm(A, axis=0) == 0)]
        reason = 'is all zeros, so its angle to anything is undefined'
    else:
        degenerate, reason = [], ''
    if degenerate:
        raise ValueError(f'{", ".join(degenerate)} {reason} under metric={metric!r}')

    def condensed_distances(matrix):
        # pdist clusters rows, so the design matrix is transposed to put references there.
        # The clip removes the ~1e-17 negatives 'correlation' can return for two nearly
        # identical references, which would otherwise become negative merge heights.
        return np.clip(pdist(np.ascontiguousarray(matrix.T), metric=metric), 0.0, None)

    distances = condensed_distances(A)
    Z = hc.linkage(distances, method=method)

    # Recluster `resample_count` randomized copies and pool every merge height they
    # produce. That pool is what "by chance" means for this particular reference set, and
    # the cutoff is one percentile of it.
    chance_merge_heights = np.empty(resample_count * (A.shape[1] - 1))
    first_permutation_digest = None
    for i in range(resample_count):
        permuted = surrogate(A, rng)
        if i == 0:
            # 40 bytes that let a comparison assert -- rather than assume -- that two arms
            # drew the same permutations, the way compare_interpolation_methods asserts its
            # holdout masks are shared.
            first_permutation_digest = hashlib.sha1(permuted.tobytes()).hexdigest()
        shuffled_Z = hc.linkage(condensed_distances(permuted), method=method)
        chance_merge_heights[i * (A.shape[1] - 1):(i + 1) * (A.shape[1] - 1)] = shuffled_Z[:, 2]
    cutoff_distance = float(np.percentile(chance_merge_heights, percentile))

    cophenetic_correlation, cophenetic_distances = hc.cophenet(Z, distances)
    labels = hc.fcluster(Z, t=cutoff_distance, criterion='distance')

    if verbose:
        print(f'clustered {len(ref_names)} references by {metric} distance, {method} linkage')
        print(f'  pairwise distance range: {distances.min():.5f}-{distances.max():.5f}')
        print(f'  merge height range:      {Z[:, 2].min():.5f}-{Z[:, 2].max():.5f} (root {Z[-1, 2]:.5f})')
        print(f'  cutoff: {cutoff_distance:.5f} ({percentile:g}th percentile of '
              f'{resample_count} randomized copies from {surrogate.__name__})')
        sizes = sorted((int(size) for size in np.bincount(labels)[1:]), reverse=True)
        print(f'  {labels.max()} cluster(s) at the cutoff, sizes {sizes}')
        print(f'  {(Z[:, 2] < cutoff_distance).sum()} of {Z.shape[0]} merges fall below it')
        print(f'  cophenetic correlation: {cophenetic_correlation:.4f}')
        if cutoff_distance >= Z[-1, 2]:
            print('  NOTE: the cutoff sits above the root, so no merge is significant at this '
                  'percentile -- every reference is its own cluster')

    return {
        'ref_names': ref_names,
        'metric': metric,
        'method': method,
        'percentile': percentile,
        'resample_count': resample_count,
        'surrogate': surrogate.__name__,
        'distances': distances,
        'Z': Z,
        'cutoff_distance': cutoff_distance,
        'chance_merge_heights': chance_merge_heights,
        'cophenetic_correlation': float(cophenetic_correlation),
        'cophenetic_distances': cophenetic_distances,
        'labels': labels,
        'n_clusters': int(labels.max()),
        'first_permutation_digest': first_permutation_digest,
    }

In [ ]:
def smallest_enclosing_subtree(Z, leaf_indices):
    """The smallest subtree of Z containing every leaf in leaf_indices.

    Walking Z once and unioning leaf sets, then taking the smallest set that contains the
    group, rather than scanning nodes by increasing height: the leaf sets of a linkage
    form a laminar family, so the smallest containing node is unique, whereas a
    height-ordered scan assumes merge heights increase monotonically -- true for the
    'complete' default but not for every method `linkage` accepts.

    Returns a dict with 'node' (linkage node id), 'height' (0 for a single leaf, which
    encloses itself), 'leaf_indices', 'size' and 'is_root'.
    """
    target = frozenset(int(i) for i in leaf_indices)
    if not target:
        raise ValueError('no references given to locate')

    n_leaves = Z.shape[0] + 1
    leaf_sets = {i: frozenset((i,)) for i in range(n_leaves)}
    for row, (left, right, _, _) in enumerate(Z):
        leaf_sets[n_leaves + row] = leaf_sets[int(left)] | leaf_sets[int(right)]

    size, node = min((len(leaves), node) for node, leaves in leaf_sets.items() if target <= leaves)
    return {
        'node': node,
        'height': 0.0 if node < n_leaves else float(Z[node - n_leaves, 2]),
        'leaf_indices': sorted(leaf_sets[node]),
        'size': size,
        'is_root': node == 2 * n_leaves - 2,
    }


def _normalize_highlight(highlight, ref_names):
    """{label: sorted leaf indices} from names, indices, or a NaN-padded ref_indices row."""
    if highlight is None:
        return {}
    groups = highlight if isinstance(highlight, dict) else {'highlighted': highlight}
    name_to_index = {name: i for i, name in enumerate(ref_names)}

    normalized = {}
    for label, refs in groups.items():
        # a bare string is iterable, and iterating it would look up one character at a time
        if isinstance(refs, str) or np.isscalar(refs):
            refs = [refs]
        indices = []
        for ref in refs:
            if isinstance(ref, str):
                if ref not in name_to_index:
                    raise ValueError(f'{ref!r} is not one of the clustered references')
                indices.append(name_to_index[ref])
            else:
                # results['ref_indices'] rows are float and NaN-padded to max_M; drop the
                # padding here, because int(nan) raises and astype(int) silently yields a
                # huge negative index
                value = float(ref)
                if np.isfinite(value):
                    indices.append(int(value))
        if not indices:
            raise ValueError(f'highlight group {label!r} names no references')
        normalized[label] = sorted(set(indices))
    return normalized


def best_subsets_by_size(results):
    """{'best M=m': [column indices]} for the lowest-median-PE combination at each size.

    The same selection `plot_best_subset_bootstrap_summaries` makes, reduced to what
    `plot_reference_dendrogram` highlights.
    """
    subsets = {}
    for m in sorted(set(results['M'])):
        m_indices = np.where(results['M'] == m)[0]
        medians = np.array([np.median(results['bootstrap_pes'][i]) for i in m_indices])
        best = m_indices[int(np.argmin(medians))]
        subsets[f'best M={m}'] = [int(j) for j in results['ref_indices'][best, :m]]
    return subsets

In [ ]:
HIGHLIGHT_COLORS = ('tab:red', 'tab:blue', 'tab:green', 'tab:purple', 'tab:brown')
HIGHLIGHT_MARKERS = ('o', 's', '^', 'D', 'v')
HIGHLIGHT_LINESTYLES = ('-', (0, (4, 2)), (0, (1, 1.5)), (0, (6, 2, 1, 2)), (0, (3, 1, 1, 1, 1, 1)))


def plot_reference_dendrogram(
    clustering, highlight=None, ax=None, color_threshold=None, show_cutoff=True, title=None,
    legend_loc='below',
):
    """Draw the reference tree, showing where whole reference combinations sit in it.

    Each highlighted group gets the smallest subtree that contains all of its references,
    shaded and bracketed at the height that subtree merges — which is the question the
    figure exists to answer. A group whose bracket sits low is a combination drawn from
    one cluster of near-interchangeable references; a group whose bracket sits at the root
    is a combination spanning the whole reference set.

    Parameters
    ----------
    clustering : dict
        The return value of `cluster_reference_spectra`.
    highlight : list or dict, optional
        One combination, or `{label: combination}` for several at once. A combination may
        be given as reference names or as column indices into A, so a row of
        `results['ref_indices']` can be passed through unchanged.
    color_threshold : float, optional
        Passed to `scipy`'s dendrogram. The default draws the whole tree in one neutral
        gray so that color belongs to the highlight; pass
        `clustering['cutoff_distance']` to color the significant clusters instead.
    legend_loc : {'below', 'inside', None}
        Where to put the legend. 'below' hangs it under the axes, which is right for a
        figure of its own but lands on whatever is drawn beneath when this panel is one
        cell of a grid; 'inside' puts it in the lower left, where a dendrogram has little
        ink because the branches crowd toward the leaves; None suppresses it.

    Returns
    -------
    (ax, groups) : the axis, and {label: smallest_enclosing_subtree(...)} so a caller can
    report the same numbers the figure draws without recomputing them.
    """
    from matplotlib.lines import Line2D
    from matplotlib.transforms import blended_transform_factory

    ref_names = clustering['ref_names']
    Z = clustering['Z']
    cutoff = clustering['cutoff_distance']
    highlighted = _normalize_highlight(highlight, ref_names)

    if ax is None:
        _, ax = plt.subplots(figsize=(9, max(4.0, 0.34 * len(ref_names))))

    dendrogram = hc.dendrogram(
        Z, ax=ax, orientation='left', labels=ref_names,
        color_threshold=0.0 if color_threshold is None else color_threshold,
        above_threshold_color='0.35',
    )
    # With orientation='left' the x axis carries distance and is inverted (root at the
    # left, leaves at 0), y carries the leaves at 5, 15, 25, ... in drawn order. scipy
    # sizes the x axis from the root height alone, so a cutoff above the root would be
    # drawn off the axes; size it from both and then stop autoscaling.
    ax.set_xlim(max(1.05 * Z[-1, 2], 1.08 * cutoff), 0.0)
    ax.set_ylim(0.0, 10.0 * len(ref_names))
    ax.set_autoscale_on(False)

    leaf_position = {leaf: position for position, leaf in enumerate(dendrogram['leaves'])}
    root_height = Z[-1, 2]
    cap_width = 0.02 * root_height
    marker_transform = blended_transform_factory(ax.transAxes, ax.transData)

    groups, handles, labels, leaf_colors = {}, [], [], {}
    for k, (label, indices) in enumerate(highlighted.items()):
        color = HIGHLIGHT_COLORS[k % len(HIGHLIGHT_COLORS)]
        marker = HIGHLIGHT_MARKERS[k % len(HIGHLIGHT_MARKERS)]
        linestyle = HIGHLIGHT_LINESTYLES[k % len(HIGHLIGHT_LINESTYLES)]
        group = smallest_enclosing_subtree(Z, indices)
        groups[label] = group

        positions = [leaf_position[leaf] for leaf in group['leaf_indices']]
        y_low, y_high = 10 * min(positions), 10 * max(positions) + 10

        # Shade the enclosing subtree, except when it is the whole tree: a full-height band
        # says nothing and only dims the figure. That case is reported by the bracket
        # sitting at the root, and by the legend.
        if group['size'] < len(ref_names):
            ax.axhspan(y_low, y_high, color=color, alpha=0.10, zorder=0)

        # Bracket at the merge height, spanning the subtree. Two combinations can share an
        # enclosing subtree -- on this data the best M=2 and M=3 subsets both reach the
        # root -- so the linestyle, not the color alone, is what tells the brackets apart.
        if group['height'] > 0:
            ax.vlines(group['height'], y_low + 1.5, y_high - 1.5, color=color, linewidth=2.0,
                      linestyles=linestyle, zorder=5)
            for y in (y_low + 1.5, y_high - 1.5):
                ax.plot([group['height'], group['height'] - cap_width], [y, y], color=color,
                        linewidth=2.0, zorder=5)

        # Markers sit inside the axes: with orientation='left' scipy puts the leaf labels
        # outside on the right, so anything past the axes edge would land on the text.
        for leaf in indices:
            leaf_colors.setdefault(leaf, []).append(color)
            ax.plot(0.985 - 0.028 * k, 10 * leaf_position[leaf] + 5, marker=marker, color=color,
                    markersize=6, transform=marker_transform, clip_on=False, zorder=6)

        extent = ('a single leaf' if group['height'] == 0 else
                  f"subtree height {group['height']:.3f} = {group['height'] / root_height:.0%} "
                  f"of root, {group['size']} of {len(ref_names)} refs")
        # a lone reference is inside every cluster trivially, so only say this of a subtree
        within = ('' if group['height'] == 0 or group['height'] > cutoff
                  else ', within the cutoff')
        handles.append(Line2D([], [], color=color, marker=marker, linestyle=linestyle,
                              linewidth=2.0, markersize=6))
        labels.append(f'{label}: {len(indices)} ref{"" if len(indices) == 1 else "s"}, '
                      f'{extent}{within}')

    # Tick labels ascend with y, the same order as dendrogram['leaves'], so the leaf each
    # one names is read off that rather than by matching the label text.
    for position, tick_label in enumerate(ax.get_ymajorticklabels()):
        colors = leaf_colors.get(dendrogram['leaves'][position])
        if colors:
            tick_label.set_fontweight('bold')
            tick_label.set_color(colors[0] if len(colors) == 1 else 'black')

    if show_cutoff:
        ax.axvline(cutoff, color='tab:orange', linestyle='--', linewidth=1.2, zorder=4)
        handles.append(Line2D([], [], color='tab:orange', linestyle='--', linewidth=1.2))
        labels.append(f'cutoff {cutoff:.3f} — tighter than {clustering["percentile"]:g}% of '
                      f'merges from {clustering["resample_count"]} randomized copies')

    ax.set_xlabel(f'{clustering["metric"]} distance ({clustering["method"]} linkage)')
    ax.set_title(title or f'{len(ref_names)} reference spectra by {clustering["metric"]} distance '
                          f'— {clustering["n_clusters"]} clusters at the cutoff')
    if handles and legend_loc is not None:
        if legend_loc == 'below':
            # the leaf labels occupy the right of the axes, so the legend goes underneath
            ax.legend(handles, labels, loc='upper left', bbox_to_anchor=(0.0, -0.11),
                      fontsize='small', borderaxespad=0.0)
        elif legend_loc == 'inside':
            ax.legend(handles, labels, loc='lower left', fontsize='small', framealpha=0.9)
        else:
            raise ValueError(f"legend_loc must be 'below', 'inside' or None, not {legend_loc!r}")
    return ax, groups

In [ ]:
# ---------------------------------------------------------------------------
# Tests for the reference clustering and its dendrogram
#
# These take A and ref_names directly rather than spectra, so they need no FakeSpectrum:
# the fixture is a design matrix with known block structure, where which references
# belong together is decided rather than discovered.
#
# The runner is an explicit list, as in the block length tests above, rather than the
# globals() scan used for the interpolation tests -- a second scanning runner would
# re-run those suites and misreport the count.
# ---------------------------------------------------------------------------

def _block_design_matrix(seed=0, n_energies=120, noise=0.01):
    """Six references: three noisy copies of one shape, three of a very different one."""
    rng = np.random.default_rng(seed)
    energies = np.linspace(0, 1, n_energies)
    shape_a = np.exp(-((energies - 0.35) ** 2) / 0.004)
    shape_b = np.exp(-((energies - 0.65) ** 2) / 0.004) + 0.5 * energies
    columns = [shape_a, shape_a, shape_a, shape_b, shape_b, shape_b]
    A = np.column_stack([c + noise * rng.standard_normal(n_energies) for c in columns])
    return A, ['a1', 'a2', 'a3', 'b1', 'b2', 'b3']


def test_references_are_the_observations():
    # The clustering is of references, not energies: 6 references over 120 energies must
    # give a 5-row linkage. Forgetting the transpose would give 119 rows.
    A, ref_names = _block_design_matrix()
    clustering = cluster_reference_spectra(A, ref_names, np.random.default_rng(0),
                                           resample_count=50, verbose=False)
    assert clustering['Z'].shape == (len(ref_names) - 1, 4)
    assert clustering['distances'].shape == (len(ref_names) * (len(ref_names) - 1) // 2,)
    assert clustering['labels'].shape == (len(ref_names),)


def test_copies_of_one_shape_cluster_together():
    A, ref_names = _block_design_matrix()
    clustering = cluster_reference_spectra(A, ref_names, np.random.default_rng(0),
                                           resample_count=200, verbose=False)
    labels = clustering['labels']
    assert labels[0] == labels[1] == labels[2], 'the copies of shape a should share a cluster'
    assert labels[3] == labels[4] == labels[5], 'the copies of shape b should share a cluster'
    assert labels[0] != labels[3], 'the two shapes should not share a cluster'
    assert clustering['cophenetic_correlation'] > 0.9

    group = smallest_enclosing_subtree(clustering['Z'], [0, 1, 2])
    assert group['leaf_indices'] == [0, 1, 2], 'no other reference belongs in that subtree'
    assert group['height'] < clustering['cutoff_distance']


def test_randomized_copies_merge_less_tightly_than_the_real_data():
    # The trap the older notebooks' permute_row_elements falls into: if the shuffle
    # silently no-ops, the randomized copies are just the real data again, the cutoff is
    # measured against the very heights it is meant to judge, and it certifies whatever it
    # is given. On data with real blocks the true merges are far tighter than chance.
    A, ref_names = _block_design_matrix()
    clustering = cluster_reference_spectra(A, ref_names, np.random.default_rng(0),
                                           resample_count=200, verbose=False)
    observed = clustering['Z'][:, 2]
    by_chance = clustering['chance_merge_heights']
    assert by_chance.mean() > observed.mean(), \
        'shuffling must destroy the block structure, not preserve it'
    assert clustering['cutoff_distance'] > observed[:len(observed) - 1].max(), \
        'every within-shape merge should be more than chance'


def test_permutation_preserves_each_energy_row():
    # Pins what is shuffled: values move within an energy row, never between rows.
    A, _ = _block_design_matrix()
    permuted = permute_within_rows(A, np.random.default_rng(0))
    np.testing.assert_array_equal(np.sort(permuted, axis=1), np.sort(A, axis=1))
    assert not np.array_equal(permuted, A), 'the shuffle did nothing at all'


def test_smallest_enclosing_subtree_on_a_hand_built_tree():
    # 4 leaves: (0,1) join at 0.1, (2,3) join at 0.2, the two pairs join at 0.5.
    Z = np.array([[0.0, 1.0, 0.1, 2.0],
                  [2.0, 3.0, 0.2, 2.0],
                  [4.0, 5.0, 0.5, 4.0]])

    group = smallest_enclosing_subtree(Z, [0, 1])
    assert group['leaf_indices'] == [0, 1] and np.isclose(group['height'], 0.1)
    assert not group['is_root']

    # a group spanning both pairs can only be enclosed by the root
    group = smallest_enclosing_subtree(Z, [0, 2])
    assert group['leaf_indices'] == [0, 1, 2, 3] and np.isclose(group['height'], 0.5)
    assert group['is_root']

    # one reference encloses itself, at height 0 -- there is no subtree to shade
    group = smallest_enclosing_subtree(Z, [3])
    assert group['leaf_indices'] == [3] and group['height'] == 0.0 and group['size'] == 1


def test_highlight_accepts_names_indices_and_nan_padding():
    ref_names = ['a1', 'a2', 'a3', 'b1']
    assert (_normalize_highlight({'best M=2': ['a1', 'b1']}, ref_names)
            == _normalize_highlight({'best M=2': [0, 3]}, ref_names)
            == {'best M=2': [0, 3]})

    # a bare list becomes one group; a bare string is one name, not four characters
    assert _normalize_highlight(['a2'], ref_names) == {'highlighted': [1]}
    assert _normalize_highlight('a2', ref_names) == {'highlighted': [1]}

    # the literal shape of a results['ref_indices'] row: float, NaN-padded
    assert _normalize_highlight({'M=2': np.array([0.0, 3.0, np.nan])}, ref_names) == {'M=2': [0, 3]}

    try:
        _normalize_highlight({'oops': ['not_a_reference']}, ref_names)
    except ValueError:
        pass
    else:
        raise AssertionError('an unknown reference name should raise')


def test_correlation_ignores_an_offset_that_cosine_sees():
    # The whole difference between the two metrics: correlation centers each reference
    # first, so adding a constant leaves the tree alone; cosine measures the angle from
    # the origin, so the same constant moves it. Both are blind to a positive rescaling.
    A, ref_names = _block_design_matrix()

    def linkage_for(matrix, metric):
        return cluster_reference_spectra(matrix, ref_names, np.random.default_rng(0),
                                         metric=metric, resample_count=10, verbose=False)['Z']

    offset = A.copy()
    offset[:, 0] += 5.0
    np.testing.assert_allclose(linkage_for(A, 'correlation'), linkage_for(offset, 'correlation'))
    assert not np.allclose(linkage_for(A, 'cosine'), linkage_for(offset, 'cosine'))

    scaled = A.copy()
    scaled[:, 0] *= 3.0
    for metric in ('correlation', 'cosine'):
        np.testing.assert_allclose(linkage_for(A, metric), linkage_for(scaled, metric), atol=1e-12)


def test_cutoff_depends_on_the_generator_but_the_tree_does_not():
    A, ref_names = _block_design_matrix()
    first = cluster_reference_spectra(A, ref_names, np.random.default_rng(7),
                                      resample_count=200, verbose=False)
    again = cluster_reference_spectra(A, ref_names, np.random.default_rng(7),
                                      resample_count=200, verbose=False)
    other = cluster_reference_spectra(A, ref_names, np.random.default_rng(8),
                                      resample_count=200, verbose=False)

    assert first['cutoff_distance'] == again['cutoff_distance'], 'one seed, one cutoff'
    assert first['first_permutation_digest'] == again['first_permutation_digest']
    assert other['cutoff_distance'] != first['cutoff_distance'], 'a different seed should move it'
    np.testing.assert_array_equal(first['Z'], other['Z'])  # the tree itself is deterministic
    assert len(first['chance_merge_heights']) == 200 * (len(ref_names) - 1)


def test_degenerate_and_non_finite_columns_are_rejected():
    A, ref_names = _block_design_matrix()

    constant = A.copy()
    constant[:, 2] = 1.0
    try:
        cluster_reference_spectra(constant, ref_names, np.random.default_rng(0),
                                  metric='correlation', resample_count=5, verbose=False)
    except ValueError as error:
        assert 'a3' in str(error), 'the offending reference should be named'
    else:
        raise AssertionError('a constant column has no correlation distance')

    with_nan = A.copy()
    with_nan[3, 2] = np.nan
    try:
        cluster_reference_spectra(with_nan, ref_names, np.random.default_rng(0),
                                  resample_count=5, verbose=False)
    except ValueError as error:
        assert 'non-finite' in str(error)
    else:
        raise AssertionError('a NaN in A should raise rather than silently poison pdist')


def test_highlight_geometry_matches_the_drawn_dendrogram():
    A, ref_names = _block_design_matrix()
    clustering = cluster_reference_spectra(A, ref_names, np.random.default_rng(0),
                                           resample_count=100, verbose=False)
    fig, ax = plt.subplots()
    _, groups = plot_reference_dendrogram(
        clustering, highlight={'one shape': ['a1', 'a2', 'a3'], 'single': ['b2']}, ax=ax)

    # the shaded band must cover exactly the enclosing subtree's leaves, in drawn order
    drawn = hc.dendrogram(clustering['Z'], no_plot=True)['leaves']
    positions = sorted(drawn.index(leaf) for leaf in groups['one shape']['leaf_indices'])
    assert positions == list(range(positions[0], positions[-1] + 1)), \
        'a subtree draws as one contiguous run of leaves'
    spans = [patch for patch in ax.patches if hasattr(patch, 'get_xy')]
    assert spans, 'the enclosing subtree should be shaded'

    # a single-reference group has no subtree to shade or bracket
    assert groups['single']['height'] == 0.0 and groups['single']['size'] == 1

    # the x axis still runs root -> leaves and still contains the cutoff
    x_left, x_right = ax.get_xlim()
    assert x_left > x_right, 'orientation=left inverts the distance axis'
    assert x_left >= clustering['cutoff_distance'], 'the cutoff must be inside the axes'

    bold = {label.get_text() for label in ax.get_ymajorticklabels()
            if label.get_fontweight() == 'bold'}
    assert bold == {'a1', 'a2', 'a3', 'b2'}, f'highlighted leaves should be bold, got {bold}'
    plt.close(fig)


def test_legend_loc_places_or_suppresses_the_legend():
    # The default hangs the legend under the axes, which is right for a figure of its own
    # and wrong inside a grid, where it lands on whatever the next row draws. The summary
    # figures ask for 'inside' instead, so both placements have to actually differ.
    A, ref_names = _block_design_matrix()
    clustering = cluster_reference_spectra(A, ref_names, np.random.default_rng(0),
                                           resample_count=50, verbose=False)

    def legend_and_axes_extents(**kwargs):
        fig, ax = plt.subplots()
        try:
            plot_reference_dendrogram(clustering, highlight=['a1'], ax=ax, **kwargs)
            fig.canvas.draw()  # window extents are only meaningful once laid out
            legend = ax.get_legend()
            if legend is None:
                return None
            return legend.get_window_extent(), ax.get_window_extent()
        finally:
            # in a finally, so the bad-value case below closes its figure too rather than
            # leaving a stray blank one to be rendered under %matplotlib inline
            plt.close(fig)

    legend_box, axes_box = legend_and_axes_extents()
    assert legend_box.y1 <= axes_box.y0 + 1, 'the default legend should sit below the axes'

    legend_box, axes_box = legend_and_axes_extents(legend_loc='inside')
    assert legend_box.y0 >= axes_box.y0 - 1, "legend_loc='inside' should sit within the axes"

    assert legend_and_axes_extents(legend_loc=None) is None, 'legend_loc=None should draw none'

    try:
        legend_and_axes_extents(legend_loc='somewhere')
    except ValueError:
        pass
    else:
        raise AssertionError('an unknown legend_loc should raise')


def test_phase_randomize_keeps_each_reference_and_breaks_the_relationships():
    # What the surrogate has to do to be a null at all: every reference keeps its own
    # character -- mean, variance, smoothness, all of which live in its power spectrum --
    # while what it shares with the other references goes away.
    A, ref_names = _block_design_matrix()
    rng = np.random.default_rng(0)
    surrogate = phase_randomize_columns(A, rng)

    assert surrogate.shape == A.shape
    np.testing.assert_allclose(np.abs(np.fft.rfft(surrogate, axis=0)),
                               np.abs(np.fft.rfft(A, axis=0)), atol=1e-10)
    np.testing.assert_allclose(surrogate.mean(axis=0), A.mean(axis=0), atol=1e-10)
    np.testing.assert_allclose(surrogate.std(axis=0), A.std(axis=0), atol=1e-10)
    assert np.isrealobj(surrogate), 'randomizing DC or Nyquist would make the inverse complex'

    def mean_abs_cross_correlation(matrix):
        correlations = np.corrcoef(matrix.T)
        return np.abs(correlations[~np.eye(correlations.shape[0], dtype=bool)]).mean()

    assert mean_abs_cross_correlation(surrogate) < mean_abs_cross_correlation(A), \
        'the surrogate references should resemble each other less than the real ones'

    # a different seed gives a different surrogate; the same seed repeats it exactly
    np.testing.assert_array_equal(
        phase_randomize_columns(A, np.random.default_rng(1)),
        phase_randomize_columns(A, np.random.default_rng(1)))
    assert not np.allclose(surrogate, phase_randomize_columns(A, np.random.default_rng(1)))


def test_surrogate_choice_reaches_the_cutoff():
    # The parameter has to actually be used: the two nulls must give different cutoffs on
    # the same data and seed, and the default must stay permute_within_rows.
    A, ref_names = _block_design_matrix()

    def cutoff_with(**kwargs):
        return cluster_reference_spectra(A, ref_names, np.random.default_rng(3),
                                         resample_count=100, verbose=False, **kwargs)

    default = cutoff_with()
    shuffled = cutoff_with(surrogate=permute_within_rows)
    phase = cutoff_with(surrogate=phase_randomize_columns)

    assert default['cutoff_distance'] == shuffled['cutoff_distance']
    assert default['surrogate'] == 'permute_within_rows'
    assert phase['surrogate'] == 'phase_randomize_columns'
    assert phase['cutoff_distance'] != shuffled['cutoff_distance']


def test_flat_labels_match_the_dendrogram_colors():
    # fcluster at the cutoff and scipy's own color_threshold must agree, which is what
    # lets the plot color clusters by passing the cutoff straight through.
    A, ref_names = _block_design_matrix()
    clustering = cluster_reference_spectra(A, ref_names, np.random.default_rng(0),
                                           resample_count=100, verbose=False)
    drawn = hc.dendrogram(clustering['Z'], no_plot=True,
                          color_threshold=clustering['cutoff_distance'])
    by_color = {}
    for leaf, color in zip(drawn['leaves'], drawn['leaves_color_list']):
        by_color.setdefault(color, set()).add(leaf)
    by_label = {}
    for leaf, label in enumerate(clustering['labels']):
        by_label.setdefault(label, set()).add(leaf)
    assert sorted(map(sorted, by_color.values())) == sorted(map(sorted, by_label.values()))


_clustering_test_fns = [
    test_references_are_the_observations,
    test_copies_of_one_shape_cluster_together,
    test_randomized_copies_merge_less_tightly_than_the_real_data,
    test_permutation_preserves_each_energy_row,
    test_smallest_enclosing_subtree_on_a_hand_built_tree,
    test_highlight_accepts_names_indices_and_nan_padding,
    test_correlation_ignores_an_offset_that_cosine_sees,
    test_cutoff_depends_on_the_generator_but_the_tree_does_not,
    test_degenerate_and_non_finite_columns_are_rejected,
    test_highlight_geometry_matches_the_drawn_dendrogram,
    test_legend_loc_places_or_suppresses_the_legend,
    test_flat_labels_match_the_dendrogram_colors,
    test_phase_randomize_keeps_each_reference_and_breaks_the_relationships,
    test_surrogate_choice_reaches_the_cutoff,
]

for _fn in _clustering_test_fns:
    _fn()
    print(f'PASSED: {_fn.__name__}')
print(f'\n{len(_clustering_test_fns)} tests passed')

In [ ]:
# Cluster the whole reference pool -- A_all, the design matrix the interpolation section
# built above for these same 24 references -- under both distances, and draw the trees
# plain. No combination has been selected at this point in the notebook, so there is
# nothing to highlight; this is the reference set on its own terms, which is also what
# plot_ref_subsets_summary draws later. The {metric: clustering} dict built here is exactly
# the shape plot_ref_subsets_summary and plot_bootstrap_summary take as their clusterings
# argument.
#
# color_threshold is set to each tree's own cutoff rather than left at its default. With no
# highlight there is nothing else spending color, so giving it to the clusters the
# shuffling test found makes a plain tree say something.
#
# The two trees are drawn together because the choice between them is real -- correlation
# centers each reference and sees shape alone, cosine keeps the mean level and measures the
# angle from the origin. Whether the difference changes any conclusion is settled after the
# fits below, by the correlation-vs-cosine comparison.
pool_ref_names = [r.file_name for r in reference_spectra_list]
pool_clusterings = {
    metric: cluster_reference_spectra(
        A_all, pool_ref_names, np.random.default_rng(42), metric=metric,
    )
    for metric in ('correlation', 'cosine')
}

fig, axs = plt.subplots(1, len(pool_clusterings), figsize=(11 * len(pool_clusterings), 9))
for ax, (metric, clustering) in zip(np.atleast_1d(axs), pool_clusterings.items()):
    plot_reference_dendrogram(
        clustering, ax=ax, legend_loc='inside',
        color_threshold=clustering['cutoff_distance'], title=f'{metric} distance',
    )
plt.tight_layout()
plt.show()

# The flat clusters behind those colors, which is what the 'labels' array holds. The fit
# knows about neither: it treats all 24 references as equally distinct.
for metric, clustering in pool_clusterings.items():
    print(f'{metric} distance, cutoff {clustering["cutoff_distance"]:.4f} '
          f'-> {clustering["n_clusters"]} clusters')
    for label in range(1, clustering['n_clusters'] + 1):
        members = [name for name, member_label
                   in zip(clustering['ref_names'], clustering['labels']) if member_label == label]
        print(f'  cluster {label} ({len(members)} references)')
        for name in members:
            print(f'      {name}')
    print()

### Does a stricter randomized comparison give a more useful cut?

The cutoff above is permissive: nearly every merge falls below it, so it separates the pool
into two clusters and says little about the structure inside them. Sweeping `percentile`
does not fix that — from p50 to p99 the count only moves between three clusters and two,
because the observed merges sit far below every merge height the randomized copies produce,
rather than anywhere inside that range.

That points at how the randomized copies are built rather than at the percentile. `permute_within_rows` shuffles values
between references at each energy, which asks "is there *any* shared structure here?" — a
low bar for a set of spectra that are all the same element's absorption edge.
`phase_randomize_columns` was written to ask a harder question: each surrogate keeps its own
power spectrum, so it is still a smooth spectrum-like curve, but nothing it shares with the
other references survives.

The cell below measures both. It is kept in the notebook because the answer is a negative
one and worth being able to re-run on a different reference set.

In [ ]:
def compare_surrogate_methods(
    A, ref_names, surrogates, metrics=('correlation', 'cosine'),
    percentiles=(50, 90, 95, 99), resample_count=300, seed=42,
):
    """What each way of building the randomized copies implies about the cut."""
    def mean_abs_cross_correlation(matrix):
        correlations = np.corrcoef(matrix.T)
        return np.abs(correlations[~np.eye(correlations.shape[0], dtype=bool)]).mean()

    print('how much the references still resemble each other after randomizing:')
    print(f'  {"real references":24s} {mean_abs_cross_correlation(A):.3f}')
    for surrogate in surrogates:
        randomized = surrogate(A, np.random.default_rng(seed))
        print(f'  {surrogate.__name__:24s} {mean_abs_cross_correlation(randomized):.3f}'
              f'   (spread across references at one energy: {randomized.std(axis=1).mean():.4f},'
              f' real {A.std(axis=1).mean():.4f})')

    for metric in metrics:
        distances = np.clip(pdist(np.ascontiguousarray(A.T), metric=metric), 0.0, None)
        Z = hc.linkage(distances, method='complete')
        print(f'\n{metric} distance: observed merges '
              f'{Z[:, 2].min():.4f}-{Z[:, 2].max():.4f} (root {Z[-1, 2]:.4f})')
        for surrogate in surrogates:
            clustering = cluster_reference_spectra(
                A, ref_names, np.random.default_rng(seed), metric=metric,
                resample_count=resample_count, surrogate=surrogate, verbose=False,
            )
            heights = clustering['chance_merge_heights']
            print(f'  {surrogate.__name__}: randomized merges '
                  f'{heights.min():.4f}-{heights.max():.4f}, '
                  f'median {np.median(heights):.4f}')
            for percentile in percentiles:
                cutoff = float(np.percentile(heights, percentile))
                n_clusters = int(hc.fcluster(Z, t=cutoff, criterion='distance').max())
                print(f'      p{percentile:<3} cutoff={cutoff:.4f} -> {n_clusters:2d} clusters '
                      f'({int((Z[:, 2] < cutoff).sum())}/{Z.shape[0]} merges below)')


compare_surrogate_methods(A_all, pool_ref_names, (permute_within_rows, phase_randomize_columns))

#### Findings: the stricter comparison is the easier one to beat

Measured on the 24-reference pool, 300 randomized copies per method, seed 42.

| | mean \|cross-reference correlation\| | spread across references at one energy |
|---|---|---|
| real references | 0.809 | 0.146 |
| `permute_within_rows` | **0.797** | 0.146 |
| `phase_randomize_columns` | **0.289** | 0.557 |

**Shuffling values within an energy barely decorrelates the references at all** — 0.797
against the real 0.809 — and that is the whole explanation. At any one energy the 24
references differ by very little (a spread of 0.146 against normalized fluorescence of order
1), so permuting which reference holds which value leaves every surrogate still tracing the
same absorption edge. Shuffling keeps the shared XANES shape without being asked to, which
makes it a *demanding* comparison: its merge heights are small (median 0.162 under
correlation), and beating it means being tighter still.

**Phase randomization destroys that shared shape**, which is exactly what makes it the
weaker comparison. Its surrogates are nearly unrelated to each other (0.289), so their merge
heights are large — median 0.575 against the real tree's root of 0.571 — and the cutoff
lands above the entire observed tree:

| randomization | p50 | p90 | p95 | p99 |
|---|---|---|---|---|
| `permute_within_rows` (correlation) | 3 clusters | 2 | 2 | 2 |
| `phase_randomize_columns` (correlation) | **1 cluster** | 1 | 1 | 1 |
| `permute_within_rows` (cosine) | 3 clusters | 2 | 2 | 2 |
| `phase_randomize_columns` (cosine) | 2 clusters | **1** | 1 | 1 |

At p95 the phase-randomized cut calls the entire pool one cluster: every merge in the real
tree is tighter than chance under that comparison, which is true and useless.

**So the default stays `permute_within_rows`, and the coarse cut is not a defect to tune
away.** The two randomizations bracket the question rather than answering it: one preserves the
common edge and one destroys it, and neither leaves a percentile that carves the pool into
interpretable groups. The reason is in the first table — these references genuinely are all
variations on one absorption edge, and no cut of a tree built from them will say otherwise.
Finer structure has to be read from merge heights, which is what the subtree brackets
report, rather than from a flat partition.

`phase_randomize_columns` stays in the notebook so the comparison can be re-run on a
reference set that is less homogeneous, where the answer could differ.

In [ ]:
energies3, A3, b3, *_ = interpolate_references_at_sample_energies(reference_spectra, unknown_spectrum)

In [ ]:
def plot_spectrum_fit(energies, b, fitted, residuals, ax):
    ax.plot(energies, b, label='unknown', color='black')
    ax.plot(energies, fitted, label='fit', color='red', linestyle='--')
    ax.scatter(energies, residuals, color='orange', label='residuals', marker='o', s=10.0)
    ax.axhline(0, color='black', linewidth=0.5)
    ax.set_xlabel('Energy (eV)')
    ax.set_ylabel('Normalized Fluorescence')
    ax.legend()

nnls_coef, nnls_fitted, nnls_residuals = fit_nnls(A3, b3)

print('=== NNLS ===')
for ref_coef, ref_name in sorted(zip(nnls_coef, ref_names), reverse=True):
    print(f"  {ref_coef:.6f}: {ref_name}")
nnls_rmse = np.sqrt(np.mean(np.square(nnls_residuals)))
print(f'RMSE: {nnls_rmse:.6f}')

olstats_coef, olstats_fitted, olstats_residuals, olstats_result = fit_ols_with_statistics(A3, b3)

print('\n=== OLS with statistics ===')
for ref_coef, ref_name in sorted(zip(olstats_coef, ref_names), reverse=True):
    print(f"  {ref_coef:.6f}: {ref_name}")
olstats_rmse = np.sqrt(np.mean(np.square(olstats_residuals)))
print(f'RMSE: {olstats_rmse:.6f}')
print(olstats_result.summary())

ols_manual_coef, ols_manual_fitted, ols_manual_residuals = fit_ols(A3, b3)

print('\n=== OLS (manual) ===')
for ref_coef, ref_name in sorted(zip(ols_manual_coef, ref_names), reverse=True):
    print(f"  {ref_coef:.6f}: {ref_name}")
ols_manual_rmse = np.sqrt(np.mean(np.square(ols_manual_residuals)))
print(f'RMSE: {ols_manual_rmse:.6f}')

fig, (ax_nnls, ax_ols, ax_ols_manual) = plt.subplots(3, 1, figsize=(10, 12))
plot_spectrum_fit(energies3, b3, nnls_fitted, nnls_residuals, ax=ax_nnls)
ax_nnls.set_title(f'NNLS Fit: {unknown_spectrum.file_name}')
plot_spectrum_fit(energies3, b3, olstats_fitted, olstats_residuals, ax=ax_ols)
ax_ols.set_title(f'OLS Fit: {unknown_spectrum.file_name}')
plot_spectrum_fit(energies3, b3, ols_manual_fitted, ols_manual_residuals, ax=ax_ols_manual)
ax_ols_manual.set_title(f'OLS (manual) Fit: {unknown_spectrum.file_name}')
plt.tight_layout()
plt.show()


In [ ]:
def calculate_acf(residuals):
    n = len(residuals)
    n_lags = min(40, n // 2)
    x = residuals - residuals.mean()
    var = np.dot(x, x) / n
    acf_values = np.array(
        [1.0] + [np.dot(x[:-lag], x[lag:]) / (n * var) for lag in range(1, n_lags + 1)]
    )
    lags = np.arange(n_lags + 1)
    return lags, acf_values


In [ ]:
lags, acf_values = calculate_acf(nnls_residuals)

print(f'ACF at lag 0: {acf_values[0]:.4f}')
print(f'ACF at lag 1: {acf_values[1]:.4f}')
print(f'ACF at lag 2: {acf_values[2]:.4f}')
print(f'ACF at lag 5: {acf_values[5]:.4f}')


In [ ]:
def plot_residuals_histogram(residuals, ax, bins=15):
    mean = residuals.mean()
    std = residuals.std()
    ax.hist(residuals, bins=bins, color="orange", alpha=0.7, edgecolor="white")
    ax.axvline(mean, color="red", linestyle="--", label=f"mean={mean:.4f}")
    ax.axvline(mean + std, color="steelblue", linestyle=":", label=f"+1 std={mean + std:.4f}")
    ax.axvline(mean - std, color="steelblue", linestyle=":", label=f"-1 std={mean - std:.4f}")
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_xlabel("Residual")
    ax.set_ylabel("Count")
    ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))
    ax.legend()

fig, ax = plt.subplots(figsize=(10, 4))
plot_residuals_histogram(nnls_residuals, ax=ax)
ax.set_title('Residuals (NNLS)')
plt.tight_layout()
plt.show()

In [ ]:
def plot_acf(lags, acf_values, n, ax):
    ci_95 = 1.96 / np.sqrt(n)
    ax.bar(lags, acf_values, color='orange', alpha=0.7)
    ax.axhline(ci_95, color='red', linestyle='--', label='95% CI (white noise)')
    ax.axhline(-ci_95, color='red', linestyle='--')
    ax.axhline(0, color='black', linewidth=0.5)
    ax.set_xlabel('Lag')
    ax.set_ylabel('Autocorrelation')
    ax.legend()


fig, ax = plt.subplots(figsize=(10, 4))
plot_acf(lags, acf_values, len(nnls_residuals), ax=ax)
ax.set_title('Autocorrelation Function of Residuals')
plt.tight_layout()
plt.show()

In [ ]:
def moving_block_holdout_bootstrap(A, b, fitted, residuals, rng, n_bootstrap=1000):
    """ Early development version """
    n = len(residuals)
    block_length = max(1, int(np.round(n ** (1 / 3))))
    n_blocks_needed = int(np.ceil(n / block_length))
    n_full_blocks = n // block_length
    n_holdout_blocks = round(n_full_blocks / 3)
    # assume n=198 and block_length=6
    # then block_starts looks like
    #  array([  0,   1,   2, ..., 192 ])
    block_starts = np.arange(n - block_length + 1)

    bootstrap_coefs = np.zeros((n_bootstrap, A.shape[1]))
    bootstrap_pes = np.zeros(n_bootstrap)
    holdout_masks = np.zeros((n_bootstrap, n), dtype=bool)

    for i in range(n_bootstrap):
        # Randomly select non-contiguous holdout blocks totaling ~1/3 of the data
        # holdout_block_indices look like
        #  array([ 6, 20,  8,  3, 16, 28, 24,  2, 32, 22, 18])
        holdout_block_indices = rng.choice(n_full_blocks, size=n_holdout_blocks, replace=False)
        holdout_mask = np.zeros(n, dtype=bool)
        for idx in holdout_block_indices:
            holdout_mask[idx * block_length:(idx + 1) * block_length] = True
        holdout_masks[i] = holdout_mask
        train_mask = ~holdout_mask

        # Restrict bootstrap block starts to positions that don't overlap any holdout block
        valid_block_starts = block_starts[
            np.array([not holdout_mask[s:s + block_length].any() for s in block_starts])
        ]

        # Build bootstrap sample from moving blocks of residuals (holdout positions excluded)
        sampled_starts = rng.choice(valid_block_starts, size=n_blocks_needed, replace=True)
        bootstrap_residuals = np.concatenate(
            [residuals[s:s + block_length] for s in sampled_starts]
        )[:n]
        bootstrap_b = fitted + bootstrap_residuals

        # Fit on bootstrap data with holdout blocks removed
        bootstrap_coef, _ = scipy.optimize.nnls(A[train_mask], bootstrap_b[train_mask])
        bootstrap_coefs[i] = bootstrap_coef

        # Prediction error on real data at the held-out positions
        holdout_residuals = A[holdout_mask] @ bootstrap_coef - b[holdout_mask]
        bootstrap_pes[i] = np.sqrt(np.mean(np.square(holdout_residuals)))

    return bootstrap_coefs, bootstrap_pes, block_length, n_holdout_blocks, holdout_masks


rng = np.random.default_rng(seed=42)
n_bootstrap = 1000

bootstrap_coefs, bootstrap_pes, block_length, n_holdout_blocks, holdout_masks = \
    moving_block_holdout_bootstrap(A3, b3, nnls_fitted, nnls_residuals, rng, n_bootstrap=n_bootstrap)

n = len(nnls_residuals)
print(f'n={n}, block_length={block_length}, n_holdout_blocks={n_holdout_blocks} (~{n_holdout_blocks * block_length / n:.0%} of data)')
print(f'Coefficient means: {bootstrap_coefs.mean(axis=0)}')
print(f'Coefficient stds:  {bootstrap_coefs.std(axis=0)}')
print(f'Prediction error mean={bootstrap_pes.mean():.6f}, std={bootstrap_pes.std():.6f}')

In [ ]:
def select_holdout_blocks(n, rng, n_bootstrap=1000, block_length=None):
    """Pre-compute holdout masks and bootstrap block samples for n_bootstrap iterations.

    Captures all randomness so the same draws can be reused across multiple
    reference subsets via do_moving_block_holdout_bootstrap.

    Returns
    -------
    holdout_masks   : (n_bootstrap, n) bool array
    sampled_starts  : (n_bootstrap, n_blocks_needed) int array — block start indices for resampling
    block_length    : int
    n_holdout_blocks : int
    """
    # block_length=None keeps the original rule of thumb, which is only the *rate* at
    # which the optimal block length grows with n. Pass an explicit value -- see
    # choose_block_length -- to size the blocks from the residuals instead.
    if block_length is None:
        block_length = max(1, int(np.round(n ** (1 / 3))))
    n_blocks_needed = int(np.ceil(n / block_length))
    n_full_blocks = n // block_length
    n_holdout_blocks = round(n_full_blocks / 3)
    block_starts = np.arange(n - block_length + 1)

    print(f'n={n}, block_length={block_length}, n_full_blocks={n_full_blocks}')
    print(f'n_holdout_blocks={n_holdout_blocks} (~{n_holdout_blocks * block_length / n:.0%} of data), '
          f'n_blocks_needed={n_blocks_needed}')

    holdout_masks = np.zeros((n_bootstrap, n), dtype=bool)
    sampled_starts = np.zeros((n_bootstrap, n_blocks_needed), dtype=int)

    for i in range(n_bootstrap):
        holdout_block_indices = rng.choice(n_full_blocks, size=n_holdout_blocks, replace=False)
        holdout_mask = np.zeros(n, dtype=bool)
        for idx in holdout_block_indices:
            holdout_mask[idx * block_length:(idx + 1) * block_length] = True
        holdout_masks[i] = holdout_mask

        valid_block_starts = block_starts[
            np.array([not holdout_mask[s:s + block_length].any() for s in block_starts])
        ]
        sampled_starts[i] = rng.choice(valid_block_starts, size=n_blocks_needed, replace=True)

    return holdout_masks, sampled_starts, block_length, n_holdout_blocks


def do_moving_block_holdout_bootstrap(A, b, fitted, residuals, holdout_masks, sampled_starts, block_length):
    """Compute holdout prediction error for each pre-drawn bootstrap iteration.

    Parameters
    ----------
    A               : (n, n_refs) design matrix for one reference subset
    b               : (n,) observed spectrum values
    fitted          : (n,) full-data fitted values
    residuals       : (n,) full-data residuals (fitted - b)
    holdout_masks   : (n_bootstrap, n) bool array from select_holdout_blocks
    sampled_starts  : (n_bootstrap, n_blocks_needed) int array from select_holdout_blocks
    block_length    : int from select_holdout_blocks

    Returns
    -------
    bootstrap_coefs : (n_bootstrap, n_refs) float array
    bootstrap_pes   : (n_bootstrap,) float array — per-iteration holdout RMSE
    """
    n = len(b)
    n_bootstrap = len(holdout_masks)
    bootstrap_coefs = np.zeros((n_bootstrap, A.shape[1]))
    # Instead of computing each iteration's holdout RMSE inline with
    # np.sqrt(np.mean(np.square(...))) (a np.square/np.mean/np.sqrt trio per iteration,
    # ~2.32M times over the notebook's workload), accumulate each iteration's holdout
    # sum-of-squares and held-out point count as scalars and take the RMSE for all
    # iterations at once after the loop. This moves the mean/sqrt reduction out of the
    # hot loop (~10-15% faster here) and, because it stores scalars rather than a
    # fixed-width residual array, stays correct when the number of held-out points
    # varies between iterations (e.g. select_holdout_blocks_v4/v5).
    holdout_sum_of_squares = np.zeros(n_bootstrap)
    holdout_point_counts = np.zeros(n_bootstrap)

    # Vectorized replacement for the per-iteration block gather this loop used to do. The
    # original code, inside `for i in range(n_bootstrap):`, called ~2.32M times total across
    # 2324 reference combinations x 1000 bootstrap iterations, was:
    #
    #     bootstrap_residuals = np.concatenate(
    #         [residuals[s:s + block_length] for s in sampled_starts[i]]
    #     )[:n]
    #     bootstrap_b = fitted + bootstrap_residuals
    #
    # sampled_starts (shape (n_bootstrap, n_blocks_needed)) is fully known before the loop
    # starts, so instead of re-gathering blocks one bootstrap iteration at a time, gather all
    # of them at once with a single fancy-indexing operation:
    #
    #   1. block_offsets = [0, 1, ..., block_length - 1].
    #   2. Broadcasting sampled_starts[:, :, None] (n_bootstrap, n_blocks_needed, 1) against
    #      block_offsets[None, None, :] (1, 1, block_length) gives block_indices of shape
    #      (n_bootstrap, n_blocks_needed, block_length), where block_indices[i, j] is the
    #      block_length run of consecutive residual indices for bootstrap i's j-th sampled
    #      block (i.e. sampled_starts[i, j] + block_offsets).
    #   3. residuals[block_indices] gathers those residual values, same shape.
    #   4. .reshape(n_bootstrap, -1) flattens the (n_blocks_needed, block_length) axes into a
    #      single axis per bootstrap row, in the same block-by-block, then-within-block order
    #      that np.concatenate produced per iteration in the original code.
    #   5. [:, :n] truncates each row to n, matching the original per-iteration `[:n]` (the
    #      concatenated blocks run past n since n_blocks_needed * block_length >= n).
    #
    # all_bootstrap_residuals ends up with shape (n_bootstrap, n): row i is exactly what the
    # old code computed as bootstrap_residuals for iteration i. Moving the gather out of the
    # hot loop cut ~27% off this function's time during profiling.
    block_offsets = np.arange(block_length)
    block_indices = sampled_starts[:, :, None] + block_offsets[None, None, :]
    all_bootstrap_residuals = residuals[block_indices].reshape(n_bootstrap, -1)[:, :n]

    for i in range(n_bootstrap):
        holdout_mask = holdout_masks[i]
        train_mask = ~holdout_mask

        bootstrap_b = fitted + all_bootstrap_residuals[i]

        bootstrap_coef, _ = scipy.optimize.nnls(A[train_mask], bootstrap_b[train_mask])
        bootstrap_coefs[i] = bootstrap_coef

        holdout_residuals = A[holdout_mask] @ bootstrap_coef - b[holdout_mask]
        holdout_sum_of_squares[i] = holdout_residuals @ holdout_residuals
        holdout_point_counts[i] = holdout_residuals.shape[0]

    # per-iteration holdout RMSE, reduced for all iterations at once
    bootstrap_pes = np.sqrt(holdout_sum_of_squares / holdout_point_counts)

    return bootstrap_coefs, bootstrap_pes


In [ ]:
def select_holdout_blocks_v2(n, rng, n_bootstrap=1000, block_length=None):
    """Like select_holdout_blocks, but shifts the holdout block grid by a random
    offset in [0, block_length) each iteration so every position has roughly
    equal probability of being held out rather than only positions aligned to
    multiples of block_length.

    Returns
    -------
    holdout_masks    : (n_bootstrap, n) bool array
    sampled_starts   : (n_bootstrap, n_blocks_needed) int array
    block_length     : int
    n_holdout_blocks : int — representative value; actual count may vary ±1 per
                       iteration depending on how many shifted blocks fit in [0, n)
    """
    # block_length=None keeps the original rule of thumb, which is only the *rate* at
    # which the optimal block length grows with n. Pass an explicit value -- see
    # choose_block_length -- to size the blocks from the residuals instead.
    if block_length is None:
        block_length = max(1, int(np.round(n ** (1 / 3))))
    n_blocks_needed = int(np.ceil(n / block_length))
    n_full_blocks = n // block_length
    n_holdout_blocks = round(n_full_blocks / 3)
    block_starts = np.arange(n - block_length + 1)

    print(f'n={n}, block_length={block_length}, n_full_blocks={n_full_blocks}')
    print(f'n_holdout_blocks={n_holdout_blocks} (~{n_holdout_blocks * block_length / n:.0%} of data), '
          f'n_blocks_needed={n_blocks_needed}')

    holdout_masks = np.zeros((n_bootstrap, n), dtype=bool)
    sampled_starts = np.zeros((n_bootstrap, n_blocks_needed), dtype=int)

    for i in range(n_bootstrap):
        offset = int(rng.integers(0, block_length))
        shifted_grid = np.arange(offset, n - block_length + 1, block_length)
        n_shifted = len(shifted_grid)
        n_shifted_holdout = round(n_shifted / 3)

        holdout_block_indices = rng.choice(n_shifted, size=n_shifted_holdout, replace=False)
        holdout_mask = np.zeros(n, dtype=bool)
        for idx in holdout_block_indices:
            start = shifted_grid[idx]
            holdout_mask[start:start + block_length] = True
        holdout_masks[i] = holdout_mask

        valid_block_starts = block_starts[
            np.array([not holdout_mask[s:s + block_length].any() for s in block_starts])
        ]
        sampled_starts[i] = rng.choice(valid_block_starts, size=n_blocks_needed, replace=True)

    return holdout_masks, sampled_starts, block_length, n_holdout_blocks


In [ ]:
def select_holdout_blocks_v3(n, rng, n_bootstrap=1000, block_length=None):
    """Like select_holdout_blocks_v2 (random per-iteration shift) but uses a circular
    (modular) block grid with exactly n_full_blocks blocks, so every position is in
    exactly one block regardless of offset. This eliminates the v2 boundary effect.

    For offsets > 0, one block wraps around the end of the array, mixing high-index
    and low-index (high-energy and low-energy) positions. That is the cost of
    achieving uniform coverage.

    Returns
    -------
    holdout_masks    : (n_bootstrap, n) bool array
    sampled_starts   : (n_bootstrap, n_blocks_needed) int array
    block_length     : int
    n_holdout_blocks : int
    """
    # block_length=None keeps the original rule of thumb, which is only the *rate* at
    # which the optimal block length grows with n. Pass an explicit value -- see
    # choose_block_length -- to size the blocks from the residuals instead.
    if block_length is None:
        block_length = max(1, int(np.round(n ** (1 / 3))))
    n_blocks_needed = int(np.ceil(n / block_length))
    n_full_blocks = n // block_length
    n_holdout_blocks = round(n_full_blocks / 3)
    resample_block_starts = np.arange(n - block_length + 1)

    print(f'n={n}, block_length={block_length}, n_full_blocks={n_full_blocks}')
    print(f'n_holdout_blocks={n_holdout_blocks} (~{n_holdout_blocks * block_length / n:.0%} of data), '
          f'n_blocks_needed={n_blocks_needed}')

    holdout_masks = np.zeros((n_bootstrap, n), dtype=bool)
    sampled_starts = np.zeros((n_bootstrap, n_blocks_needed), dtype=int)

    for i in range(n_bootstrap):
        offset = int(rng.integers(0, block_length))
        # n_full_blocks evenly-spaced blocks, wrapping circularly at n
        circular_starts = (np.arange(n_full_blocks) * block_length + offset) % n

        holdout_block_indices = rng.choice(n_full_blocks, size=n_holdout_blocks, replace=False)
        holdout_mask = np.zeros(n, dtype=bool)
        for idx in holdout_block_indices:
            start = int(circular_starts[idx])
            for j in range(block_length):
                holdout_mask[(start + j) % n] = True
        holdout_masks[i] = holdout_mask

        valid_block_starts = resample_block_starts[
            np.array([not holdout_mask[s:s + block_length].any() for s in resample_block_starts])
        ]
        sampled_starts[i] = rng.choice(valid_block_starts, size=n_blocks_needed, replace=True)

    return holdout_masks, sampled_starts, block_length, n_holdout_blocks


In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))

ax.plot(energies3, b3, label='unknown spectrum', color='black', linewidth=1.0)
ax.scatter(
    energies3[~holdout_masks[-1]], b3[~holdout_masks[-1]],
    color='steelblue', s=10, zorder=4, alpha=0.6,
    label=f'training ({(~holdout_masks[-1]).sum()} points)',
)
ax.scatter(
    energies3[holdout_masks[-1]], b3[holdout_masks[-1]],
    color='orange', s=20, zorder=5,
    label=f'holdout ({holdout_masks[-1].sum()} points)',
)

ax.set_xlabel('Energy (eV)')
ax.set_ylabel('Normalized Fluorescence')
ax.set_title(
    f'Last Holdout Mask — {unknown_spectrum.file_name}\n'
    f'block_length={block_length}, n_holdout_blocks={n_holdout_blocks}'
)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
def plot_bootstrap_summary(
    energies, b, fitted, residuals,
    lags, acf_values,
    bootstrap_coefs, bootstrap_pes,
    coef, ref_names,
    spectrum_name, n_bootstrap,
    title_prefix=None, clusterings=None,
):
    """Summarize one fitted combination: the fit, its residuals, and its coefficients.

    One figure per row rather than one shared gridspec. A single grid forces every row onto
    the same column edges, which does two bad things here: a two-panel row splits at
    n_cols // 2, so it is lopsided whenever n_cols is odd (two references gives a 1:2
    split), and the per-reference panels end up narrow with wide gutters because their
    width is set by a grid the wide rows also have to live in. Giving each row its own
    figure lets its panels divide that row evenly, and the rows still read as one unit
    because they share a width.

    Each bootstrap distribution is summarized by its median rather than its mean. A
    prediction error distribution is bounded below by zero and has a long right tail, and a
    coefficient distribution piles up against the non-negativity constraint at zero, so in
    both cases the mean sits above the bulk of the draws and reports a typical value the
    distribution rarely takes. The median is also what the combination search ranks on, so
    the marker on these figures is now the same statistic that chose the combination.

    Parameters
    ----------
    energies : ndarray, shape (n_energies,)
        The energy grid the fit was computed on.
    b : ndarray, shape (n_energies,)
        The measured spectrum, normalized.
    fitted : ndarray, shape (n_energies,)
        The fitted spectrum.
    residuals : ndarray, shape (n_energies,)
        `fitted - b`, the convention fit_nnls uses.
    lags, acf_values : ndarray, shape (n_lags,)
        Autocorrelation of `residuals` and the lags it was evaluated at.
    bootstrap_coefs : ndarray, shape (n_bootstrap, n_refs)
        One row per bootstrap iteration, one column per reference, in `ref_names` order.
    bootstrap_pes : ndarray, shape (n_bootstrap,)
        The holdout prediction error (RMSE) of each bootstrap iteration.
    coef : ndarray, shape (n_refs,)
        The coefficients of the fit to the real spectrum, drawn as `observed` against the
        bootstrap distributions.
    ref_names : sequence of str
        The references in this combination, ordered to match the columns of
        `bootstrap_coefs` and the entries of `coef`.
    spectrum_name : str
        The sample, used as the title of the fit panel.
    n_bootstrap : int
        Iterations behind the distributions, reported in the first figure's suptitle.
    title_prefix : str, optional
        Prepended to every suptitle, e.g. '1st best 3-component fit'. None leaves the
        suptitles bare, which is what the single-fit demo wants.
    clusterings : dict, optional
        {metric: cluster_reference_spectra(...)} over the whole reference pool. Each tree
        becomes a panel in one extra figure, with this combination's references
        highlighted, so the output answers "how good is this fit" and "how distinctive are
        the references it chose" together. None drops that figure.

    Returns
    -------
    tuple of matplotlib.figure.Figure
        The figures, in the order they are meant to be read: the fit, the reference trees
        (only when `clusterings` is given, so the tuple is four figures or five), the
        residual diagnostics, the coefficient histograms, and the coefficient violins.

        Nothing is drawn here -- the caller displays them, typically
        `for fig in plot_bootstrap_summary(...): display(fig)`. Each is closed with
        `plt.close` before being returned, because the inline backend draws every figure
        still open at the end of a cell: left open, each figure would appear twice, once
        from that flush and once from the caller. Closing does not discard anything, and a
        closed figure still renders when displayed, saved, or further edited.
    """
    n = len(residuals)
    n_refs = len(ref_names)
    n_cols = n_refs + 1
    # the mean here is the definition of RMSE, not a summary of a bootstrap distribution,
    # so it stays a mean
    rmse = np.sqrt(np.mean(residuals ** 2))

    # Every row shares this width so the stack lines up. Two dozen leaf labels need about
    # 11 inches per tree, which sets the floor whenever the trees are drawn -- n_cols is 2
    # for a one-reference fit, which would otherwise leave each tree four inches wide.
    fig_width = max(4 * n_cols, 22) if clusterings else 4 * n_cols

    def suptitle_for(what):
        return f'{title_prefix} — {what}' if title_prefix is not None else what

    # ---- the fit itself
    fig_fit, ax_fit = plt.subplots(figsize=(fig_width, 4.5))
    plot_spectrum_fit(energies, b, fitted, residuals, ax=ax_fit)
    ax_fit.set_title(spectrum_name)
    fig_fit.suptitle(
        suptitle_for(f'Moving Block Holdout Bootstrap Distributions ({n_bootstrap} iterations)'),
        fontsize=13,
    )
    # tight_layout doesn't account for suptitle; the rect reserves space so it doesn't overlap
    fig_fit.tight_layout(rect=[0, 0, 1, 0.93])

    # ---- where this combination sits in the reference set
    fig_trees = None
    if clusterings:
        fig_trees, axs = plt.subplots(1, len(clusterings), figsize=(fig_width, 9))
        for ax, (metric, clustering) in zip(np.atleast_1d(axs), clusterings.items()):
            plot_reference_dendrogram(
                clustering, highlight={'this fit': list(ref_names)}, ax=ax,
                legend_loc='inside', title=f'{metric} distance',
            )
        fig_trees.suptitle(suptitle_for('Reference Trees'), fontsize=13)
        fig_trees.tight_layout(rect=[0, 0, 1, 0.96])

    # ---- residual diagnostics
    fig_resid, (ax_acf, ax_resid_hist) = plt.subplots(1, 2, figsize=(fig_width, 4.5))
    plot_acf(lags, acf_values, n, ax=ax_acf)
    ax_acf.set_title('Residual Autocorrelation Function')
    ax_acf.legend(fontsize=12)

    plot_residuals_histogram(residuals, ax=ax_resid_hist)
    ax_resid_hist.set_title('Residuals Histogram')
    ax_resid_hist.legend(fontsize=12)
    fig_resid.suptitle(suptitle_for('Residual Diagnostics'), fontsize=13)
    fig_resid.tight_layout(rect=[0, 0, 1, 0.93])

    # ---- bootstrap coefficients, one column per reference plus the prediction error.
    # Two figures with the same width and the same number of columns, so a reference's
    # histogram still sits directly above its violin even though they are separate figures.
    fig_hist, hist_axs = plt.subplots(1, n_cols, figsize=(fig_width, 4.5), squeeze=False)
    fig_violin, violin_axs = plt.subplots(1, n_cols, figsize=(fig_width, 4.5), squeeze=False)
    hist_axs, violin_axs = hist_axs[0], violin_axs[0]

    # columns run from the largest median coefficient to the smallest, so the references
    # carrying the fit are read first
    for j, (coef_median, coef_i, name) \
            in enumerate(sorted(zip(np.median(bootstrap_coefs, axis=0), range(n_refs), ref_names), reverse=True)):

        p2_5 = np.percentile(bootstrap_coefs[:, coef_i], 2.5)
        p97_5 = np.percentile(bootstrap_coefs[:, coef_i], 97.5)

        hist_axs[j].hist(bootstrap_coefs[:, coef_i], bins=40, color='steelblue', alpha=0.7, edgecolor='white')
        hist_axs[j].axvline(coef[coef_i], color='red', linestyle='--', label=f'observed={coef[coef_i]:.3f}')
        hist_axs[j].axvline(coef_median, color='blue', linestyle='--', label=f'median={coef_median:.3f}')
        hist_axs[j].axvline(p2_5, color='green', linestyle=':', label=f'2.5%={p2_5:.3f}')
        hist_axs[j].axvline(p97_5, color='green', linestyle=':', label=f'97.5%={p97_5:.3f}')
        hist_axs[j].set_title(name, fontsize=9)
        hist_axs[j].set_xlabel('Coefficient')
        hist_axs[j].legend(fontsize=8)

        violin_axs[j].violinplot(bootstrap_coefs[:, coef_i])
        violin_axs[j].scatter(
            [0.95], [coef[coef_i]], color='red', zorder=5, marker='o', s=60,
            edgecolors='black', linewidths=0.8, label=f'observed={coef[coef_i]:.3f}',
        )
        violin_axs[j].scatter(
            [1.05], [coef_median], color='blue', zorder=5, marker='D', s=60,
            edgecolors='black', linewidths=0.8, label=f'median={coef_median:.3f}',
        )
        violin_axs[j].set_title(name, fontsize=9)
        violin_axs[j].legend(fontsize=8)

    pes_median = np.median(bootstrap_pes)
    pes_p2_5 = np.percentile(bootstrap_pes, 2.5)
    pes_p97_5 = np.percentile(bootstrap_pes, 97.5)

    hist_axs[n_refs].hist(bootstrap_pes, bins=40, color='darkorange', alpha=0.7, edgecolor='white')
    hist_axs[n_refs].axvline(rmse, color='red', linestyle='--', label=f'RMSE={rmse:.4f}')
    hist_axs[n_refs].axvline(pes_median, color='blue', linestyle='--', label=f'median={pes_median:.4f}')
    hist_axs[n_refs].axvline(pes_p2_5, color='green', linestyle=':', label=f'2.5%={pes_p2_5:.4f}')
    hist_axs[n_refs].axvline(pes_p97_5, color='green', linestyle=':', label=f'97.5%={pes_p97_5:.4f}')
    hist_axs[n_refs].set_title('Holdout Prediction Error')
    hist_axs[n_refs].set_xlabel('Prediction Error (RMSE)')
    hist_axs[n_refs].legend(fontsize=8)

    parts = violin_axs[n_refs].violinplot(bootstrap_pes)
    for pc in parts['bodies']:
        pc.set_facecolor('darkorange')
        pc.set_alpha(0.7)
    violin_axs[n_refs].scatter(
        [0.95], [rmse], color='red', zorder=5, marker='o', s=60,
        edgecolors='black', linewidths=0.8, label=f'RMSE={rmse:.4f}',
    )
    violin_axs[n_refs].scatter(
        [1.05], [pes_median], color='blue', zorder=5, marker='D', s=60,
        edgecolors='black', linewidths=0.8, label=f'median={pes_median:.4f}',
    )
    violin_axs[n_refs].set_title('Holdout Prediction Error')
    violin_axs[n_refs].legend(fontsize=8)

    coef_violin_axes = violin_axs[:n_refs]
    all_ylims = [ax.get_ylim() for ax in coef_violin_axes]
    global_ymin = min(lo for lo, hi in all_ylims)
    global_ymax = max(hi for lo, hi in all_ylims)
    for ax in coef_violin_axes:
        ax.set_ylim(global_ymin, global_ymax)

    for fig, what in ((fig_hist, 'Bootstrap Coefficient Distributions'),
                      (fig_violin, 'Bootstrap Coefficient Distributions (violins)')):
        fig.suptitle(suptitle_for(what), fontsize=13)
        fig.tight_layout(rect=[0, 0, 1, 0.93])

    figures = tuple(fig for fig in (fig_fit, fig_trees, fig_resid, fig_hist, fig_violin)
                    if fig is not None)
    for fig in figures:
        plt.close(fig)
    return figures


In [ ]:
# plot_bootstrap_summary draws nothing itself -- it returns its figures, so the caller
# decides what to do with them. Here that is simply showing all of them, in order.
for fig in plot_bootstrap_summary(
    energies3, b3, nnls_fitted, nnls_residuals,
    lags, acf_values,
    bootstrap_coefs, bootstrap_pes,
    nnls_coef, ref_names,
    spectrum_name=unknown_spectrum.file_name,
    n_bootstrap=n_bootstrap,
):
    display(fig)

In [ ]:
energies, A, b, *_ = interpolate_references_at_sample_energies(reference_spectra, unknown_spectrum)

In [ ]:
def select_holdout_blocks_v4(n, rng, n_bootstrap=1000, block_length_min=6, block_length_max=10,
                             block_length=None):
    """Like select_holdout_blocks but partitions the data into blocks of random
    lengths drawn uniformly from [block_length_min, block_length_max] each iteration,
    so holdout boundaries fall at different positions every iteration.

    Bootstrap resampling uses fixed length block_length_min to stay compatible
    with do_moving_block_holdout_bootstrap.

    Returns
    -------
    holdout_masks    : (n_bootstrap, n) bool array
    sampled_starts   : (n_bootstrap, n_blocks_needed) int array
    block_length     : int — block_length_min, used for resampling
    n_holdout_blocks : int — mean holdout block count across iterations (rounded)
    """
    # block_length, when given, sets the shortest holdout block (and the resample block
    # length, which v4/v5 hold fixed at the minimum). The random range is slid rather
    # than collapsed so the version keeps the varying-length behavior it exists to test.
    if block_length is not None:
        block_length_min, block_length_max = (
            block_length, block_length + (block_length_max - block_length_min),
        )
    n_blocks_needed = int(np.ceil(n / block_length_min))
    resample_block_starts = np.arange(n - block_length_min + 1)

    holdout_masks = np.zeros((n_bootstrap, n), dtype=bool)
    sampled_starts = np.zeros((n_bootstrap, n_blocks_needed), dtype=int)
    total_holdout_blocks = 0

    for i in range(n_bootstrap):
        # Partition [0, n) into blocks of random lengths
        block_starts_i, block_lengths_i = [], []
        pos = 0
        while pos < n:
            bl = int(rng.integers(block_length_min, block_length_max + 1))
            bl = min(bl, n - pos)
            block_starts_i.append(pos)
            block_lengths_i.append(bl)
            pos += bl

        n_blocks_i = len(block_lengths_i)
        n_holdout_i = round(n_blocks_i / 3)
        total_holdout_blocks += n_holdout_i

        holdout_block_indices = rng.choice(n_blocks_i, size=n_holdout_i, replace=False)
        holdout_mask = np.zeros(n, dtype=bool)
        for idx in holdout_block_indices:
            s = block_starts_i[idx]
            holdout_mask[s:s + block_lengths_i[idx]] = True
        holdout_masks[i] = holdout_mask

        valid_resample_starts = resample_block_starts[
            np.array([not holdout_mask[s:s + block_length_min].any() for s in resample_block_starts])
        ]
        sampled_starts[i] = rng.choice(valid_resample_starts, size=n_blocks_needed, replace=True)

    n_holdout_blocks = round(total_holdout_blocks / n_bootstrap)
    mean_holdout_frac = total_holdout_blocks * (block_length_min + block_length_max) / 2 / (n_bootstrap * n)
    print(f'n={n}, block_length=[{block_length_min}, {block_length_max}], n_blocks_needed={n_blocks_needed}')
    print(f'avg n_holdout_blocks={n_holdout_blocks} (~{mean_holdout_frac:.0%} of data)')

    return holdout_masks, sampled_starts, block_length_min, n_holdout_blocks


In [ ]:
def select_holdout_blocks_v5(n, rng, n_bootstrap=1000, block_length_min=6, block_length_max=10,
                             block_length=None):
    """Like select_holdout_blocks_v4 but reverses every other holdout mask, so the
    truncated final block alternates between the high-energy and low-energy end of
    the spectrum rather than always falling at the high-energy end.
    """
    # block_length, when given, sets the shortest holdout block (and the resample block
    # length, which v4/v5 hold fixed at the minimum). The random range is slid rather
    # than collapsed so the version keeps the varying-length behavior it exists to test.
    if block_length is not None:
        block_length_min, block_length_max = (
            block_length, block_length + (block_length_max - block_length_min),
        )
    n_blocks_needed = int(np.ceil(n / block_length_min))
    resample_block_starts = np.arange(n - block_length_min + 1)

    holdout_masks = np.zeros((n_bootstrap, n), dtype=bool)
    sampled_starts = np.zeros((n_bootstrap, n_blocks_needed), dtype=int)
    total_holdout_blocks = 0

    for i in range(n_bootstrap):
        block_starts_i, block_lengths_i = [], []
        pos = 0
        while pos < n:
            bl = int(rng.integers(block_length_min, block_length_max + 1))
            bl = min(bl, n - pos)
            block_starts_i.append(pos)
            block_lengths_i.append(bl)
            pos += bl

        n_blocks_i = len(block_lengths_i)
        n_holdout_i = round(n_blocks_i / 3)
        total_holdout_blocks += n_holdout_i

        holdout_block_indices = rng.choice(n_blocks_i, size=n_holdout_i, replace=False)
        holdout_mask = np.zeros(n, dtype=bool)
        for idx in holdout_block_indices:
            s = block_starts_i[idx]
            holdout_mask[s:s + block_lengths_i[idx]] = True

        if i % 2 == 1:
            holdout_mask = holdout_mask[::-1]

        holdout_masks[i] = holdout_mask

        valid_resample_starts = resample_block_starts[
            np.array([not holdout_mask[s:s + block_length_min].any() for s in resample_block_starts])
        ]
        sampled_starts[i] = rng.choice(valid_resample_starts, size=n_blocks_needed, replace=True)

    n_holdout_blocks = round(total_holdout_blocks / n_bootstrap)
    mean_holdout_frac = total_holdout_blocks * (block_length_min + block_length_max) / 2 / (n_bootstrap * n)
    print(f'n={n}, block_length=[{block_length_min}, {block_length_max}], n_blocks_needed={n_blocks_needed}')
    print(f'avg n_holdout_blocks={n_holdout_blocks} (~{mean_holdout_frac:.0%} of data)')

    return holdout_masks, sampled_starts, block_length_min, n_holdout_blocks


### Visualizing holdout block structure

The write-up below argues about *where holdout block boundaries fall* — whether the
grid is aligned or shifted, whether a block wraps around the end of the array, whether
the truncated final block always lands at the high-energy end. Those are claims about
the masks themselves, so plot the masks: `plot_holdout_block_structure` shows the
geometry each version produces, and the per-version holdout frequency std it reports
is the same statistic the results table below is built from.

Each `select_holdout_blocks` version makes two draws, and the figure covers both. Rows
1, 4 and 5 describe the **holdout** draw — which positions are set aside for scoring.
Rows 2 and 6 describe the **resample** draw: `sampled_starts`, the moving blocks that
get pasted together to build each bootstrap iteration's residual series. Row 3 puts the
two draws on one axis, each scaled by its own mean so the two can share one: 1.0 there
means "drawn exactly as often as a perfectly even draw would". Its right-hand axis
restates the holdout curve as an absolute fraction of the spectrum. The holdout curve is
the raw per-position frequency, one point per energy, whose std across positions is the
uniformity statistic the results table is built from; only the resample curve is
smoothed, since without it the block-start counts are pure Poisson noise.

The two draws are coupled, and row 2 is where that coupling is visible. Holding out a
block also withdraws every resample block overlapping it, so each iteration's holdout
choice shrinks the pool it can resample from. Row 2 shows, per iteration, which block
starts were unavailable for that reason, which were available but not drawn, and which
were drawn — once, or more than once, since blocks are drawn with replacement. It sits
directly beneath row 1, on the same two windows and the same aligned block grid, so the
masks and the availability they cause can be read against each other column for column.
Its high-energy panel ends in a blank strip one block short of the spectrum's end, where
no block start can begin; leaving that gap is what keeps the two rows aligned.

Passing `residuals=` adds two more rows showing what those blocks actually produce: the
reconstructed residual series against the original, and — the substantive check — the
autocorrelation of the reconstructed series against the original. Resampling in blocks
rather than point by point exists precisely to carry the residuals' short-range
autocorrelation into the bootstrap, and that panel is where you can see whether it does.

The structural rows need only the selector output — no NNLS and no
`do_moving_block_holdout_bootstrap` — so the figure runs in seconds either way, unlike
the prediction error comparison further down. The `n=...`, `block_length=...` lines it
prints come from the selectors themselves, which the figure calls at an explicit
`block_length=10` rather than at each version's rule-of-thumb default.

In [ ]:
def contiguous_run_lengths(holdout_masks):
    """Lengths of every contiguous run of held-out positions, pooled over iterations.

    Each row of holdout_masks is padded with False on both sides so runs touching an
    edge are still bounded by a transition; np.diff then marks run starts with +1 and
    run ends with -1. np.argwhere returns those in row-major order, so within a row
    the k-th start pairs with the k-th end and the difference of their column indices
    is the run length.

    Returns
    -------
    (n_runs,) int array -- one entry per run, pooled over all iterations
    """
    padded = np.zeros((holdout_masks.shape[0], holdout_masks.shape[1] + 2), dtype=np.int8)
    padded[:, 1:-1] = holdout_masks
    transitions = np.diff(padded, axis=1)
    run_starts = np.argwhere(transitions == 1)
    run_ends = np.argwhere(transitions == -1)
    return run_ends[:, 1] - run_starts[:, 1]


def smooth(values, window):
    """Centered moving average, with the window shrunk at the edges.

    Dividing a same-mode convolution of the values by the same convolution of an
    all-ones array normalizes each output point by however many input points actually
    contributed, so edge points are averaged over a partial window instead of being
    pulled toward zero by the implicit zero padding. That matters here because the
    edges are exactly where the versions differ.
    """
    kernel = np.ones(window)
    return (
        np.convolve(values, kernel, mode='same')
        / np.convolve(np.ones_like(values), kernel, mode='same')
    )

In [ ]:
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap, ListedColormap, to_rgb
from matplotlib.patches import Patch


def plot_holdout_mask_rasters(
    fig, subplot_spec, holdout_masks, block_length, label, color, n_raster, window,
):
    """Draw the holdout masks themselves, at both ends of the spectrum.

    This is the panel the v1-v5 write-up is really about. Every claim in it -- the grid
    is aligned, the grid is shifted, a block wraps around the end of the array, the
    truncated block always lands at the high-energy end -- is a claim about where the
    True cells of holdout_masks fall, so the mask array is drawn directly: x is position,
    y is bootstrap iteration, and a filled cell means that iteration held that position
    out. Gray vertical rules mark where an aligned grid of the given block length would
    put its boundaries; v1 sits exactly on them and the departure of the other versions
    from them is the thing to look for.

    Only the first n_raster iterations and only `window` positions at each end of the
    spectrum are shown. All n positions do not resolve into visible cells at this figure
    width, and the differences between versions are edge effects, so showing both ends
    at a legible scale beats squeezing the whole range into one strip.

    The two rasters are individual draws. Their column mean over all iterations is drawn
    full width by plot_coverage_relative_to_uniform one row further down: the row directly
    beneath this one holds plot_resample_start_rasters, which draws the same two windows
    ruled on this same grid, so the masks and the resample availability they cause can be
    read against each other column for column.

    Parameters
    ----------
    fig          : Figure -- the figure to add axes to
    subplot_spec : SubplotSpec -- cell to split into the low- and high-energy rasters
    holdout_masks: (n_bootstrap, n) bool array -- True where a position was held out
    block_length : int -- spacing of the gray aligned-grid rules
    label        : str -- version label, used in the panel's heading
    color        : matplotlib color -- version color; the raster colormap runs white ->
                   color so "held out" reads as ink on the page while hue still carries
                   version identity
    n_raster     : int -- number of iterations (rows) to show
    window       : int -- positions to show at each end of the spectrum

    Returns
    -------
    (ax_low, ax_high) -- the low-energy and high-energy rasters
    """
    n = holdout_masks.shape[1]
    cmap = LinearSegmentedColormap.from_list('holdout', ['white', color])

    inner = gridspec.GridSpecFromSubplotSpec(
        1, 2, subplot_spec=subplot_spec, wspace=0.10,
    )
    ax_low = fig.add_subplot(inner[0, 0])
    ax_high = fig.add_subplot(inner[0, 1], sharey=ax_low)

    for ax, positions in (
        (ax_low, np.arange(window)),
        (ax_high, np.arange(n - window, n)),
    ):
        ax.imshow(
            holdout_masks[:n_raster, positions],
            aspect='auto', interpolation='nearest', cmap=cmap, vmin=0, vmax=1,
            extent=[positions[0] - 0.5, positions[-1] + 0.5, n_raster, 0],
        )
        for boundary in range(0, n, block_length):
            if positions[0] <= boundary <= positions[-1]:
                ax.axvline(boundary - 0.5, color='0.45', linewidth=0.5, alpha=0.7)
        ax.set_xlabel('Position (index)', fontsize=8)
        ax.tick_params(labelsize=7)

    ax_low.set_ylabel('Bootstrap iteration', fontsize=8)
    ax_low.set_title('low-energy end', fontsize=8)
    ax_high.set_title('high-energy end', fontsize=8)
    ax_high.tick_params(labelleft=False)
    # Anchored to the left raster but centered over the pair, so the label does not
    # run into the neighbouring column.
    ax_low.annotate(
        f'{label}\nheld out, first {n_raster} iterations '
        f'(gray = aligned grid, spacing {block_length})',
        xy=(1.05, 1.22), xycoords='axes fraction', ha='center', va='bottom', fontsize=10,
    )
    return ax_low, ax_high


def plot_source_position_reuse(
    fig, subplot_spec, reuse_counts, mean_holdout_fraction, max_reuse, color,
):
    """How often a single source position is copied into one reconstructed series.

    Moving blocks are drawn with replacement, so within one bootstrap iteration a
    position of the original residual series can be copied into the reconstruction
    several times, or never. This panel is the distribution of that count, pooled over
    all (iteration, position) pairs: bar k is the fraction of pairs where the position
    was used exactly k times, and the last bar accumulates everything at or above it.

    The zero bar is the informative one. Every held-out position is unusable by
    construction, so the zero bar can never fall below the held-out fraction -- drawn
    here as a red rule. Whatever height it has above that rule is the non-held-out
    positions the draw simply missed, which is a property of drawing with replacement
    rather than of the holdout geometry.

    Parameters
    ----------
    fig                  : Figure -- the figure to add the axes to
    subplot_spec         : SubplotSpec -- cell of the outer GridSpec to draw in
    reuse_counts         : (n_bootstrap, n) int array -- times each position is used in
                           each iteration's reconstruction
    mean_holdout_fraction: float -- mean fraction of positions held out per iteration,
                           the lower bound on the zero bar
    max_reuse            : int -- largest reuse count over all versions; shared across
                           columns so the bars line up, capped at "5+" so a single
                           extreme count cannot stretch the axis
    color                : matplotlib color -- version color for the bars

    Returns
    -------
    Axes -- the axes drawn on
    """
    ax_reuse = fig.add_subplot(subplot_spec)
    bins = np.arange(0, min(max_reuse, 5) + 2)
    fractions_by_count = [
        (reuse_counts == k).mean() if k < bins[-1] else (reuse_counts >= k).mean()
        for k in bins[:-1]
    ]
    ax_reuse.bar(bins[:-1], fractions_by_count, color=color, alpha=0.8, width=0.7)
    ax_reuse.axhline(
        mean_holdout_fraction, color='red', linestyle='--', linewidth=1.0,
        label=f'held-out fraction = {mean_holdout_fraction:.3f}',
    )
    ax_reuse.set_xticks(bins[:-1])
    ax_reuse.set_xticklabels(
        [str(k) for k in bins[:-2]] + [f'{bins[-2]}+'], fontsize=8,
    )
    ax_reuse.set_ylim(0, 1)
    ax_reuse.set_xlabel('Times a source position is reused in one iteration')
    ax_reuse.set_ylabel('Fraction of\n(iteration, position)', fontsize=8)
    ax_reuse.set_title('Source position reuse', fontsize=10)
    ax_reuse.legend(fontsize=7)
    return ax_reuse


def plot_holdout_fraction_histogram(
    fig, subplot_spec, holdout_fraction_per_iteration, fraction_min, fraction_max, color,
):
    """How much of the spectrum each bootstrap iteration sets aside for scoring.

    v1-v3 hold out a fixed number of blocks of fixed length, so this collapses to a
    single spike; v4 draws random block lengths and v5 alternates, so both spread out.
    That spread is not cosmetic: an iteration that holds out more points both fits on
    less data and scores on more of it, and the resulting variation in prediction error
    per iteration is the mechanism behind the wider confidence intervals those versions
    produce in the results table.

    The x-limits are passed in rather than autoscaled so every version is drawn on the
    same axis -- a version that is genuinely a spike must look like a spike next to one
    that is genuinely broad.

    Parameters
    ----------
    fig                           : Figure -- the figure to add the axes to
    subplot_spec                  : SubplotSpec -- cell of the outer GridSpec
    holdout_fraction_per_iteration: (n_bootstrap,) array -- held-out fraction per
                                    iteration
    fraction_min, fraction_max    : float -- shared histogram range and x-limits, taken
                                    over every version so the columns are comparable
    color                         : matplotlib color -- version color for the bars

    Returns
    -------
    Axes -- the axes drawn on
    """
    ax_fraction = fig.add_subplot(subplot_spec)
    fractions = holdout_fraction_per_iteration
    ax_fraction.hist(
        fractions, bins=30, range=(fraction_min, fraction_max), color=color, alpha=0.8,
    )
    ax_fraction.axvline(
        np.median(fractions), color='red', linestyle='--', linewidth=1.0,
        label=(
            f'median={np.median(fractions):.3f}\n'
            f'range=[{fractions.min():.3f}, {fractions.max():.3f}]'
        ),
    )
    ax_fraction.set_xlim(fraction_min - 0.01, fraction_max + 0.01)
    ax_fraction.set_xlabel('Fraction of points held out')
    ax_fraction.set_ylabel('Iterations')
    ax_fraction.set_title('Held-out fraction per iteration', fontsize=10)
    ax_fraction.legend(fontsize=7)
    return ax_fraction


def plot_held_out_run_lengths(
    fig, subplot_spec, run_lengths, block_length, run_length_limit, color,
):
    """Realized lengths of the contiguous held-out stretches, not the nominal ones.

    A selector is parameterized by a block length, but what the fit actually sees is
    runs of consecutive missing points, and those are not the same thing: two selected
    blocks that happen to land adjacent merge into one run of twice the length, while a
    block truncated at the end of the array or wrapped around it shows up short. The
    red rule marks the nominal block length so the merged and truncated tails are
    obvious as departures from it.

    A few very long runs -- several selected blocks in a row -- would otherwise squeeze
    every informative bar into the first tenth of the axis, so the shared x-limit is
    clipped to a high percentile over all versions and the true maximum, along with the
    count of runs past the axis, is reported in the legend instead.

    Parameters
    ----------
    fig             : Figure -- the figure to add the axes to
    subplot_spec    : SubplotSpec -- cell of the outer GridSpec
    run_lengths     : (n_runs,) int array -- one entry per contiguous held-out run,
                      pooled over iterations, as returned by contiguous_run_lengths
    block_length    : int -- nominal block length, drawn as the red reference rule
    run_length_limit: int -- shared upper x-limit, a high percentile over all versions
    color           : matplotlib color -- version color for the bars

    Returns
    -------
    Axes -- the axes drawn on
    """
    ax_runs = fig.add_subplot(subplot_spec)
    ax_runs.hist(
        run_lengths, bins=np.arange(0.5, run_length_limit + 1.5), color=color, alpha=0.8,
    )
    ax_runs.axvline(
        block_length, color='red', linestyle='--', linewidth=1.0,
        label=(
            f'resample block_length={block_length}\n'
            f'median run={int(np.median(run_lengths))}, longest={run_lengths.max()}\n'
            f'runs beyond axis: {(run_lengths > run_length_limit).sum()}'
        ),
    )
    ax_runs.set_xlim(0, run_length_limit + 1)
    ax_runs.set_xlabel('Contiguous held-out run length (points)')
    ax_runs.set_ylabel('Runs')
    ax_runs.set_title('Realized held-out run lengths', fontsize=10)
    ax_runs.legend(fontsize=7)
    return ax_runs


def plot_coverage_relative_to_uniform(
    fig, subplot_spec, energies, holdout_frequency, relative_resample,
    block_length, ylim, color,
):
    """Both draws on one axis, each expressed as coverage relative to a uniform draw.

    A selector makes two draws per iteration: which positions to hold out, and which
    moving blocks to resample the residuals from. They are coupled -- holding out a
    block also withdraws every resample block overlapping it -- but they are counted in
    different units, holdout frequency per position against block starts drawn per start
    position. Dividing each by its own mean puts them on one scale where 1.0 means
    "drawn exactly as often as a perfectly even draw would", so a version that is even
    is a flat line at 1.0 in both curves and a version with edge effects visibly sags or
    spikes there.

    The holdout curve is drawn unsmoothed, one point per position. It is a mean of
    n_bootstrap Bernoulli draws per position, so its wobble is mostly sampling noise --
    but that noise is the scale any apparent structure has to be judged against, and
    smoothing it away invites reading a version as flat when it is merely quiet. The
    resample curve is smoothed over one block length, because unsmoothed its counts are
    dominated by Poisson noise from the tens of thousands of draws and the systematic
    structure is unreadable. Only one of the two is smoothed, so the title says which.

    The right-hand axis restates the left in absolute terms, so the panel answers "how
    evenly" and "how much" at once: coverage 1.0 is the mean held-out fraction, and a
    sag to 0.8 can be read off directly as the fraction it corresponds to. It calibrates
    the holdout curves only -- the resample curve is normalized by its own mean, which is
    a count of block starts and not a fraction of anything.

    The legend carries the mean and std of the unsmoothed holdout frequency. That std is
    precisely the uniformity statistic the v1-v5 results table is built from, so the
    number and the shape of the curve are two views of the same quantity.

    The y-limits are shared across versions, because per-column autoscaling would inflate
    each version's sampling noise to fill its own axis and make a version that is
    genuinely flat look as structured as one whose edges really do collapse.

    Parameters
    ----------
    fig              : Figure -- the figure to add the axes to
    subplot_spec     : SubplotSpec -- cell of the outer GridSpec
    energies         : (n,) array -- energy axis; the resample curve is plotted against
                       its first n_start_positions entries, since a block start only
                       exists where a whole block fits
    holdout_frequency: (n,) array -- per-position holdout frequency, the column mean of
                       the holdout masks. Divided by its own mean to give the holdout
                       curve, and that mean sets the absolute scale on the right-hand axis
    relative_resample: (n_start_positions,) array -- smoothed resample start counts /
                       their mean
    block_length     : int -- the resample curve's smoothing window, named in the title
    ylim             : (lo, hi) -- shared y-limits over every version, padded by caller
    color            : matplotlib color -- version color for the holdout curve; the
                       resample curve is always dashed gray so the two never collide

    Returns
    -------
    Axes -- the axes drawn on
    """
    ax_coverage = fig.add_subplot(subplot_spec)
    n_start_positions = len(relative_resample)
    mean_frequency = holdout_frequency.mean()

    ax_coverage.axhline(1.0, color='gray', linestyle=':', linewidth=1.0)
    ax_coverage.plot(
        energies, holdout_frequency / mean_frequency,
        color=color, linewidth=0.9, label='held out (per position)',
    )
    ax_coverage.plot(
        energies[:n_start_positions], relative_resample,
        color='dimgray', linewidth=1.4, linestyle='--', label='resample block start',
    )
    ax_coverage.set_ylim(*ylim)
    ax_coverage.set_xlabel('Energy (eV)')
    ax_coverage.set_ylabel('Coverage relative\nto uniform', fontsize=8)

    # The same axis in the units the masks are actually in. secondary_yaxis takes the
    # forward and inverse transforms rather than fixed limits, so it tracks the shared
    # ylim without being told about it.
    ax_absolute = ax_coverage.secondary_yaxis(
        'right',
        functions=(lambda relative: relative * mean_frequency,
                   lambda fraction: fraction / mean_frequency),
    )
    ax_absolute.set_ylabel('Fraction held out', fontsize=8)
    ax_absolute.tick_params(labelsize=7)

    ax_coverage.set_title(
        f'Holdout vs. resample coverage '
        f'(resample smoothed over {block_length} points)', fontsize=10,
    )
    # the std is the uniformity statistic the v1-v5 table is built from
    ax_coverage.plot(
        [], [], ' ',
        label=f'mean={mean_frequency:.3f}, std={holdout_frequency.std():.4f}',
    )
    ax_coverage.legend(fontsize=7)
    return ax_coverage


# plot_resample_start_rasters spends part of its row on the legend strip beneath the
# rasters, so a row of height h yields rasters of only START_RASTER_HEIGHT_FRACTION * h.
# A GridSpec's hspace is a fraction of the mean row height, so a two-row grid is
# (1 + hspace / 2) times as tall as the axes it holds. The caller divides its mask-raster
# row height by this to give the two raster rows rasters of the same height.
_START_RASTER_ROW_RATIOS = (9, 1.6)   # rasters : legend strip
_START_RASTER_ROW_HSPACE = 0.25       # gap between them, as a fraction of mean row height
START_RASTER_HEIGHT_FRACTION = _START_RASTER_ROW_RATIOS[0] / (
    sum(_START_RASTER_ROW_RATIOS) * (1 + _START_RASTER_ROW_HSPACE / 2)
)


def plot_resample_start_rasters(
    fig, subplot_spec, start_categories, block_length, available_fraction, color,
    n_raster, window,
):
    """Which moving-block starts each iteration could draw from, and which it drew.

    This is where the coupling between the two draws is visible. A block start is
    available to the resampler only when the whole block beginning there contains no
    held-out point, so each iteration's holdout choice shrinks the pool it can resample
    from -- and because the versions differ in where they put holdout blocks, they
    differ in which starts they leave usable.

    Four categories per cell, in the order the colormap lists them: unavailable because
    the block overlaps a held-out point (neutral gray), available but not drawn this
    iteration (a pale tint of the version color), drawn once (the version color), and
    drawn twice or more (a darkened version color), since blocks are drawn with
    replacement. Each column gets its own legend strip underneath rather than sharing
    one, because the swatches are tinted in that column's version color and so are
    genuinely a different key rather than a repeat of the same one.

    Windowed at both ends like the holdout raster and for the same reason: a start is
    one or two pixels wide across the full range, and the ends are where availability
    differs between versions. The windows are the mask raster's exactly -- the same
    positions, the same x-limits -- and gray vertical rules mark the same aligned block
    grid, so with the two rasters drawn on adjacent rows a stretch of unavailable starts
    lines up column for column with the held-out block that caused it. The high-energy
    panel ends in blank space, block_length - 1 positions wide: no block start lives that
    close to the end of the spectrum, and leaving the gap is what keeps the two rows
    aligned. The legend strip means the rasters get only
    START_RASTER_HEIGHT_FRACTION of the row, so the caller is expected to size the row
    accordingly if these rasters are to match the holdout mask rasters in height.

    Parameters
    ----------
    fig               : Figure -- the figure to add axes to
    subplot_spec      : SubplotSpec -- cell of the outer GridSpec to subdivide
    start_categories  : (n_bootstrap, n_start_positions) int array -- 0 unavailable,
                        1 available but not drawn, 2 drawn once, 3 drawn 2+ times
    block_length      : int -- spacing of the gray aligned-grid rules, the same grid
                        plot_holdout_mask_rasters draws
    available_fraction: float -- fraction of (iteration, start) pairs that were
                        available, reported in the panel heading
    color             : matplotlib color -- version color the three non-gray categories
                        are derived from
    n_raster          : int -- number of iterations (rows) to show
    window            : int -- positions to show at each end -- the same window
                        plot_holdout_mask_rasters is given, clipped to half the spectrum.
                        Values below block_length would leave the high-energy panel with
                        no starts to draw

    Returns
    -------
    (ax_low, ax_high, ax_key) -- low-energy raster, high-energy raster, legend strip
    """
    n_start_positions = start_categories.shape[1]
    inner_starts = gridspec.GridSpecFromSubplotSpec(
        2, 2, subplot_spec=subplot_spec, height_ratios=list(_START_RASTER_ROW_RATIOS),
        hspace=_START_RASTER_ROW_HSPACE, wspace=0.10,
    )
    ax_starts_low = fig.add_subplot(inner_starts[0, 0])
    ax_starts_high = fig.add_subplot(inner_starts[0, 1], sharey=ax_starts_low)
    ax_starts_key = fig.add_subplot(inner_starts[1, :])
    ax_starts_key.axis('off')

    # unavailable / available / drawn once / drawn 2+, in that order
    base_rgb = np.array(to_rgb(color))
    category_cmap = ListedColormap([
        '0.88',                      # unavailable: overlaps a held-out point
        1 - 0.30 * (1 - base_rgb),   # available but not drawn this iteration
        base_rgb,                    # drawn once
        0.55 * base_rgb,             # drawn 2+ times (blocks are drawn with replacement)
    ])

    # Window on *positions*, not on start indices, so both panels span exactly the range
    # plot_holdout_mask_rasters shows and a given index sits at the same place in both
    # rows. A block start shares an index with the position it starts at, so the low ends
    # coincide outright; at the high end starts run out block_length - 1 positions before
    # the data does, and pinning the axis to the mask raster's window leaves that tail
    # blank rather than sliding the whole panel left to fill it.
    n = n_start_positions + block_length - 1
    start_window = min(window, n // 2)
    for ax, positions in (
        (ax_starts_low, np.arange(start_window)),
        (ax_starts_high, np.arange(n - start_window, n)),
    ):
        start_indices = positions[positions < n_start_positions]
        ax.imshow(
            start_categories[:n_raster, start_indices],
            aspect='auto', interpolation='nearest', cmap=category_cmap, vmin=-0.5, vmax=3.5,
            extent=[start_indices[0] - 0.5, start_indices[-1] + 0.5, n_raster, 0],
        )
        # The same rules plot_holdout_mask_rasters draws, on the same grid and now at the
        # same x positions, so a stretch of unavailable starts can be traced straight up
        # to the held-out block that caused it.
        for boundary in range(0, n, block_length):
            if positions[0] <= boundary <= positions[-1]:
                ax.axvline(boundary - 0.5, color='0.45', linewidth=0.5, alpha=0.7)
        ax.set_xlim(positions[0] - 0.5, positions[-1] + 0.5)
        ax.set_xlabel('Block start (index)', fontsize=8)
        ax.tick_params(labelsize=7)

    ax_starts_low.set_ylabel('Bootstrap iteration', fontsize=8)
    ax_starts_low.set_title('low-energy end', fontsize=8)
    ax_starts_high.set_title('high-energy end', fontsize=8)
    ax_starts_high.tick_params(labelleft=False)
    ax_starts_low.annotate(
        f'Resample blocks: available vs. drawn\n'
        f'{available_fraction:.0%} of starts available '
        f'(rules = aligned grid, spacing {block_length})',
        xy=(1.05, 1.14), xycoords='axes fraction', ha='center', va='bottom', fontsize=9,
    )
    ax_starts_key.legend(
        handles=[
            Patch(facecolor=category_cmap(k), edgecolor='0.6', linewidth=0.4, label=text)
            for k, text in enumerate((
                'unavailable (overlaps holdout)', 'available, not drawn',
                'drawn once', 'drawn 2+ times',
            ))
        ],
        fontsize=6.5, loc='center', ncol=2, frameon=False,
    )
    return ax_starts_low, ax_starts_high, ax_starts_key


def plot_reconstructed_residual_series(
    fig, subplot_spec, energies, residuals, source_positions, color,
    n_reconstructions=2,
):
    """The original residual series against the series the resampled blocks rebuild.

    Everything else in this figure is about where blocks are; this is what they produce.
    Each reconstruction is the original residuals gathered at one iteration's source
    positions, offset downward so the three traces can be compared without overlapping.
    A reconstruction should look like the original in character -- same amplitude, same
    roughness -- and differ in that it is stitched together from blocks: the seams
    between consecutive blocks are where the reconstruction can introduce
    discontinuities the original never had.

    No block boundaries are ruled in. At this width there would be n / block_length of
    them, 33 for the real data, which reads as a gray wash rather than as structure; the
    resample-start raster is the panel where the block layout is legible.

    Parameters
    ----------
    fig              : Figure -- the figure to add the axes to
    subplot_spec     : SubplotSpec -- cell of the outer GridSpec
    energies         : (n,) array -- energy axis
    residuals        : (n,) array -- the full-data residual series being resampled
    source_positions : (n_bootstrap, n) int array -- for each iteration, the position of
                       the original series each reconstructed point is copied from
    color            : matplotlib color -- version color for the reconstructions; the
                       original is always black
    n_reconstructions: int -- how many iterations to draw beneath the original

    Returns
    -------
    Axes -- the axes drawn on
    """
    ax_series = fig.add_subplot(subplot_spec)
    offset_step = 1.3 * (residuals.max() - residuals.min())
    ax_series.plot(
        energies, residuals, color='black', linewidth=0.9, label='original residuals',
    )
    for k in range(n_reconstructions):
        ax_series.plot(
            energies, residuals[source_positions[k]] - (k + 1) * offset_step,
            color=color, linewidth=0.9, alpha=0.85,
            label='reconstructed' if k == 0 else None,
        )
    ax_series.set_yticks([])
    ax_series.set_xlabel('Energy (eV)')
    ax_series.set_ylabel('residuals (offset)', fontsize=7)
    ax_series.set_title(
        'Original vs. two reconstructed residual series', fontsize=9,
    )
    ax_series.legend(fontsize=7, loc='upper right')
    return ax_series


def plot_reconstructed_acf(
    fig, subplot_spec, residuals, source_positions, block_length, color,
    n_acf_replicates=200,
):
    """Autocorrelation of the reconstructed series against the original's.

    This is the substantive check on the whole scheme. Resampling in blocks rather than
    point by point exists precisely to carry the residuals' short-range autocorrelation
    into the bootstrap; if the reconstructions were autocorrelated like white noise, the
    blocks would be doing nothing and a point-wise bootstrap would be the simpler
    equivalent. What should be seen is the reconstructed ACF tracking the original out
    to roughly one block length -- marked by the red rule -- and losing it beyond, since
    correlation across a seam only survives by accident.

    The band is the 5-95% range over replicate reconstructions rather than a single
    trace, so the comparison is against the spread the resampling actually produces and
    not against one lucky draw.

    Parameters
    ----------
    fig             : Figure -- the figure to add the axes to
    subplot_spec    : SubplotSpec -- cell of the outer GridSpec
    residuals       : (n,) array -- the full-data residual series being resampled
    source_positions: (n_bootstrap, n) int array -- per-iteration gather indices into
                      residuals, as used by the reconstructed-series panel
    block_length    : int -- drawn as the red reference rule, the lag out to which the
                      reconstructed ACF should follow the original
    color           : matplotlib color -- version color for the band and its mean; the
                      original ACF is always dashed black
    n_acf_replicates: int -- iterations to average over; slicing clamps, so a smaller
                      n_bootstrap simply uses everything there is

    Returns
    -------
    Axes -- the axes drawn on
    """
    ax_acf = fig.add_subplot(subplot_spec)
    original_lags, original_acf = calculate_acf(residuals)
    resampled_acf = np.array([
        calculate_acf(residuals[positions])[1]
        for positions in source_positions[:n_acf_replicates]
    ])
    ax_acf.fill_between(
        original_lags, np.percentile(resampled_acf, 5, axis=0),
        np.percentile(resampled_acf, 95, axis=0),
        color=color, alpha=0.3, label='reconstructed 5–95%',
    )
    ax_acf.plot(original_lags, resampled_acf.mean(axis=0), color=color,
                linewidth=1.3, label='reconstructed mean')
    ax_acf.plot(original_lags, original_acf, color='black', linewidth=1.3,
                linestyle='--', label='original')
    ax_acf.axvline(
        block_length, color='red', linestyle=':', linewidth=1.0,
        label=f'block_length = {block_length}',
    )
    ax_acf.axhline(0, color='0.6', linewidth=0.7)
    ax_acf.set_ylim(-0.4, 1.05)
    ax_acf.set_xlabel('Lag')
    ax_acf.set_ylabel('ACF', fontsize=8)
    ax_acf.set_title('Autocorrelation preserved by resampling', fontsize=10)
    ax_acf.legend(fontsize=6)
    return ax_acf

In [ ]:
def plot_holdout_block_structure(
    n, energies, selectors, n_bootstrap=1000, block_length=10,
    n_raster_iterations=25, raster_window=54, seed=0, residuals=None,
):
    """Compare the block *geometry* produced by select_holdout_blocks v1-v5.

    The v1-v5 write-up argues about where holdout block boundaries fall: whether the
    grid is aligned or shifted, whether a block wraps around the end of the array,
    whether the truncated final block always lands at the high-energy end. Those are
    all claims about the masks themselves, but the only geometric evidence elsewhere
    in this notebook is a single number per version (the holdout frequency std). This
    figure shows the masks directly, so the claims can be read off a plot.

    Only the selector output is needed -- no NNLS, no do_moving_block_holdout_bootstrap
    -- so this runs in seconds rather than the minutes the prediction error
    comparisons take.

    This function owns three things: drawing from every selector and deriving the
    quantities the panels need, choosing the axis limits that have to be shared across
    columns, and laying out the grid. Each panel type is drawn by its own function --
    see those for what a panel shows and why it is there.

    Parameters
    ----------
    n                   : int -- number of energy points (len(b))
    energies            : (n,) array -- energy axis, for the full-width marginal plots
    selectors           : list of (label, select_holdout_blocks_fn, color)
    n_bootstrap         : int -- iterations to draw from each selector
    block_length        : int -- passed to every selector, overriding the rule-of-thumb
                          round(n ** (1/3)) each defaults to, so all versions are compared
                          at one length. v4 and v5 take it as their shortest holdout block
                          and slide their random range up from it.
    n_raster_iterations : int -- iterations to show in the rasters (rows 1 and 2)
    raster_window       : int -- positions to show at each end of the spectrum in the
                          rasters. All n positions will not resolve into visible cells
                          at this figure width, and the interesting differences between
                          versions are edge effects, so each raster row shows the
                          low-energy and high-energy ends rather than a squeezed whole.
    seed                : int -- each selector gets its own default_rng(seed), so the
                          versions are compared on the same random stream
    residuals           : (n,) array or None -- full-data residuals. When supplied, two
                          extra rows show the residual series the resampled blocks
                          actually reconstruct, and whether that series keeps the
                          autocorrelation the moving-block scheme exists to preserve.
                          Everything else works without it.

    Layout: one column per selector, one grid row per row of plots. Rows 1, 4 and 5
    describe the *holdout* draw, rows 2 and 6 the *resample* draw, row 3 puts the two
    draws on one axis, and rows 7-8 (only with residuals) show what the resampling
    produces. The two rasters are adjacent, drawn at the same two windows and ruled on
    the same aligned block grid, so they can be read against each other column for column.
    One function per row:
      1. plot_holdout_mask_rasters            -- the masks at both ends of the spectrum
      2. plot_resample_start_rasters          -- block starts available vs. drawn, at
                                                 those same two ends
      3. plot_coverage_relative_to_uniform    -- holdout draw and resample draw, both
                                                 relative to a uniform draw, the holdout
                                                 one per position and the resample one
                                                 smoothed, on an absolute right-hand axis
      4. plot_holdout_fraction_histogram      -- held-out fraction per iteration
      5. plot_held_out_run_lengths            -- realized contiguous held-out runs
      6. plot_source_position_reuse           -- times a source position is reused
                                                 within one iteration
      7. plot_reconstructed_residual_series   -- reconstructions against the original
      8. plot_reconstructed_acf               -- autocorrelation of the reconstructions
                                                 against the original's

    Returns
    -------
    matplotlib.figure.Figure
        The one figure this builds, so the caller decides what becomes of it --
        typically `display(plot_holdout_block_structure(...))`. Nothing is drawn here.

        The figure is closed with `plt.close` before being returned, because the inline
        backend draws every figure still open at the end of a cell: left open it would
        appear twice, once from that flush and once from the caller. Closing discards
        nothing -- a closed figure still renders when displayed, saved, or edited further.
    """
    # Draw from every selector up front: rows 3-5 share x-limits across columns, which
    # can only be decided once all versions are known.
    drawn = []
    for label, fn, color in selectors:
        print(f'\n=== {label} ===')
        holdout_masks, sampled_starts, block_length, n_holdout_blocks = \
            fn(n, np.random.default_rng(seed=seed), n_bootstrap=n_bootstrap,
               block_length=block_length)

        # Holding out a block also removes it from the pool of blocks available to
        # resample from (the valid_block_starts filter every selector applies), but the
        # resampling side is never plotted anywhere else in this notebook. Express both
        # draws as coverage relative to uniform so they share a single axis: 1.0 means
        # "drawn exactly as often as a perfectly even draw would". Smooth the resample
        # counts over one block length -- unsmoothed they are dominated by Poisson noise
        # from the ~33k draws and the systematic structure is unreadable. The holdout
        # frequency is drawn as it is; its own noise is the scale its structure has to be
        # judged against.
        frequency = holdout_masks.mean(axis=0)
        n_start_positions = n - block_length + 1
        resample_counts = np.bincount(
            sampled_starts.ravel(), minlength=n_start_positions,
        )[:n_start_positions].astype(float)

        # Where each position of the reconstructed residual series is copied FROM.
        # This repeats, deliberately, the gather that do_moving_block_holdout_bootstrap
        # performs on the residuals themselves -- same broadcast, same reshape order,
        # same truncation to n -- so the reconstructed series and ACF rows below show the
        # real block layout rather than an idealization of it.
        block_offsets = np.arange(block_length)
        source_positions = (
            sampled_starts[:, :, None] + block_offsets[None, None, :]
        ).reshape(n_bootstrap, -1)[:, :n]

        # Which block starts each iteration was allowed to draw from, and which it drew.
        # A start s is available when [s, s + block_length) contains no held-out point --
        # exactly the condition every selector applies when it builds valid_block_starts,
        # restated here as a cumulative-sum window count so it can be evaluated for all
        # iterations at once instead of per start.
        holdout_cumsum = np.concatenate(
            [np.zeros((n_bootstrap, 1), dtype=int), np.cumsum(holdout_masks, axis=1)], axis=1,
        )
        held_out_in_block = holdout_cumsum[:, block_length:] - holdout_cumsum[:, :-block_length]
        available_starts = held_out_in_block == 0

        chosen_counts = np.bincount(
            (np.arange(n_bootstrap)[:, None] * n_start_positions + sampled_starts).ravel(),
            minlength=n_bootstrap * n_start_positions,
        ).reshape(n_bootstrap, n_start_positions)

        # The selectors are supposed to draw only from available starts; check it rather
        # than draw a picture that quietly assumes it.
        assert not (chosen_counts > 0)[~available_starts].any(), \
            f'{label} drew a resample block overlapping its own holdout'

        # 0 unavailable, 1 available but not drawn, 2 drawn once, 3 drawn 2+ times
        start_categories = np.where(
            ~available_starts, 0,
            np.where(chosen_counts == 0, 1, np.where(chosen_counts == 1, 2, 3)),
        )

        # How many times each source position is reused within a single iteration.
        # Blocks are drawn with replacement, so counts above 1 are expected; counts of
        # 0 are positions that iteration never sees, which includes every held-out
        # position plus any non-held-out position the draw happened to miss.
        flat_counts = np.bincount(
            (np.arange(n_bootstrap)[:, None] * n + source_positions).ravel(),
            minlength=n_bootstrap * n,
        ).reshape(n_bootstrap, n)

        drawn.append({
            'label': label,
            'color': color,
            'holdout_masks': holdout_masks,
            'block_length': block_length,
            'holdout_frequency': frequency,
            'holdout_fraction_per_iteration': holdout_masks.sum(axis=1) / n,
            'run_lengths': contiguous_run_lengths(holdout_masks),
            'n_start_positions': n_start_positions,
            'relative_holdout_unsmoothed': frequency / frequency.mean(),
            'relative_resample': smooth(resample_counts / resample_counts.mean(), block_length),
            'source_positions': source_positions,
            'reuse_counts': flat_counts,
            'start_categories': start_categories,
            'available_fraction': available_starts.mean(),
        })

    fraction_min = min(d['holdout_fraction_per_iteration'].min() for d in drawn)
    fraction_max = max(d['holdout_fraction_per_iteration'].max() for d in drawn)
    # A handful of very long runs (several selected blocks landing adjacent) would
    # otherwise squeeze every informative bar into the first tenth of the axis, so
    # clip the shared x-limit to a high percentile and report the true max in-panel.
    run_length_limit = int(np.ceil(max(
        np.percentile(d['run_lengths'], 99.5) for d in drawn
    )))
    # One shared coverage scale across all five columns. Per-column autoscaling would
    # blow up each version's sampling noise to fill its own axis, making versions that
    # are in fact flat at 1.0 look as structured as v2, whose edges really do collapse.
    coverage_values = np.concatenate(
        [d['relative_holdout_unsmoothed'] for d in drawn]
        + [d['relative_resample'] for d in drawn]
    )
    coverage_lo, coverage_hi = coverage_values.min(), coverage_values.max()
    coverage_pad = 0.05 * (coverage_hi - coverage_lo)
    coverage_ylim = (coverage_lo - coverage_pad, coverage_hi + coverage_pad)

    n_raster = min(n_raster_iterations, n_bootstrap)
    window = min(raster_window, n // 2)

    # The reuse histogram and the ACF band share limits across columns for the same
    # reason rows 3-5 do: per-column autoscaling would make every version look alike.
    max_reuse = max(d['reuse_counts'].max() for d in drawn)

    show_residual_rows = residuals is not None
    # One entry per row of plots. Rows 1 and 2 are the two rasters, adjacent so the
    # holdout masks and the resample availability they cause can be compared row against
    # row. Rows 1, 4 and 5 are the holdout draw, rows 2 and 6 the resample draw, row 3
    # both draws together, rows 7-8 what the resampling reconstructs.
    # Being read against each other, the two rasters are drawn at the same height: row 2
    # is the larger ratio only because it also carries the legend strip beneath its
    # rasters (see START_RASTER_HEIGHT_FRACTION).
    # Row 3 gets 1.6 rather than the 1.3 the other marginals take: it carries three
    # curves and a second y-axis, and it absorbed a row of its own that used to sit
    # above it.
    raster_row = 2.4
    height_ratios = ([raster_row, raster_row / START_RASTER_HEIGHT_FRACTION,
                      1.6, 1.3, 1.3, 1.3]
                     + ([1.6, 1.3] if show_residual_rows else []))
    inches_per_ratio_unit = 2.32  # keeps each row the height it had in the 4-row version
    fig = plt.figure(figsize=(5 * len(selectors), inches_per_ratio_unit * sum(height_ratios)))
    outer = gridspec.GridSpec(
        len(height_ratios), len(selectors), figure=fig,
        # hspace is a fraction of the *mean* row height, so splitting the rasters and
        # their marginal into separate rows shrank every gap; 0.55 restores the room
        # row 2's two-line heading needs under row 1's raster x-axis labels.
        # wspace was 0.25 before row 3 gained a right-hand axis; its label needs room
        # that the neighbouring column's left label was occupying.
        height_ratios=height_ratios, hspace=0.55, wspace=0.42,
    )

    for j, d in enumerate(drawn):
        # -------------------------------------------------------------------- row 1
        # The holdout masks themselves, at both ends of the spectrum.
        plot_holdout_mask_rasters(
            fig, outer[0, j],
            holdout_masks=d['holdout_masks'],
            block_length=d['block_length'],
            label=d['label'],
            color=d['color'],
            n_raster=n_raster,
            window=window,
        )
        # -------------------------------------------------------------------- row 2
        # Which block starts the holdout draw left available, and which were drawn,
        # in the same two windows and on the same aligned grid as the masks above.
        plot_resample_start_rasters(
            fig, outer[1, j],
            start_categories=d['start_categories'],
            block_length=d['block_length'],
            available_fraction=d['available_fraction'],
            color=d['color'],
            n_raster=n_raster,
            window=window,
        )

        # -------------------------------------------------------------------- row 3
        # Holdout draw vs. resample draw, both relative to uniform (see the first loop):
        # the holdout frequency per position, the resample counts smoothed.
        plot_coverage_relative_to_uniform(
            fig, outer[2, j],
            energies=energies,
            holdout_frequency=d['holdout_frequency'],
            relative_resample=d['relative_resample'],
            block_length=d['block_length'],
            ylim=coverage_ylim,
            color=d['color'],
        )

        # -------------------------------------------------------------------- row 4
        # How much data each iteration holds out -- constant for v1-v3, variable for
        # v4/v5, which is the mechanism behind their wider prediction error CIs.
        plot_holdout_fraction_histogram(
            fig, outer[3, j],
            holdout_fraction_per_iteration=d['holdout_fraction_per_iteration'],
            fraction_min=fraction_min,
            fraction_max=fraction_max,
            color=d['color'],
        )

        # -------------------------------------------------------------------- row 5
        # Realized block geometry rather than the nominal block_length.
        plot_held_out_run_lengths(
            fig, outer[4, j],
            run_lengths=d['run_lengths'],
            block_length=d['block_length'],
            run_length_limit=run_length_limit,
            color=d['color'],
        )

        # -------------------------------------------------------------------- row 6
        # What drawing blocks with replacement does to a single source position.
        plot_source_position_reuse(
            fig, outer[5, j],
            reuse_counts=d['reuse_counts'],
            mean_holdout_fraction=d['holdout_fraction_per_iteration'].mean(),
            max_reuse=max_reuse,
            color=d['color'],
        )

        if not show_residual_rows:
            continue

        # -------------------------------------------------------------------- row 7
        # What the blocks actually reconstruct.
        plot_reconstructed_residual_series(
            fig, outer[6, j],
            energies=energies,
            residuals=residuals,
            source_positions=d['source_positions'],
            color=d['color'],
        )

        # -------------------------------------------------------------------- row 8
        # Whether the reconstruction keeps the autocorrelation the blocks exist for.
        plot_reconstructed_acf(
            fig, outer[7, j],
            residuals=residuals,
            source_positions=d['source_positions'],
            block_length=d['block_length'],
            color=d['color'],
        )

    fig.suptitle(
        f'select_holdout_blocks v1–v5 — holdout mask structure, '
        f'n={n}, {n_bootstrap} iterations, seed={seed}',
        fontsize=13, y=1 - 0.35 / fig.get_figheight(),
    )
    # tight_layout cannot handle the nested GridSpecFromSubplotSpec in rows 1 and 2, so
    # the spacing is set explicitly by the GridSpec above plus this margin adjustment.
    # The margins are expressed in inches so they stay constant as rows are added
    # rather than eating a fixed fraction of an ever-taller figure.
    figure_height = fig.get_figheight()
    # The right margin was 0.99 before row 3 gained a right-hand axis, which left the
    # last column's tick labels and axis label off the edge of the figure.
    figure_width = fig.get_figwidth()
    fig.subplots_adjust(
        left=0.04, right=1 - 0.75 / figure_width,
        top=1 - 1.7 / figure_height, bottom=0.5 / figure_height,
    )
    plt.close(fig)
    return fig

In [ ]:
# This figure is about holdout *geometry*. It calls the selectors directly, at
# plot_holdout_block_structure's block_length=10 rather than their own rule-of-thumb
# default of round(n ** (1/3)) = 6, so it is unaffected by
# do_ref_subsets_moving_block_holdout_bootstrap now tuning the block length by default.
# The full-data NNLS residuals are what do_moving_block_holdout_bootstrap resamples.
# Passing them adds the last two rows: the residual series the blocks reconstruct, and
# whether that series keeps the autocorrelation the block scheme exists to preserve.
_, _, structure_residuals = fit_nnls(A, b)

# The function returns its figure rather than drawing it, so displaying it is the
# caller's call.
display(plot_holdout_block_structure(
    n=len(b),
    energies=energies,
    selectors=[
        ('v1 (aligned)',        select_holdout_blocks,    'steelblue'),
        ('v2 (shifted)',        select_holdout_blocks_v2, 'darkorange'),
        ('v3 (circular shift)', select_holdout_blocks_v3, 'crimson'),
        ('v4 (random lengths)', select_holdout_blocks_v4, 'mediumseagreen'),
        ('v5 (alternating)',    select_holdout_blocks_v5, 'mediumpurple'),
    ],
    residuals=structure_residuals,
))

## Development of `select_holdout_blocks` v1–v5

### Background

The goal of `select_holdout_blocks` is to pre-draw holdout masks for a moving-block holdout bootstrap: each iteration holds out ~1/3 of the data in contiguous blocks, fits the model on the remaining 2/3, and measures prediction error on the held-out positions. Across 1000 iterations, the ideal behavior is that every position in the spectrum is held out with roughly equal frequency — otherwise, some energy regions contribute more than others to the aggregate prediction error estimate.

Two metrics were used to compare versions:
- **Holdout frequency std** across positions (lower = more uniform coverage)
- **PE distribution**: mean, median, and 95% CI of each, computed by resampling the 1000 bootstrap PE draws 5000 times and taking the 2.5th/97.5th percentiles

**The uniformity metric has a floor.** Each position's holdout frequency is a mean of
`n_bootstrap` Bernoulli draws, so a selector that is *perfectly* uniform still measures a
std across positions of about `sqrt(p (1 - p) / n_bootstrap)` — **0.0149** at p = 1/3 and
1000 iterations. Two selectors whose stds are both at that floor are indistinguishable,
and the numbers below are therefore quoted as multiples of it. This matters immediately:
the whole spread the original comparison was decided on lies inside the floor.

**And the comparison has a second parameter.** Every number in this section was measured
at one block length, `L = round(n**(1/3)) = 6`, and n = 198 = 33 × 6. Whether the holdout
grid tiles the array without a remainder turns out to decide the ranking, so the results
below are given across a sweep of block lengths rather than at that one value.

---

### v1 — Aligned grid (baseline)

The original version computes a fixed block length `L = round(n^(1/3))` (L=6 for n=198) and partitions the data into non-overlapping aligned blocks at positions 0, L, 2L, …. Each iteration randomly selects ~1/3 of these blocks for holdout. Because every position belongs to exactly one block and all blocks are selected with equal probability, holdout frequency is uniform in expectation.

*Uniform in expectation only when `L` divides `n`.* The grid covers `floor(n / L) * L` positions and the remaining `n mod L` belong to no block at all, so they are never held out. At L=6 the remainder is zero and the caveat is invisible.

---

### v2 — Random shift

**Motivation:** the aligned grid always places block boundaries at the same positions; a random per-iteration shift might diversify the holdout structure.

**Implementation:** each iteration draws a random offset in [0, L) and shifts the entire grid: `offset, offset+L, offset+2L, …`.

**Finding:** this made uniformity *worse* (std 0.0441, 3.0× the sampling floor, against v1's 0.0158). The reason is a boundary effect: when offset > 0, the shifted grid doesn't reach position 0 or the last few positions, so those edge positions are never held out in most iterations. The aligned v1 is already perfectly uniform at this block length — the shift only introduced asymmetry. v2 is the one version that stays clear of the sampling floor at *every* block length tested, which is to say it is the one version whose non-uniformity was real all along.

---

### v3 — Circular shift

**Motivation:** fix the v2 boundary effect by ensuring every position is always in exactly one block regardless of offset.

**Implementation:** instead of `np.arange(offset, n - L + 1, L)`, always generate exactly `n_full_blocks` blocks using `circular_starts = (np.arange(n_full_blocks) * L + offset) % n`. Positions are assigned to blocks modulo n, so one block wraps around from the high-energy end to the low-energy end when offset > 0.

**Trade-off:** the wrap-around block mixes high- and low-energy points, which is physically meaningless for a spectrum. This was accepted as the cost of uniform coverage.

**Finding:** best holdout uniformity of all five versions (std 0.0122) — *at L = 6*. Every position is in exactly one block only when `n_full_blocks * L == n`; otherwise the circular grid covers `floor(n / L) * L` positions and leaves an uncovered arc that rotates with the offset. See *Results across block lengths*.

---

### v4 — Random block lengths

**Motivation:** rather than shifting a fixed-length grid, vary the block lengths themselves so holdout boundaries fall at different positions each iteration.

**Implementation:** each iteration partitions [0, n) into blocks of lengths drawn uniformly from [6, 10], then holds out ~1/3 of those blocks. Bootstrap resampling uses fixed length 6 (the minimum) to keep interface compatibility. Passing an explicit `block_length=L` slides the range to [L, L+4] rather than collapsing it.

**Finding:** at L = 6, uniformity of 0.0184 — 1.2× the sampling floor, i.e. indistinguishable from uniform — and a noticeably wider PE CI than v1 and v3. The wider CI reflects extra iteration-to-iteration variance: the partition ends on a truncated block, so the holdout fraction fluctuates (53–79 points held out, against exactly 66 for v1–v3), introducing noise in the PE estimates beyond what the other versions produce. The partition itself covers [0, n) exactly at *every* block length, which is the property that matters later.

---

### v5 — Alternating direction

**Motivation:** in v4, the truncated block always falls at the high-energy (right) end of the spectrum. Reversing every other holdout mask should distribute the truncation bias symmetrically between both ends.

**Implementation:** after building the v4 holdout mask, flip it for odd iterations: `holdout_mask = holdout_mask[::-1]`.

**Finding:** slightly better uniformity than v4 (0.0148 vs 0.0184 at L = 6), confirming the reversal is doing its job — though at 1.0× and 1.2× the sampling floor, both are already as uniform as the metric can resolve. The PE CI is the widest of all five versions, for v4's reason.

---

### Results at the rule-of-thumb block length, L = 6

The comparison as originally run — 1000 iterations, seed 0, n = 198. The **×floor** column
is the holdout frequency std divided by `sqrt(p (1 - p) / 1000)`, the std a perfectly
uniform selector would still measure.

| Version | Holdout freq std | ×floor | PE mean | Mean 95% CI | PE median | Median 95% CI |
|---|---|---|---|---|---|---|
| v1 (aligned) | 0.0158 | 1.06 | 0.028588 | [0.028292, 0.028902] | 0.028044 | [0.027749, 0.028404] |
| v2 (shifted) | 0.0441 | 2.96 | 0.029023 | [0.028694, 0.029392] | 0.028288 | [0.028039, 0.028583] |
| v3 (circular) | 0.0122 | 0.82 | 0.028914 | [0.028535, 0.029323] | 0.028128 | [0.027870, 0.028409] |
| v4 (random lengths) | 0.0184 | 1.24 | 0.029202 | [0.028807, 0.029644] | 0.028376 | [0.028117, 0.028666] |
| v5 (alternating) | 0.0148 | 1.00 | 0.029423 | [0.029016, 0.029868] | 0.028339 | [0.027968, 0.028662] |

**Only v2 is distinguishable.** v1, v3, v4 and v5 all sit within a quarter of the sampling
floor — they are four measurements of the same thing, which is a uniform draw. Re-running
them at seeds 0, 1 and 2 gives v1 ∈ [0.0148, 0.0158], v3 ∈ [0.0122, 0.0151] and
v4 ∈ [0.0129, 0.0184]: the seed spread is as large as the gaps the ranking was read from.
Raising the iteration count confirms it — `sweep_holdout_frequency_convergence` below
finds all three spanning 0.0074–0.0084 at 4,000 iterations against a floor of 0.0075, and
0.0038–0.0040 at 16,000 against 0.0037, while v2 pulls away from the floor as it should:
3.0×, 5.7×, 11.3×. The metric is measuring its own noise; there is nothing left to rank.

The PE columns tell the same story from the other side: every version's CI overlaps every
other version's, so no version produces a meaningfully different prediction error.

### Results across block lengths

Holdout frequency std as a multiple of the sampling floor, n = 198, 1000 iterations,
averaged over seeds 0–2. Bold columns are block lengths that divide 198 exactly.

| version | 4 | 5 | **6** | 7 | 8 | **9** | 10 | **11** | 12 | 13 | 14 | 15 | 16 | 17 | **18** | 19 | 20 |
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
| v1 | 2.4 | 2.9 | **1.0** | 2.3 | 4.0 | **0.9** | 4.4 | **1.0** | 3.8 | 2.9 | 2.6 | 2.7 | 4.0 | 5.6 | **1.0** | 4.2 | 6.7 |
| v2 | 2.3 | 2.6 | **3.0** | 3.2 | 3.5 | **3.6** | 3.6 | **4.2** | 4.2 | 4.6 | 4.3 | 4.8 | 5.0 | 5.1 | **4.8** | 5.2 | 5.3 |
| v3 | 1.7 | 2.1 | **0.9** | 1.5 | 3.1 | **1.1** | 3.3 | **0.9** | 2.5 | 1.6 | 1.3 | 1.5 | 2.2 | 3.9 | **0.9** | 2.6 | 5.0 |
| v4 | 1.0 | 1.0 | **1.1** | 0.9 | 1.1 | **1.0** | 0.9 | **1.0** | 1.1 | 1.3 | 0.9 | 0.8 | 1.1 | 0.8 | **0.8** | 1.1 | 1.1 |
| v5 | 1.0 | 0.9 | **1.0** | 0.9 | 1.0 | **1.0** | 0.8 | **1.1** | 0.9 | 1.1 | 0.9 | 0.9 | 1.1 | 0.9 | **1.0** | 1.1 | 1.0 |

v1 and v3 are at the floor in exactly four columns — 6, 9, 11 and 18 — and nowhere else.
Those are the divisors of 198 in range. v4 and v5 are at the floor in all seventeen. The
figure below plots these curves: two sawtooths that touch bottom only at the divisor
rules, and two flat lines.

At the tuned block length for this data, `block_length = 10` (198 = 19 × 10 + 8):

| version | std | ×floor | held out | never held out | starts available | PE mean | mean 95% CI |
|---|---|---|---|---|---|---|---|
| v1 (aligned) | 0.0634 | 4.4 | 0.303 | **8 positions** | 0.490 | 0.029517 | [0.029051, 0.030007] |
| v2 (shifted) | 0.0525 | 3.6 | 0.303 | 0 | 0.488 | 0.030176 | [0.029720, 0.030647] |
| v3 (circular) | 0.0482 | 3.3 | 0.303 | 0 | 0.489 | 0.030133 | [0.029673, 0.030615] |
| v4 (random lengths) | 0.0131 | 0.9 | 0.344 | 0 | 0.479 | 0.030157 | [0.029616, 0.030704] |
| **v5 (alternating)** | **0.0128** | **0.8** | 0.344 | 0 | 0.479 | 0.030478 | [0.029907, 0.031047] |

The ordering has inverted. v1, the baseline whose uniformity was "perfect in expectation",
is now the *worst* of the five, and the versions rejected for adding complexity without
improving uniformity are the only two that are uniform. The PE CIs still overlap, so the
decision is still made on uniformity alone — it is just made the other way.

### Mechanism: `n mod L`

All three fixed-length versions build `floor(n / L)` blocks of length `L`, which covers
`n - (n mod L)` positions. The remainder has to go somewhere, and what each version does
with it is the whole difference between them:

- **v1** leaves it at the high-energy end, permanently. At L = 10 positions 190–197 are
  held out in **zero of 1000 iterations** — the frequency is exactly 0.000, not merely low.
  The count of never-held-out positions equals `n mod L` at every block length tested.
- **v2** shifts the grid by a random offset and truncates at both ends, so it loses the
  remainder plus the offset — worse, and at every block length rather than only at some.
- **v3** wraps the grid modulo n, which does not create coverage it does not have: the
  uncovered arc is still `n mod L` wide, it just rotates with the offset. Instead of a dead
  zone it produces a ramp at each end of the spectrum. At L = 10 the holdout frequency
  falls from 0.33 in the middle to 0.058 at position 0 and 0.068 at position 197.
- **v4/v5** do not have a remainder. They partition [0, n) into random-length blocks and
  truncate the last one, so every position is in exactly one block at every block length,
  and the boundaries move every iteration. v5 additionally alternates which end carries
  the truncated block.

This is arithmetic, not a property of the arsenic data. Repeating the geometry at four
spectrum lengths — std as a multiple of the sampling floor, over all 24 (n, L) pairs:

| | n mod L = 0 (7 pairs) | n mod L ≠ 0 (17 pairs) |
|---|---|---|
| v1 | 0.92 – 1.19 | 1.86 – 4.58 |
| v2 | 3.00 – 4.53 | 2.75 – 4.90 |
| v3 | 0.93 – 1.07 | 1.00 – 3.38 |
| v4 | 0.90 – 1.10 | 0.87 – 1.10 |
| v5 | 0.90 – 1.06 | 0.85 – 1.07 |

n = 200 is the control: there 10 divides n and 6 does not, the reverse of n = 198, and the
versions swap places accordingly. v3's defect scales with the size of the remainder rather
than switching on — with a remainder of 1 position (n = 233, L = 8) it is still at the
floor, with a remainder of 6 or more it is never below 2×.

**This is not a hypothetical for this pipeline.** `choose_block_length` tunes the block
length from the residuals and returns **10** for this data, so the notebook's own fits run
at a block length that leaves a remainder of 8. It is a knife-edge, too: the Politis–White
p10 estimate is 10.28, and restricting the reference pool to M ≤ 2 moves it to 11 — which
divides 198, where the defect disappears entirely. Uniformity should not depend on whether
a tuned estimate happens to round onto a divisor of the number of energy points.

### Structural evidence from `plot_holdout_block_structure`

The metrics above are outcomes. The block-structure figure earlier in the notebook shows
the *geometry* they come from, at seed 0, 1000 iterations, n = 198. It now draws
`block_length=10`, so `summarize_block_structure` below tabulates both: L = 6, where the
section's original claims were made, and L = 10, where the figure is now drawn.

| | v1 | v2 | v3 | v4 | v5 |
|---|---|---|---|---|---|
| held-out points per iteration, L=6 | 66 | 66 | 66 | 53–79 | 53–79 |
| held-out points per iteration, L=10 | 60 | 60 | 60 | 52–82 | 52–82 |
| held-out fraction, L=6 | 0.333 | 0.333 | 0.333 | 0.268–0.399 | 0.268–0.399 |
| held-out fraction, L=10 | 0.303 | 0.303 | 0.303 | 0.263–0.414 | 0.263–0.414 |
| run length, median / longest, L=6 | 6 / 48 | 6 / 42 | 6 / 36 | 9 / 56 | 9 / 56 |
| run length, median / longest, L=10 | 10 / 50 | 10 / 50 | 10 / 50 | 13 / 66 | 13 / 66 |
| both ends held out together, L=6 | 10.7% | 1.5% | **27.3%** | 10.1% | 10.1% |
| both ends held out together, L=10 | **0.0%** | **0.0%** | 3.0% | 9.8% | 9.8% |
| resample block starts available, L=6 | 47.7% | 47.0% | 47.6% | **52.7%** | **52.7%** |
| resample block starts available, L=10 | **49.0%** | 48.8% | 48.9% | 47.9% | 47.9% |
| source positions never drawn, L=6 | 52.6% | 52.9% | 52.8% | 50.9% | 50.8% |
| source positions never drawn, L=10 | 52.1% | 52.1% | 52.0% | 52.9% | 52.8% |

#### Confirmations

**v2's boundary effect is exactly what was claimed.** The shifted grid rarely reaches
position 0 or n−1, and the figure puts a number on it: at L = 6 the two ends are held out
together in only **1.5%** of iterations, against ~10.7% for v1. The raster shows blank
columns at both edges. This is the whole of v2's std penalty, and it is an edge artifact
rather than anything distributed.

**v3's wrap-around is real and not rare.** When the offset is non-zero, one v3 block
contains both position 197 and position 0, so holding out both ends together is the same
event as selecting that block. Blocks are drawn *without* replacement, which
`predicted_both_ends_held_out` accounts for: (5/6)(11/33) + (1/6)(11/33)(10/32) =
**29.5%**. v1 has no wrap and reaches both ends only by selecting the two end blocks
separately, (11/33)(10/32) = **10.4%**. Measured at 1000 iterations the two come out at
27.3% and 10.7%; a rate near 0.3 carries a standard error of 1.4 points there, so the
cell re-measures at 40,000 iterations, where both predictions land — 29.3% and 10.5%.
The physically-artificial construct is confirmed to be unique to v3 and to affect nearly
three iterations in ten at the rule-of-thumb block length.

At L = 10 v3's rate falls to 3.0%, which is not the wrap becoming rare but the remainder
appearing: the uncovered arc is 8 positions wide and rotates with the offset, and only one
of the ten offsets leaves both ends inside a block at all — (1/10)(6/19) = 3.2% predicted,
3.1% measured. v1's rate falls to exactly zero, since position 197 is in its dead zone and
can never be held out at all.

**v4/v5's wider CIs have the mechanism attributed to them.** The write-up above explains
their wider prediction error CIs by the holdout fraction fluctuating when random block
lengths do not sum cleanly to n. The figure shows that fluctuation directly: v1–v3 hold out
exactly 66 points in every iteration at L = 6, while v4/v5 range over **53–79**. The
hypothesized mechanism is real and sizeable — and it is the price of the exact tiling that
makes them the only uniform versions at an arbitrary block length.

#### Corrections

**The "v4/v5 leave a larger resample pool" finding was a property of L = 6.** Holding out a
block also withdraws every resample block overlapping it, and at L = 6 v4/v5 leave 52.7% of
block starts available against ~47.7% for v1–v3 — attributed to their holding out fewer,
longer regions. Swept, that advantage does not survive: it is gone by L = 10 (47.9% against
49.0%) and reversed by L = 15. It is not a reason to prefer any version.

**v2's "both ends held out" rate also collapses to zero at L = 10**, for v1's reason:
its shifted grid leaves the high-energy end uncovered often enough that the two ends are
never held out in the same iteration.

**Nominal block length is not realized block length.** All of v1–v3 use blocks of exactly
L, but because ~1/3 of blocks are selected independently, adjacent selections merge: at
L = 6 the median run is 6 as designed while the longest reaches 48, 42 and 36 points. This
holds at every block length and for every version, and it means "block length 10" describes
the resampling exactly and the holdout only on average.

#### The block length question this raised, now answered downstream

Resampling in blocks exists to carry the residuals' autocorrelation into the bootstrap, and
at L = 6 the reconstructed series only partly does so. Mean ACF over 200 reconstructions
(v3), against the original:

| lag | 1 | 2 | 3 | 5 | 6 |
|---|---|---|---|---|---|
| original ACF | +0.461 | +0.297 | +0.312 | +0.183 | +0.121 |
| reconstructed, L=6 | +0.352 | +0.165 | +0.128 | +0.004 | −0.011 |
| fraction retained, L=6 | 0.76 | 0.55 | 0.41 | 0.02 | −0.09 |
| reconstructed, L=10 | +0.373 | +0.190 | +0.189 | +0.067 | +0.034 |
| fraction retained, L=10 | 0.81 | 0.64 | 0.60 | 0.37 | 0.28 |

At L = 6 essentially none of the lag-5 correlation survives, while the original residuals
still carry +0.183 there; a block of 6 can only preserve correlation at lags up to 5
internally, and every boundary is a splice to a randomly chosen distant part of the
spectrum. Ten-point blocks retain roughly a third of it. This applies to all five versions
equally — they share the block length and differ only in holdout selection — and it is what
motivated `choose_block_length`, in the section that follows.

---

### Analysis

**Recommendation: v5.** v3's apparent win was an artifact of 6 dividing 198 exactly. v5 is
within 1.1× of the sampling floor at every block length from 4 to 20 and at every spectrum
length tested, because it is the only version that both tiles [0, n) exactly at an arbitrary
block length and shares the truncation between the two ends of the spectrum.

**The ranking was never about the versions; it was about `n mod L`.** Any fixed-length grid
covers `floor(n / L) * L` positions, and the three versions that use one differ only in
where they put the shortfall: v1 at the high-energy end permanently, v2 at both ends plus
its offset, v3 rotating around the ends. When the shortfall is zero they are all uniform
and indistinguishable from each other and from v4/v5; when it is not, they are all
non-uniform. The block length is tuned from the data, so `n mod L == 0` is a coincidence
the pipeline should not be relying on.

**v5 over v4** on the same reasoning that motivated it: v4's truncated block always lands at
the high-energy end. The difference is inside the sampling floor at every length measured,
so it is a structural argument rather than a measured one — but it costs nothing, and the
asymmetry it removes is the kind that a longer run would eventually resolve.

**The cost of v5 is a fluctuating holdout fraction**, 0.263–0.414 at L = 10 against v3's
fixed 0.303, and with it the widest PE confidence interval of the five. That is a real
price and it is the reason the earlier comparison rejected v4/v5. It buys uniform coverage
at any block length, which the alternative does not provide at the block length this
pipeline actually uses.

**v1, v2 and v3 are not recommended.** v2 is dominated everywhere. v1 and v3 are equal-best
when `L` divides `n` and clearly worst when it does not, and nothing in the pipeline
guarantees the former.

**Prediction error still does not discriminate.** Every version's mean-PE CI overlaps every
other version's at L = 6, 10 and 15. The versions are being chosen on coverage geometry, as
they always were. Note also that PE cannot be compared *across* block lengths: v1–v3 hold
out `round(floor(n / L) / 3)` blocks of L, so the fraction they hold out drifts down as the
remainder grows (0.333 at L = 6, 0.303 at L = 10), and an iteration that holds out less
both fits on more data and scores on less of it.

In [ ]:

# Compare v1–v5
# The v1-v5 comparison is about holdout *geometry* at the rule-of-thumb block length, so
# it pins block_length=6 rather than tracking the tuned default. Its write-up and stored
# figures stay meaningful that way.

import matplotlib.gridspec as gridspec


def holdout_frequency_sampling_floor(holdout_fraction, n_bootstrap):
    """The holdout frequency std a *perfectly uniform* selector still measures.

    Each position's holdout frequency is the mean of n_bootstrap Bernoulli draws, so even
    a selector that holds out every position with exactly equal probability p produces
    per-position estimates scattered by sqrt(p (1 - p) / n_bootstrap). The std across
    positions therefore has a floor, and any comparison of two selectors whose stds are
    both at that floor is comparing sampling noise.

    This matters here because the floor is 0.0149 at p = 1/3 and n_bootstrap = 1000 --
    larger than the entire spread the original v1/v3/v4/v5 comparison was decided on.

    Parameters
    ----------
    holdout_fraction : float -- mean fraction of positions held out per iteration
    n_bootstrap      : int -- iterations the frequency is averaged over

    Returns
    -------
    float -- the expected std across positions under perfect uniformity
    """
    return float(np.sqrt(holdout_fraction * (1 - holdout_fraction) / n_bootstrap))


coef_cmp, fitted_cmp, residuals_cmp = fit_nnls(A, b)
n_cmp = len(b)
n_bootstrap_cmp = 1000

holdout_masks_v1, sampled_starts_v1, block_length_v1, _ = \
    select_holdout_blocks(n_cmp, np.random.default_rng(seed=0), n_bootstrap=n_bootstrap_cmp,
                                   block_length=6)
holdout_masks_v2, sampled_starts_v2, block_length_v2, _ = \
    select_holdout_blocks_v2(n_cmp, np.random.default_rng(seed=0), n_bootstrap=n_bootstrap_cmp,
                                   block_length=6)
holdout_masks_v3, sampled_starts_v3, block_length_v3, _ = \
    select_holdout_blocks_v3(n_cmp, np.random.default_rng(seed=0), n_bootstrap=n_bootstrap_cmp,
                                   block_length=6)
holdout_masks_v4, sampled_starts_v4, block_length_v4, _ = \
    select_holdout_blocks_v4(n_cmp, np.random.default_rng(seed=0), n_bootstrap=n_bootstrap_cmp,
                                   block_length=6)
holdout_masks_v5, sampled_starts_v5, block_length_v5, _ = \
    select_holdout_blocks_v5(n_cmp, np.random.default_rng(seed=0), n_bootstrap=n_bootstrap_cmp,
                                   block_length=6)

_, bootstrap_pes_v1 = do_moving_block_holdout_bootstrap(
    A, b, fitted_cmp, residuals_cmp, holdout_masks_v1, sampled_starts_v1, block_length_v1,
)
_, bootstrap_pes_v2 = do_moving_block_holdout_bootstrap(
    A, b, fitted_cmp, residuals_cmp, holdout_masks_v2, sampled_starts_v2, block_length_v2,
)
_, bootstrap_pes_v3 = do_moving_block_holdout_bootstrap(
    A, b, fitted_cmp, residuals_cmp, holdout_masks_v3, sampled_starts_v3, block_length_v3,
)
_, bootstrap_pes_v4 = do_moving_block_holdout_bootstrap(
    A, b, fitted_cmp, residuals_cmp, holdout_masks_v4, sampled_starts_v4, block_length_v4,
)
_, bootstrap_pes_v5 = do_moving_block_holdout_bootstrap(
    A, b, fitted_cmp, residuals_cmp, holdout_masks_v5, sampled_starts_v5, block_length_v5,
)

versions = [
    ('v1 (aligned)',        holdout_masks_v1.mean(axis=0), bootstrap_pes_v1, 'steelblue'),
    ('v2 (shifted)',        holdout_masks_v2.mean(axis=0), bootstrap_pes_v2, 'darkorange'),
    ('v3 (circular shift)', holdout_masks_v3.mean(axis=0), bootstrap_pes_v3, 'crimson'),
    ('v4 (random lengths)', holdout_masks_v4.mean(axis=0), bootstrap_pes_v4, 'mediumseagreen'),
    ('v5 (alternating)',    holdout_masks_v5.mean(axis=0), bootstrap_pes_v5, 'mediumpurple'),
]

# Pre-compute 95% CIs (done once so plots and printed table are consistent)
def bootstrap_ci(values, stat_fn, n_resamples=5000, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    idx = rng.integers(0, len(values), size=(n_resamples, len(values)))
    stats = stat_fn(values[idx], axis=1)
    return np.percentile(stats, [2.5, 97.5])

ci_rng = np.random.default_rng(seed=1)
mean_cis   = [bootstrap_ci(pes, np.mean,   rng=ci_rng) for _, _, pes, _ in versions]
median_cis = [bootstrap_ci(pes, np.median, rng=ci_rng) for _, _, pes, _ in versions]

# Layout: row 0 = frequency plots, row 1 = hist + violin, row 2 = CI plots
fig = plt.figure(figsize=(25, 15))
gs = gridspec.GridSpec(3, 5, figure=fig, hspace=0.45)
ax_freqs     = [fig.add_subplot(gs[0, j]) for j in range(5)]
ax_hist      = fig.add_subplot(gs[1, :4])
ax_violin    = fig.add_subplot(gs[1, 4])
ax_mean_ci   = fig.add_subplot(gs[2, :2])
ax_median_ci = fig.add_subplot(gs[2, 3:])

# Row 0: holdout frequency per energy position
all_freq_vals = np.concatenate([v[1] for v in versions])
freq_ymin = all_freq_vals.min() * 0.9
freq_ymax = all_freq_vals.max() * 1.1

for ax, (label, freq, pes, color) in zip(ax_freqs, versions):
    ax.plot(energies, freq, color=color, linewidth=0.8)
    ax.axhline(freq.mean(), color='red', linestyle='--', label=f'mean={freq.mean():.3f}')
    ax.set_xlabel('Energy (eV)')
    ax.set_ylabel('Fraction held out')
    ax.set_title(label, fontsize=10)
    ax.set_ylim(freq_ymin, freq_ymax)
    ax.legend(fontsize=8)

# Row 1 left: overlaid PE histograms
for label, freq, pes, color in versions:
    ax_hist.hist(pes, bins=40, alpha=0.5, color=color, label=f'{label} mean={pes.mean():.5f}', density=True)
    ax_hist.axvline(pes.mean(), color=color, linestyle='--', linewidth=1.5)
ax_hist.set_xlabel('Prediction Error (RMSE)')
ax_hist.set_ylabel('Density')
ax_hist.set_title('Holdout prediction error distributions')
ax_hist.legend(fontsize=9)

# Row 1 right: violin comparison
parts = ax_violin.violinplot([v[2] for v in versions], positions=range(1, 6), showmedians=True)
for pc, (_, _, _, color) in zip(parts['bodies'], versions):
    pc.set_facecolor(color)
    pc.set_alpha(0.7)
ax_violin.set_xticks(range(1, 6))
ax_violin.set_xticklabels([f'v{i}' for i in range(1, 6)], fontsize=9)
ax_violin.set_ylabel('Prediction Error (RMSE)')
ax_violin.set_title('PE violin comparison')

# Shared y-axis limits for the two CI plots
all_ci_bounds = [b for lo, hi in mean_cis + median_cis for b in (lo, hi)]
ci_ymin = min(all_ci_bounds) * 0.9995
ci_ymax = max(all_ci_bounds) * 1.0005

# Row 2 left: 95% CI of mean PE
for j, ((label, _, pes, color), (lo, hi)) in enumerate(zip(versions, mean_cis)):
    ax_mean_ci.errorbar(
        j + 1, pes.mean(),
        yerr=[[pes.mean() - lo], [hi - pes.mean()]],
        fmt='o', color=color, capsize=5, linewidth=1.5, markersize=7, label=label,
    )
ax_mean_ci.set_xticks(range(1, 6))
ax_mean_ci.set_xticklabels([f'v{i}' for i in range(1, 6)])
ax_mean_ci.set_ylabel('Prediction Error (RMSE)')
ax_mean_ci.set_title('95% CI of Mean Prediction Error')
ax_mean_ci.set_ylim(ci_ymin, ci_ymax)
ax_mean_ci.legend(fontsize=8)

# Row 2 right: 95% CI of median PE
for j, ((label, _, pes, color), (lo, hi)) in enumerate(zip(versions, median_cis)):
    ax_median_ci.errorbar(
        j + 1, np.median(pes),
        yerr=[[np.median(pes) - lo], [hi - np.median(pes)]],
        fmt='s', color=color, capsize=5, linewidth=1.5, markersize=7, label=label,
    )
ax_median_ci.set_xticks(range(1, 6))
ax_median_ci.set_xticklabels([f'v{i}' for i in range(1, 6)])
ax_median_ci.set_ylabel('Prediction Error (RMSE)')
ax_median_ci.set_title('95% CI of Median Prediction Error')
ax_median_ci.set_ylim(ci_ymin, ci_ymax)
ax_median_ci.legend(fontsize=8)

plt.suptitle(
    f'select_holdout_blocks v1–v5 — {n_bootstrap_cmp} iterations',
    fontsize=13,
)
plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()

print('Prediction error summary (95% CI computed by resampling the bootstrap PE draws):')
for (label, freq, pes, color), (mean_lo, mean_hi), (median_lo, median_hi) \
        in zip(versions, mean_cis, median_cis):
    print(f'  {label}:')
    print(f'    mean={pes.mean():.6f}  95% CI [{mean_lo:.6f}, {mean_hi:.6f}]')
    print(f'    median={np.median(pes):.6f}  95% CI [{median_lo:.6f}, {median_hi:.6f}]')

# A perfectly uniform selector still measures a std of sqrt(p (1 - p) / n_bootstrap),
# so the multiple of that floor is what says whether a difference between two versions
# is a difference at all. The block length sweep below is built on this.
print('\nHoldout frequency std across positions (lower = more uniform),')
print('against the sampling floor a perfectly uniform draw would still measure:')
for label, freq, pes, color in versions:
    floor = holdout_frequency_sampling_floor(float(freq.mean()), n_bootstrap_cmp)
    print(f'  {label}: {freq.std():.4f}  = {freq.std() / floor:.2f}x the {floor:.4f} floor')


In [ ]:
# Block length was the variable the v1-v5 comparison above never varied. Everything from
# here on treats it as the independent variable, so these helpers take it as an argument
# and never fall back on a selector's own round(n ** (1/3)) rule of thumb.
#
# The sweeps below are hundreds of independent draws, so they run in parallel. What makes
# that safe is that every arm builds its own default_rng(seed) and shares nothing else:
# the results are bit-identical to running the arms in a loop, and the sweep cell asserts
# exactly that on one arm every time it runs. joblib's loky backend pickles these
# notebook functions by value rather than relying on forked workers inheriting the
# kernel, which imposes two rules on anything run as an arm -- it must take everything it
# needs as an argument, including the selector, and it must hand back whatever it printed,
# since a worker's stdout never reaches the notebook.

import contextlib
import io

from joblib import Parallel, delayed

N_JOBS = -1   # all cores; set to 1 to run every sweep in this process instead


def run_arms(arms, n_jobs=N_JOBS):
    """Evaluate independent (callable, kwargs) tasks, returning results in task order.

    Task order rather than completion order, so that printed tables and stored figures do
    not depend on how the work happened to be scheduled.

    Parameters
    ----------
    arms   : list of (callable, kwargs dict) -- each evaluated as callable(**kwargs)
    n_jobs : int -- passed to joblib; -1 uses every core, 1 runs in this process, which
             is what the determinism check and any debugging want

    Returns
    -------
    list -- one result per task, in the order given
    """
    if n_jobs == 1:
        return [arm_fn(**kwargs) for arm_fn, kwargs in arms]
    return Parallel(n_jobs=n_jobs)(delayed(arm_fn)(**kwargs) for arm_fn, kwargs in arms)


def draw_holdout_arm(
    select_holdout_blocks_fn, n, block_length, seed, n_bootstrap, pe_inputs=None,
    acf_residuals=None, n_acf_replicates=200,
):
    """One (version, block length, seed) draw, reduced to everything the sweeps report.

    This is the unit of work every sweep below is built from, and it runs in a worker
    process, so it takes the selector as an argument rather than looking one up and
    returns the selector's own printed report rather than writing it to a stdout the
    notebook would never see.

    What it measures divides into two groups. The uniformity group -- the holdout
    frequency's std, its extremes and where they fall, and the count of positions never
    held out at all -- is what the block length sweep ranks versions on. The geometry
    group -- points held out per iteration, realized run lengths, whether both ends of the
    spectrum are held out together, how much of the resample pool survives, how much of
    the original series a reconstruction never touches -- is what the structural table
    reports. Both come from the same draw, so both are computed here.

    Parameters
    ----------
    select_holdout_blocks_fn : callable -- one of the v1-v5 selectors
    n                        : int -- number of energy points
    block_length             : int -- passed to the selector explicitly
    seed                     : int -- the arm's whole source of randomness
    n_bootstrap              : int -- iterations to draw
    pe_inputs                : (A, b, fitted, residuals) or None -- when given, the arm
                               also scores this draw with
                               do_moving_block_holdout_bootstrap and returns the
                               prediction error and its 95% CI. Done here rather than in
                               the caller so the holdout masks, which are the bulky part
                               of a draw, never have to leave the worker
    acf_residuals            : (n,) array or None -- when given, the arm also rebuilds
                               that series from its own block draw and returns the mean
                               autocorrelation of the reconstructions, i.e. how much of
                               the dependence this block length carried through
    n_acf_replicates         : int -- reconstructions averaged for that autocorrelation

    Returns
    -------
    dict -- the measurements listed above, plus 'printed', the selector's report
    """
    printed = io.StringIO()
    with contextlib.redirect_stdout(printed):
        holdout_masks, sampled_starts, realized_block_length, n_holdout_blocks = \
            select_holdout_blocks_fn(
                n, np.random.default_rng(seed=seed),
                n_bootstrap=n_bootstrap, block_length=block_length,
            )

    frequency = holdout_masks.mean(axis=0)
    never_held_out = np.flatnonzero(frequency == 0)
    held_out_per_iteration = holdout_masks.sum(axis=1)
    run_lengths = contiguous_run_lengths(holdout_masks)

    # Where each position of a reconstructed series is copied from, and so how much of the
    # original series a given iteration never touches. The same gather
    # do_moving_block_holdout_bootstrap performs, restated for counting.
    block_offsets = np.arange(realized_block_length)
    source_positions = (
        sampled_starts[:, :, None] + block_offsets[None, None, :]
    ).reshape(n_bootstrap, -1)[:, :n]
    reuse_counts = np.bincount(
        (np.arange(n_bootstrap)[:, None] * n + source_positions).ravel(),
        minlength=n_bootstrap * n,
    ).reshape(n_bootstrap, n)

    arm = {
        'block_length': block_length,
        'realized_block_length': int(realized_block_length),
        'n_holdout_blocks': int(n_holdout_blocks),
        'seed': seed,
        'n_bootstrap': n_bootstrap,
        'holdout_frequency': frequency,
        'std': float(frequency.std()),
        'holdout_fraction': float(frequency.mean()),
        'sampling_floor': holdout_frequency_sampling_floor(
            float(frequency.mean()), n_bootstrap,
        ),
        'frequency_min': float(frequency.min()),
        'frequency_min_position': int(np.argmin(frequency)),
        'frequency_max': float(frequency.max()),
        'frequency_max_position': int(np.argmax(frequency)),
        'never_held_out': int(never_held_out.size),
        'never_held_out_first': int(never_held_out[0]) if never_held_out.size else None,
        'never_held_out_last': int(never_held_out[-1]) if never_held_out.size else None,
        'held_out_min': int(held_out_per_iteration.min()),
        'held_out_max': int(held_out_per_iteration.max()),
        'run_length_median': int(np.median(run_lengths)),
        'run_length_max': int(run_lengths.max()),
        'mean_run_length': float(run_lengths.mean()),
        # Both ends held out in the same iteration. For a version whose grid can wrap this
        # is the wrap showing up in a statistic; for one whose grid cannot, it is two
        # separately-selected end blocks landing adjacent.
        'both_ends_held_out': float((holdout_masks[:, 0] & holdout_masks[:, -1]).mean()),
        'available_fraction': available_start_fraction(holdout_masks, realized_block_length),
        'never_drawn_fraction': float((reuse_counts == 0).mean()),
        'printed': printed.getvalue(),
    }
    if acf_residuals is not None:
        arm['reconstructed_acf'] = np.array([
            calculate_acf(acf_residuals[positions])[1]
            for positions in source_positions[:n_acf_replicates]
        ]).mean(axis=0)

    if pe_inputs is not None:
        A_pe, b_pe, fitted_pe, residuals_pe = pe_inputs
        _, bootstrap_pes = do_moving_block_holdout_bootstrap(
            A_pe, b_pe, fitted_pe, residuals_pe,
            holdout_masks, sampled_starts, realized_block_length,
        )
        arm['pe_mean'] = float(bootstrap_pes.mean())
        arm['pe_median'] = float(np.median(bootstrap_pes))
        arm['pe_mean_ci'] = bootstrap_ci(
            bootstrap_pes, np.mean, rng=np.random.default_rng(seed=1),
        )
    return arm


def available_start_fraction(holdout_masks, block_length):
    """Fraction of (iteration, block start) pairs the holdout draw leaves resampleable.

    A start s is available when [s, s + block_length) contains no held-out point -- the
    condition every selector applies when it builds valid_block_starts. Restating it as a
    cumulative-sum window count evaluates it for all iterations at once instead of per
    start.

    Parameters
    ----------
    holdout_masks : (n_bootstrap, n) bool array -- True where a position was held out
    block_length  : int -- resample block length, the width of the window

    Returns
    -------
    float -- fraction of (iteration, start) pairs that were available
    """
    n_bootstrap = holdout_masks.shape[0]
    holdout_cumsum = np.concatenate(
        [np.zeros((n_bootstrap, 1), dtype=int), np.cumsum(holdout_masks, axis=1)], axis=1,
    )
    held_out_in_block = holdout_cumsum[:, block_length:] - holdout_cumsum[:, :-block_length]
    return float((held_out_in_block == 0).mean())


def predicted_both_ends_held_out(n, block_length, circular):
    """Probability that one iteration holds out both ends of the spectrum, in closed form.

    Holding out position 0 and position n-1 in the same iteration is the statistic v3's
    wrap-around block shows up in, and the write-up reads the wrap rate off the comparison
    between this and the measurement, so it is derived rather than asserted.

    A grid of n // block_length blocks covers the first (n // block_length) * block_length
    positions and leaves the remainder uncovered. A circular grid slides that coverage by
    an offset drawn uniformly from [0, block_length) and lets one block wrap around the
    end of the array; an aligned grid has no offset and no wrap. For each offset the two
    ends are either uncovered, and so never held out; or inside a single block, and held
    out together exactly when that block is drawn, with probability k / N; or in two
    different blocks, and held out together when both are drawn, which for a draw without
    replacement is k (k - 1) / N (N - 1).

    Parameters
    ----------
    n            : int -- number of energy points
    block_length : int -- holdout block length
    circular     : bool -- True for v3's wrapping grid, False for v1's aligned one

    Returns
    -------
    float -- the probability, averaged over the offset
    """
    n_full_blocks = n // block_length
    n_holdout_blocks = round(n_full_blocks / 3)
    one_block = n_holdout_blocks / n_full_blocks
    two_blocks = one_block * (n_holdout_blocks - 1) / (n_full_blocks - 1)

    offsets = list(range(block_length)) if circular else [0]
    total = 0.0
    for offset in offsets:
        block_of_position = {}
        for block_index in range(n_full_blocks):
            start = block_index * block_length + offset
            for j in range(block_length):
                position = (start + j) % n if circular else start + j
                if position < n:
                    block_of_position[position] = block_index
        first, last = block_of_position.get(0), block_of_position.get(n - 1)
        if first is None or last is None:
            continue
        total += one_block if first == last else two_blocks
    return total / len(offsets)


def sweep_selector_block_lengths(
    n, selectors, block_lengths, n_bootstrap=1000, seeds=(0, 1, 2), pe_inputs=None,
    n_jobs=N_JOBS,
):
    """Run every selector at every block length and record what separates them.

    The v1-v5 results table was measured at a single block length, L = round(n ** (1/3)),
    which for n = 198 is 6 -- and 198 = 33 x 6 exactly. Whether a version's holdout grid
    tiles the array without a remainder turns out to decide the ranking, so the sweep
    covers a contiguous range of block lengths rather than a few round numbers: the
    divisibility sawtooth is only visible if consecutive lengths are all present.

    Every version is run at several seeds, because the headline metric has a sampling
    floor (see holdout_frequency_sampling_floor) and a single seed cannot be told from it.
    The spread across seeds is kept alongside the mean so the figure can show it.

    Prediction error is scored inside the arms, on the first seed only, so that the
    holdout masks never have to be shipped back from a worker; the cross-n sweep passes
    no design matrix and gets pure geometry.

    Parameters
    ----------
    n             : int -- number of energy points
    selectors     : list of (label, select_holdout_blocks_fn, color)
    block_lengths : iterable of int -- block lengths to run every version at
    n_bootstrap   : int -- iterations per (version, block length, seed)
    seeds         : tuple of int -- each gets its own default_rng; the first seed's draw
                    is the one whose geometry and prediction errors are kept, so those
                    stay comparable with the single-seed tables elsewhere
    pe_inputs     : (A, b, fitted, residuals) or None -- when given, each version's first
                    seed is also scored with do_moving_block_holdout_bootstrap. The
                    geometry does not need it, so the cross-n sweep omits it.
    n_jobs        : int -- passed to run_arms

    Returns
    -------
    dict with 'n', 'block_lengths', 'n_bootstrap', 'seeds', 'labels', 'colors', and
    'arms': {(label, block_length): row}, where each row is what draw_holdout_arm returns
    for the first seed, plus 'std' averaged over seeds with 'std_min'/'std_max' bounding
    it, plus -- with pe_inputs -- prediction error and its 95% CI.
    """
    block_lengths = list(block_lengths)
    tasks = [
        (draw_holdout_arm, {
            'select_holdout_blocks_fn': select_holdout_blocks_fn,
            'n': n, 'block_length': block_length, 'seed': seed,
            'n_bootstrap': n_bootstrap,
            'pe_inputs': pe_inputs if seed == seeds[0] else None,
        })
        for _, select_holdout_blocks_fn, _ in selectors
        for block_length in block_lengths
        for seed in seeds
    ]
    results = run_arms(tasks, n_jobs=n_jobs)

    arms, position = {}, 0
    for label, _, _ in selectors:
        for block_length in block_lengths:
            by_seed = results[position:position + len(seeds)]
            position += len(seeds)
            stds = np.array([arm['std'] for arm in by_seed])
            row = dict(by_seed[0])
            row.update({
                'std': float(stds.mean()),
                'std_min': float(stds.min()),
                'std_max': float(stds.max()),
                'std_by_seed': stds,
            })
            arms[(label, block_length)] = row

    return {
        'n': n,
        'block_lengths': block_lengths,
        'n_bootstrap': n_bootstrap,
        'seeds': tuple(seeds),
        'labels': [label for label, _, _ in selectors],
        'colors': [color for _, _, color in selectors],
        'arms': arms,
    }


def sweep_uniformity_across_n(
    n_values, selectors, block_lengths, n_bootstrap=1000, seeds=(0, 1, 2), n_jobs=N_JOBS,
):
    """The same sweep at several spectrum lengths, to separate a rule from a coincidence.

    The block-length effect found at n = 198 is a claim about n mod block_length, not
    about the arsenic data, and the way to show that is to change n. Geometry only -- no
    design matrix is involved, so no prediction error is computed.

    Parameters
    ----------
    n_values      : iterable of int -- spectrum lengths to repeat the sweep at
    selectors     : list of (label, select_holdout_blocks_fn, color)
    block_lengths : iterable of int -- block lengths to run at each n
    n_bootstrap   : int -- iterations per arm
    seeds         : tuple of int -- as in sweep_selector_block_lengths
    n_jobs        : int -- passed to run_arms

    Returns
    -------
    dict {n: sweep}, each value exactly what sweep_selector_block_lengths returns
    """
    return {
        n: sweep_selector_block_lengths(
            n, selectors, block_lengths, n_bootstrap=n_bootstrap, seeds=seeds,
            n_jobs=n_jobs,
        )
        for n in n_values
    }


def sweep_holdout_frequency_convergence(
    n, selectors, block_length, n_bootstrap_values, seeds=(0, 1, 2), n_jobs=N_JOBS,
):
    """Does the uniformity metric measure the selector, or its own sampling noise?

    A per-position holdout frequency averaged over B iterations carries a standard error
    of sqrt(p (1 - p) / B) whatever the selector does, so the std across positions has a
    floor. The way to tell a real difference from that floor is to move B: a systematic
    non-uniformity is a constant the extra iterations cannot average away, while noise
    falls as 1 / sqrt(B). Run at a block length that divides n, every version except v2
    should track the floor down.

    Parameters
    ----------
    n                 : int -- number of energy points
    selectors         : list of (label, select_holdout_blocks_fn, color)
    block_length      : int -- held fixed; the point of the sweep is B, not L
    n_bootstrap_values: iterable of int -- iteration counts to compare
    seeds             : tuple of int -- the spread across these is the other half of the
                        argument: a gap smaller than the seed spread is not a gap
    n_jobs            : int -- passed to run_arms

    Returns
    -------
    dict with 'n', 'block_length', 'labels', 'n_bootstrap_values' and
    'arms': {(label, n_bootstrap): {'std', 'std_min', 'std_max', 'sampling_floor'}}
    """
    n_bootstrap_values = list(n_bootstrap_values)
    tasks = [
        (draw_holdout_arm, {
            'select_holdout_blocks_fn': select_holdout_blocks_fn,
            'n': n, 'block_length': block_length, 'seed': seed,
            'n_bootstrap': n_bootstrap,
        })
        for _, select_holdout_blocks_fn, _ in selectors
        for n_bootstrap in n_bootstrap_values
        for seed in seeds
    ]
    results = run_arms(tasks, n_jobs=n_jobs)

    arms, position = {}, 0
    for label, _, _ in selectors:
        for n_bootstrap in n_bootstrap_values:
            by_seed = results[position:position + len(seeds)]
            position += len(seeds)
            stds = np.array([arm['std'] for arm in by_seed])
            arms[(label, n_bootstrap)] = {
                'std': float(stds.mean()),
                'std_min': float(stds.min()),
                'std_max': float(stds.max()),
                'sampling_floor': by_seed[0]['sampling_floor'],
            }
    return {
        'n': n,
        'block_length': block_length,
        'labels': [label for label, _, _ in selectors],
        'n_bootstrap_values': n_bootstrap_values,
        'seeds': tuple(seeds),
        'arms': arms,
    }


def summarize_block_structure(
    n, selectors, block_lengths, residuals=None, n_bootstrap=1000, seed=0,
    n_acf_replicates=200, n_jobs=N_JOBS,
):
    """The geometry behind the uniformity numbers, at one draw per version and length.

    The block structure figure shows this geometry for one block length at a time; this
    tabulates it at several, which is what the write-up compares. Nothing here is new
    machinery -- every quantity comes from draw_holdout_arm -- but collecting them at two
    block lengths side by side is what turns "v4/v5 leave a larger resample pool" into
    "at L = 6 they do", and what shows the wrap rate and the dead zone moving.

    The reconstructed autocorrelation is included because it is the one quantity that
    should depend on the block length and not on the version: the blocks exist to carry
    the residuals' dependence, and every version resamples with blocks of the same length.

    Parameters
    ----------
    n               : int -- number of energy points
    selectors       : list of (label, select_holdout_blocks_fn, color)
    block_lengths   : iterable of int -- block lengths to tabulate
    residuals       : (n,) array or None -- the series the reconstructions are built
                      from. Omit it for the geometry alone, which is what a re-measurement
                      at a high iteration count wants
    n_bootstrap     : int -- iterations per arm
    seed            : int -- single seed; this is geometry of one draw, not a spread
    n_acf_replicates: int -- reconstructions averaged for the autocorrelation
    n_jobs          : int -- passed to run_arms

    Returns
    -------
    dict with 'n', 'block_lengths', 'labels', 'original_acf', 'original_lags' and
    'arms': {(label, block_length): arm}, each arm as draw_holdout_arm returns it
    """
    block_lengths = list(block_lengths)
    tasks = [
        (draw_holdout_arm, {
            'select_holdout_blocks_fn': select_holdout_blocks_fn,
            'n': n, 'block_length': block_length, 'seed': seed,
            'n_bootstrap': n_bootstrap, 'acf_residuals': residuals,
            'n_acf_replicates': n_acf_replicates,
        })
        for _, select_holdout_blocks_fn, _ in selectors
        for block_length in block_lengths
    ]
    results = run_arms(tasks, n_jobs=n_jobs)
    original_lags, original_acf = (
        calculate_acf(residuals) if residuals is not None else (None, None)
    )
    return {
        'n': n,
        'block_lengths': block_lengths,
        'labels': [label for label, _, _ in selectors],
        'n_bootstrap': n_bootstrap,
        'original_acf': original_acf,
        'original_lags': original_lags,
        'arms': dict(zip(
            [(label, block_length)
             for label, _, _ in selectors for block_length in block_lengths],
            results,
        )),
    }


def most_uniform_version(sweep, block_length):
    """Label of the version with the lowest holdout frequency std at one block length."""
    return min(sweep['labels'], key=lambda label: sweep['arms'][(label, block_length)]['std'])


def sweep_array(sweep, key):
    """One row per version, one column per block length, for the panel functions.

    The panels take explicit named arrays rather than the sweep dict, matching the rest
    of the notebook's plotting functions, so this is the adapter between the two.
    """
    return np.array([
        [sweep['arms'][(label, block_length)][key] for block_length in sweep['block_lengths']]
        for label in sweep['labels']
    ])

In [ ]:
# Panels for the block length sweep, in the same style as the block structure figure
# above: one function per panel, each taking (fig, subplot_spec, <explicit named arrays>)
# and creating its own axes, with a layout function that owns the grid and the limits
# that have to be shared.


def plot_uniformity_vs_block_length(
    fig, subplot_spec, block_lengths, std, std_min, std_max, sampling_floor,
    labels, colors, divisor_block_lengths,
):
    """Holdout frequency std against block length, against the noise it has to beat.

    This is the panel the section's conclusion now rests on. The original comparison read
    a ranking off five stds measured at one block length and one seed; two things are
    added here. The block length is swept, which turns the metric from a number into a
    curve. And the sampling floor is drawn -- the std a perfectly uniform selector still
    measures from n_bootstrap Bernoulli draws per position -- which turns "lower is
    better" into "distinguishable from uniform, or not".

    The gray rules mark block lengths that divide n exactly. A fixed-length grid tiles the
    array without a remainder only there, so those are the block lengths where v1 and v3
    have nothing to drop, and the curves fall to the floor at precisely those points.

    Log y, because the range from the floor to v1's worst is a factor of ten and the
    interesting comparison at the bottom would otherwise be a flat line.

    Parameters
    ----------
    fig                    : Figure -- the figure to add the axes to
    subplot_spec           : SubplotSpec -- cell of the outer GridSpec
    block_lengths          : (n_lengths,) int array -- the swept block lengths
    std                    : (n_versions, n_lengths) array -- holdout frequency std,
                             averaged over seeds
    std_min, std_max       : (n_versions, n_lengths) arrays -- its range over seeds, drawn
                             as a band so a difference smaller than the seed spread cannot
                             be mistaken for a difference between versions
    sampling_floor         : (n_versions, n_lengths) array -- the per-arm floor; the shaded
                             band spans its min and max, which differ only because the
                             versions hold out slightly different fractions
    labels, colors         : lists, one per version
    divisor_block_lengths  : list of int -- swept block lengths that divide n exactly

    Returns
    -------
    Axes -- the axes drawn on
    """
    ax = fig.add_subplot(subplot_spec)
    for boundary in divisor_block_lengths:
        ax.axvline(boundary, color='0.45', linewidth=0.5, alpha=0.7)
    ax.fill_between(
        block_lengths, sampling_floor.min(axis=0), sampling_floor.max(axis=0),
        color='0.75', alpha=0.55, zorder=0,
        label='sampling floor $\\sqrt{p(1-p)/B}$',
    )
    for label, color, arm_std, arm_lo, arm_hi in zip(labels, colors, std, std_min, std_max):
        ax.fill_between(block_lengths, arm_lo, arm_hi, color=color, alpha=0.20, linewidth=0)
        ax.plot(block_lengths, arm_std, color=color, linewidth=1.6, marker='o',
                markersize=3.5, label=label)
    ax.set_yscale('log')
    # Headroom for the legend, which sits inside the axes to stay next to the curves.
    ax.set_ylim(top=std_max.max() * 3.0)
    ax.set_xticks(block_lengths)
    ax.set_xlabel('Holdout block length')
    ax.set_ylabel('Holdout frequency std\n(log scale)', fontsize=9)
    ax.set_title(
        'Uniformity vs. block length — gray rules mark block lengths that divide n',
        fontsize=11,
    )
    ax.legend(fontsize=7.5, ncol=2, loc='upper left')
    return ax


def plot_frequency_profiles(
    fig, subplot_spec, holdout_frequency, labels, colors, block_length, n_bootstrap,
):
    """Where the non-uniformity sits, at one block length that does not divide n.

    A std is a single number and hides the shape of the defect, which is the part that
    matters: a selector that is 3% non-uniform everywhere is a different problem from one
    that never holds out the last eight positions at all. Plotted per position, the three
    fixed-grid versions separate visibly -- v1 leaves a dead zone of exactly n mod L
    positions at the high-energy end, v2 and v3 rotate that same deficit into ramps at the
    ends -- while the variable-length versions stay flat across the whole spectrum.

    Parameters
    ----------
    fig              : Figure -- the figure to add the axes to
    subplot_spec     : SubplotSpec -- cell of the outer GridSpec
    holdout_frequency: (n_versions, n) array -- per-position holdout frequency
    labels, colors   : lists, one per version
    block_length     : int -- the block length these profiles were drawn at, for the title
    n_bootstrap      : int -- iterations behind each frequency, for the title

    Returns
    -------
    Axes -- the axes drawn on
    """
    ax = fig.add_subplot(subplot_spec)
    positions = np.arange(holdout_frequency.shape[1])
    for label, color, frequency in zip(labels, colors, holdout_frequency):
        ax.plot(positions, frequency, color=color, linewidth=1.1, label=label)
    ax.axhline(1 / 3, color='gray', linestyle=':', linewidth=1.0, label='1/3')
    ax.set_ylim(0, max(0.45, holdout_frequency.max() * 1.1))
    ax.set_xlim(positions[0], positions[-1])
    ax.set_xlabel('Position (index)')
    ax.set_ylabel('Fraction\nheld out', fontsize=9)
    ax.set_title(
        f'Per-position holdout frequency at block_length={block_length} '
        f'({n_bootstrap} iterations)', fontsize=11,
    )
    ax.legend(fontsize=7.5, ncol=3, loc='lower center', framealpha=0.85)
    return ax


def plot_holdout_budget_vs_block_length(
    fig, subplot_spec, block_lengths, holdout_fraction, mean_run_length, labels, colors,
):
    """What each version actually holds out as the block length grows.

    Uniformity is not the only thing that moves with block length, and these two curves
    are why prediction error cannot be compared across a sweep of it. A fixed grid holds
    out round(floor(n / L) / 3) blocks of L, so as L grows and the remainder grows with
    it, v1-v3 hold out a steadily smaller share of the spectrum -- an iteration that holds
    out less both fits on more data and scores on less of it. v4 and v5 partition the
    whole array every iteration and hold out ~1/3 at every length.

    The right-hand panel is the honest caveat to the left one: an explicit block_length
    sets v4/v5's *shortest* holdout block and slides their random range up from it, so at
    a nominal L they hold out blocks averaging longer than the fixed-grid versions do.
    Both curves are drawn so the comparison can be audited rather than assumed.

    Several curves coincide exactly rather than being missing. v1 and v3 hold out the same
    round(floor(n / L) / 3) blocks of L and differ only in where they put them, and v5 is
    v4 with every other mask reversed, so each pair draws a single line.

    Parameters
    ----------
    fig             : Figure -- the figure to add axes to
    subplot_spec    : SubplotSpec -- cell of the outer GridSpec, split into two panels
    block_lengths   : (n_lengths,) int array -- the swept block lengths
    holdout_fraction: (n_versions, n_lengths) array -- mean fraction held out per iteration
    mean_run_length : (n_versions, n_lengths) array -- mean realized contiguous held-out
                      run length, which merging of adjacent selected blocks pushes above
                      the nominal block length for every version
    labels, colors  : lists, one per version

    Returns
    -------
    (ax_fraction, ax_run) -- the held-out fraction and realized run length axes
    """
    inner = gridspec.GridSpecFromSubplotSpec(1, 2, subplot_spec=subplot_spec, wspace=0.22)
    ax_fraction = fig.add_subplot(inner[0, 0])
    ax_run = fig.add_subplot(inner[0, 1])

    for label, color, fraction, run_length in zip(
        labels, colors, holdout_fraction, mean_run_length,
    ):
        ax_fraction.plot(block_lengths, fraction, color=color, linewidth=1.5,
                         marker='o', markersize=3.5, label=label)
        ax_run.plot(block_lengths, run_length, color=color, linewidth=1.5,
                    marker='o', markersize=3.5, label=label)

    ax_fraction.axhline(1 / 3, color='red', linestyle='--', linewidth=1.0, label='1/3')
    ax_fraction.set_xticks(block_lengths)
    ax_fraction.set_xlabel('Holdout block length')
    ax_fraction.set_ylabel('Fraction held out\nper iteration', fontsize=9)
    ax_fraction.set_title(
        'How much each version holds out (v1/v3 coincide, as do v4/v5)', fontsize=10,
    )
    ax_fraction.legend(fontsize=7, ncol=2)

    ax_run.plot(block_lengths, block_lengths, color='0.45', linestyle=':', linewidth=1.0,
                label='nominal')
    ax_run.set_xticks(block_lengths)
    ax_run.set_xlabel('Holdout block length')
    ax_run.set_ylabel('Mean realized\nrun length', fontsize=9)
    ax_run.set_title('Realized vs. nominal block length', fontsize=10)
    ax_run.legend(fontsize=7, ncol=2)
    return ax_fraction, ax_run


def plot_availability_vs_block_length(
    fig, subplot_spec, block_lengths, available_fraction, labels, colors,
):
    """Resample block starts left available, against block length.

    The structural comparison at L=6 found that v4/v5 leave a larger pool of resampleable
    block starts than v1-v3 -- fewer, longer holdout regions contaminate fewer overlapping
    windows -- and recorded it as a point in their favor. Swept, that advantage turns out
    to be a property of L=6 rather than of the versions: it is about five percentage
    points there, gone by L=10, and reversed by L=15, where the fixed-grid versions leave
    the larger pool. It is not a reason to prefer any version.

    As in the panel above, v1 and v3 draw a single line and so do v4 and v5: availability
    depends on how much is held out in how many pieces, which those pairs share.

    Parameters
    ----------
    fig               : Figure -- the figure to add the axes to
    subplot_spec      : SubplotSpec -- cell of the outer GridSpec
    block_lengths     : (n_lengths,) int array -- the swept block lengths
    available_fraction: (n_versions, n_lengths) array -- fraction of (iteration, start)
                        pairs that contained no held-out point
    labels, colors    : lists, one per version

    Returns
    -------
    Axes -- the axes drawn on
    """
    ax = fig.add_subplot(subplot_spec)
    for label, color, availability in zip(labels, colors, available_fraction):
        ax.plot(block_lengths, availability, color=color, linewidth=1.5,
                marker='o', markersize=3.5, label=label)
    ax.set_xticks(block_lengths)
    ax.set_xlabel('Holdout block length')
    ax.set_ylabel('Block starts\navailable', fontsize=9)
    ax.set_title(
        'Resample pool left by the holdout draw (v1/v3 coincide, as do v4/v5)', fontsize=10,
    )
    ax.legend(fontsize=7, ncol=3)
    return ax


def plot_block_length_sensitivity(sweep, profile_block_length):
    """Lay out the block length sweep: is the best version a property of the version?

    Four rows, each answering one question about the sweep. The first is the finding --
    which versions are distinguishable from a uniform draw, at which block lengths. The
    second shows the shape of the defect at a block length that does not divide n, since
    the summary statistic cannot. The third and fourth are the controls: what each version
    holds out, and what it leaves behind to resample from, so that neither the ranking nor
    the earlier availability finding can be read without their caveats.

    Parameters
    ----------
    sweep                : dict from sweep_selector_block_lengths
    profile_block_length : int -- the block length whose per-position profiles are drawn
                           in row 2. Choose one that does not divide n; that is where the
                           versions differ

    Returns
    -------
    Figure -- the figure drawn
    """
    block_lengths = np.array(sweep['block_lengths'])
    labels, colors = sweep['labels'], sweep['colors']
    n, n_bootstrap = sweep['n'], sweep['n_bootstrap']
    divisor_block_lengths = [int(L) for L in block_lengths if n % L == 0]

    fig = plt.figure(figsize=(13, 15))
    outer = gridspec.GridSpec(4, 1, figure=fig, height_ratios=[1.35, 1.0, 1.0, 0.9],
                              hspace=0.42)

    plot_uniformity_vs_block_length(
        fig, outer[0, 0],
        block_lengths=block_lengths,
        std=sweep_array(sweep, 'std'),
        std_min=sweep_array(sweep, 'std_min'),
        std_max=sweep_array(sweep, 'std_max'),
        sampling_floor=sweep_array(sweep, 'sampling_floor'),
        labels=labels, colors=colors,
        divisor_block_lengths=divisor_block_lengths,
    )
    plot_frequency_profiles(
        fig, outer[1, 0],
        holdout_frequency=np.array([
            sweep['arms'][(label, profile_block_length)]['holdout_frequency']
            for label in labels
        ]),
        labels=labels, colors=colors,
        block_length=profile_block_length, n_bootstrap=n_bootstrap,
    )
    plot_holdout_budget_vs_block_length(
        fig, outer[2, 0],
        block_lengths=block_lengths,
        holdout_fraction=sweep_array(sweep, 'holdout_fraction'),
        mean_run_length=sweep_array(sweep, 'mean_run_length'),
        labels=labels, colors=colors,
    )
    plot_availability_vs_block_length(
        fig, outer[3, 0],
        block_lengths=block_lengths,
        available_fraction=sweep_array(sweep, 'available_fraction'),
        labels=labels, colors=colors,
    )

    fig.suptitle(
        f'select_holdout_blocks v1–v5 — sensitivity to block length, '
        f'n={n}, {n_bootstrap} iterations, seeds={sweep["seeds"]}',
        fontsize=13, y=0.995,
    )
    fig.subplots_adjust(left=0.07, right=0.98, top=0.955, bottom=0.04)
    plt.show()
    return fig

In [ ]:
# The comparison above ran at one block length. This runs every version at every length
# from 4 to 20, three seeds each, on every core, and repeats the geometry at four spectrum
# lengths --
# because the ranking that comparison produced turns out to be a statement about
# n mod block_length rather than about the versions. Prediction error reuses A, b and the
# full-data fit from the cell above, so the PE column is comparable with its table, and
# the CIs are computed with the same bootstrap_ci.

# The block length the figure profiles, and the one choose_block_length picks for this
# data further down. 198 = 19 * 10 + 8, so it is exactly the case the sweep is about.
profile_block_length = 10
rule_of_thumb_block_length = 6   # round(198 ** (1/3)), and a divisor of 198

sweep_selectors = [
    ('v1 (aligned)',        select_holdout_blocks,    'steelblue'),
    ('v2 (shifted)',        select_holdout_blocks_v2, 'darkorange'),
    ('v3 (circular shift)', select_holdout_blocks_v3, 'crimson'),
    ('v4 (random lengths)', select_holdout_blocks_v4, 'mediumseagreen'),
    ('v5 (alternating)',    select_holdout_blocks_v5, 'mediumpurple'),
]
sweep_block_lengths = list(range(4, 21))

block_length_sweep = sweep_selector_block_lengths(
    n_cmp, sweep_selectors, sweep_block_lengths, n_bootstrap=n_bootstrap_cmp,
    seeds=(0, 1, 2), pe_inputs=(A, b, fitted_cmp, residuals_cmp), n_jobs=N_JOBS,
)
# Same geometry at other spectrum lengths. n = 200 is the control: there 10 divides n and
# 6 does not, the reverse of n = 198, so a result that follows n mod L rather than the
# data has to swap over with it.
cross_n_sweep = sweep_uniformity_across_n(
    (150, 198, 200, 233), sweep_selectors, (6, 8, 9, 10, 11, 12),
    n_bootstrap=n_bootstrap_cmp, seeds=(0, 1, 2), n_jobs=N_JOBS,
)


# The arms above ran in worker processes. Re-run one of them in this process and
# compare it element for element: an arm that shared anything with its neighbours, or
# that drew from anything but its own seeded generator, would show up here.
_serial_arm = run_arms([(draw_holdout_arm, {
    'select_holdout_blocks_fn': select_holdout_blocks_v3, 'n': n_cmp,
    'block_length': profile_block_length, 'seed': 0, 'n_bootstrap': n_bootstrap_cmp,
})], n_jobs=1)[0]
_parallel_arm = block_length_sweep['arms'][('v3 (circular shift)', profile_block_length)]
assert np.array_equal(
    _serial_arm['holdout_frequency'], _parallel_arm['holdout_frequency'],
), 'running the sweep in parallel changed the holdout draw'
assert _serial_arm['available_fraction'] == _parallel_arm['available_fraction']


def floor_ratio(sweep, label, block_length):
    """Holdout frequency std in units of the sampling floor: 1.0 means indistinguishable."""
    arm = sweep['arms'][(label, block_length)]
    return arm['std'] / arm['sampling_floor']


# The write-up's claims, asserted rather than narrated: a re-run that breaks one of them
# should fail here instead of leaving the section quietly wrong.
v1_label, v2_label, v3_label, v4_label, v5_label = [s[0] for s in sweep_selectors]
divisor_lengths = [L for L in sweep_block_lengths if n_cmp % L == 0]
remainder_lengths = [L for L in sweep_block_lengths if n_cmp % L != 0]

for block_length in sweep_block_lengths:
    # v1's aligned grid covers floor(n / L) * L positions, so the remainder is not in any
    # block and is never held out by any iteration. This is the defect, exactly stated.
    assert (block_length_sweep['arms'][(v1_label, block_length)]['never_held_out']
            == n_cmp % block_length), f'v1 dead zone != n mod L at L={block_length}'
    for label in (v4_label, v5_label):
        assert floor_ratio(block_length_sweep, label, block_length) <= 1.5, \
            f'{label} left the sampling floor at L={block_length}'
    assert floor_ratio(block_length_sweep, v2_label, block_length) >= 2.0, \
        f'v2 reached the sampling floor at L={block_length}'
for block_length in divisor_lengths:
    for label in (v1_label, v3_label):
        assert floor_ratio(block_length_sweep, label, block_length) <= 1.5, \
            f'{label} is off the floor at L={block_length}, which divides n'
for block_length in remainder_lengths:
    assert floor_ratio(block_length_sweep, v1_label, block_length) >= 1.5, \
        f'v1 is at the floor at L={block_length}, which does not divide n'

for n_value, sweep in cross_n_sweep.items():
    for block_length in sweep['block_lengths']:
        for label in (v4_label, v5_label):
            assert floor_ratio(sweep, label, block_length) <= 1.5, \
                f'{label} left the floor at n={n_value}, L={block_length}'
        if n_value % block_length == 0:
            assert floor_ratio(sweep, v1_label, block_length) <= 1.5, \
                f'v1 is off the floor at n={n_value}, L={block_length}, which divides n'
        else:
            assert floor_ratio(sweep, v1_label, block_length) >= 1.5, \
                f'v1 is at the floor at n={n_value}, L={block_length}, which does not divide n'
            if n_value % block_length >= 6:
                assert floor_ratio(sweep, v3_label, block_length) >= 2.0, \
                    f'v3 absorbed a {n_value % block_length}-position remainder at L={block_length}'

plot_block_length_sensitivity(block_length_sweep, profile_block_length=profile_block_length)

short_labels = [label.split()[0] for label in block_length_sweep['labels']]
print(f'Holdout frequency std as a multiple of the sampling floor sqrt(p(1-p)/B), '
      f'n={n_cmp}, B={n_bootstrap_cmp}, mean over seeds {block_length_sweep["seeds"]}.')
print('1.0 = indistinguishable from a perfectly uniform draw.  * = L divides n.\n')
print(f'{"L":>3} {"n%L":>4}  ' + '  '.join(f'{s:>13}' for s in short_labels) + '   most uniform')
for block_length in sweep_block_lengths:
    cells = []
    for label in block_length_sweep['labels']:
        arm = block_length_sweep['arms'][(label, block_length)]
        cells.append(f'{arm["std"]:.4f} ({floor_ratio(block_length_sweep, label, block_length):.1f}x)')
    marker = '*' if n_cmp % block_length == 0 else ' '
    print(f'{block_length:>3}{marker}{n_cmp % block_length:>4}  ' + '  '.join(f'{c:>13}' for c in cells)
          + f'   {most_uniform_version(block_length_sweep, block_length).split()[0]}')

print(f'\nSpread over seeds {block_length_sweep["seeds"]} at block_length='
      f'{rule_of_thumb_block_length}, where every version but v2 is at the floor -- the '
      f'gap the original ranking was read from is smaller than this spread:')
for label in block_length_sweep['labels']:
    arm = block_length_sweep['arms'][(label, rule_of_thumb_block_length)]
    print(f'  {label:22} std {arm["std"]:.4f} over [{arm["std_min"]:.4f}, '
          f'{arm["std_max"]:.4f}]   floor {arm["sampling_floor"]:.4f}')

print(f'\nWhere the non-uniformity sits at block_length={profile_block_length} '
      f'(seed {block_length_sweep["seeds"][0]}):')
for label in block_length_sweep['labels']:
    arm = block_length_sweep['arms'][(label, profile_block_length)]
    frequency = arm['holdout_frequency']
    dead = ('none' if arm['never_held_out'] == 0 else
            f'{arm["never_held_out"]} positions, '
            f'{arm["never_held_out_first"]}-{arm["never_held_out_last"]}')
    print(f'  {label:22} ends {frequency[0]:.3f} at position 0 and {frequency[-1]:.3f} at '
          f'{n_cmp - 1};  lowest {arm["frequency_min"]:.3f} at '
          f'{arm["frequency_min_position"]};  never held out: {dead}')

print(f'\nAt block_length={profile_block_length} (the tuned value for this data; '
      f'{n_cmp} = {n_cmp // profile_block_length} x {profile_block_length} '
      f'+ {n_cmp % profile_block_length}):\n')
print(f'  {"version":22} {"std":>7} {"xfloor":>7} {"held out":>9} {"never":>6} {"avail":>7} '
      f'{"PE mean":>9}  {"95% CI":>22}')
for label in block_length_sweep['labels']:
    arm = block_length_sweep['arms'][(label, profile_block_length)]
    lo, hi = arm['pe_mean_ci']
    print(f'  {label:22} {arm["std"]:7.4f} '
          f'{floor_ratio(block_length_sweep, label, profile_block_length):6.1f}x '
          f'{arm["holdout_fraction"]:9.3f} {arm["never_held_out"]:6d} '
          f'{arm["available_fraction"]:7.3f} {arm["pe_mean"]:9.6f}  '
          f'[{lo:.6f}, {hi:.6f}]')

print('\nSame geometry at other spectrum lengths (std as a multiple of the sampling floor):\n')
print(f'{"n":>5} {"L":>3} {"n%L":>4}  ' + '  '.join(f'{s:>6}' for s in short_labels))
for n_value, sweep in cross_n_sweep.items():
    for block_length in sweep['block_lengths']:
        ratios = [floor_ratio(sweep, label, block_length) for label in sweep['labels']]
        marker = '*' if n_value % block_length == 0 else ' '
        print(f'{n_value:>5} {block_length:>3}{marker}{n_value % block_length:>3}  '
              + '  '.join(f'{r:6.2f}' for r in ratios))

divides = [(n_value, block_length)
           for n_value, sweep in cross_n_sweep.items()
           for block_length in sweep['block_lengths'] if n_value % block_length == 0]
remainder = [(n_value, block_length)
             for n_value, sweep in cross_n_sweep.items()
             for block_length in sweep['block_lengths'] if n_value % block_length != 0]
print(f'\nSummarized over those {len(divides) + len(remainder)} (n, L) pairs:\n')
print(f'  {"version":22} {f"n mod L = 0 ({len(divides)} pairs)":>24} '
      f'{f"n mod L != 0 ({len(remainder)} pairs)":>24}')
for label in block_length_sweep['labels']:
    bands = []
    for pairs in (divides, remainder):
        ratios = [floor_ratio(cross_n_sweep[n_value], label, block_length)
                  for n_value, block_length in pairs]
        bands.append(f'{min(ratios):.2f} - {max(ratios):.2f}')
    print(f'  {label:22} {bands[0]:>24} {bands[1]:>24}')

In [ ]:
# The claim that the original v1-v5 ranking was noise rests on two measurements. One is
# the spread across seeds at a fixed iteration count, printed above. The other is what
# happens when the iteration count moves: sampling noise falls as 1/sqrt(B), while a
# systematic non-uniformity is a constant that more iterations cannot average away, so its
# ratio to the floor grows instead. Run at a block length that divides n, where no version
# has a remainder to mishandle and the only thing left to measure is the noise itself.

convergence = sweep_holdout_frequency_convergence(
    n_cmp, sweep_selectors, block_length=rule_of_thumb_block_length,
    n_bootstrap_values=(1000, 4000, 16000), seeds=(0, 1, 2), n_jobs=N_JOBS,
)

# v2's non-uniformity is the one that is real at every block length, so it is the one
# whose ratio to the floor has to grow as the floor drops; the others have to track it.
for label in convergence['labels']:
    ratios = [convergence['arms'][(label, B)]['std'] / convergence['arms'][(label, B)]['sampling_floor']
              for B in convergence['n_bootstrap_values']]
    if label.startswith('v2'):
        assert ratios[-1] > 2 * ratios[0], \
            f'{label} did not separate from the floor as iterations grew: {ratios}'
    else:
        assert max(ratios) <= 1.5, f'{label} left the sampling floor at some B: {ratios}'

print(f'Holdout frequency std against iteration count, n={n_cmp}, '
      f'block_length={rule_of_thumb_block_length} (divides n), '
      f'mean over seeds {convergence["seeds"]} with the seed range beneath.\n')
print(f'{"B":>7} {"floor":>8}  ' + '  '.join(f'{label.split()[0]:>16}'
                                             for label in convergence['labels']))
for n_bootstrap in convergence['n_bootstrap_values']:
    means, ranges = [], []
    for label in convergence['labels']:
        arm = convergence['arms'][(label, n_bootstrap)]
        means.append(f'{arm["std"]:.4f} ({arm["std"] / arm["sampling_floor"]:.1f}x)')
        ranges.append(f'[{arm["std_min"]:.4f}, {arm["std_max"]:.4f}]')
    floor = convergence['arms'][(convergence['labels'][0], n_bootstrap)]['sampling_floor']
    print(f'{n_bootstrap:>7} {floor:>8.4f}  ' + '  '.join(f'{m:>16}' for m in means))
    print(f'{"":>7} {"":>8}  ' + '  '.join(f'{r:>16}' for r in ranges))

print('\nEvery version except v2 tracks the floor down as 1/sqrt(B): what the metric was')
print('measuring at 1000 iterations was its own sampling noise, not the selector. v2 is')
print('the exception, and its ratio to the floor grows because its defect is real.')

In [ ]:
# The geometry behind the uniformity numbers, at the rule-of-thumb block length and at the
# tuned one. Same quantities the block structure figure draws, tabulated at two block
# lengths side by side, which is what turns "v4/v5 leave a larger resample pool" into
# "at L=6 they do" and shows the dead zone and the wrap rate moving with the remainder.

structure_summary = summarize_block_structure(
    n_cmp, sweep_selectors,
    block_lengths=(rule_of_thumb_block_length, profile_block_length),
    residuals=residuals_cmp, n_bootstrap=n_bootstrap_cmp, seed=0, n_jobs=N_JOBS,
)

short_labels = [label.split()[0] for label in structure_summary['labels']]
rows = [
    ('held out / iteration', lambda a: f'{a["held_out_min"]}-{a["held_out_max"]}'),
    ('held-out fraction', lambda a: f'{a["held_out_min"] / n_cmp:.3f}-'
                                    f'{a["held_out_max"] / n_cmp:.3f}'),
    ('run median / longest', lambda a: f'{a["run_length_median"]} / {a["run_length_max"]}'),
    ('both ends held out', lambda a: f'{a["both_ends_held_out"]:.1%}'),
    ('starts available', lambda a: f'{a["available_fraction"]:.1%}'),
    ('positions never drawn', lambda a: f'{a["never_drawn_fraction"]:.1%}'),
    ('never held out', lambda a: f'{a["never_held_out"]}'),
]
for block_length in structure_summary['block_lengths']:
    print(f'=== block_length={block_length}   n={n_cmp} = '
          f'{n_cmp // block_length} x {block_length} + {n_cmp % block_length}, '
          f'{n_bootstrap_cmp} iterations, seed 0')
    print(f'  {"":22} ' + '  '.join(f'{label:>13}' for label in short_labels))
    for title, render in rows:
        cells = [render(structure_summary['arms'][(label, block_length)])
                 for label in structure_summary['labels']]
        print(f'  {title:22} ' + '  '.join(f'{cell:>13}' for cell in cells))
    print()

# Holding out both ends of the spectrum in one iteration is the statistic v3's wrap shows
# up in, so the closed form is worth stating next to it rather than in prose. At 1000
# iterations the standard error on a rate near 0.3 is about 0.014, which is why the
# v3 rows below agree with the prediction only to about that.
print('Both ends held out together: predicted against measured\n')
print(f'  {"":22} ' + '  '.join(f'{f"L={L}":>19}'
                                for L in structure_summary['block_lengths']))
for label, circular in (('v1 (aligned)', False), ('v3 (circular shift)', True)):
    cells = []
    for block_length in structure_summary['block_lengths']:
        predicted = predicted_both_ends_held_out(n_cmp, block_length, circular=circular)
        measured = structure_summary['arms'][(label, block_length)]['both_ends_held_out']
        cells.append(f'{predicted:.1%} vs {measured:.1%}')
    print(f'  {label:22} ' + '  '.join(f'{cell:>19}' for cell in cells))

# The gap in the v3 row is sampling error, not a wrong model: a rate near 0.3 carries a
# standard error of 1.4 points at 1000 iterations. Re-measure the two versions the closed
# form covers with forty times the iterations, where that error is 0.2 points, and hold
# the prediction to it. Geometry only -- no reconstructions, so no residuals needed.
wrap_check = summarize_block_structure(
    n_cmp,
    [selector for selector in sweep_selectors
     if selector[0] in ('v1 (aligned)', 'v3 (circular shift)')],
    block_lengths=structure_summary['block_lengths'],
    n_bootstrap=40 * n_bootstrap_cmp, seed=0, n_jobs=N_JOBS,
)
print(f'\nRe-measured at {40 * n_bootstrap_cmp} iterations:\n')
print(f'  {"":22} ' + '  '.join(f'{f"L={L}":>19}' for L in wrap_check['block_lengths']))
for label, circular in (('v1 (aligned)', False), ('v3 (circular shift)', True)):
    cells = []
    for block_length in wrap_check['block_lengths']:
        predicted = predicted_both_ends_held_out(n_cmp, block_length, circular=circular)
        measured = wrap_check['arms'][(label, block_length)]['both_ends_held_out']
        standard_error = np.sqrt(max(measured * (1 - measured), 1e-12)
                                 / wrap_check['n_bootstrap'])
        assert abs(measured - predicted) <= 4 * standard_error + 1e-9, \
            f'{label} at L={block_length}: predicted {predicted:.4f}, measured {measured:.4f}'
        cells.append(f'{predicted:.1%} vs {measured:.1%}')
    print(f'  {label:22} ' + '  '.join(f'{cell:>19}' for cell in cells))

# Whether the blocks deliver the dependence they exist to preserve. This is the one
# quantity that should follow the block length and not the version -- every version
# resamples with blocks of the same length -- so all five are printed to show that it does.
print('\nAutocorrelation of the reconstructed residual series, mean of 200 reconstructions')
print('(the fraction of the original retained is in parentheses):\n')
acf_lags = (1, 2, 3, 5, 6)
original_acf = structure_summary['original_acf']
print(f'  {"":26} ' + '  '.join(f'{f"lag {lag}":>16}' for lag in acf_lags))
print(f'  {"original residuals":26} '
      + '  '.join(f'{original_acf[lag]:>+16.3f}' for lag in acf_lags))
for block_length in structure_summary['block_lengths']:
    for label in structure_summary['labels']:
        reconstructed = structure_summary['arms'][(label, block_length)]['reconstructed_acf']
        cells = [f'{reconstructed[lag]:+.3f} ({reconstructed[lag] / original_acf[lag]:+.2f})'
                 for lag in acf_lags]
        print(f'  {f"L={block_length}, {label.split()[0]}":26} '
              + '  '.join(f'{cell:>16}' for cell in cells))

### Choosing the block length from the data

Every `select_holdout_blocks` version above sets

```python
block_length = max(1, int(np.round(n ** (1 / 3))))   # = 6 for n = 198
```

`n ** (1/3)` is the *rate* at which the MSE-optimal moving-block length grows with
sample size. It is not the optimal length — the constant in front of that rate depends
on how much dependence the series actually carries, and the code above silently takes
that constant to be 1.

The v1–v5 write-up flagged this: at the rule-of-thumb length the resampled residual
series keeps only about three quarters of the lag-1 autocorrelation and essentially none
by lag 5, where the original residuals still carry +0.183. Blocks of 6 are cutting the
dependence they exist to preserve. (Row 9 of `plot_holdout_block_structure` is the live
version of that panel; it now draws `block_length=10`, where retention is markedly
better. Both are tabulated in the write-up above.)

**The estimator.** `politis_white_block_length` implements Politis & White (2004), the
standard data-driven block length for the moving block bootstrap. Three steps:

1. Choose a bandwidth `M = 2 * m_hat`, where `m_hat` is the smallest lag past which the
   sample autocorrelation stays inside `±2 sqrt(log10(n) / n)` for `k_n` consecutive
   lags — the lag past which the series stops looking correlated.
2. Form flat-top-weighted sums over `|k| <= M` of the autocovariances: the long-run
   variance `G_hat = sum R(k)` and the curvature `g_hat = sum |k| R(k)`.
3. `b_opt = (2 g_hat**2 / D_hat) ** (1/3) * n ** (1/3)`, with `D_hat = (4/3) G_hat**2`
   for the moving block bootstrap, capped at `ceil(min(3 sqrt(n), n/3))`.

**One block length for 2,324 combinations.** The holdout draw is shared across every
reference combination — that is what makes their prediction errors comparable — so a
single block length has to serve all of them, while Politis–White gives one estimate per
combination and those estimates disagree wildly.

`choose_block_length` reduces them with a **low quantile rather than the median**, for a
structural reason rather than a tuning preference. An underfit combination's residuals
are not noise: they contain the part of the spectrum the model failed to explain, which
is smooth and spectrum-shaped. Politis–White cannot distinguish that deterministic
misfit from genuine dependence and reads it as very long-range autocorrelation — rank
correlation between fit RMSE and `b_opt` is +0.54, and every one-reference subset pins
the cap.

The key point is that this contamination is **one-sided**. Unmodeled structure can only
ever inflate the estimate, never deflate it, so the low order statistics are the
trustworthy end of the distribution and the median is meaningless. The percentile is a
parameter and the p1/p5/p10/p25 sweep is printed on every call, so the choice stays
auditable rather than magic.

These cells define the estimator; the section at the end of the notebook sweeps the
block length through the whole pipeline to check whether it changes any answer.

In [ ]:
def flat_top_lag_window(t):
    """Politis & Romano flat-top lag window: 1 on |t|<=1/2, tapering to 0 at |t|=1."""
    abs_t = np.abs(t)
    return np.where(abs_t <= 0.5, 1.0, np.where(abs_t <= 1.0, 2.0 * (1.0 - abs_t), 0.0))


def politis_white_block_length(x, k_n=None):
    """Data-driven optimal moving-block-bootstrap block length (Politis & White 2004).

    The notebook's selectors all set block_length = round(n ** (1/3)), which is only
    the *rate* at which the optimal block length grows with n; the constant in front
    of it depends on how much dependence the series actually carries. This estimates
    that constant from the data.

    The estimator has three steps:

      1. Pick a bandwidth M. Find the smallest lag m past which the sample
         autocorrelation stays inside the +/- 2 sqrt(log10(n) / n) band for k_n
         consecutive lags -- i.e. the lag past which the series looks uncorrelated --
         and set M = 2m.
      2. Form flat-top-weighted estimates over lags |k| <= M of the long-run variance
         G_hat_0 = sum R(k) and of the "curvature" g_hat = sum |k| R(k).
      3. b_opt = (2 g_hat^2 / D_hat) ** (1/3) * n ** (1/3), with
         D_hat = (4/3) G_hat_0^2 for the moving block bootstrap.

    Returns
    -------
    dict with the estimate and the intermediate quantities, so the number can be
    audited rather than taken on faith.
    """
    x = np.asarray(x, dtype=float)
    n = len(x)
    if k_n is None:
        k_n = max(5, int(np.ceil(np.sqrt(np.log10(n)))))

    # autocovariances R(0..n-1) and autocorrelations
    centered = x - x.mean()
    max_lag = min(n - 1, int(np.ceil(np.sqrt(n))) + k_n + 40)
    autocovariance = np.array(
        [np.dot(centered[:n - k], centered[k:]) / n for k in range(max_lag + 1)]
    )
    autocorrelation = autocovariance / autocovariance[0]

    # step 1: smallest m with k_n consecutive insignificant autocorrelations after it
    significance_bound = 2.0 * np.sqrt(np.log10(n) / n)
    m_hat = 0
    for m in range(1, max_lag - k_n + 1):
        window = np.abs(autocorrelation[m + 1:m + 1 + k_n])
        if len(window) == k_n and (window < significance_bound).all():
            m_hat = m
            break
    else:
        # no such run: fall back to the largest lag that is still significant
        significant = np.where(np.abs(autocorrelation[1:]) >= significance_bound)[0]
        m_hat = int(significant[-1]) + 1 if len(significant) else 1

    bandwidth_M = min(2 * m_hat, max_lag)

    # step 2: flat-top-weighted sums over -M..M (symmetric, so double the k>0 terms)
    lags = np.arange(1, bandwidth_M + 1)
    weights = flat_top_lag_window(lags / bandwidth_M) if bandwidth_M > 0 else np.array([])
    long_run_variance = autocovariance[0] + 2.0 * np.sum(weights * autocovariance[lags])
    curvature = 2.0 * np.sum(weights * lags * autocovariance[lags])

    # step 3
    d_hat = (4.0 / 3.0) * long_run_variance ** 2
    if curvature == 0 or d_hat == 0:
        b_opt = 1.0
    else:
        b_opt = (2.0 * curvature ** 2 / d_hat) ** (1.0 / 3.0) * n ** (1.0 / 3.0)

    # Politis & White cap the estimate; without it a near-unit-root series can ask for
    # a block longer than the sample can support.
    b_max = np.ceil(min(3.0 * np.sqrt(n), n / 3.0))
    b_opt_capped = float(np.clip(b_opt, 1.0, b_max))

    return {
        'n': n,
        'k_n': k_n,
        'm_hat': m_hat,
        'bandwidth_M': bandwidth_M,
        'significance_bound': significance_bound,
        'long_run_variance': long_run_variance,
        'variance': autocovariance[0],
        'curvature': curvature,
        'b_opt_raw': float(b_opt),
        'b_opt': b_opt_capped,
        'b_max': float(b_max),
        'rule_of_thumb': float(max(1, round(n ** (1 / 3)))),
    }

In [ ]:
def choose_block_length(residual_matrix, percentile=10, verbose=True):
    """Pick one moving-block length for a whole set of reference combinations.

    The holdout draw is shared across every reference combination -- that is what makes
    their prediction errors comparable -- so one block length has to serve all of them.
    politis_white_block_length gives a per-combination answer, and those answers
    disagree wildly (7 to the cap of 43 on the arsenic data), so they have to be reduced
    to a single number.

    That reduction is a *low* quantile rather than the median, and the reason is
    structural. An underfit combination's residuals are not noise: they contain the part
    of the spectrum the model failed to explain, which is smooth and spectrum-shaped.
    Politis-White cannot tell that deterministic misfit from genuine dependence, and
    reads it as very long-range autocorrelation. On the arsenic data every M=1 subset
    pins the cap and rank correlation between fit RMSE and b_opt is +0.54.

    The key point is that this contamination is *one-sided*: unmodeled structure can only
    ever inflate the estimate, never deflate it. So the low order statistics are the
    trustworthy end of the distribution, and the median -- drawn mostly from combinations
    whose residuals are mostly signal -- is meaningless.

    Parameters
    ----------
    residual_matrix : (n_combinations, n) array of full-data residuals, one row per
                      reference combination. This is exactly the all_residuals array
                      do_ref_subsets_moving_block_holdout_bootstrap already builds.
    percentile      : which low quantile to take. The p1/p5/p10/p25 sweep is reported
                      alongside so the choice can be audited rather than trusted.

    Returns
    -------
    dict with 'block_length' (int, the value to use), 'b_opt' (per-combination
    estimates), 'percentile_sweep', 'n_at_cap' and 'rule_of_thumb'.
    """
    residual_matrix = np.atleast_2d(np.asarray(residual_matrix, dtype=float))
    n = residual_matrix.shape[1]

    estimates = np.array([
        politis_white_block_length(row)['b_opt'] for row in residual_matrix
    ])
    b_max = np.ceil(min(3.0 * np.sqrt(n), n / 3.0))

    sweep_percentiles = [1, 5, 10, 25, 50]
    sweep = dict(zip(sweep_percentiles, np.percentile(estimates, sweep_percentiles)))

    block_length = max(1, int(np.round(np.percentile(estimates, percentile))))
    rule_of_thumb = max(1, int(np.round(n ** (1 / 3))))

    if verbose:
        print(f'block length tuned from {len(estimates)} combinations '
              f'(Politis-White, p{percentile}):')
        print('  percentile sweep: ' + '  '.join(f'p{p}={v:.2f}' for p, v in sweep.items()))
        print(f'  {int((estimates >= b_max).sum())} of {len(estimates)} combinations pinned '
              f'the cap ({b_max:.0f}) -- these are underfit, not strongly dependent')
        print(f'  block_length={block_length} (rule of thumb round(n**(1/3))={rule_of_thumb})')

    return {
        'block_length': block_length,
        'b_opt': estimates,
        'percentile': percentile,
        'percentile_sweep': sweep,
        'n_at_cap': int((estimates >= b_max).sum()),
        'b_max': float(b_max),
        'rule_of_thumb': rule_of_thumb,
    }

In [ ]:
# ---------------------------------------------------------------------------
# Tests for politis_white_block_length / choose_block_length
#
# The estimator has no reference implementation here to check against, so the
# tests pin the behavior that makes it trustworthy: it should ask for short
# blocks when there is nothing to preserve, longer blocks as dependence grows,
# never more than the cap, and -- the property the whole aggregate rests on --
# it should not let a structure-dominated series drag the chosen length up.
# ---------------------------------------------------------------------------

def _ar1_series(phi, n=500, seed=0):
    rng = np.random.default_rng(seed)
    innovations = rng.standard_normal(n)
    series = np.zeros(n)
    for i in range(1, n):
        series[i] = phi * series[i - 1] + innovations[i]
    return series


def test_white_noise_wants_short_blocks():
    """Independent data has no dependence to preserve, so blocks should be ~1 point."""
    white_noise = np.random.default_rng(0).standard_normal(198)
    result = politis_white_block_length(white_noise)
    assert result['b_opt'] < 2.0, f"expected b_opt < 2 for white noise, got {result['b_opt']:.3f}"


def test_ar1_block_length_increases_with_dependence():
    """More persistent series need longer blocks to carry their autocorrelation."""
    block_lengths = [politis_white_block_length(_ar1_series(phi))['b_opt']
                     for phi in (0.2, 0.5, 0.8)]
    assert block_lengths == sorted(block_lengths), \
        f'expected block length to increase with phi, got {np.round(block_lengths, 2)}'
    assert block_lengths[0] < block_lengths[-1], \
        f'expected a strict increase from phi=0.2 to phi=0.8, got {np.round(block_lengths, 2)}'


def test_block_length_is_capped():
    """A near-unit-root series would otherwise ask for a block the sample cannot support."""
    result = politis_white_block_length(_ar1_series(0.99, n=198))
    assert result['b_opt'] <= result['b_max'], \
        f"b_opt {result['b_opt']:.2f} exceeded the cap {result['b_max']:.2f}"
    assert result['b_opt'] <= np.ceil(198 / 3)


def test_aggregate_ignores_inflated_estimates():
    """The low quantile must survive rows whose residuals are misfit rather than noise.

    This is the assumption choose_block_length is built on, so it is tested rather
    than asserted in a comment: mixing in structure-dominated rows -- a smooth
    half-cosine, which is what an underfit spectrum's residuals look like -- must
    raise the *median* while leaving the low quantile on the noise-like rows.
    """
    n = 198
    rng = np.random.default_rng(0)
    noise_like = np.array([_ar1_series(0.5, n=n, seed=s) for s in range(30)])
    structure_like = np.array([
        np.cos(np.linspace(0, np.pi, n)) * (1.0 + 0.05 * rng.standard_normal(n))
        for _ in range(30)
    ])

    clean = choose_block_length(noise_like, percentile=10, verbose=False)
    contaminated = choose_block_length(
        np.vstack([noise_like, structure_like]), percentile=10, verbose=False,
    )

    assert contaminated['block_length'] == clean['block_length'], (
        f"low quantile moved from {clean['block_length']} to "
        f"{contaminated['block_length']} when structure-dominated rows were added"
    )
    assert contaminated['percentile_sweep'][50] > clean['percentile_sweep'][50], \
        'expected the median to be inflated by the structure-dominated rows'
    assert contaminated['n_at_cap'] > 0, 'expected the structure-dominated rows to pin the cap'


_block_length_test_fns = [
    test_white_noise_wants_short_blocks,
    test_ar1_block_length_increases_with_dependence,
    test_block_length_is_capped,
    test_aggregate_ignores_inflated_estimates,
]
for _fn in _block_length_test_fns:
    _fn()
    print(f'PASSED: {_fn.__name__}')
print(f'\n{len(_block_length_test_fns)} tests passed')

In [ ]:
from itertools import combinations
from math import comb


def do_ref_subsets_moving_block_holdout_bootstrap(b, A, M, rng, select_holdout_blocks_fn,
                                                  n_bootstrap=1000, block_length='auto'):
    """Bootstrap prediction error for every combination of references at each size in M.

    select_holdout_blocks_fn is called once and its draws are shared across all
    combinations so that per-combination prediction errors are directly comparable.

    Parameters
    ----------
    b                        : (n,) observed spectrum normalized fluorescence values
    A                        : (n, n_refs) interpolated reference spectrum fluorescence values
    M                        : list of combination sizes to evaluate (each between 1 and n_refs)
    rng                      : numpy random Generator
    select_holdout_blocks_fn : callable with signature
                               (n, rng, n_bootstrap=..., block_length=...) ->
                               (holdout_masks, sampled_starts, block_length, n_holdout_blocks)
    n_bootstrap              : number of bootstrap iterations
    block_length             : 'auto' (default) tunes the block length from the residuals
                               with choose_block_length; an int pins it; None leaves the
                               selector on its own round(n ** (1/3)) rule of thumb

    Returns
    -------
    results : dict of numpy arrays, one row per combination across all sizes in M:
        'M'               : (n_combinations,) int
        'ref_indices'     : (n_combinations, max_M) float — column indices into A, NaN for unused
        'bootstrap_coefs' : (n_combinations, n_bootstrap, max_M) float — NaN for unused coefficients
        'bootstrap_pes'   : (n_combinations, n_bootstrap) float
        'coef'            : (n_combinations, max_M) float — full-data NNLS coefficients, NaN for unused
        'fitted'          : (n_combinations, n) float
        'residuals'       : (n_combinations, n) float
        'lags'            : (n_lags+1,) int — ACF lag indices (shared across all combinations)
        'acf_values'      : (n_combinations, n_lags+1) float — ACF of full-data residuals
    holdout_masks    : (n_bootstrap, n) bool array
    sampled_starts   : (n_bootstrap, n_blocks_needed) int array
    block_length     : int
    n_holdout_blocks : int
    """
    n, n_refs = A.shape
    for m in M:
        if not (1 <= m <= n_refs):
            raise ValueError(f'M value {m} must be between 1 and n_refs={n_refs}')

    max_M = max(M)
    combos = [(m, ref_indices) for m in sorted(M) for ref_indices in combinations(range(n_refs), m)]
    n_combinations = len(combos)
    n_lags = min(40, n // 2)

    all_M = np.zeros(n_combinations, dtype=int)
    all_ref_indices = np.full((n_combinations, max_M), np.nan)
    all_coef = np.full((n_combinations, max_M), np.nan)
    all_fitted = np.zeros((n_combinations, n))
    all_residuals = np.zeros((n_combinations, n))

    for i, (m, ref_indices) in enumerate(combos):
        coef, fitted, residuals = fit_nnls(A[:, ref_indices], b)
        all_M[i] = m
        all_ref_indices[i, :m] = ref_indices
        all_coef[i, :m] = coef
        all_fitted[i] = fitted
        all_residuals[i] = residuals

    all_acf_values = np.zeros((n_combinations, n_lags + 1))
    ci_95 = 1.96 / np.sqrt(n)
    max_sig_lags = []

    for i in range(n_combinations):
        lags, acf_values = calculate_acf(all_residuals[i])
        all_acf_values[i] = acf_values
        sig_mask = np.abs(acf_values[1:]) > ci_95
        max_sig_lags.append(int(np.where(sig_mask)[0][-1]) + 1 if sig_mask.any() else 0)

    print(f'Significant autocorrelation lags across all subsets: {min(max_sig_lags)}–{max(max_sig_lags)}')

    # The residuals are already in hand, so the block length can be sized from the
    # dependence they actually carry rather than from n alone. See choose_block_length
    # for why the aggregate over combinations is a low quantile.
    if isinstance(block_length, str):
        assert block_length == 'auto', f'block_length must be an int, None or "auto", got {block_length!r}'
        block_length = choose_block_length(all_residuals)['block_length']

    holdout_masks, sampled_starts, block_length, n_holdout_blocks = \
        select_holdout_blocks_fn(n, rng, n_bootstrap=n_bootstrap, block_length=block_length)

    all_bootstrap_coefs = np.full((n_combinations, n_bootstrap, max_M), np.nan)
    all_bootstrap_pes = np.zeros((n_combinations, n_bootstrap))

    for i, (m, ref_indices) in enumerate(combos):
        bootstrap_coefs, bootstrap_pes = do_moving_block_holdout_bootstrap(
            A[:, ref_indices], b, all_fitted[i], all_residuals[i],
            holdout_masks, sampled_starts, block_length,
        )
        all_bootstrap_coefs[i, :, :m] = bootstrap_coefs
        all_bootstrap_pes[i] = bootstrap_pes

    results = {
        'M': all_M,
        'ref_indices': all_ref_indices,
        'bootstrap_coefs': all_bootstrap_coefs,
        'bootstrap_pes': all_bootstrap_pes,
        'coef': all_coef,
        'fitted': all_fitted,
        'residuals': all_residuals,
        'lags': lags,
        'acf_values': all_acf_values,
    }
    return results, holdout_masks, sampled_starts, block_length, n_holdout_blocks


In [ ]:
def plot_worst_median_best_pe_distributions(results, spectrum_name, ax):
    from matplotlib.patches import Patch

    unique_M = sorted(set(results['M']))
    n_M = len(unique_M)
    colors = plt.cm.tab10(np.linspace(0, 0.4, n_M))
    M_to_color = {m: c for m, c in zip(unique_M, colors)}

    violin_pes = []
    violin_positions = []
    violin_color_list = []
    violin_labels = []

    pos = 0
    for m in unique_M:
        m_indices = np.where(results['M'] == m)[0]
        medians_m = np.array([np.median(results['bootstrap_pes'][i]) for i in m_indices])
        asc = np.argsort(medians_m)  # ascending: asc[0] = best (lowest median PE)
        n_m = len(asc)

        if n_m == 1:
            picks = [(asc[0], 'only')]
        elif n_m == 2:
            picks = [(asc[-1], 'worst'), (asc[0], 'best')]
        else:
            picks = [(asc[-1], 'worst'), (asc[n_m // 2], 'median'), (asc[0], 'best')]

        for local_idx, rank_label in picks:
            violin_pes.append(results['bootstrap_pes'][m_indices[local_idx]])
            violin_positions.append(pos)
            violin_color_list.append(M_to_color[m])
            violin_labels.append(rank_label)
            pos += 1

        pos += 1  # gap between M groups

    parts = ax.violinplot(violin_pes, positions=violin_positions, showmedians=True)
    for idx, color in enumerate(violin_color_list):
        parts['bodies'][idx].set_facecolor(color)
        parts['bodies'][idx].set_alpha(0.7)

    ax.set_xticks(violin_positions)
    ax.set_xticklabels(violin_labels, fontsize=8)
    ax.set_ylabel('Holdout Prediction Error (RMSE)')
    ax.set_xlabel('Reference Subset')
    ax.set_title(f'Bootstrap Prediction Error — Worst / Median / Best per Subset Size — {spectrum_name}')

    legend_elements = [Patch(facecolor=M_to_color[m], alpha=0.7, label=f'M={m}') for m in unique_M]
    ax.legend(handles=legend_elements)


def plot_descending_median_pe(results, axs):
    unique_M = sorted(set(results['M']))
    n_M = len(unique_M)
    colors = plt.cm.tab10(np.linspace(0, 0.4, n_M))
    M_to_color = {m: c for m, c in zip(unique_M, colors)}

    all_pe_vals = []
    for j, m in enumerate(unique_M):
        m_indices = np.where(results['M'] == m)[0]
        m_order = m_indices[np.argsort(np.median(results['bootstrap_pes'][m_indices], axis=1))[::-1]]
        medians = np.median(results['bootstrap_pes'][m_order], axis=1)
        all_pe_vals.extend(medians.tolist())

        axs[j].plot(range(len(m_order)), medians, 's-', color=M_to_color[m], linewidth=1.5, label='median')
        axs[j].set_xlabel('Reference Subset')
        axs[j].set_title(f'M={m}')
        axs[j].legend(fontsize=8)

    pe_ymin = min(all_pe_vals) * 0.999
    pe_ymax = max(all_pe_vals) * 1.001
    for ax in axs:
        ax.set_ylim(pe_ymin, pe_ymax)
    axs[0].set_ylabel('Median Prediction Error (RMSE)')


def plot_ref_subsets_summary(
    results, ref_names, spectrum_name,
    sample_energies, interp_energies, elapsed_time,
    clusterings=None,
):
    """The whole combination search: the reference pool, then the prediction errors.

    clusterings, when given, is {metric: cluster_reference_spectra(...)} over the pool.
    The trees are drawn first and unhighlighted -- this figure covers every combination at
    once, so there is no one subset to mark, and the tree is here as context for what the
    references are. Where a *selected* combination sits is marked by
    plot_bootstrap_summary, one figure per combination.
    """
    import pandas as pd

    n_combinations = len(results['M'])
    unique_M = sorted(set(results['M']))
    n_M = len(unique_M)

    if clusterings:
        # 11 x 9 inches per tree is what plot_cluster_metric_comparison settled on for a
        # 24-name leaf column; anything narrower clips the file names
        fig, axs = plt.subplots(1, len(clusterings), figsize=(11 * len(clusterings), 9))
        for ax, (metric, clustering) in zip(np.atleast_1d(axs), clusterings.items()):
            plot_reference_dendrogram(
                clustering, ax=ax, legend_loc='inside',
                title=f'{metric} distance — {spectrum_name}',
            )
        plt.tight_layout()
        plt.show()

    fig, ax = plt.subplots(figsize=(max(10, 5 * n_M), 5))
    plot_worst_median_best_pe_distributions(results, spectrum_name, ax)
    plt.tight_layout()
    plt.show()

    fig, axs = plt.subplots(1, n_M, figsize=(max(10, 5 * n_M), 4))
    plot_descending_median_pe(results, np.atleast_1d(axs))
    plt.tight_layout()
    plt.show()

    rows = {
        'Sample':                      spectrum_name,
        'Sample energy points':        len(sample_energies),
        'Sample energy range':         f'{sample_energies[0]:.1f}–{sample_energies[-1]:.1f} eV',
        'References':                  len(ref_names),
        'Interpolation energy range':  f'{interp_energies[0]:.1f}–{interp_energies[-1]:.1f} eV',
        'Interpolation energy points': len(interp_energies),
        **{f'Subsets (M={m})': int((results['M'] == m).sum()) for m in unique_M},
        'Total bootstrap time':        f'{elapsed_time:.1f} s',
        'Avg time per subset':         f'{elapsed_time / n_combinations:.2f} s',
    }
    display(pd.DataFrame({'Value': rows}))


In [ ]:
def _ordinal(n):
    if 11 <= n % 100 <= 13:
        suffix = 'th'
    else:
        suffix = {1: 'st', 2: 'nd', 3: 'rd'}.get(n % 10, 'th')
    return f'{n}{suffix}'


def plot_best_subset_bootstrap_summaries(results, ref_names, energies, b, n_bootstrap, spectrum_name,
                                          N=3, clusterings=None):
    """Call plot_bootstrap_summary for the top N best subsets of each subset size.

    "Best" is defined as lowest median bootstrap prediction error.

    Parameters
    ----------
    results       : dict returned by do_ref_subset_moving_block_holdout_bootstrap
    ref_names     : list of reference spectrum names
    energies      : energy grid array
    b             : observed spectrum values
    n_bootstrap   : number of bootstrap iterations (passed to plot_bootstrap_summary)
    spectrum_name : str — used in plot titles
    N             : number of best subsets per subset size to plot (default 3)
    clusterings   : optional {metric: clustering} over the whole reference pool, forwarded
                    to plot_bootstrap_summary so each figure shows where its combination
                    sits in the reference trees

    Displays the figures plot_bootstrap_summary returns, best subset first within each
    subset size, and returns nothing. A caller wanting the figures themselves should call
    plot_bootstrap_summary directly.
    """
    for m in sorted(set(results['M'])):
        m_indices = np.where(results['M'] == m)[0]
        medians_m = np.array([np.median(results['bootstrap_pes'][i]) for i in m_indices])
        for rank, local_idx in enumerate(np.argsort(medians_m)[:N], start=1):
            i = m_indices[local_idx]
            ref_idx = [int(j) for j in results['ref_indices'][i, :m]]
            subset_ref_names = [ref_names[j] for j in ref_idx]
            title_prefix = f'{_ordinal(rank)} best {m}-component fit'
            for fig in plot_bootstrap_summary(
                energies, b,
                results['fitted'][i],
                results['residuals'][i],
                results['lags'],
                results['acf_values'][i],
                results['bootstrap_coefs'][i, :, :m],
                results['bootstrap_pes'][i],
                results['coef'][i, :m],
                subset_ref_names,
                spectrum_name=spectrum_name,
                n_bootstrap=n_bootstrap,
                title_prefix=title_prefix,
                clusterings=clusterings,
            ):
                display(fig)

In [ ]:
import time


def version_block_length_arm(
    label, select_holdout_blocks_fn, b, A, M, n_bootstrap, block_length, seed,
):
    """One (holdout version, block length) arm of the v1-v5 prediction error comparison.

    Runs in a worker process (see run_arms), so it takes the selector as an argument and
    returns what it printed. The full bootstrap result comes back because the caller draws
    a prediction error distribution from each arm.

    Parameters
    ----------
    label                    : str -- how the arm is named in the figure it feeds
    select_holdout_blocks_fn : callable -- one of the v1-v5 selectors
    b, A                     : response vector and design matrix, shared by every arm
    M                        : list of subset sizes to evaluate
    n_bootstrap              : int -- iterations
    block_length             : int -- pinned, not tuned; the comparison is about geometry
    seed                     : int -- the arm's whole source of randomness

    Returns
    -------
    dict -- 'label', 'results', 'elapsed' and 'printed'
    """
    printed = io.StringIO()
    t0 = time.perf_counter()
    with contextlib.redirect_stdout(printed):
        results, _, _, _, _ = do_ref_subsets_moving_block_holdout_bootstrap(
            b, A, M=list(M), rng=np.random.default_rng(seed=seed),
            select_holdout_blocks_fn=select_holdout_blocks_fn,
            n_bootstrap=n_bootstrap, block_length=block_length,
        )
    return {'label': label, 'results': results, 'elapsed': time.perf_counter() - t0,
            'printed': printed.getvalue()}


def plot_compare_select_holdout_blocks_functions(sample_spectrum, reference_spectra,
                                                 block_lengths=(6, 10), n_jobs=N_JOBS):
    # The v1-v5 comparison is about holdout *geometry*, and which version has the best
    # geometry depends on the block length: a fixed-length grid tiles the array exactly
    # only when the block length divides n. So both are pinned here rather than tracking
    # the tuned default -- 6, the rule of thumb, which divides n=198, and 10, the tuned
    # value, which leaves a remainder of 8. See the write-up above.
    n_bootstrap = 1000
    
    valid_energies, interpolated_ref_spectra, b, *_ = interpolate_references_at_sample_energies(
        reference_spectra=reference_spectra,
        sample_spectrum=sample_spectrum
    )
    
    holdout_fn_versions = [
        ('v1 (aligned)',        select_holdout_blocks),
        ('v2 (shifted)',        select_holdout_blocks_v2),
        ('v3 (circular shift)', select_holdout_blocks_v3),
        ('v4 (random lengths)', select_holdout_blocks_v4),
        ('v5 (alternating)',    select_holdout_blocks_v5),
    ]
    
    # Ten independent, identically seeded arms; they run on every core and come back in
    # task order, each carrying the report it printed.
    arms = run_arms([
        (version_block_length_arm, {
            'label': f'{label}, L={block_length}', 'select_holdout_blocks_fn': fn,
            'b': b, 'A': interpolated_ref_spectra, 'M': [1, 2, 3],
            'n_bootstrap': n_bootstrap, 'block_length': block_length, 'seed': 42,
        })
        for block_length in block_lengths
        for label, fn in holdout_fn_versions
    ], n_jobs=n_jobs)

    all_version_results = []
    for arm in arms:
        print(f'\n=== {arm["label"]} ===')
        print(arm['printed'], end='')
        all_version_results.append((arm['label'], arm['results'], arm['elapsed']))
    
    unique_M = sorted(set(all_version_results[0][1]['M']))
    n_M = len(unique_M)
    
    for label, results, elapsed in all_version_results:
        fig, ax = plt.subplots(figsize=(max(10, 5 * n_M), 5))
        plot_worst_median_best_pe_distributions(results, f'{sample_spectrum.file_name} [{label}]', ax)
        plt.tight_layout()
        plt.show()

sample_spectrum = filter_spectra_by_name(sample_spectra_list, "OTT3_55*")[0]
print(f'sample spectrum: {sample_spectrum.file_name}')
reference_spectra = filter_spectra_by_name(
    reference_spectra_list,
    "Arsenopyrite_Jul*", "orpiment_all*", "arsenate*_diop*", "Fh2l**"
)
ref_names = [r.file_name for r in reference_spectra]
print(f"reference spectra:\n{chr(10).join(chr(9) + r for r in ref_names)}")
    
plot_compare_select_holdout_blocks_functions(sample_spectrum, reference_spectra)

In [ ]:
import time

def do_fits_and_plot_summaries(sample_spectrum, reference_spectra):
    print(f'sample spectrum: {sample_spectrum.file_name}')
    ref_names = [r.file_name for r in reference_spectra]
    print(f"reference spectra:\n{chr(10).join(chr(9) + r for r in ref_names)}")
    
    n_bootstrap = 1000

    valid_energies_24, interpolated_ref_spectra, b, *_ = interpolate_references_at_sample_energies(
        reference_spectra=reference_spectra,
        sample_spectrum=sample_spectrum
    )
    
    t0 = time.perf_counter()
    results, holdout_masks, sampled_starts, block_length, n_holdout_blocks = \
        do_ref_subsets_moving_block_holdout_bootstrap(
            b, interpolated_ref_spectra, M=[1, 2, 3], rng=np.random.default_rng(seed=42),
            # v5, not v3: the block length is tuned from the residuals and does not
            # generally divide n, and v5 is the version whose coverage is uniform either
            # way. See "Development of select_holdout_blocks v1-v5" above.
            select_holdout_blocks_fn=select_holdout_blocks_v5,
            n_bootstrap=n_bootstrap,
        )
    elapsed = time.perf_counter() - t0

    # Cluster the pool once, here, rather than inside the plotting functions: they draw the
    # same two trees on every figure -- up to nine of them -- and this way one place fixes
    # the seed and the parameters. verbose=False keeps the clustering's own report out of a
    # figure-building path.
    reference_clusterings = {
        metric: cluster_reference_spectra(
            interpolated_ref_spectra, ref_names, np.random.default_rng(42),
            metric=metric, verbose=False,
        )
        for metric in ('correlation', 'cosine')
    }

    plot_ref_subsets_summary(
        results, ref_names, spectrum_name=sample_spectrum.file_name,
        sample_energies=sample_spectrum.data_df.index.values,
        interp_energies=valid_energies_24,
        elapsed_time=elapsed,
        clusterings=reference_clusterings,
    )
    
    plot_best_subset_bootstrap_summaries(
        results, ref_names, valid_energies_24, b, n_bootstrap,
        spectrum_name=sample_spectrum.file_name,
        clusterings=reference_clusterings,
    )

    # Hand back the design matrix and the results so the reference clustering below works on
    # exactly what was fitted, rather than interpolating the same spectra a second time --
    # and the block length, holdout count and elapsed time, which the fit produces and would
    # otherwise be lost. block_length especially: under block_length='auto' it is tuned from
    # the residuals, so it is a result rather than a setting, and write_fit_results records it.
    return {
        'results': results,
        'A': interpolated_ref_spectra,
        'b': b,
        'ref_names': ref_names,
        'valid_energies': valid_energies_24,
        'sample_energies': sample_spectrum.data_df.index.values,
        'spectrum_name': sample_spectrum.file_name,
        'clusterings': reference_clusterings,
        'block_length': block_length,
        'n_holdout_blocks': n_holdout_blocks,
        'elapsed_time': elapsed,
        'seed': 42,
    }

sample_spectrum = filter_spectra_by_name(sample_spectra_list, "OTT3_55*")[0]
reference_spectra = filter_spectra_by_name(
    reference_spectra_list,
    "Arsenopyrite_Jul*", "orpiment_all*", "arsenate*_diop*", "*"
)

fit_summary = do_fits_and_plot_summaries(sample_spectrum, reference_spectra)

In [ ]:
# The fit above hands back its design matrix and results, so the tree is built on exactly
# the matrix that was fitted rather than on a second interpolation of the same spectra.
reference_clustering = cluster_reference_spectra(
    fit_summary['A'], fit_summary['ref_names'], np.random.default_rng(42),
)

_, highlighted_groups = plot_reference_dendrogram(
    reference_clustering, highlight=best_subsets_by_size(fit_summary['results']),
)
plt.show()

## Comparing confidence interval estimators

The combination search ranks reference subsets by their median holdout prediction error,
and `plot_best_subset_bootstrap_summaries` reports the top three at each subset size as
"1st best", "2nd best", "3rd best". That ranking is a bare point estimate: it orders the
subsets but says nothing about whether the gaps between them are real. The v1–v5 findings
above already showed prediction error failing to separate five holdout selectors whose
intervals all overlapped, so the question is a live one here too.

Answering it needs a confidence interval on each subset's prediction error — a statement of
how precisely 1,000 bootstrap draws pin down the median, as opposed to how widely those
draws scatter. The two are different quantities, and the interval is the narrower one by
roughly a factor of `sqrt(n_bootstrap)`.

Three estimators are already in reach, and this repository currently calls all three
"95% CI":

- **percentile bootstrap of the median** — `bootstrap_ci` above, used for the v1–v5
  comparison: resample the draws, recompute the median, take the middle 95%.
- **bias-corrected and accelerated (BCa)** — `scipy.stats.bootstrap` at its defaults, which
  is what `mrfitty/prediction_error_fit.py` uses to choose a component count. It corrects
  the percentile interval for skew in the draws.
- **order statistic** — a distribution-free interval read straight off the sorted draws. No
  resampling, no random numbers.

They are not interchangeable, and the selector in the next section calls one of them
thousands of times, so the choice is worth measuring rather than inheriting. The study below
asks two things: do they *disagree* on real data from the fit above, and which one actually
delivers its advertised 95% coverage on data shaped like this.

In [ ]:
# Three estimators of a 95% confidence interval on a median, behind one signature:
# (values, rng) -> (lo, hi). The selector below needs exactly one of them.
#
# What they estimate is the precision of the median, not the spread of the draws. The
# draws scatter because every bootstrap iteration held out a different set of energies;
# the interval says how well 1000 such draws locate the number in the middle, which is
# tighter than the draws themselves by roughly sqrt(n).

import warnings

import scipy.stats


def median_ci_percentile_bootstrap(values, rng, n_resamples=5000):
    """Percentile bootstrap of the median: resample the draws, take the middle 95%.

    Reuses bootstrap_ci from the v1-v5 comparison above so the notebook keeps one
    definition of this interval rather than two that drift apart.
    """
    lo, hi = bootstrap_ci(np.asarray(values), np.median, n_resamples=n_resamples, rng=rng)
    return float(lo), float(hi)


def median_ci_scipy_bca(values, rng):
    """scipy's bias-corrected and accelerated interval, at the defaults.

    The same call mrfitty/prediction_error_fit.py makes to choose a component count, so
    picking this one would make the notebook and the package agree exactly. BCa shifts and
    stretches the percentile interval to correct for skew in the draws, which is the reason
    to prefer it -- prediction errors are bounded below by zero and have a long right tail.
    """
    # BCa fails on these arrays often enough that the warning would bury everything else
    # the study prints -- and the failures are counted and reported rather than ignored, so
    # silencing the warning loses nothing. See the findings below for why they happen.
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        result = scipy.stats.bootstrap(
            data=(np.asarray(values),), statistic=np.median, rng=rng,
        )
    return float(result.confidence_interval.low), float(result.confidence_interval.high)


def median_ci_order_statistic(values, rng=None):
    """A distribution-free interval on the median, read off the sorted draws.

    The number of draws below the true median is Binomial(n, 1/2), so counting
    1.96 * sqrt(n) / 2 draws out from the middle in each direction brackets it about 95% of
    the time whatever shape the draws have. No resampling and no random numbers, which is
    why `rng` is accepted and ignored -- it exists so the three estimators can be called
    interchangeably.

    The endpoints are actual draws, so the interval cannot be narrower than the gap between
    neighboring draws. At n = 1000 that granularity is far below the interval's width.
    """
    sorted_values = np.sort(np.asarray(values))
    n = len(sorted_values)
    half_width = 1.96 * np.sqrt(n) / 2.0
    lo_index = max(0, int(np.floor(n / 2.0 - half_width)))
    hi_index = min(n - 1, int(np.ceil(n / 2.0 + half_width)))
    return float(sorted_values[lo_index]), float(sorted_values[hi_index])


MEDIAN_CI_METHODS = {
    'percentile bootstrap': median_ci_percentile_bootstrap,
    'scipy BCa': median_ci_scipy_bca,
    'order statistic': median_ci_order_statistic,
}

In [ ]:
# The study behind the choice of estimator. Two questions, because agreement alone cannot
# settle it: three estimators of the same quantity can agree and all three be wrong.

import time

import pandas as pd


def _representative_combinations(results):
    """(label, row index, best row index at that size) for four fits spanning each size.

    The runner-up is the point that matters: it is the closest call the selector below ever
    has to make, and an estimator that disagrees with the others anywhere will disagree
    there. Best, median and worst are included so the comparison is not read off one
    borderline case -- the estimators have the easiest job where the draws are tightest.
    """
    targets = []
    for m in sorted(set(results['M'])):
        m_indices = np.where(results['M'] == m)[0]
        order = m_indices[np.argsort(np.median(results['bootstrap_pes'][m_indices], axis=1))]
        best = int(order[0])
        picks = [('best', order[0]), ('runner-up', order[min(1, len(order) - 1)]),
                 ('median', order[len(order) // 2]), ('worst', order[-1])]
        targets.extend((f'M={m} {rank}', int(i), best) for rank, i in picks)
    return targets


def ci_method_agreement(results, seed=0):
    """Every estimator applied to the same real arrays: do they disagree, and does it matter?

    Each row is one (combination, estimator) pair, carrying both intervals the selector
    below deals with -- the combination's own interval on its median prediction error, and
    the interval on its paired difference against the best combination at its size. The
    `tied` column is the verdict that interval produces, so a disagreement between
    estimators shows up as a changed decision rather than a changed decimal.
    """
    rows = []
    for label, i, best_i in _representative_combinations(results):
        prediction_errors = results['bootstrap_pes'][i]
        differences = prediction_errors - results['bootstrap_pes'][best_i]
        for method_index, (method_name, median_ci) in enumerate(MEDIAN_CI_METHODS.items()):
            # one rng per (combination, estimator), seeded from position rather than from
            # str.hash, which is salted per process and would make the table irreproducible
            rng = np.random.default_rng([seed, i, method_index])
            pe_lo, pe_hi = median_ci(prediction_errors, rng)
            d_lo, d_hi = median_ci(differences, rng)
            rows.append({
                'subset': label,
                'is_best': i == best_i,
                'method': method_name,
                'pe_median': float(np.median(prediction_errors)),
                'pe_ci_lo': pe_lo,
                'pe_ci_hi': pe_hi,
                'pe_ci_width': pe_hi - pe_lo,
                'd_median': float(np.median(differences)),
                'd_ci_lo': d_lo,
                'd_ci_hi': d_hi,
                'd_ci_width': d_hi - d_lo,
                # NaN propagates to False here, which is the honest reading: an estimator
                # that cannot produce an interval has not called anything tied
                'tied': bool(d_lo <= 0.0 <= d_hi),
            })
    return pd.DataFrame(rows)


def ci_coverage_arm(values, arm_label, method_name, median_ci, n_replicates, seed):
    """How often one estimator's interval covers a median that is known exactly.

    The observed draws are treated as the population. Its median is then known without
    assuming any distribution, and resampling n draws from it reproduces the real,
    right-skewed shape of the data the estimator will be used on -- which a normal or
    lognormal stand-in would not. An estimator that deserves the label "95%" should cover
    that known median in about 95% of the replicates.

    Runs in a worker process (see run_arms), so the estimator arrives as an argument rather
    than being looked up in a global, and nothing is printed.

    Parameters
    ----------
    values       : ndarray -- the draws standing in as the population
    arm_label    : str -- which array this is, carried through to the figure
    method_name  : str -- which estimator, carried through to the figure
    median_ci    : callable (values, rng) -> (lo, hi)
    n_replicates : int -- resampled data sets; the coverage estimate's own noise is
                   sqrt(0.95 * 0.05 / n_replicates), about 0.010 at 500
    seed         : int -- the arm's whole source of randomness

    Returns
    -------
    dict -- 'arm', 'method', 'coverage', 'failed', 'mean_width', 'n_replicates', 'truth'.
        'failed' is the fraction of replicates where the estimator returned no interval at
        all. Those count against coverage, because an estimator that cannot answer has not
        covered anything, and they are reported separately so the two failure modes -- an
        interval that misses, and no interval -- are not read as one number.
    """
    values = np.asarray(values)
    truth = float(np.median(values))
    n = len(values)
    rng = np.random.default_rng(seed)

    covered = 0
    failed = 0
    widths = np.empty(n_replicates)
    for replicate in range(n_replicates):
        resampled = values[rng.integers(0, n, size=n)]
        lo, hi = median_ci(resampled, rng)
        if not (np.isfinite(lo) and np.isfinite(hi)):
            failed += 1
            widths[replicate] = np.nan
            continue
        covered += lo <= truth <= hi
        widths[replicate] = hi - lo

    return {
        'arm': arm_label,
        'method': method_name,
        'coverage': covered / n_replicates,
        'failed': failed / n_replicates,
        'mean_width': float(np.nanmean(widths)) if failed < n_replicates else np.nan,
        'n_replicates': n_replicates,
        'truth': truth,
    }


def ci_method_timing(values, n_repeats=10, seed=0):
    """Seconds per interval, measured serially so the numbers are not distorted by load."""
    rows = []
    for method_name, median_ci in MEDIAN_CI_METHODS.items():
        rng = np.random.default_rng(seed)
        start = time.perf_counter()
        for _ in range(n_repeats):
            median_ci(values, rng)
        rows.append({
            'method': method_name,
            'seconds_per_interval': (time.perf_counter() - start) / n_repeats,
        })
    return pd.DataFrame(rows)


def compare_median_ci_methods(results, seed=0, n_replicates=500, n_jobs=N_JOBS):
    """Measure the three estimators on the fit above: agreement, coverage, and cost.

    Coverage is run for two arms, because the selector below uses these estimators on two
    differently shaped arrays: prediction errors, which are positive and right-skewed, and
    paired differences between two subsets' prediction errors, which are roughly symmetric
    and sit near zero. An estimator can be well calibrated on one and not the other.

    Returns
    -------
    dict of pandas.DataFrame -- 'agreement', 'coverage', 'timing', plus 'n_combinations'
    """
    targets = _representative_combinations(results)
    best_at_largest_m = targets[-1][2]          # every pick at a size carries its best
    runner_up = targets[-3][1]                  # the 'runner-up' pick at that size
    coverage_arrays = {
        'prediction error': results['bootstrap_pes'][best_at_largest_m],
        'paired difference': (results['bootstrap_pes'][runner_up]
                              - results['bootstrap_pes'][best_at_largest_m]),
    }

    arms = run_arms([
        (ci_coverage_arm, {
            'values': values, 'arm_label': arm_label, 'method_name': method_name,
            'median_ci': median_ci, 'n_replicates': n_replicates,
            'seed': seed + 1000 * arm_index + method_index,
        })
        for arm_index, (arm_label, values) in enumerate(coverage_arrays.items())
        for method_index, (method_name, median_ci) in enumerate(MEDIAN_CI_METHODS.items())
    ], n_jobs=n_jobs)

    return {
        'agreement': ci_method_agreement(results, seed=seed),
        'coverage': pd.DataFrame(arms),
        'timing': ci_method_timing(coverage_arrays['prediction error'], seed=seed),
        'n_combinations': len(results['M']),
    }

In [ ]:
CI_METHOD_COLORS = {
    'percentile bootstrap': 'steelblue',
    'scipy BCa': 'darkorange',
    'order statistic': 'mediumseagreen',
}


def plot_ci_method_intervals(agreement, column_prefix, ax, zero_line=False):
    """One group of three intervals per combination, one interval per estimator.

    column_prefix selects which pair of intervals to draw: 'pe' for each combination's own
    interval on its median prediction error, 'd' for the interval on its paired difference
    against the best combination at its size.
    """
    subsets = list(dict.fromkeys(agreement['subset']))
    methods = list(CI_METHOD_COLORS)
    # the three estimators of one combination are nudged apart so their intervals can be
    # compared without overplotting
    offsets = np.linspace(-0.22, 0.22, len(methods))

    for offset, method in zip(offsets, methods):
        rows = agreement[agreement['method'] == method].set_index('subset').loc[subsets]
        center = rows[f'{column_prefix}_median'].to_numpy()
        lo = rows[f'{column_prefix}_ci_lo'].to_numpy()
        hi = rows[f'{column_prefix}_ci_hi'].to_numpy()
        ax.errorbar(
            np.arange(len(subsets)) + offset, center,
            yerr=[center - lo, hi - center],
            fmt='o', color=CI_METHOD_COLORS[method], capsize=4,
            linewidth=1.5, markersize=5, linestyle='none', label=method,
        )

    if zero_line:
        ax.axhline(0.0, color='red', linestyle='--', linewidth=1.0, label='no difference')
    ax.set_xticks(np.arange(len(subsets)))
    ax.set_xticklabels(subsets, rotation=30, ha='right', fontsize=8)
    ax.legend(fontsize=8)


def plot_ci_method_comparison(comparison, spectrum_name):
    """The estimator study as figures: do they disagree, and is any of them calibrated?

    Parameters
    ----------
    comparison    : dict returned by compare_median_ci_methods
    spectrum_name : str -- used in the suptitles

    Returns
    -------
    tuple of matplotlib.figure.Figure
        Two figures: the intervals the estimators produce on real arrays, and their
        measured coverage, width and cost. Both are closed before being returned, so the
        caller displays them -- `for fig in plot_ci_method_comparison(...): display(fig)`.
    """
    agreement = comparison['agreement']
    coverage = comparison['coverage']
    timing = comparison['timing']
    methods = list(CI_METHOD_COLORS)

    # ---- what the three estimators say about the same arrays
    fig_intervals, (ax_pe, ax_diff) = plt.subplots(1, 2, figsize=(16, 5))
    plot_ci_method_intervals(agreement, 'pe', ax_pe)
    ax_pe.set_ylabel('Median Prediction Error (RMSE)')
    ax_pe.set_xlabel('Reference Subset')
    ax_pe.set_title("95% CI of each subset's own median prediction error")

    plot_ci_method_intervals(agreement, 'd', ax_diff, zero_line=True)
    ax_diff.set_ylabel('Median Paired Difference in PE')
    ax_diff.set_xlabel('Reference Subset')
    # an interval that crosses the red line calls the subset tied with the best one, so a
    # disagreement between estimators is visible here as a changed decision
    ax_diff.set_title('95% CI of the paired difference against the best subset at that size')

    fig_intervals.suptitle(
        f'Confidence interval estimators on the same draws — {spectrum_name}', fontsize=13,
    )
    fig_intervals.tight_layout(rect=[0, 0, 1, 0.93])

    # ---- whether any of them is calibrated, and what it costs
    fig_quality, (ax_coverage, ax_width, ax_cost) = plt.subplots(1, 3, figsize=(18, 5))
    arms = list(dict.fromkeys(coverage['arm']))
    bar_width = 0.8 / len(arms)
    arm_hatches = {arm: hatch for arm, hatch in zip(arms, ('', '//'))}

    for arm_index, arm in enumerate(arms):
        rows = coverage[coverage['arm'] == arm].set_index('method').loc[methods]
        x = np.arange(len(methods)) + (arm_index - (len(arms) - 1) / 2) * bar_width
        n_replicates = int(rows['n_replicates'].iloc[0])
        # the coverage estimate has noise of its own: a perfectly calibrated estimator
        # measured over n_replicates still scatters by sqrt(p (1 - p) / n_replicates), so
        # the error bar is what says whether a bar is really off 0.95
        coverage_error = np.sqrt(0.95 * 0.05 / n_replicates)
        ax_coverage.bar(
            x, rows['coverage'], bar_width, yerr=coverage_error, capsize=3,
            color=[CI_METHOD_COLORS[m] for m in methods], alpha=0.85,
            hatch=arm_hatches[arm], edgecolor='white', label=arm,
        )
        ax_width.bar(
            x, rows['mean_width'], bar_width,
            color=[CI_METHOD_COLORS[m] for m in methods], alpha=0.85,
            hatch=arm_hatches[arm], edgecolor='white', label=arm,
        )

    ax_coverage.axhline(0.95, color='red', linestyle='--', label='nominal 95%')
    ax_coverage.set_ylim(0.80, 1.0)
    ax_coverage.set_ylabel('Measured coverage')
    ax_coverage.set_title(
        f'Coverage of a known median ({int(coverage["n_replicates"].iloc[0])} replicates)'
    )

    ax_width.set_ylabel('Mean interval width')
    # coverage on its own rewards an interval for being wide, so the two panels are read
    # together: the estimator to want is the narrowest one still sitting on 0.95
    ax_width.set_title('Mean interval width (narrower is better, at equal coverage)')

    n_combinations = comparison['n_combinations']
    seconds = timing.set_index('method').loc[methods, 'seconds_per_interval']
    # the selector builds two intervals per combination, its own and its paired difference
    ax_cost.bar(np.arange(len(methods)), seconds * 2 * n_combinations / 60.0,
                color=[CI_METHOD_COLORS[m] for m in methods], alpha=0.85, edgecolor='white')
    ax_cost.set_ylabel('Minutes')
    ax_cost.set_yscale('log')
    ax_cost.set_title(f'Cost of two intervals for each of {n_combinations} combinations')

    for ax in (ax_coverage, ax_width, ax_cost):
        ax.set_xticks(np.arange(len(methods)))
        ax.set_xticklabels(methods, rotation=15, ha='right', fontsize=9)
    ax_coverage.legend(fontsize=8)
    ax_width.legend(fontsize=8)

    fig_quality.suptitle('Calibration, width and cost of the three estimators', fontsize=13)
    fig_quality.tight_layout(rect=[0, 0, 1, 0.93])

    figures = (fig_intervals, fig_quality)
    for fig in figures:
        plt.close(fig)
    return figures

In [ ]:
# Reuses the fit from the section above rather than refitting: the study is about the
# estimators, so it wants exactly the draws the selector below will be given.
ci_comparison = compare_median_ci_methods(fit_summary['results'], seed=7, n_replicates=500)

for fig in plot_ci_method_comparison(ci_comparison, fit_summary['spectrum_name']):
    display(fig)

display(ci_comparison['coverage'])

timing = ci_comparison['timing'].copy()
# what the selector below would spend: one interval for each combination's own prediction
# error and one for its paired difference
timing['minutes for the whole search'] = (
    timing['seconds_per_interval'] * 2 * ci_comparison['n_combinations'] / 60.0
)
display(timing)

# A comparison of the best subset with itself is a column of exact zeros. It is in the
# table because what the estimators do with it is the most visible difference between them,
# but it is not a case any of them is really being asked to judge, so it is counted apart
# from the comparisons that carry information.
agreement = ci_comparison['agreement']
verdicts = agreement.pivot(index='subset', columns='method', values='tied')
degenerate = sorted(set(agreement[agreement['is_best']]['subset']))
real = verdicts.drop(index=degenerate)

print(f'{int((real.nunique(axis=1) > 1).sum())} of {len(real)} genuine comparisons get a '
      f'different tied/not-tied verdict depending on the estimator.')
print(f'{int((verdicts.loc[degenerate].nunique(axis=1) > 1).sum())} of {len(degenerate)} '
      f'self-comparisons do, where an estimator either returns zero or returns nothing:')
display(verdicts)

### Findings

**The three estimators agree about the data and disagree about nothing that matters — except
that one of them frequently declines to answer.**

*They produce the same intervals.* On every non-degenerate comparison above, the three
intervals differ in the fourth significant figure and their mean widths match to within about
1%. No tied/not-tied verdict changes because of the estimator. Whatever else is uncertain
here, the choice between these three is not where the uncertainty lives.

*Two of them are calibrated; BCa often fails outright.* Measured against a median known
exactly, the percentile bootstrap covers about 94–95% and the order statistic about 95–96%,
both within the ±1% noise of a 500-replicate estimate, and neither ever fails to return an
interval. `scipy.stats.bootstrap` returns NaN on a substantial share of these inputs — a few
percent of the prediction error arrays and considerably more of the paired differences.

The reason is worth knowing, because it is a property of this data rather than a bug. BCa
estimates an acceleration constant by jackknifing, and when the jackknife values are all
equal that estimate divides by zero. A bootstrap prediction error array is full of ties —
each iteration's value is an RMSE over a resampled set of held-out points, so the same value
recurs — and the median of a heavily tied sample is often unmoved by dropping any one point.
The degenerate case is the extreme version of this: the best subset's difference against
itself is a column of exact zeros, and BCa returns NaN on it every time. `peci_tie_table`
special-cases that comparison rather than relying on any estimator to handle it.

*So the choice comes down to cost.* Two intervals per combination over the full search costs
minutes for either resampling estimator and milliseconds for the order statistic — a factor
of roughly ten thousand, for the same answer. `peci_tie_table` therefore defaults to
`median_ci_order_statistic`. This is not a claim that it is the best estimator of a median in
general; it is that on these arrays it is calibrated, it never fails, it agrees with the
alternatives, and it is free.

The cost matters less than it looks like it should, though, because of what the next section
finds: the interval this estimator produces is not the one that should be deciding ties.

## Selecting subsets by prediction error confidence interval

`plot_best_subset_bootstrap_summaries` reports the three lowest median prediction errors at
each subset size as "1st best", "2nd best" and "3rd best". The section below reports
something weaker and more defensible instead: the subsets whose prediction error cannot be
told apart from the lowest one. Those are equally good, and presenting them in rank order
would be reading noise as a result.

The comparison is *paired*, and that is worth being explicit about because it is the whole
reason a sharp answer is available.
`do_ref_subsets_moving_block_holdout_bootstrap` draws the holdout blocks once and scores
every combination on those same draws, so on iteration *k* every subset was asked to predict
the same held-out energies. Subtracting two subsets' prediction errors iteration by
iteration therefore cancels whatever made iteration *k* easy or hard for both of them — an
unlucky holdout that lands on the whiteline hurts every subset at once — and what remains is
the difference between the subsets themselves.

Comparing each subset's own interval against the best subset's interval for overlap would
throw that away. Two subsets can have thoroughly overlapping intervals while one of them
loses on every single iteration, and the overlap test would call that a tie.

What remains is to choose what interval to put around those differences, and that turns out
to matter more than the choice of estimator did.

In [ ]:
# Deciding when two subsets are equally good.
#
# Both rules below work on the same paired differences and return the same three things --
# (tied, lo, hi) -- so the table and the figures do not care which one was used. They
# differ in what interval they put around the difference, and the sensitivity study below
# shows that the difference between them is not cosmetic: one of them reports the data and
# the other partly reports how long the bootstrap was allowed to run.


def tie_by_paired_median_ci(differences, median_ci, rng):
    """Tied when a confidence interval on the *median* difference contains zero.

    Not the default; tie_by_paired_distribution is. Kept because it is the more literal
    reading of "compare the prediction error confidence intervals", and because it is the
    comparison mrfitty/prediction_error_fit.py makes when it chooses a component count.

    This asks how precisely the center of the difference distribution is known. Its width
    falls as 1 / sqrt(n_bootstrap), because more iterations locate a median more precisely
    -- so it measures how long the bootstrap ran as much as it measures the data, and
    running longer will eventually separate any two subsets that differ at all. See
    plot_tie_rule_sensitivity.
    """
    lo, hi = median_ci(differences, rng)
    return bool(lo <= 0.0 <= hi), lo, hi


def tie_by_paired_distribution(differences, median_ci=None, rng=None):
    """Tied when the middle 95% of the differences themselves straddles zero. The default.

    What the distribution is
    ------------------------
    `differences` holds one number per bootstrap iteration:

        differences[k] = bootstrap_pes[subset][k] - bootstrap_pes[best][k]

    and each `bootstrap_pes[...][k]` is that subset's holdout prediction error on
    iteration k -- the RMSE of its fit against the measured spectrum at the energies
    iteration k held out (see do_moving_block_holdout_bootstrap). Because
    do_ref_subsets_moving_block_holdout_bootstrap draws the holdout blocks once and scores
    every combination on those same draws, entry k of both arrays refers to the same held
    out energies, so the subtraction is a like-for-like comparison rather than a difference
    of two independent numbers.

    One entry is therefore one answer to "on this particular set of held out energies, by
    how much did this subset predict worse than the best one?" -- negative where it
    predicted better. The n_bootstrap entries together are the distribution of that answer
    over the holdout draws, and its spread is how much the answer depends on which energies
    happened to be held out.

    How the interval is taken
    ------------------------
    The 2.5th and 97.5th percentiles of those entries, via np.percentile, which sorts them
    and interpolates linearly between the two order statistics bracketing each percentile.
    This is the range the differences themselves occupy, *not* a confidence interval on
    their median: no resampling happens here, and `median_ci` and `rng` are accepted only
    so that the two tie rules can be called interchangeably.

    Why this is the default
    -----------------------
    Straddling zero means the subset beat the best one on a non-negligible share of the
    holdout draws -- roughly, at least 2.5% of them -- so the two change places often
    enough that the data does not establish an order between them. That is a property of
    the spectrum and the references. Widening the interval on the *median* instead measures
    how precisely the center is pinned down, which improves as 1 / sqrt(n_bootstrap): it
    reports the iteration count as much as the data, and running the bootstrap longer will
    eventually separate any two subsets that differ at all. The percentiles here do not
    move with the iteration count, only become better estimated. See
    plot_tie_rule_sensitivity, which measures exactly that difference.

    The cost is that this rule is lenient: a subset that loses 90% of the time is still
    called tied. It answers "is the order between these two established?", not "which one
    is better on average" -- the d_win_rate column peci_tie_table records is the reading to
    reach for when the question is the latter.
    """
    lo, hi = np.percentile(differences, [2.5, 97.5])
    return bool(lo <= 0.0 <= hi), float(lo), float(hi)


def peci_tie_table(results, ref_names, tie_rule=tie_by_paired_distribution,
                   median_ci=median_ci_order_statistic, seed=0):
    """One row per combination: its prediction error, and whether it ties the best at its size.

    Ties are decided by a *paired* difference rather than by comparing two subsets'
    intervals for overlap, and the reason is in how the search was run.
    do_ref_subsets_moving_block_holdout_bootstrap calls select_holdout_blocks_fn once and
    reuses those draws for every combination, so on iteration k every subset was scored on
    the same held-out energies. Subtracting two subsets' prediction errors iteration by
    iteration therefore cancels whatever made iteration k easy or hard for both of them,
    and what is left is the difference between the subsets themselves.

    Comparing each subset's own interval against the best subset's interval instead would
    throw that pairing away. Two intervals can overlap comfortably while every paired
    difference has the same sign, so the overlap test calls subsets tied that the data
    plainly separates -- it answers a question nobody asked ("could these two medians be
    equal if the subsets had been scored independently?") rather than the question the
    search poses ("is this subset worse than the best one on the same holdouts?").

    Parameters
    ----------
    results   : dict returned by do_ref_subsets_moving_block_holdout_bootstrap
    ref_names : list of reference spectrum names, indexed by results['ref_indices']
    tie_rule  : callable (differences, median_ci, rng) -> (tied, lo, hi). The default
                brackets the middle 95% of the paired differences themselves -- read its
                docstring, which sets out how that distribution is formed.
                tie_by_paired_median_ci puts a confidence interval on their median instead;
                on this data the two disagree about almost everything, and
                plot_tie_rule_sensitivity is the measurement behind preferring the default.
    median_ci : callable (values, rng) -> (lo, hi), used by whichever tie_rule wants one.
                The default is the order statistic interval, which the study above measured
                as calibrated on both the prediction errors and the paired differences,
                never failing, and fast enough to run on every combination without thinking
                about it. A resampling estimator works here too and costs minutes rather
                than milliseconds.
    seed      : int. Each combination gets its own generator derived from this, so a row
                does not depend on how many rows were computed before it.

    Returns
    -------
    pandas.DataFrame -- one row per combination, columns:
        'combination'  : row index into the results arrays
        'M'            : subset size
        'ref_indices'  : tuple of column indices into the design matrix
        'ref_names'    : tuple of reference names
        'pe_median'    : median bootstrap prediction error
        'pe_ci_lo/hi'  : interval on that median
        'is_best'      : lowest pe_median at this subset size
        'd_median'     : median paired difference against the best subset at this size
        'd_lo/hi'      : the interval tie_rule put around that difference -- what it means
                         depends on the rule, which is why these are not named 'ci'
        'd_win_rate'   : fraction of iterations where this subset beat the best one, a
                         reading of the same comparison that no interval convention
                         mediates
        'tied'         : whether this subset is as good as the best one
        'rank'         : position within the subset size, by pe_median ascending
    """
    rows = []
    for m in sorted(set(results['M'])):
        m_indices = np.where(results['M'] == m)[0]
        order = m_indices[np.argsort(np.median(results['bootstrap_pes'][m_indices], axis=1))]
        best = int(order[0])
        best_prediction_errors = results['bootstrap_pes'][best]

        for rank, i in enumerate(order, start=1):
            i = int(i)
            prediction_errors = results['bootstrap_pes'][i]
            rng = np.random.default_rng([seed, i])
            pe_ci_lo, pe_ci_hi = median_ci(prediction_errors, rng)

            if i == best:
                # the best subset against itself is a column of exact zeros, which has no
                # spread for a rule to work with -- scipy's BCa returns NaN on it. It ties
                # itself by definition, so say so rather than asking.
                tied, d_median, d_lo, d_hi, win_rate = True, 0.0, 0.0, 0.0, 0.0
            else:
                differences = prediction_errors - best_prediction_errors
                tied, d_lo, d_hi = tie_rule(differences, median_ci, rng)
                d_median = float(np.median(differences))
                win_rate = float((differences < 0).mean())

            # results['ref_indices'] rows are float and NaN-padded to max_M, so the row is
            # sliced to its own m before int() -- int(nan) raises
            ref_idx = tuple(int(j) for j in results['ref_indices'][i, :m])
            rows.append({
                'combination': i,
                'M': int(m),
                'ref_indices': ref_idx,
                'ref_names': tuple(ref_names[j] for j in ref_idx),
                'pe_median': float(np.median(prediction_errors)),
                'pe_ci_lo': pe_ci_lo,
                'pe_ci_hi': pe_ci_hi,
                'is_best': i == best,
                'd_median': d_median,
                'd_lo': d_lo,
                'd_hi': d_hi,
                'd_win_rate': win_rate,
                'tied': tied,
                'rank': rank,
            })

    return pd.DataFrame(rows)


def peci_tie_counts(tie_table):
    """How many combinations tie the best one at each subset size, against how many exist."""
    return tie_table.groupby('M').agg(
        tied=('tied', 'sum'), combinations=('tied', 'size'),
    ).astype(int)


def tie_rule_sensitivity(results, ref_names, n_bootstrap_grid, seed=0):
    """Tie-set size under both rules as the bootstrap is allowed to run longer.

    The point of the sweep is that one of these rules is a statement about the data and the
    other is partly a statement about the iteration count. Truncating the existing draws to
    the first n of them is exactly a shorter run, since the iterations are independent and
    identically distributed, so no refitting is needed.
    """
    rules = {'CI on the median difference': tie_by_paired_median_ci,
             'middle 95% of the differences': tie_by_paired_distribution}
    rows = []
    for n_bootstrap in n_bootstrap_grid:
        truncated = dict(results)
        truncated['bootstrap_pes'] = results['bootstrap_pes'][:, :n_bootstrap]
        for rule_name, tie_rule in rules.items():
            counts = peci_tie_counts(
                peci_tie_table(truncated, ref_names, tie_rule=tie_rule, seed=seed)
            )
            rows.extend({'n_bootstrap': n_bootstrap, 'rule': rule_name, 'M': m,
                         'tied': int(counts.loc[m, 'tied']),
                         'combinations': int(counts.loc[m, 'combinations'])}
                        for m in counts.index)
    return pd.DataFrame(rows)

In [ ]:
def plot_peci_tie_panel(tie_table, m, ax, n_show=40):
    """The tie structure at one subset size: paired difference against the best subset.

    Everything is drawn relative to the best subset, which therefore sits at exactly zero.
    A combination whose error bar touches the dashed line is one the data cannot separate
    from the best -- the tie set is read straight off the figure, without consulting the
    table.

    The error bars are whatever interval peci_tie_table's tie_rule put around the
    difference, so the figure shows the rule that was actually applied rather than a second
    convention drawn alongside it.

    Only the n_show lowest-error combinations are drawn; at 24 references there are 2,024
    of them at M=3, and the ones far to the right are not close calls. The title says how
    many there are in total.
    """
    rows = tie_table[tie_table['M'] == m].sort_values('rank')
    n_total = len(rows)
    n_tied = int(rows['tied'].sum())
    shown = rows.head(n_show)

    x = np.arange(len(shown))
    center = shown['d_median'].to_numpy()
    yerr = [center - shown['d_lo'].to_numpy(), shown['d_hi'].to_numpy() - center]
    tied = shown['tied'].to_numpy()

    for mask, color, label in ((tied, 'mediumseagreen', f'tied with the best ({n_tied})'),
                               (~tied, 'steelblue', 'worse than the best')):
        if not mask.any():
            continue
        ax.errorbar(
            x[mask], center[mask], yerr=[yerr[0][mask], yerr[1][mask]],
            fmt='o', color=color, capsize=3, linewidth=1.2, markersize=4,
            linestyle='none', label=label,
        )

    ax.axhline(0.0, color='red', linestyle='--', linewidth=1.0, label='the best subset')
    ax.set_xlabel(f'Reference subset, by median prediction error '
                  f'({len(shown)} of {n_total} shown)')
    ax.set_ylabel('Median paired difference in PE')
    ax.set_title(f'M={m}: {n_tied} of {n_total} subsets are as good as the best')
    ax.legend(fontsize=8)


def plot_best_peci_subset_bootstrap_summaries(
    results, ref_names, energies, b, n_bootstrap, spectrum_name, tie_table,
    max_subsets_per_size=3, n_show=40, clusterings=None,
):
    """Call plot_bootstrap_summary for the subsets that are as good as the best one.

    The sibling of plot_best_subset_bootstrap_summaries, differing in what it means by
    "best". That function takes the three lowest median prediction errors at each subset
    size and labels them 1st, 2nd and 3rd, which asserts an ordering the numbers may not
    support. This one reports the subsets whose prediction error is not distinguishable
    from the lowest -- see peci_tie_table for how that is decided -- and says so in the
    titles, so nothing implies a ranking within the tie set.

    Both agree on the winner: the tie set is built around the same lowest-median-prediction-
    error subset that best_subsets_by_size picks.

    Parameters
    ----------
    results              : dict returned by do_ref_subsets_moving_block_holdout_bootstrap
    ref_names            : list of reference spectrum names
    energies             : energy grid array
    b                    : observed spectrum values
    n_bootstrap          : number of bootstrap iterations, passed to plot_bootstrap_summary
    spectrum_name        : str, used in plot titles
    tie_table            : the DataFrame from peci_tie_table, computed by the caller. It is
                           a separate argument rather than computed here because a tie set
                           is worth looking at on its own, and because a resampling
                           estimator makes it a minutes-long computation that should not be
                           hidden inside a plotting call.
    max_subsets_per_size : how many tied subsets to draw summaries for at each subset size,
                           lowest median prediction error first. This is the only thing
                           bounding the output: a tie set can hold hundreds of subsets and
                           each one costs four or five figures. None draws all of them.
    n_show               : combinations per tie-structure panel, passed through
    clusterings          : optional {metric: clustering} over the whole reference pool,
                           forwarded to plot_bootstrap_summary so each figure shows where
                           its combination sits in the reference trees

    Returns
    -------
    tuple of matplotlib.figure.Figure
        For each subset size, the tie structure at that size followed by the summaries of
        the subsets drawn from it. Every figure is closed before being returned, for the
        reason plot_bootstrap_summary gives; the caller displays them, typically
        `for fig in plot_best_peci_subset_bootstrap_summaries(...): display(fig)`.
    """
    figures = []
    for m in sorted(set(tie_table['M'])):
        rows = tie_table[tie_table['M'] == m].sort_values('rank')
        tied_rows = rows[rows['tied']]
        n_tied = len(tied_rows)

        fig_ties, ax_ties = plt.subplots(figsize=(12, 4.5))
        plot_peci_tie_panel(tie_table, m, ax_ties, n_show=n_show)
        fig_ties.suptitle(
            f'Subsets not distinguishable from the best {m}-component fit — {spectrum_name}',
            fontsize=13,
        )
        # tight_layout doesn't account for suptitle; the rect reserves space for it
        fig_ties.tight_layout(rect=[0, 0, 1, 0.91])
        plt.close(fig_ties)
        figures.append(fig_ties)

        selected = tied_rows if max_subsets_per_size is None \
            else tied_rows.head(max_subsets_per_size)

        for rank, row in enumerate(selected.itertuples(), start=1):
            i = row.combination
            if n_tied == 1:
                title_prefix = f'the only {m}-component fit not beaten by the best'
            else:
                # "of n tied" rather than "2nd best": the order within a tie set is the
                # median prediction error, which is exactly what these subsets were found
                # not to differ on
                title_prefix = f'{_ordinal(rank)} of {n_tied} tied {m}-component fits'
            figures.extend(plot_bootstrap_summary(
                energies, b,
                results['fitted'][i],
                results['residuals'][i],
                results['lags'],
                results['acf_values'][i],
                results['bootstrap_coefs'][i, :, :m],
                results['bootstrap_pes'][i],
                results['coef'][i, :m],
                list(row.ref_names),
                spectrum_name=spectrum_name,
                n_bootstrap=n_bootstrap,
                title_prefix=title_prefix,
                clusterings=clusterings,
            ))

    return tuple(figures)


def plot_tie_rule_sensitivity(sensitivity, spectrum_name):
    """How each tie rule's answer moves as the bootstrap is allowed to run longer.

    One panel per subset size, tie-set size against n_bootstrap, one line per rule. A rule
    whose line drifts downward is reporting the iteration count: run it longer and fewer
    subsets survive, until only the winner is left. A rule whose line is flat is reporting
    the data, and would give the same answer at any iteration count.

    Returns
    -------
    matplotlib.figure.Figure
        The one figure this builds, closed, for the caller to display.
    """
    subset_sizes = sorted(set(sensitivity['M']))
    rules = list(dict.fromkeys(sensitivity['rule']))
    colors = {rule: color for rule, color in zip(rules, ('steelblue', 'mediumseagreen'))}

    fig, axs = plt.subplots(1, len(subset_sizes),
                            figsize=(max(10, 5 * len(subset_sizes)), 4.5), squeeze=False)
    for ax, m in zip(axs[0], subset_sizes):
        at_m = sensitivity[sensitivity['M'] == m]
        for rule in rules:
            rows = at_m[at_m['rule'] == rule].sort_values('n_bootstrap')
            ax.plot(rows['n_bootstrap'], rows['tied'], 'o-', color=colors[rule],
                    linewidth=1.5, markersize=5, label=rule)
        ax.set_xscale('log')
        ax.set_xlabel('Bootstrap iterations')
        ax.set_title(f'M={m} ({int(at_m["combinations"].iloc[0])} combinations)')
        ax.legend(fontsize=8)
    axs[0][0].set_ylabel('Subsets tied with the best')

    fig.suptitle(
        f'Does the tie rule describe the data or the iteration count? — {spectrum_name}',
        fontsize=13,
    )
    fig.tight_layout(rect=[0, 0, 1, 0.91])
    plt.close(fig)
    return fig

In [ ]:
# Both rules on the same differences, so the gap between them is visible before either one
# is used to select anything.
tie_table = peci_tie_table(fit_summary['results'], fit_summary['ref_names'], seed=11)
median_ci_tie_table = peci_tie_table(
    fit_summary['results'], fit_summary['ref_names'],
    tie_rule=tie_by_paired_median_ci, seed=11,
)
print('subsets tied with the best, by rule:')
display(
    peci_tie_counts(tie_table)
    .rename(columns={'tied': 'tied (middle 95% of the differences)'})
    .join(peci_tie_counts(median_ci_tie_table)['tied']
          .rename('tied (CI on the median)'))
)

# Truncating the draws to the first n of them is exactly a shorter bootstrap run, so this
# sweep costs no refitting.
sensitivity = tie_rule_sensitivity(
    fit_summary['results'], fit_summary['ref_names'],
    n_bootstrap_grid=[50, 100, 250, 500, 1000], seed=11,
)
display(plot_tie_rule_sensitivity(sensitivity, fit_summary['spectrum_name']))

# The summaries below use the default rule, which brackets the differences themselves.
# Pass tie_rule=tie_by_paired_median_ci to peci_tie_table to select on an interval around
# their median instead -- see the findings for why that is not the default.
for fig in plot_best_peci_subset_bootstrap_summaries(
    fit_summary['results'], fit_summary['ref_names'], fit_summary['valid_energies'],
    fit_summary['b'], n_bootstrap=1000, spectrum_name=fit_summary['spectrum_name'],
    tie_table=tie_table, max_subsets_per_size=3,
    clusterings=fit_summary['clusterings'],
):
    display(fig)

display(tie_table[tie_table['tied']][
    ['M', 'rank', 'ref_names', 'pe_median', 'd_median', 'd_lo', 'd_hi', 'd_win_rate']
])

In [ ]:
# ---------------------------------------------------------------------------
# Tests for the confidence interval estimators, peci_tie_table and
# plot_best_peci_subset_bootstrap_summaries.
#
# The selector's whole claim is that a paired comparison can separate subsets that
# unpaired intervals cannot, and that one of the two tie rules answers a question about
# the data while the other answers one about the iteration count. Both are properties
# rather than numbers, so both are tested rather than asserted in a comment.
# ---------------------------------------------------------------------------

def _paired_pe_results(offsets, n_bootstrap=1000, shared_scale=1.0, noise_scale=0.02,
                       seed=0, subset_sizes=None, n_energies=30):
    """A results dict whose prediction errors are paired the way a real search's are.

    Every combination gets the same per-iteration term -- standing in for the holdout draw
    all of them were scored on -- plus its own offset and a little independent noise. The
    offset is the truth the tie rules have to recover: a combination with a larger offset
    really is worse, by exactly the amount the rules are asked to detect.

    The arrays the tie rules never look at -- fits, residuals, autocorrelations,
    coefficients -- are filled in with the right shapes and NaN padding so that
    plot_bootstrap_summary can be driven from the same fixture.
    """
    rng = np.random.default_rng(seed)
    offsets = np.asarray(offsets, dtype=float)
    n_combinations = len(offsets)
    if subset_sizes is None:
        subset_sizes = np.ones(n_combinations, dtype=int)
    subset_sizes = np.asarray(subset_sizes, dtype=int)
    max_M = int(subset_sizes.max())

    shared = shared_scale * rng.random(n_bootstrap)
    prediction_errors = (
        shared[None, :]
        + offsets[:, None]
        + noise_scale * rng.standard_normal((n_combinations, n_bootstrap))
    )

    b = rng.normal(size=n_energies)
    residuals = rng.normal(size=(n_combinations, n_energies)) * 0.01
    results = {
        'M': subset_sizes,
        'ref_indices': np.full((n_combinations, max_M), np.nan),
        'coef': np.full((n_combinations, max_M), np.nan),
        'bootstrap_coefs': np.full((n_combinations, n_bootstrap, max_M), np.nan),
        'bootstrap_pes': prediction_errors,
        'fitted': b[None, :] + residuals,
        'residuals': residuals,
        'lags': np.arange(6),
        'acf_values': rng.random((n_combinations, 6)),
    }
    n_refs = max(max_M, 2)
    for i, m in enumerate(subset_sizes):
        # a distinct combination of the reference pool for each row, wrapped so that any
        # number of rows can be built without running out
        results['ref_indices'][i, :m] = [(i + j) % n_refs for j in range(m)]
        results['coef'][i, :m] = rng.random(m)
        results['bootstrap_coefs'][i, :, :m] = rng.random((n_bootstrap, m))

    ref_names = [f'ref{i}.e' for i in range(n_refs)]
    return results, ref_names, b


def test_median_ci_estimators_bracket_the_median():
    """Whatever else they disagree about, an interval has to contain the point estimate."""
    values = np.random.default_rng(0).lognormal(size=500)
    for name, median_ci in MEDIAN_CI_METHODS.items():
        lo, hi = median_ci(values, np.random.default_rng(0))
        assert lo <= np.median(values) <= hi, f'{name} produced [{lo}, {hi}]'


def test_order_statistic_ci_survives_a_constant_sample():
    """The best subset's difference against itself is all zeros, and must not blow up.

    scipy's BCa returns NaN here, which is why peci_tie_table special-cases the best
    subset rather than trusting whatever the estimator says.
    """
    lo, hi = median_ci_order_statistic(np.zeros(100), None)
    assert (lo, hi) == (0.0, 0.0)


def test_peci_tie_table_has_one_row_per_combination():
    results, ref_names, _ = _paired_pe_results([0.0, 0.05, 0.10, 0.15])
    tie_table = peci_tie_table(results, ref_names)

    assert len(tie_table) == len(results['M'])
    assert sorted(tie_table['rank']) == [1, 2, 3, 4]
    assert set(tie_table['combination']) == set(range(4))


def test_peci_best_subset_ties_itself():
    results, ref_names, _ = _paired_pe_results([0.0, 0.05, 0.10])
    tie_table = peci_tie_table(results, ref_names)
    best = tie_table[tie_table['is_best']]

    assert len(best) == 1, 'exactly one combination is the best at this subset size'
    assert best['rank'].iloc[0] == 1
    assert best['d_median'].iloc[0] == 0.0
    assert bool(best['tied'].iloc[0]), 'a subset is always as good as itself'


def test_peci_excludes_a_clearly_worse_subset():
    """An offset far larger than the noise is not a tie under either rule."""
    results, ref_names, _ = _paired_pe_results([0.0, 1.0], noise_scale=0.01)
    for tie_rule in (tie_by_paired_median_ci, tie_by_paired_distribution):
        tie_table = peci_tie_table(results, ref_names, tie_rule=tie_rule)
        loser = tie_table[~tie_table['is_best']]
        assert not loser['tied'].iloc[0], f'{tie_rule.__name__} called a 1.0 offset a tie'
        assert loser['d_win_rate'].iloc[0] == 0.0


def test_peci_paired_comparison_is_sharper_than_overlapping_intervals():
    """The argument for pairing, as a test rather than a comment.

    Both combinations are dominated by the same per-iteration term, so their prediction
    errors scatter over a wide range and their own intervals overlap almost completely.
    The paired difference removes that shared term and separates them anyway.
    """
    results, ref_names, _ = _paired_pe_results([0.0, 0.05], shared_scale=10.0, noise_scale=0.01)
    tie_table = peci_tie_table(results, ref_names)
    best, other = tie_table.iloc[0], tie_table.iloc[1]

    overlapping = (best['pe_ci_lo'] <= other['pe_ci_hi']
                   and other['pe_ci_lo'] <= best['pe_ci_hi'])
    assert overlapping, 'the setup is only interesting if the unpaired intervals overlap'
    assert not other['tied'], 'the paired comparison should separate what the overlap cannot'


def test_tie_rules_differ_in_what_more_iterations_buy():
    """The property that decides which rule to select on.

    A confidence interval on a median narrows as sqrt(n), so running the bootstrap longer
    eventually separates any two subsets that differ at all -- its answer is partly a
    statement about the iteration count. Bracketing the differences themselves measures how
    far apart the two subsets are across holdout draws, which is a property of the data and
    does not sharpen with more iterations.

    The widths are what is asserted, not the tie verdicts: a verdict flips only once the
    width crosses the offset, and where that happens for one particular draw is luck.
    """
    results, ref_names, _ = _paired_pe_results(
        [0.0, 0.05], shared_scale=1.0, noise_scale=0.5, n_bootstrap=4000,
    )
    widths = {}
    for tie_rule in (tie_by_paired_median_ci, tie_by_paired_distribution):
        for n_bootstrap in (250, 4000):
            truncated = dict(results)
            truncated['bootstrap_pes'] = results['bootstrap_pes'][:, :n_bootstrap]
            tie_table = peci_tie_table(truncated, ref_names, tie_rule=tie_rule)
            other = tie_table[~tie_table['is_best']].iloc[0]
            widths[(tie_rule.__name__, n_bootstrap)] = other['d_hi'] - other['d_lo']

    # sixteen times the iterations should quarter the width of an interval on a median;
    # halving is the loose version of that, and is what is asserted
    assert widths[('tie_by_paired_median_ci', 4000)] \
        < 0.5 * widths[('tie_by_paired_median_ci', 250)], \
        f'CI on the median did not narrow with more iterations: {widths}'

    # the spread of the differences is a property of the data, so it should barely move
    width_ratio = (widths[('tie_by_paired_distribution', 4000)]
                   / widths[('tie_by_paired_distribution', 250)])
    assert 0.8 < width_ratio < 1.25, \
        f'bracketing the differences moved by {width_ratio:.2f}x with more iterations'


def test_tie_rule_sensitivity_reports_both_rules_at_every_size():
    """The sweep the finding above is read off: one row per (iteration count, rule, size)."""
    results, ref_names, _ = _paired_pe_results(
        [0.0, 0.01, 0.02, 0.0, 0.01, 0.02], noise_scale=0.1, subset_sizes=[1, 1, 1, 2, 2, 2],
    )
    sensitivity = tie_rule_sensitivity(results, ref_names, n_bootstrap_grid=[100, 1000])

    assert len(sensitivity) == 2 * 2 * 2, 'two iteration counts, two rules, two subset sizes'
    assert set(sensitivity['combinations']) == {3}
    assert (sensitivity['tied'] >= 1).all(), 'the best subset always ties itself'
    assert (sensitivity['tied'] <= sensitivity['combinations']).all()


def test_plot_best_peci_returns_closed_figures_and_respects_the_cap():
    """Figures come back for the caller to display, and the cap is what bounds how many."""
    # offsets well inside the noise at both subset sizes, so there is a tie set with more
    # than one member for the cap to actually bite on
    results, ref_names, b = _paired_pe_results(
        [0.0, 0.002, 0.004, 0.0, 0.002, 0.004], noise_scale=0.05, n_bootstrap=200,
        subset_sizes=[1, 1, 1, 2, 2, 2],
    )
    energies = np.linspace(11800, 12000, results['residuals'].shape[1])
    tie_table = peci_tie_table(results, ref_names)
    tied_per_size = peci_tie_counts(tie_table)['tied']
    assert (tied_per_size > 1).any(), 'the cap is only exercised if something ties'

    figure_counts = {}
    for max_subsets_per_size in (1, 2, None):
        figures = plot_best_peci_subset_bootstrap_summaries(
            results, ref_names, energies, b,
            n_bootstrap=results['bootstrap_pes'].shape[1],
            spectrum_name='SYNTH.e', tie_table=tie_table,
            max_subsets_per_size=max_subsets_per_size,
        )
        # one tie-structure figure per subset size, then four per summary -- five when
        # clusterings are given, which they are not here
        drawn = tied_per_size if max_subsets_per_size is None \
            else tied_per_size.clip(upper=max_subsets_per_size)
        assert len(figures) == len(tied_per_size) + 4 * drawn.sum()
        assert all(not plt.fignum_exists(figure.number) for figure in figures), \
            'every figure should be closed before it is returned'
        figure_counts[max_subsets_per_size] = len(figures)

    assert figure_counts[1] <= figure_counts[2] <= figure_counts[None]


_peci_test_fns = [
    test_median_ci_estimators_bracket_the_median,
    test_order_statistic_ci_survives_a_constant_sample,
    test_peci_tie_table_has_one_row_per_combination,
    test_peci_best_subset_ties_itself,
    test_peci_excludes_a_clearly_worse_subset,
    test_peci_paired_comparison_is_sharper_than_overlapping_intervals,
    test_tie_rules_differ_in_what_more_iterations_buy,
    test_tie_rule_sensitivity_reports_both_rules_at_every_size,
    test_plot_best_peci_returns_closed_figures_and_respects_the_cap,
]
for _fn in _peci_test_fns:
    _fn()
    print(f'PASSED: {_fn.__name__}')
print(f'\n{len(_peci_test_fns)} tests passed')

### Findings

**The paired comparison works, and it is sharp enough to expose a problem with asking it for
a confidence interval on the median.**

*Pairing does what it was supposed to do.* The shared holdout draws cancel, and what is left
separates subsets whose unpaired intervals overlap almost completely — that is what
`test_peci_paired_comparison_is_sharper_than_overlapping_intervals` pins down. An
overlap-of-intervals test on this data would have been useless in the opposite direction,
calling nearly everything tied.

*But a confidence interval on the median difference is too sharp, and for the wrong reason.*
Its width falls as `1 / sqrt(n_bootstrap)`. The sensitivity figure above shows the
consequence directly: as the bootstrap is allowed to run longer, the CI-on-the-median rule
reports steadily fewer ties until only the winner survives, while the number of combinations
and the data behind them never changed. Run the bootstrap ten times longer and it will
separate subsets it previously called equal. That makes its answer partly a statement about
how long the computation ran — a knob the analyst set, not a fact about the sample.

The middle-95%-of-the-differences line on the same figure is flat. It measures how often the
two subsets change places across holdout draws, which is a property of the spectrum and the
references, and it gives the same answer at 250 iterations as at 1,000.

*The practical effect is stark.* Under the CI-on-the-median rule the tie set collapses to the
winner alone at every subset size, so the section above reports exactly what
`plot_best_subset_bootstrap_summaries` already reported, minus the unsupported 2nd- and
3rd-place labels. Under the other rule the tie sets have real membership, and the members are
subsets that beat the nominal winner on a sizeable fraction of holdout draws — read the
`d_win_rate` column, which is the same comparison with no interval convention in the way.

*A subset ranked below the winner can beat it head to head.* Because the ranking is by median
prediction error and the comparison is by median paired difference, the two orderings need
not agree, and in the table above they do not: at least one subset has a higher median
prediction error than the winner and a *negative* median paired difference, meaning it
predicts the held-out energies better on most iterations. Ranking by median prediction error
is not the same as ranking by head-to-head performance, and the pairing is what makes the
difference visible.

**What this changed.** `peci_tie_table` defaults to `tie_by_paired_distribution` on the
strength of the sensitivity figure above. `tie_by_paired_median_ci` is still there and is one
argument away:

```python
tie_table = peci_tie_table(results, ref_names, tie_rule=tie_by_paired_median_ci)
```

It is worth keeping for two reasons. It is the more literal reading of "compare the
prediction error confidence intervals", and it is the interval
`mrfitty/prediction_error_fit.py` compares when it chooses a component count — which makes
its behavior here a reason to look at that choice again. The component counts that function
compares are few and far apart, where the same `1 / sqrt(n_bootstrap)` sharpening is less
likely to bite, but nothing makes it immune, and it has the same hard dependence on an
iteration count the analyst chose.

**What the default costs.** Bracketing the differences is lenient: a subset that loses 90% of
the holdout draws still straddles zero and is still called tied. That is the honest reading of
"the data does not establish an order between these two", but it is not "these two are equally
good on average", and the tie sets it produces are correspondingly generous. The `d_win_rate`
column is the reading to reach for when the question is which subset is actually better more
often — and it is the column that makes the inversion described above visible.

## Choosing the reference distance: correlation vs. cosine

The tree above is built on `correlation` distance, which is what
`mrfitty/combination_fit.py` uses and therefore the continuity-preserving default. It is
not the only defensible choice, and the difference is not cosmetic: correlation centers
each reference before comparing, so it sees shape alone, while `cosine` leaves the mean
level in and measures the angle from the origin. The uncentered version is the closer
analogue of what a non-negative fit can actually trade between two columns.

**"Merge" and "merge height".** Both this section and the tables below are read in those
terms, so: a dendrogram is built by repeatedly joining the two closest things — first two
individual references, then the groups those joins create — until everything hangs
together in one tree. Each join is a **merge**, drawn as the fork where two branches meet,
and the distance at which it happened is its **merge height**, the position of that fork
along the distance axis. Under `complete` linkage, which these trees use, a merge's height
is the distance between the two furthest-apart members of the two groups being joined, so
it is the width of the group the merge creates rather than the closest approach between
them. In these figures the leaves sit at distance 0 on the right and heights grow
leftward: a fork near the leaves joins references that are nearly the same vector, and the
final merge — the **root**, at the far left — joins the two halves of the whole reference
set at the largest distance in the tree. Merge heights are what the cutoff is compared
against, and what each highlighted combination's bracket reports.

The choice sits underneath every reading taken from the tree — which references count as
one cluster, where the cutoff falls, and therefore whether a selected combination is
reported as spanning the reference set or as coming from one corner of it. This section
measures what it changes, in the style of the interpolation comparison further down: one design
matrix, one linkage method, the same number of shuffled copies behind each cutoff, and the
same combinations located in both trees, so the metric is the only thing that differs. Both
arms are seeded identically and — since the shuffling depends only on the shape of `A` —
scramble the references in exactly the same way, which means the two cutoffs are measured
against the same randomized data rather than against two different draws of it.
`compare_cluster_metrics` asserts that rather than assuming it.

In [ ]:
def compare_cluster_metrics(
    A, ref_names, highlight, metrics=('correlation', 'cosine'), method='complete',
    seed=42, resample_count=1000, percentile=95.0,
):
    """Cluster one design matrix under each metric and measure what the choice changes.

    Controlled the way `compare_interpolation_methods` is: one A, one linkage method, the
    same number of randomized copies behind each cutoff, and the same combinations located
    in both trees, so the metric is the only thing that differs. Each arm gets a freshly
    seeded generator, and because `rng.permuted` depends only on A's shape, both arms
    shuffle the references in exactly the same way -- so the two cutoffs are measured
    against the same randomized data. That is asserted here through the stored digest
    rather than assumed.
    """
    from scipy.stats import spearmanr
    from sklearn.metrics import adjusted_rand_score

    clusterings = {
        metric: cluster_reference_spectra(
            A, ref_names, np.random.default_rng(seed), metric=metric, method=method,
            resample_count=resample_count, percentile=percentile, verbose=False,
        )
        for metric in metrics
    }
    digests = {c['first_permutation_digest'] for c in clusterings.values()}
    assert len(digests) == 1, 'the metric arms did not shuffle the references the same way'

    groups = _normalize_highlight(highlight, ref_names)

    # nearest neighbor of every reference, under each metric
    nearest = {}
    for metric, clustering in clusterings.items():
        square = squareform(clustering['distances'])
        np.fill_diagonal(square, np.inf)
        nearest[metric] = [ref_names[j] for j in square.argmin(axis=1)]
    first, second = metrics
    changed_neighbor = [
        (ref_names[i], nearest[first][i], nearest[second][i])
        for i in range(len(ref_names)) if nearest[first][i] != nearest[second][i]
    ]

    # where each combination sits in each tree -- the question the section is asked to settle
    subtree_spread = {}
    for label, indices in groups.items():
        subtree_spread[label] = {}
        for metric, clustering in clusterings.items():
            group = smallest_enclosing_subtree(clustering['Z'], indices)
            subtree_spread[label][metric] = {
                **group,
                'height_fraction': group['height'] / clustering['Z'][-1, 2],
                'within_cutoff': group['height'] <= clustering['cutoff_distance'],
            }

    return {
        'metrics': tuple(metrics),
        'ref_names': list(ref_names),
        'clusterings': clusterings,
        'groups': groups,
        'nearest_neighbor': nearest,
        'changed_neighbor': changed_neighbor,
        'subtree_spread': subtree_spread,
        'distance_spearman': float(spearmanr(*[clusterings[m]['distances'] for m in metrics]).statistic),
        'adjusted_rand': float(adjusted_rand_score(*[clusterings[m]['labels'] for m in metrics])),
    }


def summarize_cluster_metric_comparison(comparison):
    """Print the comparison: the trees' own statistics, then what actually moved."""
    metrics = comparison['metrics']
    clusterings = comparison['clusterings']

    print(f"{'':<30}" + ''.join(f'{m:>16}' for m in metrics))
    rows = [
        ('largest distance', lambda c: f"{c['distances'].max():.4f}"),
        ('root merge height', lambda c: f"{c['Z'][-1, 2]:.4f}"),
        ('cutoff distance', lambda c: f"{c['cutoff_distance']:.4f}"),
        ('clusters at the cutoff', lambda c: f"{c['n_clusters']:d}"),
        ('cluster sizes', lambda c: '/'.join(str(s) for s in sorted(np.bincount(c['labels'])[1:], reverse=True))),
        ('cophenetic correlation', lambda c: f"{c['cophenetic_correlation']:.5f}"),
    ]
    for name, value_of in rows:
        print(f'{name:<30}' + ''.join(f'{value_of(clusterings[m]):>16}' for m in metrics))

    print(f'\npairwise distance rank correlation (Spearman): {comparison["distance_spearman"]:.5f}')
    print(f'flat clustering agreement (adjusted Rand):     {comparison["adjusted_rand"]:.4f}')
    print(f'references whose nearest neighbor changes:     '
          f'{len(comparison["changed_neighbor"])} of {len(comparison["ref_names"])}')
    for name, first_neighbor, second_neighbor in comparison['changed_neighbor']:
        print(f'  {name}\n      {metrics[0]}: {first_neighbor}\n      {metrics[1]}: {second_neighbor}')

    print('\nwhere each selected combination sits:')
    print(f"{'combination':<14}" + ''.join(f'{m:>34}' for m in metrics))
    for label, per_metric in comparison['subtree_spread'].items():
        cells = ''
        for metric in metrics:
            spread = per_metric[metric]
            verdict = 'within one cluster' if spread['within_cutoff'] else 'spans the tree'
            cells += (f"{spread['height']:>8.4f} ({spread['height_fraction']:>3.0%}) "
                      f"{spread['size']:>2d} leaves {verdict:>18}")
        print(f'{label:<14}' + cells)

In [ ]:
def plot_cluster_metric_comparison(comparison, spectrum_name):
    """The two trees on one figure, then the two diagnostics on another.

    Two figures rather than one grid: 24 reference file names need a dendrogram panel's
    full width, and squeezing the diagnostics in beside them clips the labels.
    """
    metrics = comparison['metrics']
    first, second = metrics

    fig_trees, axs = plt.subplots(1, len(metrics), figsize=(11 * len(metrics), 9))
    for ax, metric in zip(np.atleast_1d(axs), metrics):
        plot_reference_dendrogram(
            comparison['clusterings'][metric], highlight=comparison['groups'], ax=ax,
            title=f'{metric} distance',
        )
    fig_trees.suptitle(f'Reference clustering, {" vs ".join(metrics)} distance — {spectrum_name}',
                       fontsize=13)
    fig_trees.tight_layout()

    fig_diagnostics, (ax_scatter, ax_chance) = plt.subplots(1, 2, figsize=(13, 4.5))

    ax_scatter.scatter(comparison['clusterings'][first]['distances'],
                       comparison['clusterings'][second]['distances'],
                       s=12, alpha=0.5, color='tab:blue')
    ax_scatter.set_xlabel(f'{first} distance')
    ax_scatter.set_ylabel(f'{second} distance')
    ax_scatter.set_title(f'Every reference pair under both metrics\n'
                         f'Spearman rho = {comparison["distance_spearman"]:.5f}', fontsize=10)

    for metric, color in zip(metrics, ('tab:blue', 'tab:green')):
        clustering = comparison['clusterings'][metric]
        # each metric is scaled by its own root height, so two very different distance
        # scales can be read on one axis
        scale = clustering['Z'][-1, 2]
        ax_chance.hist(clustering['chance_merge_heights'] / scale, bins=60, histtype='step',
                       density=True, color=color, label=f'{metric}: randomized')
        ax_chance.axvline(clustering['cutoff_distance'] / scale, color=color, linestyle='--',
                          label=f'{metric}: cutoff')
        ax_chance.plot(clustering['Z'][:, 2] / scale, np.full(clustering['Z'].shape[0], -0.35),
                       marker='|', linestyle='none', color=color, markersize=8)
    ax_chance.set_xlabel('merge height / root merge height')
    ax_chance.set_ylabel('density')
    ax_chance.set_title('How tightly randomized references merge, and the cutoff that gives\n'
                        '(ticks below the axis are the real merges)', fontsize=10)
    ax_chance.legend(fontsize='small')
    fig_diagnostics.tight_layout()

    return fig_trees, fig_diagnostics

In [ ]:
# Reuses the fit from the section above rather than refitting: the study is about the
# distance metric, so it wants precisely that design matrix and those selected subsets.
metric_comparison = compare_cluster_metrics(
    fit_summary['A'], fit_summary['ref_names'],
    highlight=best_subsets_by_size(fit_summary['results']),
)
summarize_cluster_metric_comparison(metric_comparison)

plot_cluster_metric_comparison(metric_comparison, spectrum_name=fit_summary['spectrum_name'])
plt.show()

### Findings

Run on `OTT3_55_spot0.e` against the full 24-reference pool (276 reference pairs), cubic
spline interpolation, `complete` linkage, and a cutoff set at the 95th percentile of the
merge heights from 1,000 shuffled copies of the reference set, seed 42. Both arms shuffled
identically, asserted rather than assumed. The clustering costs about 0.1 s per metric.

#### The two metrics agree about neighbors and disagree about clusters

| quantity | correlation | cosine |
|---|---|---|
| largest pairwise distance | 0.5710 | 0.1930 |
| cutoff distance | 0.3264 | 0.1029 |
| clusters at the cutoff | 2 (19 / 5) | 2 (14 / 10) |
| cophenetic correlation | 0.90051 | 0.84091 |

| across the two metrics | |
|---|---|
| Spearman rho of the 276 pairwise distances | 0.99077 |
| adjusted Rand of the two flat clusterings | 0.3147 |
| references whose nearest neighbor changes | 1 of 24 |

The local structure is essentially identical — the metrics rank the 276 pairs the same way
(rho = 0.991), and exactly one reference changes nearest neighbor
(`arsenate_sorbed_anth`, from `Fh2l_sorbed_arsenate_pH7_10uM` to `arsenate_sorbed_ram`).
What differs is where the *deep* merges fall, and that is enough to move the cut: an
adjusted Rand of 0.31 between two 2-cluster partitions of the same 24 references.

**The two cuts are not equally interpretable.** Cosine splits the pool 14 / 10 into
arsenate species (the sorbed arsenates, `arsenate_aqueous`, `as2o5`, `scorodite`,
`sodium_arsenate`, `goethite_sorbed_arsenate`) against arsenite plus the sulfides and
arsenides (`arsenite_aqueous`, `As2O3`, the `Fh2l_sorbed_arsenite` pair,
`sodium_arsenite_13BM`, `orpiment`, both arsenopyrites, `As_pyrite`, `Lollingite`) — an
oxidation-state split. Correlation's 5 / 19 isolates only the sulfides and arsenides and
leaves As(V) and As(III) species together in one lump of 19. Centering removes the mean
level, which on a normalized XANES spectrum is a large part of what separates oxidation
states.

#### And it changes how one selection reads

| combination | correlation | cosine |
|---|---|---|
| best M=1 (`Lollingite`) | a single leaf | a single leaf |
| best M=2 (`As2O3` + `As_pyrite`) | **root, 24 leaves — spans the tree** | **0.0777 = 40% of root, 10 leaves — within the cutoff** |
| best M=3 (`Arsenopyrite_Julcani` + `arsenate_sorbed_diop` + `orpiment`) | root, 24 leaves — spans the tree | root, 24 leaves — spans the tree |

This is the one place the metric choice reaches a conclusion. Under correlation the best
two-component fit combines references from opposite ends of the tree; under cosine the same
two references sit inside a single significant cluster, which says the specific pair may be
partly interchangeable with others in that cluster. Same data, same fit, opposite reading.
The M=3 selection spans the root either way, and M=1 is a leaf either way.

#### Recommendation

**Keep `correlation` as the default, and read the cosine tree alongside it whenever a
selection's subtree falls below the cutoff.** Correlation is what the package already uses,
and it summarizes its own distances more faithfully here (cophenetic 0.901 vs 0.841). But
the metrics disagree precisely in the case the figure exists to adjudicate — whether a
combination is one cluster or several — so a selection that lands inside one cluster under
either metric deserves the second look. Both trees cost a tenth of a second.

One caveat on the cutoff itself: it is high relative to the tree. Twenty-two of the 23
merges fall below the correlation cutoff, which makes the shuffling test a weak and
conservative screen here — it establishes that nearly every merge in this reference set is
far tighter than chance would give, not that the two clusters left above the cutoff are the
only real structure. The cutoff answers "is this grouping more than an accident?", and on
this data the answer is yes almost everywhere. Reading finer structure off the tree means
reading merge heights, which is what the subtree brackets show, rather than reading the
cut.

## Saving a fit: results to one file, and back

Every figure above is drawn from a live fit, so redrawing any of them — after changing a
plot, or to look again at a run from last week — means re-running the 2,324-combination
bootstrap. Nothing about a finished fit survives the kernel.

`write_fit_results` puts the fit in one Parquet file and `read_fit_results` hands back
exactly the dict `do_fits_and_plot_summaries` returns, so the plotting functions cannot tell
a file-driven session from a fresh one. On this fit that is 24 MB written in half a second,
against 24 seconds to compute it again.

Parquet rather than a bag of arrays because the interesting axis of this data *is* tabular —
one row per reference combination — so the ranking every summary figure is built on can be
read straight off the table by pandas, DuckDB, Polars or R, without unpacking a thousand
bootstrap draws per row. The draws ride along as list columns, and everything that is not
per-combination — the design matrix, the energy grids, the reference names, the two
clusterings, and the provenance — goes in the file's own key-value metadata, which is where
GeoParquet keeps its spec.

In [ ]:
# Writing a fit to one file, and reading it back into what the plotting functions expect.
#
# The figures in this notebook are all drawn from a live fit, so redrawing anything means
# re-running the 2,324-combination bootstrap. These two functions end that: the fit goes
# into a Parquet file, and the reader hands back exactly the dict `do_fits_and_plot_summaries`
# returns, so a file-driven session is indistinguishable from a fresh one.
#
# Parquet rather than a bag of arrays because the interesting axis of this data *is* tabular
# -- one row per reference combination -- and a table lets the ranking be queried from
# pandas, DuckDB, Polars or R without unpacking anything. The bootstrap draws ride along as
# list columns, which Parquet stores variable-length, so the NaN padding a live fit carries
# is not written at all.
#
# Needs pyarrow, which is declared in requirements.txt and so comes with `pip install -e .`
# -- declared there rather than only here so these two functions can move into the mrfitty
# package without a dependency change.
import base64
import datetime
import json

import pyarrow as pa
import pyarrow.parquet as pq

# 2 adds the writing mrfitty version and timestamp. The writer only ever writes the current
# version; the reader accepts every version listed here, so a file written before a field
# existed still opens -- the missing field comes back as None rather than the file being
# refused. A version outside this tuple is refused, which is what makes the check worth
# having: it is a file this code cannot reconstruct, not one missing a label.
RESULTS_SCHEMA_VERSION = 2
READABLE_SCHEMA_VERSIONS = (1, 2)
RESULTS_METADATA_KEY = b'mrfitty_fit_results'

# Read by plot_reference_dendrogram and worth keeping; the two large clustering arrays
# (chance_merge_heights, cophenetic_distances) are read only by plot_cluster_metric_comparison,
# which is outside what this file is meant to redraw, so they are left out and restored as
# None rather than silently missing.
_CLUSTERING_SCALARS = ('metric', 'method', 'percentile', 'resample_count', 'surrogate',
                       'cutoff_distance', 'cophenetic_correlation', 'n_clusters',
                       'first_permutation_digest')
_CLUSTERING_ARRAYS = ('Z', 'labels', 'distances')
_CLUSTERING_OMITTED = ('chance_merge_heights', 'cophenetic_distances')


def _encode_array(array):
    """A numpy array as JSON-safe base64 plus the shape and dtype needed to rebuild it."""
    array = np.ascontiguousarray(array)
    return {'b64': base64.b64encode(array.tobytes()).decode('ascii'),
            'dtype': array.dtype.str,          # includes byte order, so a big-endian reader is safe
            'shape': list(array.shape)}


def _decode_array(encoded):
    return np.frombuffer(base64.b64decode(encoded['b64']),
                         dtype=np.dtype(encoded['dtype'])).reshape(encoded['shape'])


def write_fit_results(path, fit_summary, block_length, n_holdout_blocks, elapsed_time, seed):
    """Write everything the fit summaries draw from into one Parquet file.

    Parameters
    ----------
    path : str
        Destination `.parquet` file.
    fit_summary : dict
        What `do_fits_and_plot_summaries` returns: 'results', 'A', 'b', 'ref_names',
        'valid_energies', 'sample_energies', 'spectrum_name' and 'clusterings'.
    block_length, n_holdout_blocks, elapsed_time, seed
        Provenance the fit produced but does not carry in `results`. `block_length` matters
        most: under the default `block_length='auto'` it is tuned from the residuals, so it
        is a result of the fit rather than a setting, and cannot be recovered later.

    The file also records the mrfitty version that wrote it and the time it was written,
    so a file found later says which code produced it and when -- neither is recoverable
    from the fit itself, and the version is what tells a reader whether the numbers in the
    file predate a change in how they are computed.

    The bootstrap draws are stored as float32. They are 75 MB of the 83 MB at float64, and
    nothing drawn from them -- medians, percentiles, violins -- resolves anywhere near
    float32's precision. Everything else keeps full precision.
    """
    results = fit_summary['results']
    n_combinations = len(results['M'])
    n_bootstrap = results['bootstrap_pes'].shape[1]

    # One row per combination, each list column holding only that row's own M values: the
    # NaN padding to max_M that the in-memory arrays carry is an artifact of rectangular
    # numpy, and the reader puts it back.
    rows = {
        'M': [], 'ref_indices': [], 'coef': [], 'bootstrap_coefs': [],
        'bootstrap_pes': [], 'residuals': [], 'acf_values': [], 'median_pe': [],
    }
    for i in range(n_combinations):
        m = int(results['M'][i])
        rows['M'].append(m)
        rows['ref_indices'].append(results['ref_indices'][i, :m].astype(np.int16).tolist())
        rows['coef'].append(results['coef'][i, :m].tolist())
        rows['bootstrap_coefs'].append(
            results['bootstrap_coefs'][i, :, :m].astype(np.float32).reshape(-1).tolist())
        rows['bootstrap_pes'].append(results['bootstrap_pes'][i].astype(np.float32).tolist())
        rows['residuals'].append(results['residuals'][i].tolist())
        rows['acf_values'].append(results['acf_values'][i].tolist())
        # stored so the ranking every summary figure is built on can be read off the table
        # without unnesting a thousand draws per row
        rows['median_pe'].append(float(np.median(results['bootstrap_pes'][i])))

    table = pa.table(rows, schema=pa.schema([
        ('M', pa.int8()),
        ('ref_indices', pa.list_(pa.int16())),
        ('coef', pa.list_(pa.float64())),
        ('bootstrap_coefs', pa.list_(pa.float32())),
        ('bootstrap_pes', pa.list_(pa.float32())),
        ('residuals', pa.list_(pa.float64())),
        ('acf_values', pa.list_(pa.float64())),
        ('median_pe', pa.float64()),
    ]))

    # Everything that is not per-combination goes in the file's key-value metadata as one
    # JSON document -- the same place GeoParquet keeps its spec -- so the table itself stays
    # a clean per-combination table for anyone querying it.
    clusterings = {}
    for metric, clustering in (fit_summary.get('clusterings') or {}).items():
        clusterings[metric] = {
            'ref_names': list(clustering['ref_names']),
            **{key: clustering[key] for key in _CLUSTERING_SCALARS},
            **{key: _encode_array(clustering[key]) for key in _CLUSTERING_ARRAYS},
        }

    metadata = {
        'schema_version': RESULTS_SCHEMA_VERSION,
        'mrfitty_version': mrfitty.__version__,
        # UTC with an explicit offset, so the timestamp means the same thing to a reader in
        # another timezone and sorts lexicographically
        'written_at': datetime.datetime.now(datetime.timezone.utc).isoformat(),
        'spectrum_name': fit_summary['spectrum_name'],
        'ref_names': list(fit_summary['ref_names']),
        'n_bootstrap': int(n_bootstrap),
        'n_combinations': int(n_combinations),
        'max_M': int(results['M'].max()),
        'block_length': int(block_length),
        'n_holdout_blocks': int(n_holdout_blocks),
        'elapsed_time': float(elapsed_time),
        'seed': int(seed),
        'omitted': list(_CLUSTERING_OMITTED),
        # the sample's own measured grid, wider than the fitted window; two kilobytes, and
        # storing it whole means a caller can hand it straight to plot_ref_subsets_summary
        'sample_energies': _encode_array(fit_summary['sample_energies']),
        'valid_energies': _encode_array(fit_summary['valid_energies']),
        'b': _encode_array(fit_summary['b']),
        'A': _encode_array(fit_summary['A']),
        'lags': _encode_array(results['lags']),
        'clusterings': clusterings,
    }
    table = table.replace_schema_metadata({RESULTS_METADATA_KEY: json.dumps(metadata).encode()})
    pq.write_table(table, path, compression='zstd')
    return path


def read_fit_results(path):
    """Rebuild the dict a live fit would have returned, from a file written by write_fit_results.

    The returned dict has the same keys `do_fits_and_plot_summaries` returns, so every
    plotting function takes it unchanged, plus the provenance the writer recorded.

    Reads every schema version in READABLE_SCHEMA_VERSIONS. The keys are the same at each
    one, so a caller does not branch on the version: a version 1 file, written before the
    writer recorded them, comes back with 'mrfitty_version' and 'written_at' set to None --
    "this file does not say" rather than a missing key.
    """
    table = pq.read_table(path)
    raw = table.schema.metadata.get(RESULTS_METADATA_KEY)
    if raw is None:
        raise ValueError(f'{path} carries no {RESULTS_METADATA_KEY.decode()} metadata; '
                         'it was not written by write_fit_results')
    metadata = json.loads(raw)
    if metadata['schema_version'] not in READABLE_SCHEMA_VERSIONS:
        raise ValueError(f'{path} is schema version {metadata["schema_version"]}, '
                         f'this reader understands {READABLE_SCHEMA_VERSIONS}')

    columns = {name: table.column(name).to_pylist() for name in table.column_names}
    n_combinations = len(columns['M'])
    max_M = metadata['max_M']
    n_bootstrap = metadata['n_bootstrap']
    n_energies = len(columns['residuals'][0])
    n_lags = len(columns['acf_values'][0])

    # Rebuild the rectangular, NaN-padded arrays a live fit produces, so nothing downstream
    # can tell the difference -- including the `[:, :m]` slicing the summary plots do.
    results = {
        'M': np.array(columns['M'], dtype=int),
        'ref_indices': np.full((n_combinations, max_M), np.nan),
        'coef': np.full((n_combinations, max_M), np.nan),
        'bootstrap_coefs': np.full((n_combinations, n_bootstrap, max_M), np.nan, dtype=np.float32),
        'bootstrap_pes': np.zeros((n_combinations, n_bootstrap), dtype=np.float32),
        'fitted': np.zeros((n_combinations, n_energies)),
        'residuals': np.zeros((n_combinations, n_energies)),
        'lags': _decode_array(metadata['lags']),
        'acf_values': np.zeros((n_combinations, n_lags)),
    }
    b = _decode_array(metadata['b'])
    for i in range(n_combinations):
        m = int(columns['M'][i])
        results['ref_indices'][i, :m] = columns['ref_indices'][i]
        results['coef'][i, :m] = columns['coef'][i]
        results['bootstrap_coefs'][i, :, :m] = np.asarray(
            columns['bootstrap_coefs'][i], dtype=np.float32).reshape(n_bootstrap, m)
        results['bootstrap_pes'][i] = columns['bootstrap_pes'][i]
        results['residuals'][i] = columns['residuals'][i]
        results['acf_values'][i] = columns['acf_values'][i]
        # fit_nnls defines residuals = fitted - b, so the fitted spectrum is recoverable
        # exactly and is not worth the 3.7 MB it would take to store
        results['fitted'][i] = b + results['residuals'][i]

    clusterings = {}
    for metric, stored in metadata['clusterings'].items():
        clusterings[metric] = {
            'ref_names': list(stored['ref_names']),
            **{key: stored[key] for key in _CLUSTERING_SCALARS},
            **{key: _decode_array(stored[key]) for key in _CLUSTERING_ARRAYS},
            # present and empty rather than absent, so a caller that wants them sees why
            **{key: None for key in metadata['omitted']},
        }

    return {
        'results': results,
        'A': _decode_array(metadata['A']),
        'b': b,
        'ref_names': list(metadata['ref_names']),
        'valid_energies': _decode_array(metadata['valid_energies']),
        'spectrum_name': metadata['spectrum_name'],
        'clusterings': clusterings,
        'sample_energies': _decode_array(metadata['sample_energies']),
        'block_length': metadata['block_length'],
        'n_holdout_blocks': metadata['n_holdout_blocks'],
        'elapsed_time': metadata['elapsed_time'],
        'seed': metadata['seed'],
        'schema_version': metadata['schema_version'],
        # absent in version 1, and None says so
        'mrfitty_version': metadata.get('mrfitty_version'),
        'written_at': metadata.get('written_at'),
    }

In [ ]:
# ---------------------------------------------------------------------------
# Tests for write_fit_results / read_fit_results
#
# Built on a miniature fit -- 3 references, 5 combinations, 20 bootstrap iterations --
# so the file is written and read in milliseconds, but with the same shapes, the same
# NaN padding and real clusterings, which is what the round trip has to preserve.
# ---------------------------------------------------------------------------
import tempfile

def _synthetic_fit_summary(seed=0, n_energies=30, n_bootstrap=20):
    """A miniature fit: 3 references, combinations of sizes 1 and 2, NaN padding and all."""
    rng = np.random.default_rng(seed)
    ref_names = ['a.e', 'b.e', 'c.e']
    combos = [(1, [0]), (1, [1]), (1, [2]), (2, [0, 1]), (2, [1, 2])]
    max_M = max(m for m, _ in combos)
    n = len(combos)
    b = rng.normal(size=n_energies)
    results = {
        'M': np.array([m for m, _ in combos], dtype=int),
        'ref_indices': np.full((n, max_M), np.nan),
        'coef': np.full((n, max_M), np.nan),
        'bootstrap_coefs': np.full((n, n_bootstrap, max_M), np.nan),
        'bootstrap_pes': rng.random((n, n_bootstrap)),
        'fitted': np.zeros((n, n_energies)),
        'residuals': rng.normal(size=(n, n_energies)) * 0.01,
        'lags': np.arange(6),
        'acf_values': rng.random((n, 6)),
    }
    for i, (m, indices) in enumerate(combos):
        results['ref_indices'][i, :m] = indices
        results['coef'][i, :m] = rng.random(m)
        results['bootstrap_coefs'][i, :, :m] = rng.random((n_bootstrap, m))
        results['fitted'][i] = b + results['residuals'][i]

    A = rng.random((n_energies, len(ref_names)))
    clusterings = {
        metric: cluster_reference_spectra(A, ref_names, np.random.default_rng(1),
                                          metric=metric, resample_count=20, verbose=False)
        for metric in ('correlation', 'cosine')
    }
    return {
        'results': results, 'A': A, 'b': b, 'ref_names': ref_names,
        'valid_energies': np.linspace(11800, 12000, n_energies),
        'spectrum_name': 'SYNTH.e', 'clusterings': clusterings,
        'sample_energies': np.linspace(11790, 12010, n_energies + 7),
    }


def _round_trip(fit_summary, **kwargs):
    with tempfile.TemporaryDirectory() as directory:
        path = os.path.join(directory, 'fit.parquet')
        write_fit_results(path, fit_summary, block_length=10, n_holdout_blocks=6,
                          elapsed_time=1.5, seed=42, **kwargs)
        return read_fit_results(path), os.path.getsize(path)


def test_results_round_trip():
    fit_summary = _synthetic_fit_summary()
    loaded, _ = _round_trip(fit_summary)
    original = fit_summary['results']

    for key in original:
        assert loaded['results'][key].shape == original[key].shape, key

    np.testing.assert_array_equal(loaded['results']['M'], original['M'])
    np.testing.assert_array_equal(loaded['results']['lags'], original['lags'])
    for key in ('ref_indices', 'coef', 'residuals', 'acf_values'):
        np.testing.assert_allclose(loaded['results'][key], original[key], atol=0, rtol=0,
                                   err_msg=f'{key} should survive exactly')
    # the draws are stored as float32, which is the one deliberate loss
    for key in ('bootstrap_coefs', 'bootstrap_pes'):
        np.testing.assert_allclose(loaded['results'][key], original[key], rtol=1e-6,
                                   err_msg=f'{key} should survive to float32 precision')


def test_padding_and_fitted_are_reconstructed():
    fit_summary = _synthetic_fit_summary()
    loaded, _ = _round_trip(fit_summary)
    original = fit_summary['results']

    # the file stores only each row's own M values; the padding has to come back
    for i, m in enumerate(original['M']):
        assert np.isnan(loaded['results']['ref_indices'][i, m:]).all()
        assert np.isnan(loaded['results']['coef'][i, m:]).all()
        assert np.isnan(loaded['results']['bootstrap_coefs'][i, :, m:]).all()
        assert np.isfinite(loaded['results']['ref_indices'][i, :m]).all()

    # fitted is not stored at all -- it is b + residuals, exactly
    np.testing.assert_allclose(loaded['results']['fitted'], original['fitted'], atol=1e-12)


def test_inputs_and_provenance_survive():
    fit_summary = _synthetic_fit_summary()
    loaded, _ = _round_trip(fit_summary)

    np.testing.assert_array_equal(loaded['A'], fit_summary['A'])
    np.testing.assert_array_equal(loaded['b'], fit_summary['b'])
    np.testing.assert_array_equal(loaded['valid_energies'], fit_summary['valid_energies'])
    assert loaded['ref_names'] == fit_summary['ref_names']
    assert loaded['spectrum_name'] == 'SYNTH.e'
    np.testing.assert_array_equal(loaded['sample_energies'], fit_summary['sample_energies'])
    assert (loaded['block_length'], loaded['n_holdout_blocks']) == (10, 6)
    assert loaded['seed'] == 42 and loaded['elapsed_time'] == 1.5
    assert loaded['schema_version'] == RESULTS_SCHEMA_VERSION


def test_the_file_says_what_wrote_it_and_when():
    before = datetime.datetime.now(datetime.timezone.utc)
    loaded, _ = _round_trip(_synthetic_fit_summary())

    assert loaded['mrfitty_version'] == mrfitty.__version__
    # parsed rather than compared as text, so a malformed timestamp fails here
    written_at = datetime.datetime.fromisoformat(loaded['written_at'])
    assert written_at.tzinfo is not None, 'the timestamp should carry its UTC offset'
    assert before <= written_at <= datetime.datetime.now(datetime.timezone.utc)


def _rewrite_metadata(path, **changes):
    """Copy a results file with its metadata document edited; None deletes a key.

    How a file from an older writer is manufactured without keeping one around: the table
    is untouched, so what the reader sees differs from a current file only in the metadata.
    """
    table = pq.read_table(path)
    metadata = json.loads(table.schema.metadata[RESULTS_METADATA_KEY])
    for key, value in changes.items():
        if value is None:
            metadata.pop(key, None)
        else:
            metadata[key] = value
    table = table.replace_schema_metadata({RESULTS_METADATA_KEY: json.dumps(metadata).encode()})
    older_path = path.replace('.parquet', '_rewritten.parquet')
    pq.write_table(table, older_path, compression='zstd')
    return older_path


def test_a_version_1_file_still_reads():
    fit_summary = _synthetic_fit_summary()
    with tempfile.TemporaryDirectory() as directory:
        path = os.path.join(directory, 'fit.parquet')
        write_fit_results(path, fit_summary, block_length=10, n_holdout_blocks=6,
                          elapsed_time=1.5, seed=42)
        # exactly what version 1 was: the same table, without the two fields version 2 added
        version_1_path = _rewrite_metadata(path, schema_version=1,
                                           mrfitty_version=None, written_at=None)
        loaded = read_fit_results(version_1_path)

    # everything version 1 did carry comes back unchanged
    np.testing.assert_array_equal(loaded['results']['M'], fit_summary['results']['M'])
    np.testing.assert_array_equal(loaded['A'], fit_summary['A'])
    assert loaded['spectrum_name'] == 'SYNTH.e'
    assert (loaded['block_length'], loaded['seed']) == (10, 42)
    # and the two it does not: present and None, so a caller reads them without branching
    assert loaded['schema_version'] == 1
    assert loaded['mrfitty_version'] is None
    assert loaded['written_at'] is None


def test_an_unreadable_schema_version_is_still_refused():
    fit_summary = _synthetic_fit_summary()
    with tempfile.TemporaryDirectory() as directory:
        path = os.path.join(directory, 'fit.parquet')
        write_fit_results(path, fit_summary, block_length=10, n_holdout_blocks=6,
                          elapsed_time=1.5, seed=42)
        future_path = _rewrite_metadata(path, schema_version=max(READABLE_SCHEMA_VERSIONS) + 1)
        try:
            read_fit_results(future_path)
        except ValueError as error:
            assert 'schema version' in str(error)
        else:
            raise AssertionError('a version this reader cannot reconstruct should be refused')


def test_clusterings_survive_and_still_draw():
    fit_summary = _synthetic_fit_summary()
    loaded, _ = _round_trip(fit_summary)

    for metric, original in fit_summary['clusterings'].items():
        restored = loaded['clusterings'][metric]
        np.testing.assert_array_equal(restored['Z'], original['Z'])
        np.testing.assert_array_equal(restored['labels'], original['labels'])
        np.testing.assert_allclose(restored['distances'], original['distances'])
        assert restored['cutoff_distance'] == original['cutoff_distance']
        assert restored['surrogate'] == original['surrogate']
        # the two big arrays are deliberately not written; say so rather than omit the key
        for key in ('chance_merge_heights', 'cophenetic_distances'):
            assert restored[key] is None

    # the dendrogram reads none of the omitted keys, so it draws from the reloaded dict
    fig, ax = plt.subplots()
    plot_reference_dendrogram(loaded['clusterings']['correlation'], highlight=['a.e'], ax=ax)
    plt.close(fig)


def test_the_table_is_queryable_without_unnesting():
    # the reason for Parquet over a bag of arrays: the ranking every summary figure is built
    # on has to be readable from the table itself
    import pandas as pd
    fit_summary = _synthetic_fit_summary()
    with tempfile.TemporaryDirectory() as directory:
        path = os.path.join(directory, 'fit.parquet')
        write_fit_results(path, fit_summary, block_length=10, n_holdout_blocks=6,
                          elapsed_time=1.5, seed=42)
        frame = pd.read_parquet(path, columns=['M', 'median_pe'])

    live_medians = np.median(fit_summary['results']['bootstrap_pes'], axis=1)
    np.testing.assert_allclose(frame['median_pe'].to_numpy(), live_medians, rtol=1e-6)
    np.testing.assert_array_equal(np.argsort(frame['median_pe'].to_numpy()),
                                  np.argsort(live_medians))
    np.testing.assert_array_equal(frame['M'].to_numpy(), fit_summary['results']['M'])


def test_a_foreign_parquet_is_rejected():
    import pandas as pd
    with tempfile.TemporaryDirectory() as directory:
        path = os.path.join(directory, 'other.parquet')
        pd.DataFrame({'x': [1, 2, 3]}).to_parquet(path)
        try:
            read_fit_results(path)
        except ValueError as error:
            assert 'mrfitty_fit_results' in str(error)
        else:
            raise AssertionError('a file this writer did not produce should be refused')


_results_io_test_fns = [
    test_results_round_trip,
    test_padding_and_fitted_are_reconstructed,
    test_inputs_and_provenance_survive,
    test_the_file_says_what_wrote_it_and_when,
    test_a_version_1_file_still_reads,
    test_an_unreadable_schema_version_is_still_refused,
    test_clusterings_survive_and_still_draw,
    test_the_table_is_queryable_without_unnesting,
    test_a_foreign_parquet_is_rejected,
]

for _fn in _results_io_test_fns:
    _fn()
    print(f'PASSED: {_fn.__name__}')
print(f'\n{len(_results_io_test_fns)} tests passed')

In [ ]:
# Write the fit above, read it back, and draw from the copy that came off disk. The file
# goes to a temporary directory so nothing lands in the repository; a real caller passes a
# path they intend to keep.
import tempfile

results_path = os.path.join(tempfile.mkdtemp(), f'{fit_summary["spectrum_name"]}.parquet')
write_fit_results(
    results_path, fit_summary,
    block_length=fit_summary['block_length'],
    n_holdout_blocks=fit_summary['n_holdout_blocks'],
    elapsed_time=fit_summary['elapsed_time'],
    seed=fit_summary['seed'],
)
print(f'wrote {results_path}')
print(f'  {os.path.getsize(results_path) / 1e6:.1f} MB on disk, against '
      f'{sum(a.nbytes for a in fit_summary["results"].values()) / 1e6:.1f} MB of float64 in memory')

loaded = read_fit_results(results_path)
print(f'  read back: {len(loaded["results"]["M"])} combinations, '
      f'block_length={loaded["block_length"]}, fit took {loaded["elapsed_time"]:.1f} s')

# The ranking the summary figures are built on, read off the table without touching the
# draws -- the reason this is a Parquet table rather than a bag of arrays.
import pandas as pd

combination_table = pd.read_parquet(results_path, columns=['M', 'median_pe'])
print('\nbest combination at each size, queried from the table:')
for row_index in combination_table.groupby('M')['median_pe'].idxmin():
    row = combination_table.loc[row_index]
    chosen = loaded['results']['ref_indices'][row_index, :int(row.M)].astype(int)
    print(f'  M={int(row.M)}  median PE={row.median_pe:.6f}  '
          f'{", ".join(loaded["ref_names"][j] for j in chosen)}')

# and the figures themselves, from the file alone
plot_ref_subsets_summary(
    loaded['results'], loaded['ref_names'], loaded['spectrum_name'],
    sample_energies=loaded['sample_energies'],
    interp_energies=loaded['valid_energies'],
    elapsed_time=loaded['elapsed_time'],
    clusterings=loaded['clusterings'],
)

### What the table answers without a refit

The point of storing the fit as a table rather than a bag of arrays is that most questions
about a fit are questions about *combinations*, and combinations are rows. The queries below
each answer one, reading only the columns they need — the bootstrap draws are 22 of the
file's 24 MB, and none of the first five touch them.

In [ ]:
# Six things the file answers without recomputing anything. `pq` and `json` come from the
# writer cell above.

# 1. What run is this? The provenance lives in the file's metadata, so this reads no data at
#    all -- not one row, not one column.
metadata = json.loads(pq.read_schema(results_path).metadata[RESULTS_METADATA_KEY])
print('1. provenance, without reading any data')
print(f'   {metadata["spectrum_name"]}: {metadata["n_combinations"]} combinations, '
      f'{metadata["n_bootstrap"]} bootstrap iterations, block_length={metadata["block_length"]}, '
      f'seed {metadata["seed"]}, fitted in {metadata["elapsed_time"]:.1f} s')
print(f'   written by mrfitty {metadata["mrfitty_version"]} at {metadata["written_at"]}')

# 2. Where did the megabytes go? Parquet records the compressed size of every column, which
#    is what makes it worth knowing that the draws are almost all of it.
parquet_file = pq.ParquetFile(results_path)
column_bytes = {}
for group in range(parquet_file.metadata.num_row_groups):
    row_group = parquet_file.metadata.row_group(group)
    for column in range(row_group.num_columns):
        chunk = row_group.column(column)
        column_bytes[chunk.path_in_schema] = column_bytes.get(chunk.path_in_schema, 0) + chunk.total_compressed_size
print('\n2. where the bytes are')
for name, size in sorted(column_bytes.items(), key=lambda item: -item[1]):
    print(f'   {name:32s} {size / 1e6:6.2f} MB')

# 3. The ranking every summary figure is built on, from two small columns.
ranking = pd.read_parquet(results_path, columns=['M', 'median_pe'])
print('\n3. the five best combinations overall')
for row_index, row in ranking.nsmallest(5, 'median_pe').iterrows():
    chosen = loaded['results']['ref_indices'][row_index, :int(row.M)].astype(int)
    print(f'   median PE={row.median_pe:.6f}  M={int(row.M)}  '
          f'{", ".join(loaded["ref_names"][j] for j in chosen)}')

# 4. A question the prediction-error ranking alone does not answer: which references keep
#    turning up in the good fits? A reference in nearly every one of the best combinations is
#    a more robust claim than one that happens to be in the single winner.
best_three = (pd.read_parquet(results_path, columns=['M', 'ref_indices', 'median_pe'])
              .query('M == 3').nsmallest(25, 'median_pe'))
appearances = pd.Series([j for row in best_three['ref_indices'] for j in row]).value_counts()
print('\n4. references appearing in the best 25 three-component fits')
for reference_index, count in appearances.head(6).items():
    print(f'   {count:2d}/25  {loaded["ref_names"][reference_index]}')

# 5. What does another reference buy? The whole distribution per subset size, not just its best.
print('\n5. median prediction error by combination size')
print(ranking.groupby('M')['median_pe'].describe()[['min', '50%', 'max']].to_string())

# 6. And when a question does need the draws, fetch that column alone and take the row.
#    Note this reads the whole bootstrap_pes column, not one row: the file is written as a
#    single row group, so column pruning saves I/O here but row filtering does not.
best_row = int(ranking['median_pe'].idxmin())
draws = np.asarray(pq.read_table(results_path, columns=['bootstrap_pes'])
                   .column('bootstrap_pes')[best_row].as_py())
print(f'\n6. the best combination\'s {draws.size} draws: 95% interval '
      f'[{np.percentile(draws, 2.5):.5f}, {np.percentile(draws, 97.5):.5f}]')

## Comparing interpolation methods: linear vs. cubic spline

Every reference spectrum is measured on its own energy grid, so all of them have to be
resampled onto the sample's grid before any fitting can happen. That resampling is the
very first step in the pipeline, which makes the interpolation method an assumption
sitting underneath the design matrix, the NNLS coefficients, the prediction error, and
ultimately the reference combination reported as the best fit.

Until now the method was fixed: `ReferenceSpectrum` builds an
`InterpolatedUnivariateSpline` (a cubic spline) at construction time, and
`interpolate_references_at_sample_energies` simply used it. `make_interpolant` now makes
that choice explicit and selectable, and this section measures what the choice is worth.

The comparison is deliberately controlled. The common energy range depends only on the
measured ranges of the sample and references, never on how they are interpolated, so
both arms fit the same `n` points and the same response vector `b`. Because `n` matches,
each arm's freshly seeded generator draws *identical* holdout masks — every bootstrap
iteration trains and scores on exactly the same positions in both arms. The design
matrix is the only thing that differs, and `compare_interpolation_methods` asserts all
three invariants rather than assuming them.

In [ ]:
import time


def interpolation_method_arm(
    method_name, make_interpolant, sample_spectrum, reference_spectra, M, n_bootstrap,
    seed, select_holdout_blocks_fn,
):
    """One interpolation arm: build the design matrix that way, then run the bootstrap.

    Runs in a worker process (see run_arms), so it takes everything it needs as an
    argument -- including the interpolant factory and the selector -- and returns what it
    printed rather than writing to a stdout the notebook would never see.

    It deliberately returns the *whole* result of the run rather than a summary: the
    controlled-comparison invariants are checked by the caller, on the energies, response
    vector and holdout masks the two arms actually used, and the comparison figure needs
    the full bootstrap output of both arms.

    Parameters
    ----------
    method_name              : str -- 'linear' or 'cubic', echoed back to label the arm
    make_interpolant         : callable -- the interpolant factory under test
    sample_spectrum          : Spectrum -- the unknown being fit
    reference_spectra        : list of ReferenceSpectrum -- the pool
    M                        : list of subset sizes to evaluate
    n_bootstrap, seed        : iterations, and the arm's whole source of randomness
    select_holdout_blocks_fn : callable -- the holdout selector, the same for both arms

    Returns
    -------
    dict -- 'method_name', 'valid_energies', 'b', 'A', 'results', 'holdout_masks',
    'elapsed' and 'printed'
    """
    printed = io.StringIO()
    with contextlib.redirect_stdout(printed):
        valid_energies, A, b, *_ = interpolate_references_at_sample_energies(
            reference_spectra=reference_spectra,
            sample_spectrum=sample_spectrum,
            make_interpolant=make_interpolant,
        )
        t0 = time.perf_counter()
        results, holdout_masks, _, _, _ = do_ref_subsets_moving_block_holdout_bootstrap(
            b, A, M=list(M), rng=np.random.default_rng(seed=seed),
            select_holdout_blocks_fn=select_holdout_blocks_fn,
            n_bootstrap=n_bootstrap,
        )
        elapsed = time.perf_counter() - t0
    return {
        'method_name': method_name, 'valid_energies': valid_energies, 'b': b, 'A': A,
        'results': results, 'holdout_masks': holdout_masks, 'elapsed': elapsed,
        'printed': printed.getvalue(),
    }


def compare_interpolation_methods(
    sample_spectrum, reference_spectra, M=(1, 2, 3), n_bootstrap=1000, seed=42,
    select_holdout_blocks_fn=select_holdout_blocks_v5, n_jobs=N_JOBS,
):
    """Run the full subset bootstrap twice, once per interpolation method.

    The two arms are matched as tightly as possible so that any difference in the
    results is attributable to the interpolation method alone:

    * The common energy range depends only on the measured energy ranges of the
      sample and references, never on how they are interpolated, so both arms get
      the same valid_energies, the same n, and the same response vector b.
    * Because n is identical, select_holdout_blocks_fn draws *identical* holdout
      masks and resample block starts from a freshly seeded generator in each arm.
      Every bootstrap iteration therefore trains and scores on exactly the same
      positions in both arms, and the only thing that differs is the design matrix.

    Both invariants are asserted below rather than assumed.

    Returns
    -------
    comparison : dict with keys 'valid_energies', 'b', 'ref_names', and one entry per
                 method name ('linear', 'cubic') holding {'A', 'results', 'elapsed'}
    """
    methods = [
        ('linear', make_linear_interpolant),
        ('cubic', make_cubic_spline_interpolant),
    ]

    comparison = {
        'ref_names': [r.file_name for r in reference_spectra],
        # the references' own measured points, kept so the comparison plot can show
        # the nodes the two interpolants are drawn through
        'reference_energies': [r.data_df.index.values for r in reference_spectra],
        'reference_norm': [r.data_df['norm'].values for r in reference_spectra],
    }
    holdout_masks_by_method = {}

    # The two arms are independent and identically seeded, so they run side by side; the
    # invariants that make this a controlled comparison are checked afterwards, on what
    # comes back, exactly as they were when the arms ran one after the other.
    arm_results = run_arms([
        (interpolation_method_arm, {
            'method_name': method_name, 'make_interpolant': make_interpolant,
            'sample_spectrum': sample_spectrum, 'reference_spectra': reference_spectra,
            'M': M, 'n_bootstrap': n_bootstrap, 'seed': seed,
            'select_holdout_blocks_fn': select_holdout_blocks_fn,
        })
        for method_name, make_interpolant in methods
    ], n_jobs=n_jobs)

    for arm in arm_results:
        method_name = arm['method_name']
        print(f'\n=== {method_name} ===')
        print(arm['printed'], end='')
        if 'valid_energies' in comparison:
            # the controlled-comparison invariants
            np.testing.assert_array_equal(comparison['valid_energies'], arm['valid_energies'])
            np.testing.assert_array_equal(comparison['b'], arm['b'])
        comparison['valid_energies'] = arm['valid_energies']
        comparison['b'] = arm['b']
        comparison[method_name] = {
            'A': arm['A'], 'results': arm['results'], 'elapsed': arm['elapsed'],
        }
        holdout_masks_by_method[method_name] = arm['holdout_masks']

    np.testing.assert_array_equal(
        holdout_masks_by_method['linear'], holdout_masks_by_method['cubic'],
    )
    print('\nboth arms used identical energies, response vector, and holdout masks')

    return comparison


def summarize_interpolation_comparison(comparison, top_n=10):
    """Print the headline comparison: does the interpolation method change the answer?"""
    import pandas as pd
    from scipy.stats import spearmanr

    A_linear = comparison['linear']['A']
    A_cubic = comparison['cubic']['A']
    ref_names = comparison['ref_names']

    print('Design matrix')
    print(f'  max |cubic - linear| : {np.abs(A_cubic - A_linear).max():.5f} norm units')
    print(f'  rms |cubic - linear| : {np.sqrt(np.mean((A_cubic - A_linear) ** 2)):.5f} norm units')
    worst = np.argsort(np.abs(A_cubic - A_linear).max(axis=0))[::-1][:5]
    print('  largest-disagreeing references:')
    for i in worst:
        print(f'    {ref_names[i]}: max |diff| {np.abs(A_cubic[:, i] - A_linear[:, i]).max():.5f}')

    median_pe = {
        method: np.median(comparison[method]['results']['bootstrap_pes'], axis=1)
        for method in ('linear', 'cubic')
    }
    results = comparison['cubic']['results']

    print('\nPrediction error over all reference combinations')
    rho, _ = spearmanr(median_pe['linear'], median_pe['cubic'])
    print(f'  combinations: {len(results["M"])}')
    print(f'  Spearman rank correlation of median PE: {rho:.5f}')
    for method in ('linear', 'cubic'):
        print(f'  {method:6s}: median PE over combinations = {np.median(median_pe[method]):.6f}, '
              f'best = {median_pe[method].min():.6f}, {comparison[method]["elapsed"]:.1f}s')

    def subset_names(index):
        ref_indices = results['ref_indices'][index]
        return ' + '.join(ref_names[int(j)] for j in ref_indices[~np.isnan(ref_indices)])

    print('\nBest subset by median PE, per subset size')
    rows = []
    for m in sorted(set(results['M'])):
        m_indices = np.where(results['M'] == m)[0]
        best = {
            method: m_indices[np.argmin(median_pe[method][m_indices])]
            for method in ('linear', 'cubic')
        }
        rows.append({
            'M': m,
            'best (linear)': subset_names(best['linear']),
            'best (cubic)': subset_names(best['cubic']),
            'same?': 'yes' if best['linear'] == best['cubic'] else 'NO',
            'PE linear': f'{median_pe["linear"][best["linear"]]:.6f}',
            'PE cubic': f'{median_pe["cubic"][best["cubic"]]:.6f}',
        })
    display(pd.DataFrame(rows).set_index('M'))

    print(f'\nTop-{top_n} agreement (by median PE, across all subset sizes)')
    top = {method: set(np.argsort(median_pe[method])[:top_n]) for method in ('linear', 'cubic')}
    shared = top['linear'] & top['cubic']
    print(f'  {len(shared)} of {top_n} subsets appear in both top-{top_n} lists')
    overall_best = {method: int(np.argmin(median_pe[method])) for method in ('linear', 'cubic')}
    print(f'  overall best (linear): {subset_names(overall_best["linear"])}')
    print(f'  overall best (cubic) : {subset_names(overall_best["cubic"])}')
    print(f'  overall best agrees: '
          f'{"yes" if overall_best["linear"] == overall_best["cubic"] else "NO"}')

    return median_pe

In [ ]:
def plot_interpolation_method_comparison(comparison, median_pe, spectrum_name, n_emphasize=4):
    """Visualize how linear vs. cubic spline interpolation propagates to the results.

    Six panels, reading top-left to bottom-right as the causal chain: where the two
    design matrices differ, why they differ there, and whether that difference
    survives into the prediction errors and the selected reference subset.
    """
    import matplotlib.gridspec as gridspec
    from matplotlib.patches import Patch
    from matplotlib.ticker import NullFormatter
    from scipy.stats import spearmanr

    energies = comparison['valid_energies']
    A_linear = comparison['linear']['A']
    A_cubic = comparison['cubic']['A']
    ref_names = comparison['ref_names']
    results = comparison['cubic']['results']
    difference = A_cubic - A_linear

    unique_M = sorted(set(results['M']))
    colors = plt.cm.tab10(np.linspace(0, 0.4, len(unique_M)))
    M_to_color = {m: c for m, c in zip(unique_M, colors)}

    # rank references by how much the two methods disagree about them
    per_reference_max = np.abs(difference).max(axis=0)
    emphasized = np.argsort(per_reference_max)[::-1][:n_emphasize]
    # explicit hues rather than a colormap slice: the un-emphasized references are
    # gray, so an emphasis color that lands near gray would be invisible
    emphasis_colors = ['crimson', 'darkorange', 'seagreen', 'mediumpurple',
                       'steelblue', 'saddlebrown'][:n_emphasize]

    fig = plt.figure(figsize=(21, 14))
    gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.35, wspace=0.25)

    # ---------------------------------------------------------------- panel 1
    # Where the methods disagree, per reference, across the fitted energy range.
    ax = fig.add_subplot(gs[0, :2])
    for i in range(difference.shape[1]):
        if i not in emphasized:
            ax.plot(energies, difference[:, i], color='0.75', linewidth=0.7, zorder=1)
    for rank, i in enumerate(emphasized):
        ax.plot(
            energies, difference[:, i], color=emphasis_colors[rank], linewidth=1.3, zorder=3,
            label=f'{ref_names[i]} (max {per_reference_max[i]:.4f})',
        )
    ax.axhline(0, color='black', linewidth=0.8, zorder=2)
    ax.plot([], [], color='0.75', linewidth=0.7,
            label=f'other references ({difference.shape[1] - n_emphasize})')
    ax.set_xlabel('Energy (eV)')
    ax.set_ylabel('cubic − linear (norm units)')
    ax.set_title('Where the two interpolations disagree, per reference', fontsize=11)
    ax.legend(fontsize=7, loc='upper right')

    # ---------------------------------------------------------------- panel 2
    # Why they disagree: the worst reference's own measured points, zoomed on the
    # window where the disagreement peaks, with both interpolants drawn through them.
    ax = fig.add_subplot(gs[0, 2])
    worst_reference = int(emphasized[0])
    peak_energy = energies[np.abs(difference[:, worst_reference]).argmax()]
    # a tight window: over any wider span the two interpolants are visually identical
    # and the panel says nothing. The disagreement is a local, few-eV effect.
    window = 2.0
    dense = np.linspace(peak_energy - window, peak_energy + window, 600)
    dense = dense[(dense >= energies[0]) & (dense <= energies[-1])]

    reference_spectrum_energies = comparison['reference_energies'][worst_reference]
    reference_spectrum_norm = comparison['reference_norm'][worst_reference]
    in_window = (
        (reference_spectrum_energies >= dense[0] - 2) & (reference_spectrum_energies <= dense[-1] + 2)
    )
    ax.plot(dense, make_cubic_spline_interpolant(
        reference_spectrum_energies, reference_spectrum_norm)(dense),
        color='crimson', linewidth=1.5, label='cubic spline')
    ax.plot(dense, make_linear_interpolant(
        reference_spectrum_energies, reference_spectrum_norm)(dense),
        color='steelblue', linewidth=1.5, linestyle='--', label='linear')
    ax.plot(
        reference_spectrum_energies[in_window], reference_spectrum_norm[in_window],
        'o', color='black', markersize=5, zorder=5, label='measured points',
    )
    ax.set_xlim(dense[0], dense[-1])
    # absolute eV, not an offset like '+1.187e4' -- the window is only a few eV wide
    ax.ticklabel_format(useOffset=False, axis='x')
    ax.set_xlabel('Energy (eV)')
    ax.set_ylabel('norm')
    ax.set_title(
        f'{ref_names[worst_reference]}\nnode spacing '
        f'{np.median(np.diff(reference_spectrum_energies)):.2f} eV',
        fontsize=9,
    )
    ax.legend(fontsize=7)

    # ---------------------------------------------------------------- panel 3
    # Does the difference survive into prediction error? One point per reference
    # combination; the y=x line is where the method makes no difference at all.
    ax = fig.add_subplot(gs[1, 0])
    for m in unique_M:
        m_mask = results['M'] == m
        ax.scatter(
            median_pe['linear'][m_mask], median_pe['cubic'][m_mask],
            s=8, alpha=0.5, color=M_to_color[m], label=f'M={m}',
        )
    lims = [
        min(median_pe['linear'].min(), median_pe['cubic'].min()) * 0.98,
        max(median_pe['linear'].max(), median_pe['cubic'].max()) * 1.02,
    ]
    ax.plot(lims, lims, color='black', linewidth=1.0, linestyle=':', label='no difference')
    # log-log: prediction errors span an order of magnitude across combinations, and on
    # a linear scale the well-fitting combinations -- the only ones model selection can
    # ever choose between -- collapse into one corner
    ax.set_xscale('log')
    ax.set_yscale('log')
    # log minor ticks label every 2x/3x/4x decade step here and collide into mush
    ax.xaxis.set_minor_formatter(NullFormatter())
    ax.yaxis.set_minor_formatter(NullFormatter())
    ax.set_xlim(lims)
    ax.set_ylim(lims)
    ax.set_aspect('equal')
    rho, _ = spearmanr(median_pe['linear'], median_pe['cubic'])
    ax.set_xlabel('Median PE — linear')
    ax.set_ylabel('Median PE — cubic')
    ax.set_title(f'Median prediction error per combination\nSpearman ρ = {rho:.5f}', fontsize=10)
    ax.legend(fontsize=7)

    # ---------------------------------------------------------------- panel 4
    # Is one method systematically optimistic? A shift away from zero means the
    # choice biases the PE level, not just its ranking.
    ax = fig.add_subplot(gs[1, 1])
    delta = median_pe['cubic'] - median_pe['linear']
    for m in unique_M:
        ax.hist(delta[results['M'] == m], bins=40, alpha=0.6, color=M_to_color[m], label=f'M={m}')
    ax.axvline(0, color='black', linewidth=1.0, linestyle=':')
    ax.axvline(
        np.median(delta), color='red', linestyle='--', linewidth=1.2,
        label=f'median Δ = {np.median(delta):+.6f}',
    )
    ax.set_xlabel('Median PE: cubic − linear')
    ax.set_ylabel('Combinations')
    ax.set_title('Signed prediction error shift', fontsize=10)
    ax.legend(fontsize=7)

    # ---------------------------------------------------------------- panel 5
    # Ranking stability where it actually matters. Model selection only ever looks
    # at the best few combinations, so a high overall rank correlation can still
    # hide a reordering at the top.
    ax = fig.add_subplot(gs[1, 2])
    n_top = 30
    rank_linear = np.argsort(np.argsort(median_pe['linear']))
    rank_cubic = np.argsort(np.argsort(median_pe['cubic']))
    top_mask = (rank_cubic < n_top) | (rank_linear < n_top)
    for m in unique_M:
        m_mask = top_mask & (results['M'] == m)
        if not m_mask.any():
            continue  # subset sizes absent from the top N would add an empty legend row
        ax.scatter(
            rank_linear[m_mask], rank_cubic[m_mask],
            s=25, alpha=0.75, color=M_to_color[m], label=f'M={m}',
        )
    ax.plot([0, n_top], [0, n_top], color='black', linewidth=1.0, linestyle=':')
    ax.set_xlim(-1, n_top)
    ax.set_ylim(-1, n_top)
    ax.set_aspect('equal')
    ax.set_xlabel('Rank — linear (0 = best)')
    ax.set_ylabel('Rank — cubic (0 = best)')
    ax.set_title(f'Rank agreement among the top {n_top} combinations', fontsize=10)
    ax.legend(fontsize=7)

    # ---------------------------------------------------------------- panel 6
    # The practical output: the coefficients a user would report. Shown for the
    # subset each method selects as best overall, with bootstrap 95% intervals.
    ax = fig.add_subplot(gs[2, :])

    # Group by reference rather than by method, so the two methods' coefficients for
    # the same reference sit side by side and are directly comparable. Taking the union
    # of both selected subsets means this still reads correctly if the methods pick
    # different subsets -- a reference chosen by only one method simply has one bar.
    per_method = {}
    for method in ('linear', 'cubic'):
        best = int(np.argmin(median_pe[method]))
        method_results = comparison[method]['results']
        ref_indices = method_results['ref_indices'][best]
        ref_indices = ref_indices[~np.isnan(ref_indices)].astype(int)
        n_used = len(ref_indices)
        bootstrap_coefficients = method_results['bootstrap_coefs'][best][:, :n_used]
        per_method[method] = {
            'ref_indices': list(ref_indices),
            'coef': method_results['coef'][best][:n_used],
            'lo': np.percentile(bootstrap_coefficients, 2.5, axis=0),
            'hi': np.percentile(bootstrap_coefficients, 97.5, axis=0),
        }

    union_refs = list(dict.fromkeys(
        per_method['linear']['ref_indices'] + per_method['cubic']['ref_indices']
    ))
    same_subset = set(per_method['linear']['ref_indices']) == set(per_method['cubic']['ref_indices'])

    width = 0.36  # < half the 0.4 offset spacing, so the paired bars keep a visible gap
    for offset, (method, color) in zip((-0.20, 0.20),
                                       (('linear', 'steelblue'), ('cubic', 'crimson'))):
        entry = per_method[method]
        for position, ref_index in enumerate(union_refs):
            if ref_index not in entry['ref_indices']:
                continue  # this method did not select this reference
            k = entry['ref_indices'].index(ref_index)
            ax.bar(position + offset, entry['coef'][k], color=color, alpha=0.8, width=width)
            ax.errorbar(
                position + offset, entry['coef'][k],
                yerr=[[entry['coef'][k] - entry['lo'][k]], [entry['hi'][k] - entry['coef'][k]]],
                fmt='none', ecolor='black', capsize=4, linewidth=1.2,
            )

    ax.set_xticks(range(len(union_refs)))
    ax.set_xticklabels([ref_names[i] for i in union_refs], fontsize=8)
    ax.set_ylabel('NNLS coefficient')
    ax.set_title(
        'Best subset selected by each method — full-data coefficients with bootstrap 95% intervals'
        + ('  (both methods selected the same subset)' if same_subset
           else '  (METHODS SELECTED DIFFERENT SUBSETS)'),
        fontsize=11,
    )
    ax.legend(
        handles=[
            Patch(facecolor='steelblue', alpha=0.8, label='linear'),
            Patch(facecolor='crimson', alpha=0.8, label='cubic'),
        ],
        fontsize=8,
    )

    fig.suptitle(
        f'Linear vs. cubic spline interpolation of reference spectra — {spectrum_name}',
        fontsize=13, y=0.98,
    )
    fig.subplots_adjust(left=0.05, right=0.98, top=0.92, bottom=0.10)
    plt.show()

In [ ]:
sample_spectrum = filter_spectra_by_name(sample_spectra_list, "OTT3_55*")[0]
reference_spectra = filter_spectra_by_name(reference_spectra_list, "*")

comparison = compare_interpolation_methods(sample_spectrum, reference_spectra)
median_pe = summarize_interpolation_comparison(comparison)
plot_interpolation_method_comparison(comparison, median_pe, sample_spectrum.file_name)

### Findings

Run on `OTT3_55_spot0.e` against the full 24-reference pool, M = [1, 2, 3] (2,324
combinations), 1,000 bootstrap iterations, `select_holdout_blocks_v5`, seed 42, block
length tuned to 10. Each arm took ~23.3 s.

#### The design matrices genuinely differ

| quantity | value |
|---|---|
| max &#124;cubic − linear&#124; | 0.05353 norm units |
| rms &#124;cubic − linear&#124; | 0.00382 norm units |

For scale, the best fit's prediction error is ~0.0287 RMSE, so the *peak* disagreement
between the two interpolations is nearly **twice** the error the model is judged by.
This is not a rounding-level difference.

The largest-disagreeing references:

| reference | max &#124;diff&#124; | node spacing |
|---|---|---|
| scorodite_A_Foster_sln.e | 0.05353 | 0.50 eV |
| arsenate_sorbed_anth_avg_als_cal.e | 0.05300 | 1.05 eV |
| arsenate_sorbed_diop_avg_als_cal.e | 0.04052 | 1.05 eV |
| arsenate_sorbed_opal_avg1_8_als_cal.e | 0.03604 | 1.05 eV |
| arsenate_sorbed_calcite_avg1_4_als_cal.e | 0.03312 | 1.05 eV |

**Curvature matters more than node spacing.** The expectation going in was that the six
references measured every 1.05 eV — coarser than the sample's 0.5 eV grid — would
dominate. They are indeed over-represented, but the single worst offender, `scorodite`,
sits on a perfectly ordinary 0.50 eV grid. Panel 1 shows why: the disagreement is
essentially zero above ~11900 eV and concentrates entirely in 11845–11880 eV, the
absorption edge and white line. Where a spectrum is nearly straight between nodes, a
chord and a spline agree no matter how far apart the nodes are; where it turns sharply,
they diverge. Panel 2 shows the mechanism directly — the linear interpolant cuts a chord
across the white-line peak while the cubic spline rounds over it.

#### But it does not change any conclusion

| quantity | linear | cubic |
|---|---|---|
| Spearman ρ of median PE (2,324 combinations) | 0.99994 | 0.99994 |
| median PE across all combinations | 0.117429 | 0.117713 |
| best median PE | 0.028843 | **0.028666** |
| best subset at M=1 | Lollingite_Lolling_avg_OA | *same* |
| best subset at M=2 | As2O3_ref_avg + As_pyrite_A_Foster | *same* |
| best subset at M=3 | Arsenopyrite_Julcani + arsenate_sorbed_diop + orpiment_all_ref | *same* |
| top-10 combinations | 9 of 10 shared | 9 of 10 shared |

Model selection is completely insensitive to the choice here. The best subset agrees at
every subset size, nine of the ten best combinations overall are shared and the single
best agrees, and the rank correlation across all 2,324 combinations is 0.99994. The fitted coefficients for the
selected M=3 subset are indistinguishable between the methods, well inside their
bootstrap 95% intervals (panel 6).

#### One asymmetry worth noting

Cubic is very slightly *worse* on the typical combination (median PE higher by
+0.000284) but slightly *better* at the optimum (0.028666 vs 0.028843). A plausible
reading is that where a combination genuinely explains the sample, the smoother cubic
references track the real spectrum more faithfully and shave a little error; where the
combination is a poor explanation, the interpolation difference is just one more
uncorrelated perturbation. The effect is small either way and this data cannot really
separate those explanations — it is offered as an observation, not a conclusion.

#### Recommendation

**Keep cubic spline as the default.** It matches what `ReferenceSpectrum` already builds
(and reproduces it to ~1e-15), it is marginally better at the selected optimum, and the
comparison shows nothing to be gained by switching. The useful result is the negative
one: the hardcoded spline in `mrfitty/base.py` is not quietly steering model selection
on this dataset.

That conclusion is dataset-specific in one identifiable way. The disagreement lives
entirely at the absorption edge, so a reference set measured coarsely *through the edge*
— rather than coarsely overall, as here — could behave differently. Re-running this
section is the way to check, which is why it lives in the notebook as a permanent,
re-runnable comparison rather than a one-off answer.

> **Re-run twice, for reasons that have nothing to do with interpolation.** These
> numbers were first produced with the `round(n ** (1/3)) = 6` rule of thumb, then
> re-run when `do_ref_subsets_moving_block_holdout_bootstrap` began tuning the block
> length by default (10 for this data — see *Is the tuned block length better?* below),
> and re-run again when the holdout selector changed from `select_holdout_blocks_v3` to
> `select_holdout_blocks_v5`, which is uniform at a block length that does not divide n
> (see *Development of `select_holdout_blocks` v1–v5* above). Across the selector change
> the Spearman ρ moved from 0.99995 to 0.99994, the best median PE from 0.028670 to
> 0.028666, and nothing structural: the same subset is selected at every size under both
> interpolation methods, and the same combination is best overall. The interpolation
> conclusion depends on neither the block length nor the holdout geometry.


## Is the tuned block length better?

`choose_block_length` says the arsenic data wants blocks of 10 rather than 6. That it is
a better estimate of the residual dependence does not by itself make it a better choice
for *this* pipeline, where the block length also sizes the holdout blocks and constrains
which positions are available to resample from.

The cells below sweep the block length through the entire model-selection pipeline and
ask three questions in order:

1. **Does the resampling deliver the dependence?** Autocorrelation of the resampled
   residual series against the original, at each block length.
2. **Does an independent criterion agree?** The bootstrap's own long-run variance
   estimate, `n · Var(mean)`, which rises with block length and plateaus once the blocks
   are long enough. This shares no machinery with Politis–White.
3. **Does anything actually change?** Whether the prediction-error ranking reorders, and
   whether the selected reference subset at each size moves. This is the question that
   decides whether any of the rest matters.

Each arm gets the same seed and the same design matrix, so block geometry is the only
thing varying between them.

In [ ]:
from functools import partial


def make_block_length_selector(block_length, base_selector=None):
    """A selector with block_length pinned, for sweeping it as an independent variable.

    Now that the v1-v5 selectors take an explicit block_length this is just a partial
    application, but it names the intent and keeps the sweep readable. Defaults to v5
    (random block lengths, alternating truncation), the version the notebook settled on --
    and the only one whose coverage stays uniform at a block length that does not divide n,
    which is exactly what a sweep over block lengths produces.
    """
    if base_selector is None:
        base_selector = select_holdout_blocks_v5
    selector = partial(base_selector, block_length=block_length)
    selector.__name__ = f'{base_selector.__name__}_L{block_length}'
    return selector


def resample_availability(n, block_length, n_bootstrap=200, seed=0):
    """Fraction of block start positions a holdout draw leaves available to resample from.

    Reported alongside every block length in the sweep, because a block length can be
    right for the dependence in the residuals and still be a bad fit here: the same
    length also sizes the holdout blocks, and those eat the contiguous runs the resample
    blocks need. Long enough blocks and there is nothing legal left to draw.
    """
    selector = make_block_length_selector(block_length)
    holdout_masks, _, _, _ = selector(n, np.random.default_rng(seed), n_bootstrap=n_bootstrap)
    return available_start_fraction(holdout_masks, block_length), n - block_length + 1

In [ ]:
import time

from scipy.stats import spearmanr


def reconstruct_from_sampled_starts(residuals, sampled_starts, block_length):
    """Rebuild the residual series each bootstrap iteration actually fits.

    Identical to the gather inside do_moving_block_holdout_bootstrap, so what this
    measures is what the bootstrap really used, not a re-derivation of it.
    """
    n = len(residuals)
    block_offsets = np.arange(block_length)
    block_indices = sampled_starts[:, :, None] + block_offsets[None, None, :]
    return residuals[block_indices].reshape(len(sampled_starts), -1)[:, :n]


def block_length_study_arm(
    block_length, b, A, M, n_bootstrap, seed, reference_residuals, original_acf,
    original_long_run_variance, n_acf_replicates,
):
    """One arm of the block length sweep: the whole pipeline at a single block length.

    Runs in a worker process (see run_arms), which is what the argument list is about --
    it takes everything it needs rather than reading notebook globals, and it hands back
    whatever it printed rather than writing to a stdout the notebook would never see.

    Everything an arm computes gets reduced here to a few hundred numbers: the
    per-combination median prediction errors, the reconstructed autocorrelation band, and
    a handful of scalars. The bootstrap results themselves -- tens of megabytes per arm --
    never leave the worker.

    Parameters
    ----------
    block_length               : int -- the arm's block length, passed to every selector
    b, A                       : the response vector and design matrix, identical across
                                 arms so that block geometry is the only thing varying
    M                          : list of subset sizes to evaluate
    n_bootstrap, seed          : iterations, and the arm's whole source of randomness
    reference_residuals        : (n,) array -- the series the ACF panels reconstruct
    original_acf               : (n_lags,) array -- its autocorrelation, for the error
    original_long_run_variance : float -- flat-top estimate the resampled series is
                                 measured against
    n_acf_replicates           : int -- reconstructions averaged for the ACF

    Returns
    -------
    dict -- the arm's summary, plus 'printed'
    """
    printed = io.StringIO()
    t0 = time.perf_counter()
    with contextlib.redirect_stdout(printed):
        print(f'=== block_length = {block_length} ===')
        selector = make_block_length_selector(block_length)
        # block_length has to be passed explicitly: do_ref_subsets_moving_block_holdout_bootstrap
        # defaults to 'auto', which would tune one length and hand it to every arm --
        # silently collapsing the sweep into seven copies of the same run.
        results, holdout_masks, sampled_starts, realized_block_length, n_holdout_blocks = \
            do_ref_subsets_moving_block_holdout_bootstrap(
                b, A, M=list(M), rng=np.random.default_rng(seed=seed),
                select_holdout_blocks_fn=selector, n_bootstrap=n_bootstrap,
                block_length=block_length,
            )
    elapsed = time.perf_counter() - t0
    n = len(b)

    # The sweep is only a controlled comparison if each arm got the geometry it asked
    # for, so check rather than assume.
    assert realized_block_length == block_length, \
        f'asked for block_length={block_length}, selector returned {realized_block_length}'
    assert sampled_starts.max() + block_length <= n, \
        'a resample block ran past the end of the residual series'

    availability = available_start_fraction(holdout_masks, block_length)
    assert availability > 0, f'block_length={block_length} left nothing to resample from'

    # The ACF is computed on a subset (one calculate_acf call per replicate is the
    # expensive part) but the variance estimate uses every iteration -- at 200
    # replicates its sampling error is large enough to put visible kinks in a curve
    # that is supposed to show a trend.
    reconstructed = reconstruct_from_sampled_starts(
        reference_residuals, sampled_starts, block_length,
    )
    reconstructed_acf = np.array([
        calculate_acf(row)[1] for row in reconstructed[:n_acf_replicates]
    ])
    # The moving-block bootstrap's own long-run variance estimate: n * Var(mean of the
    # resampled series). As the block length grows this rises and then plateaus, and
    # the plateau is where the blocks have become long enough to carry the series'
    # dependence. Where it plateaus is an empirical check on Politis-White that does
    # not share any of its machinery. Reported relative to the flat-top estimate only
    # to put it on a readable scale -- the plateau, not the value 1, is the signal.
    long_run_variance_ratio = float(
        n * reconstructed.mean(axis=1).var() / original_long_run_variance
    )

    return {
        'median_pe': np.median(results['bootstrap_pes'], axis=1),
        'holdout_fraction': float(holdout_masks.mean()),
        'availability': availability,
        'n_start_positions': n - block_length + 1,
        'n_holdout_blocks': n_holdout_blocks,
        'reconstructed_acf_mean': reconstructed_acf.mean(axis=0),
        'reconstructed_acf_lo': np.percentile(reconstructed_acf, 5, axis=0),
        'reconstructed_acf_hi': np.percentile(reconstructed_acf, 95, axis=0),
        'long_run_variance_ratio': long_run_variance_ratio,
        'acf_error': float(np.abs(
            reconstructed_acf.mean(axis=0)[1:11] - original_acf[1:11]
        ).mean()),
        'elapsed': elapsed,
        'printed': printed.getvalue(),
    }


def compare_block_lengths(b, A, block_lengths, M=(1, 2, 3), n_bootstrap=1000, seed=42,
                          reference_combination=None, n_acf_replicates=200,
                          n_jobs=N_JOBS):
    """Run the whole model-selection pipeline once per candidate block length.

    Politis-White answers "what block length preserves the dependence in the residuals".
    This answers the question that actually matters here: does changing the block length
    change which reference combination the prediction error picks? A tuned block length
    that reproduces the ACF beautifully but reorders the model ranking is a much bigger
    deal than one that does not.

    Every block length gets the same seed and the same design matrix, so the only thing
    varying between arms is the block geometry.

    Parameters
    ----------
    reference_combination : tuple of column indices whose residuals are used for the
                            ACF-preservation panels. Fixed across arms so the ACF
                            comparison is about block length and nothing else.
                            Defaults to the lowest-RSS combination.

    Returns
    -------
    dict keyed by block length, plus 'combos', 'reference_combination' and
    'original_acf' shared across arms.
    """
    block_lengths = list(block_lengths)
    n, n_refs = A.shape
    combos = [(m, ref_indices) for m in sorted(M) for ref_indices in combinations(range(n_refs), m)]

    if reference_combination is None:
        residual_sums = [np.sum(fit_nnls(A[:, c], b)[2] ** 2) for _, c in combos]
        reference_combination = combos[int(np.argmin(residual_sums))][1]
    print(f'ACF panels use reference combination {reference_combination}')

    _, _, reference_residuals = fit_nnls(A[:, list(reference_combination)], b)
    original_lags, original_acf = calculate_acf(reference_residuals)
    # Flat-top long-run variance of the original series -- what the resampled series
    # should reproduce if the blocks are long enough.
    original_long_run_variance = politis_white_block_length(reference_residuals)['long_run_variance']

    # One arm per block length, each independent and identically seeded, so they run on
    # every core. Results come back in task order and each arm's own report is printed
    # here, in that order, so the output does not depend on how the work was scheduled.
    arms = dict(zip(block_lengths, run_arms([
        (block_length_study_arm, {
            'block_length': block_length, 'b': b, 'A': A, 'M': M,
            'n_bootstrap': n_bootstrap, 'seed': seed,
            'reference_residuals': reference_residuals,
            'original_acf': original_acf,
            'original_long_run_variance': original_long_run_variance,
            'n_acf_replicates': n_acf_replicates,
        })
        for block_length in block_lengths
    ], n_jobs=n_jobs)))
    for block_length in block_lengths:
        print('\n' + arms[block_length].pop('printed'), end='')

    # Best combination per subset size, and how the full ranking moves, both measured
    # against the rule-of-thumb arm when it is present.
    baseline = 6 if 6 in arms else block_lengths[0]
    subset_sizes = np.array([m for m, _ in combos])
    for block_length, arm in arms.items():
        arm['best_by_M'] = {
            m: combos[int(np.where(subset_sizes == m)[0][
                np.argmin(arm['median_pe'][subset_sizes == m])
            ])][1]
            for m in sorted(M)
        }
        arm['spearman_vs_baseline'] = float(
            spearmanr(arm['median_pe'], arms[baseline]['median_pe']).statistic
        )

    return {
        'arms': arms,
        'combos': combos,
        'subset_sizes': subset_sizes,
        'baseline': baseline,
        'reference_combination': reference_combination,
        'original_acf': original_acf,
        'original_lags': original_lags,
        'n': n,
    }

In [ ]:
def plot_block_length_study(study, tuned, spectrum_name, rule_of_thumb=None):
    """Six panels answering, in order: what does the data want, does the resampling
    deliver it, and does any of it change the answer.

    Panels 1-2 are about the *estimate*: where the per-combination Politis-White values
    fall, and what block length preserves the residual autocorrelation. Panel 3 is an
    independent empirical check on that estimate. Panels 4-6 are about the
    *consequences*: whether the prediction error, its ranking, and the selected subsets
    move when the block length does.

    Block length is an ordered quantity, so it is drawn on a single-hue light-to-dark
    ramp rather than categorical colors, and the two block lengths that matter -- the
    rule of thumb and the tuned value -- are marked in every panel that has an L axis.
    """
    arms = study['arms']
    block_lengths = sorted(arms)
    n = study['n']
    if rule_of_thumb is None:
        rule_of_thumb = max(1, int(np.round(n ** (1 / 3))))
    tuned_block_length = tuned['block_length']

    ramp = plt.cm.Blues(np.linspace(0.35, 0.95, len(block_lengths)))
    length_to_color = dict(zip(block_lengths, ramp))
    rule_color = 'darkorange'
    tuned_color = 'crimson'

    def mark_reference_lengths(ax, label=True):
        ax.axvline(rule_of_thumb, color=rule_color, linestyle='--', linewidth=1.3,
                   label=f'rule of thumb = {rule_of_thumb}' if label else None)
        ax.axvline(tuned_block_length, color=tuned_color, linestyle='-', linewidth=1.3,
                   label=f'tuned = {tuned_block_length}' if label else None)

    fig = plt.figure(figsize=(19, 11))
    gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.32, wspace=0.24)

    # -------------------------------------------------------------- panel 1
    # Where the per-combination estimates fall, and why the aggregate is a low quantile.
    ax = fig.add_subplot(gs[0, 0])
    estimates = tuned['b_opt']
    ax.hist(estimates, bins=40, color='0.75', edgecolor='white', linewidth=0.4)
    # p1/p5/p10/p25 land within a few points of each other, so the labels are staggered
    # vertically rather than stacked on one line where they would collide.
    label_heights = [0.97, 0.82, 0.67, 0.52]
    for (percentile, value), height in zip(
        [(p, v) for p, v in tuned['percentile_sweep'].items() if p != 50], label_heights,
    ):
        ax.axvline(value, color='0.35', linestyle=':', linewidth=1.0)
        ax.text(value, ax.get_ylim()[1] * height, f' p{percentile}={value:.1f}',
                fontsize=7, color='0.35', ha='left', va='top')
    ax.axvline(tuned['b_max'], color='0.2', linestyle='-', linewidth=1.0)
    ax.text(tuned['b_max'], ax.get_ylim()[1] * 0.97,
            f"cap ({tuned['n_at_cap']} combos)", rotation=90, fontsize=7,
            color='0.2', ha='right', va='top')
    mark_reference_lengths(ax)
    ax.set_xlabel('Politis–White block length estimate')
    ax.set_ylabel('Reference combinations')
    ax.set_title('Per-combination estimates\n(underfit subsets pile up at the cap)', fontsize=10)
    ax.legend(fontsize=8)

    # -------------------------------------------------------------- panel 2
    # Does the resampled series keep the autocorrelation the blocks exist to preserve?
    ax = fig.add_subplot(gs[0, 1])
    lags = study['original_lags']
    # The two block lengths that matter are pulled out of the ramp and given the same
    # colors they carry as vertical rules in the other panels -- inside a single-hue
    # ramp they are only a shade apart and cannot be picked out.
    for block_length in block_lengths:
        if block_length == rule_of_thumb:
            color, width = rule_color, 2.4
        elif block_length == tuned_block_length:
            color, width = tuned_color, 2.4
        else:
            color, width = length_to_color[block_length], 1.1
        ax.plot(lags, arms[block_length]['reconstructed_acf_mean'],
                color=color, linewidth=width, label=f'L = {block_length}')
    ax.plot(lags, study['original_acf'], color='black', linestyle='--', linewidth=1.6,
            label='original residuals')
    ax.axhline(0, color='0.6', linewidth=0.7)
    ax.set_xlim(0, 20)
    ax.set_xlabel('Lag')
    ax.set_ylabel('ACF')
    ax.set_title('Autocorrelation of the resampled residuals\n'
                 f'(combination {study["reference_combination"]})', fontsize=10)
    ax.legend(fontsize=7, ncol=2)

    # -------------------------------------------------------------- panel 3
    # Two summaries of panel 2 against block length. The mean absolute ACF error should
    # fall; the bootstrap's own long-run variance estimate should rise and then plateau.
    # They are plotted on separate axes rather than twinned -- different units.
    ax = fig.add_subplot(gs[0, 2])
    ax.plot(block_lengths, [arms[L]['acf_error'] for L in block_lengths],
            color='steelblue', marker='o', markersize=7, linewidth=2.0)
    mark_reference_lengths(ax)
    ax.set_xlabel('Block length')
    ax.set_ylabel('Mean |ACF error|, lags 1–10')
    ax.set_title('Autocorrelation fidelity vs. block length\n'
                 '(monotone — rules 6 out, cannot pick a best)', fontsize=10)
    ax.legend(fontsize=8)

    # -------------------------------------------------------------- panel 4
    ax = fig.add_subplot(gs[1, 0])
    ax.plot(block_lengths, [arms[L]['long_run_variance_ratio'] for L in block_lengths],
            color='mediumseagreen', marker='o', markersize=7, linewidth=2.0)
    mark_reference_lengths(ax)
    ax.set_xlabel('Block length')
    ax.set_ylabel('n · Var(mean) / flat-top estimate')
    ax.set_title('Bootstrap long-run variance\n(still rising at L = 20 — no plateau)', fontsize=10)
    ax.legend(fontsize=8)

    # -------------------------------------------------------------- panel 5
    # The consequence that matters: does the block length reorder the model ranking?
    ax = fig.add_subplot(gs[1, 1])
    ax.plot(block_lengths, [arms[L]['spearman_vs_baseline'] for L in block_lengths],
            color='mediumpurple', marker='o', markersize=7, linewidth=2.0)
    mark_reference_lengths(ax)
    ax.set_xlabel('Block length')
    ax.set_ylabel(f"Spearman rho vs. L = {study['baseline']}")
    ax.set_title('Does the prediction-error ranking move?', fontsize=10)
    ax.legend(fontsize=8)

    # -------------------------------------------------------------- panel 6
    # Does the *answer* move -- the selected subset at each size. Agreement is encoded by
    # text as well as color, so it never depends on color alone.
    ax = fig.add_subplot(gs[1, 2])
    subset_sizes = sorted({m for m, _ in study['combos']})
    baseline_best = arms[study['baseline']]['best_by_M']
    agree_color, differ_color = '#cfe3f3', '#f6d5c4'
    for row, m in enumerate(subset_sizes):
        for col, block_length in enumerate(block_lengths):
            best = arms[block_length]['best_by_M'][m]
            agrees = best == baseline_best[m]
            ax.add_patch(plt.Rectangle(
                (col - 0.5, row - 0.5), 1, 1,
                facecolor=agree_color if agrees else differ_color,
                edgecolor='white', linewidth=2,
            ))
            ax.text(col, row, ','.join(str(i) for i in best),
                    ha='center', va='center', fontsize=7.5, color='0.15')
    ax.set_xlim(-0.5, len(block_lengths) - 0.5)
    ax.set_ylim(len(subset_sizes) - 0.5, -0.5)
    ax.set_xticks(range(len(block_lengths)))
    ax.set_xticklabels([str(L) for L in block_lengths])
    ax.set_yticks(range(len(subset_sizes)))
    ax.set_yticklabels([f'M = {m}' for m in subset_sizes])
    ax.set_xlabel('Block length')
    ax.set_title(f'Selected subset (reference indices)\n'
                 f'shaded where it differs from L = {study["baseline"]}', fontsize=10)
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.tick_params(length=0)

    fig.suptitle(
        f'Tuning the moving-block length — {spectrum_name}, n = {n}, '
        f'rule of thumb {rule_of_thumb} vs. tuned {tuned_block_length}',
        fontsize=13,
    )
    fig.subplots_adjust(top=0.88, left=0.05, right=0.985, bottom=0.07)
    plt.show()

In [ ]:
# The block length study, on the same spectrum and reference pool as
# do_fits_and_plot_summaries so the numbers are comparable to everything above.
sample_spectrum = filter_spectra_by_name(sample_spectra_list, "OTT3_55*")[0]
reference_spectra = filter_spectra_by_name(reference_spectra_list, "*")
print(f'sample spectrum: {sample_spectrum.file_name}')

valid_energies_bl, A_bl, b_bl, *_ = interpolate_references_at_sample_energies(
    reference_spectra=reference_spectra, sample_spectrum=sample_spectrum,
)

# choose_block_length needs one residual series per reference combination. These are the
# same full-data NNLS fits do_ref_subsets_moving_block_holdout_bootstrap does internally.
bl_combos = [c for m in (1, 2, 3) for c in combinations(range(A_bl.shape[1]), m)]
bl_residual_matrix = np.array([fit_nnls(A_bl[:, c], b_bl)[2] for c in bl_combos])
tuned_block_length = choose_block_length(bl_residual_matrix, percentile=10)

# The tuned length is an integer produced by rounding a continuous estimate, and whether
# it divides n is what decides how uniform v1's and v3's coverage is -- see the v1-v5
# write-up above. Restricting the pool to one- and two-reference subsets moves the p10
# estimate across the rounding boundary and onto a divisor of n, which is how narrow the
# margin is that the earlier recommendation was resting on.
small_subset_rows = [i for i, combo in enumerate(bl_combos) if len(combo) <= 2]
small_subset_tuned = choose_block_length(
    bl_residual_matrix[small_subset_rows], percentile=10, verbose=False,
)
print('\nHow close that integer sits to the other side of the rounding boundary:')
for description, count, tuned in (
    ('all subsets', len(bl_combos), tuned_block_length),
    ('M <= 2 only', len(small_subset_rows), small_subset_tuned),
):
    tuned_length = tuned['block_length']
    print(f'  {description:12} ({count:>4} combinations)  p10 = '
          f'{tuned["percentile_sweep"][10]:5.2f} -> block_length {tuned_length:>2}, '
          f'n mod block_length = {len(b_bl) % tuned_length}')

block_length_study = compare_block_lengths(
    b_bl, A_bl, block_lengths=[3, 6, 9, 10, 12, 15, 20], n_bootstrap=1000,
)

plot_block_length_study(
    block_length_study, tuned_block_length, spectrum_name=sample_spectrum.file_name,
)

print('\nBlock length sweep:')
print(f"{'L':>4}  {'|ACF err|':>10}  {'LRV ratio':>10}  {'rho vs L=6':>11}  "
      f"{'avail':>7}  best subset per M")
for block_length, arm in sorted(block_length_study['arms'].items()):
    best = '  '.join(f"M{m}={','.join(str(i) for i in subset)}"
                     for m, subset in sorted(arm['best_by_M'].items()))
    print(f"{block_length:>4}  {arm['acf_error']:>10.4f}  "
          f"{arm['long_run_variance_ratio']:>10.3f}  {arm['spearman_vs_baseline']:>11.5f}  "
          f"{arm['availability']:>7.3f}  {best}")

### Findings

**The rule of thumb is too short, and not marginally.** Politis–White over all 2,324
reference combinations puts the low quantiles at p1 = 8.0, p5 = 10.0, p10 = 10.3,
p25 = 11.7 — a flat, stable band well above the `round(n ** (1/3)) = 6` the selectors
used. Not one combination in 2,324 asked for a block as short as 6. The tuned default is
**10**.

The median of those estimates is 25.9 and 533 of 2,324 combinations pin the 43-point cap,
which is not a sign of enormous dependence — it is the one-sided contamination the
estimator cannot see past. Those are underfit subsets whose residuals still contain the
spectrum. Rank correlation between fit RMSE and `b_opt` is +0.54. This is why
`choose_block_length` takes a low quantile.

**Both direct diagnostics are monotone in block length, so neither one picks a value.**
This is worth stating plainly, because it is tempting to read an optimum off them:

| block length | mean \|ACF error\|, lags 1–10 | n · Var(mean) / flat-top | avail. | Spearman vs. L = 6 |
|---|---|---|---|---|
| 3  | 0.1258 | 1.081 | 0.577 | 0.99767 |
| 6  | 0.0933 | 1.501 | 0.527 | 1.00000 |
| 9  | 0.0685 | 1.866 | 0.504 | 0.99793 |
| 10 | 0.0751 | 2.040 | 0.480 | 0.99659 |
| 12 | 0.0612 | 2.105 | 0.472 | 0.98760 |
| 15 | 0.0627 | 2.125 | 0.469 | 0.98845 |
| 20 | 0.0538 | 2.465 | 0.468 | 0.98203 |

Autocorrelation fidelity improves out to L = 20 — not strictly, since the arms differ by
more than block length now that v5 draws holdout blocks of [L, L+4], but by 0.126 → 0.054
over the range — and the bootstrap's own long-run variance estimate is still climbing
there, with no plateau inside the range the holdout scheme can support. Longer blocks
reproduce the dependence better; what they cost is resampling variety, and these two
curves cannot see that cost. So they establish only the negative result — that 6 sits on
the steep part of the curve, where dependence is still being actively destroyed — and it
is Politis–White that supplies the bias–variance tradeoff and therefore the actual
number.

**Changing the block length does not change the answer, up to a point.** This is the
result that matters, and over the range that matters it is a null one. The selected
subset is identical at every block length from 3 to 15 — `(7,)` at M=1, `(1, 2)` at M=2,
`(0, 11, 19)` at M=3 — and the Spearman rank correlation of the 2,324 median prediction
errors against the L = 6 arm stays at or above 0.987 across that whole range. The tuned
value, 10, sits comfortably inside it at ρ = 0.997.

At L = 20 the selection does move, and at all three sizes: `(0,)`, `(0, 19)` and
`(0, 19, 20)`. Twenty-point holdout blocks are a tenth of the spectrum each, and v5 draws
them from [20, 24], so an iteration holds out three or four contiguous stretches and
scores on very little that is independent of them. This is the cost the ACF and long-run
variance curves cannot see, showing up at last. It is well outside the range Politis–White
points at, so it does not affect the recommendation — it bounds it.

Resample availability sits between 0.468 and 0.577 across the range, so the feasibility
constraint — holdout blocks eating the contiguous runs the resample blocks need — never
binds at these lengths. It would eventually: availability falls with L, and the assertion
in `make_block_length_selector`'s underlying selector is what would catch it.

**Recommendation: adopt the tuned block length as the default, and treat the null result
as the reassurance it is.** `do_ref_subsets_moving_block_holdout_bootstrap` now defaults
to `block_length='auto'`. The case for it is that 10 is estimated from the residuals
rather than assumed from `n`, and the block scheme's whole purpose is to preserve
dependence it was measurably cutting. The case for not worrying about it is this section's
own evidence: on the arsenic data the selected model is invariant across a factor of
seven in block length, which means none of the conclusions elsewhere in this notebook are
artifacts of the block length being 6.

> **Re-run under `select_holdout_blocks_v5`.** This study was first run with v3, whose
> holdout coverage is uniform only when the block length divides n = 198 — true at 6, 9
> and 11, false at 10, 12, 15 and 20, which is to say false in most of the arms above.
> The selector was switched to v5 and the study re-run; see *Development of
> `select_holdout_blocks` v1–v5*. The recommendation is unchanged and the null result is
> slightly stronger than it was: the rank correlation against L = 6 no longer decays the
> way it did (0.982 at worst, against 0.969 under v3), which suggests some of the earlier
> drift with block length was v3's coverage defect growing with the remainder rather than
> the block length itself. What is new is the L = 20 breakdown described above.

The v1–v5 comparison cells are pinned at `block_length=6` *and* `block_length=10`, rather
than tracking the tuned default, because which version has the best holdout geometry
depends on whether the block length divides n — and those two values sit on either side
of that.

**Caveats.** The estimate is noisy — re-estimating from block-resampled residuals spans
roughly 3–10 — though that check is biased by construction, since the resampled series
has less dependence than the original, which is the very effect under study. A clean
interval needs subsampling. And all of this is one spectrum: the tuning is now automatic,
so a different sample with genuinely different residual structure will get a different
block length, which is the point, but the invariance result above has only been
demonstrated here.